#   The Complete Workflow for the Start-up

In [8]:
# The code is ready for any type of data for cleaning 


In [33]:
# Note: 01
1. the code need review and edit as well as improvement 
2. the app need improve : should include concel icon: like when I need to upload need data should ask me whether I need to add eit to existing data or this is need data 
and  open new interface for the analysis.and also should include icons for save , restore the process or back and concel the process 

SyntaxError: invalid syntax (1445321003.py, line 2)

In [1]:
# create_icon.py
from PIL import Image, ImageDraw

# Create a 256x256 image
size = 256
img = Image.new('RGBA', (size, size), (0, 0, 0, 0))
draw = ImageDraw.Draw(img)

# Draw a flask shape
# Flask body
draw.ellipse([40, 60, 216, 200], fill=(26, 115, 232), outline=(33, 33, 33), width=4)
# Flask neck
draw.rectangle([100, 20, 156, 60], fill=(26, 115, 232), outline=(33, 33, 33), width=4)
# Flask top
draw.ellipse([100, 10, 156, 30], fill=(26, 115, 232), outline=(33, 33, 33), width=4)

# Draw data points (Au, Cu, Zn)
draw.ellipse([80, 100, 100, 120], fill=(255, 193, 7))
draw.ellipse([120, 130, 140, 150], fill=(255, 193, 7))
draw.ellipse([90, 160, 110, 180], fill=(255, 193, 7))

# Save as ICO
img.save('icon.ico', format='ICO', sizes=[(256, 256)])

print("✅ icon.ico created!")

✅ icon.ico created!


In [ ]:
# ============================================================#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Elemental Society Analysis - Enterprise Edition
Project Code: 1010-PRO
Developer: Mohamed Almoufty Abdoalmoutleb Abaker
Refactored by: AI Senior Software Architect

Description:
A high-performance, non-blocking GUI application for geochemical data analysis.
Incorporates Multithreading, MVC architecture, and SOLID principles.
"""

import sys
import os
import re
import traceback
import warnings
from datetime import datetime
from pathlib import Path
from typing import Optional, Dict, List, Any

# Core Data Science
import numpy as np
import pandas as pd

# GUI Framework
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QTabWidget, QTableWidget, QTableWidgetItem, QToolBar, QStatusBar,
    QLabel, QPushButton, QFileDialog, QMessageBox, QProgressBar,
    QComboBox, QSpinBox, QCheckBox, QInputDialog, QTextEdit, QDialog,
    QSizePolicy, QAbstractItemView
)
from PyQt5.QtCore import Qt, QThread, pyqtSignal, QSize
from PyQt5.QtGui import QFont, QIcon

# Visualization
import matplotlib
matplotlib.use("Qt5Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
import seaborn as sns

# Statistics & ML
from scipy import stats as scipy_stats
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

# ============================================================
# 1. CONSTANTS & CONFIGURATION
# ============================================================

class AppMetadata:
    NAME = "Elemental Society Analysis"
    VERSION = "3.1.0"
    CODE = "1010-PRO"
    DEVELOPER = "Mohamed Almoufty Abdoalmoutleb Abaker"

class Theme:
    """Design System Palette"""
    PRIMARY = "#1a73e8"
    PRIMARY_DARK = "#1557b0"
    PRIMARY_LIGHT = "#e8f0fe"
    SUCCESS = "#1e8e3e"
    WARNING = "#e37400"
    DANGER = "#d93025"
    BG = "#f0f2f5"
    SURFACE = "#ffffff"
    TEXT_PRIMARY = "#202124"
    TEXT_SECONDARY = "#5f6368"
    BORDER = "#dadce0"

class StyleSheets:
    """QSS Styles for Consistency"""
    TOOLBAR = f"""
        QToolBar {{ background-color: {Theme.SURFACE}; border-bottom: 1px solid {Theme.BORDER}; spacing: 6px; padding: 8px; }}
        QToolButton {{ border: none; padding: 8px 16px; border-radius: 4px; font-weight: 500; }}
        QToolButton:hover {{ background-color: {Theme.BG}; }}
        QToolButton#primary_btn {{ background-color: {Theme.PRIMARY}; color: white; }}
        QToolButton#primary_btn:hover {{ background-color: {Theme.PRIMARY_DARK}; }}
        QToolButton#success_btn {{ background-color: {Theme.SUCCESS}; color: white; }}
        QToolButton#success_btn:hover {{ background-color: #137333; }}
        QToolButton#warning_btn {{ background-color: {Theme.WARNING}; color: white; }}
    """
    
    MAIN_WINDOW = f"""
        QMainWindow {{ background-color: {Theme.BG}; }}
        QLabel {{ color: {Theme.TEXT_PRIMARY}; }}
        QTabWidget::pane {{ background: {Theme.SURFACE}; border: 1px solid {Theme.BORDER}; border-radius: 6px; top: -1px; }}
        QTabBar::tab {{ background: {Theme.BG}; padding: 12px 24px; margin-right: 2px; border-top-left-radius: 6px; border-top-right-radius: 6px; color: {Theme.TEXT_SECONDARY}; }}
        QTabBar::tab:selected {{ background: {Theme.SURFACE}; color: {Theme.PRIMARY}; border-bottom: 2px solid {Theme.PRIMARY}; font-weight: bold; }}
    """

# Element Standardization Map
ELEMENT_ALIAS_MAP = {
    'K': 'K_pct', 'K2O': 'K_pct', 'K_PCT': 'K_pct', 'K%': 'K_pct',
    'MG': 'Mg_pct', 'MGO': 'Mg_pct', 'MG_PCT': 'Mg_pct', 'MG%': 'Mg_pct',
    'CA': 'Ca_pct', 'CAO': 'Ca_pct', 'CA_PCT': 'Ca_pct', 'CA%': 'Ca_pct',
    'NA': 'Na_pct', 'NA2O': 'Na_pct', 'NA_PCT': 'Na_pct', 'NA%': 'Na_pct',
    'FE': 'Fe_pct', 'FE2O3': 'Fe_pct', 'FEO': 'Fe_pct', 'FE_PCT': 'Fe_pct', 'FE%': 'Fe_pct',
    'AU': 'Au_ppm', 'AU_PPM': 'Au_ppm', 'AU_PPB': 'Au_ppm',
    'CU': 'Cu_ppm', 'CU_PPM': 'Cu_ppm',
    'ZN': 'Zn_ppm', 'ZN_PPM': 'Zn_ppm',
    'PB': 'Pb_ppm', 'PB_PPM': 'Pb_ppm',
    'AG': 'Ag_ppm', 'AG_PPM': 'Ag_ppm',
    'AS': 'As_ppm', 'AS_PPM': 'As_ppm',
    'SB': 'Sb_ppm', 'SB_PPM': 'Sb_ppm',
    'MO': 'Mo_ppm', 'MO_PPM': 'Mo_ppm',
    'SIO2': 'SiO2_pct', 'SIO2_PCT': 'SiO2_pct', 'SIO2%': 'SiO2_pct',
    'AL': 'Al_pct', 'AL2O3': 'Al_pct', 'AL_PCT': 'Al_pct', 'AL%': 'Al_pct',
}

# ============================================================
# 2. MODEL: BACKEND LOGIC
# ============================================================

class DataProcessor:
    """Handles Data Loading and Cleaning."""
    
    @staticmethod
    def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
        renames = {}
        for col in df.columns:
            clean_col = re.sub(r'[^a-zA-Z0-9%_]', '', str(col)).upper()
            if clean_col in ELEMENT_ALIAS_MAP:
                renames[col] = ELEMENT_ALIAS_MAP[clean_col]
            else:
                # Partial matching
                for key, std in ELEMENT_ALIAS_MAP.items():
                    if key in clean_col:
                        renames[col] = std
                        break
        return df.rename(columns=renames)

    @staticmethod
    def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
        if df is None or df.empty: return df
        
        # 1. Clean Headers
        df.columns = [str(c).strip() for c in df.columns]
        df = df.loc[:, ~df.columns.duplicated()]
        
        # 2. Standardize
        df = DataProcessor.standardize_columns(df)
        
        # 3. Handle Missing Values
        missing_vals = ['', 'nan', 'NaN', 'null', 'None', 'NA', '-', '--', 'n/a', '?', '<nd>']
        df = df.replace(missing_vals, np.nan)
        
        # 4. Convert to Numeric
        text_cols = ['sample', 'id', 'point', 'lithology', 'comments', 'unit', 'rock_type']
        for col in df.columns:
            if not any(t in col.lower() for t in text_cols):
                # Strip detection limits (<, >)
                cleaned = df[col].astype(str).str.replace(r'^[<>]', '', regex=True)
                cleaned = cleaned.str.replace(',', '.')
                df[col] = pd.to_numeric(cleaned, errors='coerce')
        
        # 5. Cleanup
        df = df.dropna(how='all')
        return df

    @staticmethod
    def read_file(path: str) -> pd.DataFrame:
        """Robust file reader with encoding/delimiter sniffing."""
        ext = Path(path).suffix.lower()
        if ext in ['.xlsx', '.xls', '.xlsm']:
            return pd.read_excel(path, engine='openpyxl')

        encodings = ['utf-8-sig', 'utf-8', 'cp1252', 'latin-1', 'iso-8859-1']
        seps = [";", ",", "\t", "|"]

        for enc in encodings:
            for sep in seps:
                try:
                    df = pd.read_csv(path, sep=sep, encoding=enc, engine="python", on_bad_lines="skip")
                    if df.shape[1] > 1: return df
                except Exception:
                    continue
        raise ValueError("Could not determine file format or encoding.")

class GeoChemEngine:
    """Handles Geochemical Calculations."""
    
    @staticmethod
    def calculate_all(df: pd.DataFrame) -> Dict[str, Any]:
        results = {
            'alteration_indices': {},
            'element_ratios': {},
            'mineralization_indices': {},
            'grade_stats': {},
            'available': {'alteration': False, 'ratios': False, 'mineralization': False, 'grades': False}
        }

        cols = df.columns
        
        # --- Alteration Indices ---
        if {'K_pct', 'Mg_pct', 'Ca_pct', 'Na_pct'}.issubset(cols):
            try:
                denom = df['K_pct'] + df['Mg_pct'] + df['Ca_pct'] + df['Na_pct'] + 1e-6
                results['alteration_indices']['Ishikawa AI'] = 100 * (df['K_pct'] + df['Mg_pct']) / denom
                results['available']['alteration'] = True
            except Exception: pass

        if {'Fe_pct', 'Mg_pct', 'Na_pct', 'K_pct'}.issubset(cols):
            try:
                denom = df['Fe_pct'] + df['Mg_pct'] + df['Na_pct'] + df['K_pct'] + 1e-6
                results['alteration_indices']['CCPI'] = 100 * (df['Fe_pct'] + df['Mg_pct']) / denom
                results['available']['alteration'] = True
            except Exception: pass

        # --- Ratios ---
        ratios = [('Au_ppm', 'Cu_ppm', 'Au/Cu'), ('Zn_ppm', 'Pb_ppm', 'Zn/Pb'), ('Cu_ppm', 'Zn_ppm', 'Cu/Zn')]
        for c1, c2, name in ratios:
            if c1 in cols and c2 in cols:
                try:
                    results['element_ratios'][name] = df[c1] / (df[c2] + 1e-6)
                    results['available']['ratios'] = True
                except Exception: pass

        # --- Mineralization ---
        if 'Au_ppm' in cols:
            try:
                results['mineralization_indices']['Gold Index'] = df['Au_ppm'] * 100
                results['available']['mineralization'] = True
            except Exception: pass
            
        base_metals = [c for c in ['Cu_ppm', 'Zn_ppm', 'Pb_ppm'] if c in cols]
        if base_metals:
            try:
                results['mineralization_indices']['Base Metal Index'] = sum(df[c].fillna(0) for c in base_metals) / len(base_metals)
                results['available']['mineralization'] = True
            except Exception: pass

        # --- Grade Stats ---
        key_elements = ['Au_ppm', 'Cu_ppm', 'Zn_ppm', 'Pb_ppm', 'Ag_ppm', 'As_ppm']
        for elem in key_elements:
            if elem in cols:
                data = df[elem].dropna()
                if len(data) > 0:
                    threshold = data.quantile(0.75)
                    mean_val = data.mean()
                    
                    grade_class = 'HIGH' if mean_val > threshold * 2 else 'MEDIUM' if mean_val > threshold else 'LOW'
                    
                    results['grade_stats'][elem] = {
                        'Mean': mean_val, 'Median': data.median(), 'Std Dev': data.std(),
                        'Min': data.min(), 'Max': data.max(), 'N': len(data),
                        'Threshold': threshold, 'Grade': grade_class
                    }
                    results['available']['grades'] = True
                    
        return results

# ============================================================
# 3. WORKER THREADS (CONTROLLER)
# ============================================================

class FileLoaderWorker(QThread):
    """Background thread for file loading."""
    finished = pyqtSignal(object, str)  # dataframe, filename
    error = pyqtSignal(str)
    progress = pyqtSignal(str)

    def __init__(self, path: str):
        super().__init__()
        self.path = path

    def run(self):
        try:
            self.progress.emit("Reading file structure...")
            df = DataProcessor.read_file(self.path)
            self.progress.emit("Cleaning and standardizing data...")
            df = DataProcessor.clean_dataframe(df)
            self.progress.emit("Finalizing...")
            self.finished.emit(df, os.path.basename(self.path))
        except Exception as e:
            self.error.emit(f"Failed to load file: {str(e)}")

class AnalysisWorker(QThread):
    """Background thread for heavy computations."""
    finished = pyqtSignal(dict)
    error = pyqtSignal(str)
    progress = pyqtSignal(str)

    def __init__(self, df: pd.DataFrame):
        super().__init__()
        self.df = df

    def run(self):
        try:
            self.progress.emit("Computing Geochemistry...")
            geo_results = GeoChemEngine.calculate_all(self.df)
            self.progress.emit("Computing Statistics...")
            stats_results = self.df.describe().to_dict()
            self.progress.emit("Done.")
            self.finished.emit({'geochem': geo_results, 'stats': stats_results})
        except Exception as e:
            self.error.emit(str(e))

# ============================================================
# 4. VIEW: GUI COMPONENTS
# ============================================================

class MplCanvas(FigureCanvas):
    """Custom Matplotlib Canvas."""
    def __init__(self, parent=None, width=5, height=4, dpi=100):
        self.fig = Figure(figsize=(width, height), dpi=dpi, facecolor='white')
        self.axes = self.fig.add_subplot(111)
        super().__init__(self.fig)
        FigureCanvas.setSizePolicy(self, QSizePolicy.Expanding, QSizePolicy.Expanding)
        FigureCanvas.updateGeometry(self)

class WelcomeTab(QWidget):
    def __init__(self):
        super().__init__()
        layout = QVBoxLayout()
        layout.setAlignment(Qt.AlignCenter)
        
        lbl = QLabel(
            "<h1 style='color: #1a73e8;'>Elemental Society Analysis</h1>"
            "<h3 style='color: #5f6368;'>Professional Suite for Exploration Geochemistry</h3>"
            "<p><b>Project Code 1010-PRO</b><br>Upload data to begin analysis.</p>"
        )
        lbl.setAlignment(Qt.AlignCenter)
        layout.addWidget(lbl)
        self.setLayout(layout)

class DataTab(QWidget):
    def __init__(self):
        super().__init__()
        layout = QVBoxLayout(self)
        
        # Controls
        controls = QHBoxLayout()
        self.lbl_info = QLabel("No Data")
        self.lbl_info.setStyleSheet("font-weight: bold; color: #5f6368;")
        controls.addWidget(self.lbl_info)
        controls.addStretch()
        controls.addWidget(QLabel("Rows Limit:"))
        self.spin_limit = QSpinBox()
        self.spin_limit.setRange(10, 10000)
        self.spin_limit.setValue(500)
        self.spin_limit.valueChanged.connect(self.refresh_table) # Connected later
        controls.addWidget(self.spin_limit)
        
        layout.addLayout(controls)
        
        # Table
        self.table = QTableWidget()
        self.table.setAlternatingRowColors(True)
        self.table.setSortingEnabled(True)
        self.table.horizontalHeader().setStretchLastSection(True)
        self.table.setEditTriggers(QAbstractItemView.NoEditTriggers)
        layout.addWidget(self.table)

        self.df = None

    def set_data(self, df: pd.DataFrame):
        self.df = df
        self.lbl_info.setText(f"{len(df)} rows x {len(df.columns)} columns")
        self.refresh_table()

    def refresh_table(self):
        if self.df is None: return
        limit = min(self.spin_limit.value(), len(self.df))
        
        self.table.setRowCount(limit)
        self.table.setColumnCount(len(self.df.columns))
        self.table.setHorizontalHeaderLabels(self.df.columns)
        
        for r in range(limit):
            for c, col in enumerate(self.df.columns):
                val = self.df.iloc[r, c]
                item = QTableWidgetItem(str(val) if pd.notna(val) else "")
                self.table.setItem(r, c, item)

class AnalysisTab(QWidget):
    def __init__(self):
        super().__init__()
        layout = QVBoxLayout(self)
        
        # Controls
        controls = QHBoxLayout()
        self.combo_plot = QComboBox()
        self.combo_plot.addItems(["Correlation Matrix", "PCA Scores", "Scatter Plot", "Histogram"])
        self.combo_plot.currentTextChanged.connect(self.plot)
        
        self.combo_x = QComboBox()
        self.combo_y = QComboBox()
        
        controls.addWidget(QLabel("Type:"))
        controls.addWidget(self.combo_plot)
        controls.addWidget(QLabel("X:"))
        controls.addWidget(self.combo_x)
        controls.addWidget(QLabel("Y:"))
        controls.addWidget(self.combo_y)
        
        btn = QPushButton("Update Plot")
        btn.clicked.connect(self.plot)
        controls.addWidget(btn)
        
        layout.addLayout(controls)
        
        # Canvas
        self.canvas = MplCanvas(self, width=5, height=4, dpi=100)
        layout.addWidget(self.canvas)
        
        self.df = None

    def set_data(self, df: pd.DataFrame):
        self.df = df
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        self.combo_x.clear()
        self.combo_x.addItems(numeric_cols)
        self.combo_y.clear()
        self.combo_y.addItems(numeric_cols)
        if len(numeric_cols) > 1:
            self.combo_y.setCurrentIndex(1)

    def plot(self):
        if self.df is None: return
        
        self.canvas.fig.clear()
        ax = self.canvas.fig.add_subplot(111)
        plot_type = self.combo_plot.currentText()
        
        numeric_df = self.df.select_dtypes(include=[np.number])
        
        try:
            if plot_type == "Correlation Matrix":
                if len(numeric_df.columns) > 1:
                    sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", ax=ax)
            elif plot_type == "PCA Scores":
                if len(numeric_df.columns) >= 2:
                    scaler = StandardScaler()
                    scaled = scaler.fit_transform(numeric_df.dropna())
                    pca = SklearnPCA(n_components=2)
                    scores = pca.fit_transform(scaled)
                    ax.scatter(scores[:, 0], scores[:, 1], alpha=0.6, edgecolor='k')
                    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
                    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
            elif plot_type == "Scatter Plot":
                x, y = self.combo_x.currentText(), self.combo_y.currentText()
                if x in self.df.columns and y in self.df.columns:
                    ax.scatter(self.df[x], self.df[y], alpha=0.6, edgecolor='k', color='steelblue')
                    ax.set_xlabel(x)
                    ax.set_ylabel(y)
            elif plot_type == "Histogram":
                x = self.combo_x.currentText()
                if x in self.df.columns:
                    ax.hist(self.df[x].dropna(), bins=20, color='steelblue', edgecolor='white')
                    ax.set_xlabel(x)
            
            self.canvas.fig.tight_layout()
            self.canvas.draw()
        except Exception as e:
            print(f"Plot Error: {e}")

class GeochemTab(QWidget):
    def __init__(self):
        super().__init__()
        layout = QVBoxLayout(self)
        
        controls = QHBoxLayout()
        self.combo_type = QComboBox()
        self.combo_type.addItems(["All Results", "Alteration Indices", "Element Ratios", "Mineralization"])
        self.combo_type.currentTextChanged.connect(self.render)
        controls.addWidget(self.combo_type)
        controls.addStretch()
        
        layout.addLayout(controls)
        
        self.viewer = QTextEdit()
        self.viewer.setReadOnly(True)
        self.viewer.setFont(QFont("Segoe UI", 10))
        layout.addWidget(self.viewer)
        
        self.results = None

    def set_results(self, results: dict):
        self.results = results
        self.render()

    def render(self):
        if not self.results: return
        selection = self.combo_type.currentText()
        geochem = self.results.get('geochem', {})
        
        html = "<html><body style='font-family: sans-serif;'>"
        
        if selection == "All Results":
            html += "<h2>Comprehensive Report</h2>"
            # Simplified report for brevity
            idxs = geochem.get('alteration_indices', {})
            html += "<h3>Alteration Indices</h3><ul>"
            for k, v in idxs.items():
                html += f"<li><b>{k}</b>: Mean = {v.mean():.2f}</li>"
            html += "</ul>"
            
        elif selection == "Alteration Indices":
            html += "<h2>Alteration Indices</h2>"
            for k, v in geochem.get('alteration_indices', {}).items():
                html += f"<p><b>{k}</b>: Mean={v.mean():.4f}</p>"
                
        elif selection == "Element Ratios":
            html += "<h2>Element Ratios</h2>"
            for k, v in geochem.get('element_ratios', {}).items():
                html += f"<p><b>{k}</b>: Mean={v.mean():.4f}</p>"
                
        elif selection == "Mineralization":
            html += "<h2>Mineralization Indices</h2>"
            for k, v in geochem.get('mineralization_indices', {}).items():
                html += f"<p><b>{k}</b>: Mean={v.mean():.4f}</p>"

        html += "</body></html>"
        self.viewer.setHtml(html)

# ============================================================
# 5. CONTROLLER: MAIN WINDOW
# ============================================================

class ElementalSocietyPro(QMainWindow):
    def __init__(self):
        super().__init__()
        self.df: Optional[pd.DataFrame] = None
        self.worker: Optional[QThread] = None
        self.init_ui()

    def init_ui(self):
        self.setWindowTitle(f"{AppMetadata.NAME} v{AppMetadata.VERSION}")
        self.resize(1200, 800)
        self.setStyleSheet(StyleSheets.MAIN_WINDOW)

        # Central Widget
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout(central)
        main_layout.setContentsMargins(0, 0, 0, 0)

        # 1. Toolbar
        self.toolbar = QToolBar("Main Toolbar")
        self.toolbar.setStyleSheet(StyleSheets.TOOLBAR)
        self.toolbar.setMovable(False)
        self.addToolBar(self.toolbar)
        self.setup_toolbar()

        # 2. Status Bar
        self.status_bar = QStatusBar()
        self.setStatusBar(self.status_bar)
        
        self.progress_bar = QProgressBar()
        self.progress_bar.setVisible(False)
        self.progress_bar.setFixedWidth(200)
        self.status_bar.addPermanentWidget(self.progress_bar)
        
        self.status_label = QLabel("Ready")
        self.status_bar.addWidget(self.status_label)

        # 3. Main Content (Tabs)
        self.tabs = QTabWidget()
        main_layout.addWidget(self.tabs)

        # Initialize Tabs
        self.tab_welcome = WelcomeTab()
        self.tab_data = DataTab()
        self.tab_analysis = AnalysisTab()
        self.tab_geochem = GeochemTab()
        
        self.tabs.addTab(self.tab_welcome, "🏠 Welcome")
        self.tabs.addTab(self.tab_data, "📊 Data")
        self.tabs.addTab(self.tab_analysis, "📈 Analysis")
        self.tabs.addTab(self.tab_geochem, "🧪 Geochemistry")

    def setup_toolbar(self):
        # Upload
        btn_upload = QPushButton("Upload File")
        btn_upload.setObjectName("primary_btn")
        btn_upload.clicked.connect(self.on_upload_click)
        self.toolbar.addWidget(btn_upload)

        # Analyze
        self.btn_analyze = QPushButton("Run Analysis")
        self.btn_analyze.setObjectName("success_btn")
        self.btn_analyze.setEnabled(False)
        self.btn_analyze.clicked.connect(self.on_analyze_click)
        self.toolbar.addWidget(self.btn_analyze)
        
        self.toolbar.addSeparator()
        
        # Export
        btn_export = QPushButton("Export")
        btn_export.clicked.connect(self.on_export_click)
        self.toolbar.addWidget(btn_export)

        spacer = QWidget()
        spacer.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Preferred)
        self.toolbar.addWidget(spacer)
        
        self.file_label = QLabel("No File")
        self.toolbar.addWidget(self.file_label)

    # ==========================
    # LOGIC & SLOTS
    # ==========================

    def on_upload_click(self):
        path, _ = QFileDialog.getOpenFileName(self, "Open Data", "", "Files (*.csv *.xlsx *.xls);;All Files (*)")
        if not path: return

        self.set_busy(True, "Loading file...")
        
        self.worker = FileLoaderWorker(path)
        self.worker.progress.connect(self.status_label.setText)
        self.worker.finished.connect(self.on_data_loaded)
        self.worker.error.connect(self.on_worker_error)
        self.worker.start()

    def on_data_loaded(self, df: pd.DataFrame, filename: str):
        self.df = df
        self.set_busy(False)
        
        # Update UI
        self.file_label.setText(filename)
        self.btn_analyze.setEnabled(True)
        self.tab_data.set_data(df)
        self.tab_analysis.set_data(df)
        
        self.tabs.setCurrentWidget(self.tab_data)
        self.status_bar.showMessage(f"Loaded {len(df)} rows successfully.", 3000)

    def on_analyze_click(self):
        if self.df is None: return
        
        self.set_busy(True, "Running Analysis...")
        self.tabs.setCurrentWidget(self.tab_geochem)
        
        self.worker = AnalysisWorker(self.df)
        self.worker.progress.connect(self.status_label.setText)
        self.worker.finished.connect(self.on_analysis_done)
        self.worker.error.connect(self.on_worker_error)
        self.worker.start()

    def on_analysis_done(self, results: dict):
        self.set_busy(False)
        self.tab_geochem.set_results(results)
        self.status_bar.showMessage("Analysis Complete", 3000)

    def on_worker_error(self, msg: str):
        self.set_busy(False)
        QMessageBox.critical(self, "Error", msg)
        self.status_bar.showMessage("Error occurred", 5000)

    def on_export_click(self):
        if self.df is None:
            QMessageBox.warning(self, "Warning", "No data to export.")
            return
            
        path, _ = QFileDialog.getSaveFileName(self, "Save Data", "", "CSV (*.csv);;Excel (*.xlsx)")
        if path:
            try:
                if path.endswith('.csv'):
                    self.df.to_csv(path, index=False)
                elif path.endswith('.xlsx'):
                    self.df.to_excel(path, index=False)
                QMessageBox.information(self, "Success", "Data exported successfully.")
            except Exception as e:
                QMessageBox.critical(self, "Error", str(e))

    def set_busy(self, busy: bool, message: str = ""):
        self.progress_bar.setVisible(busy)
        self.progress_bar.setRange(0, 0) if busy else self.progress_bar.setRange(0, 1)
        self.status_label.setText(message if busy else "Ready")
        self.setEnabled(not busy)

    def closeEvent(self, event):
        if self.worker and self.worker.isRunning():
            self.worker.terminate()
            self.worker.wait()
        event.accept()

if __name__ == "__main__":
    app = QApplication(sys.argv)
    app.setStyle('Fusion') 
    window = ElementalSocietyPro()
    window.show()
    sys.exit(app.exec_())
# ELEMENTAL SOCIETY ANALYSIS - PROFESSIONAL EDITION v1.0
# Project Code: 1010
# Developer: Mohamed Almoufty Abdoalmoutleb Abaker
# ============================================================

import sys
import os
import re
from pathlib import Path
from datetime import datetime
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Elemental Society Analysis - Enterprise Edition
Project Code: 1010-PRO
Developer: Mohamed Almoufty Abdoalmoutleb Abaker
Refactored by: AI Senior Software Architect

Description:
A high-performance, non-blocking GUI application for geochemical data analysis.
Incorporates Multithreading, MVC architecture, and SOLID principles.
"""

import sys
import os
import re
import traceback
import warnings
from datetime import datetime
from pathlib import Path
from typing import Optional, Dict, List, Any

# Core Data Science
import numpy as np
import pandas as pd

# GUI Framework
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QTabWidget, QTableWidget, QTableWidgetItem, QToolBar, QStatusBar,
    QLabel, QPushButton, QFileDialog, QMessageBox, QProgressBar,
    QComboBox, QSpinBox, QCheckBox, QInputDialog, QTextEdit, QDialog,
    QSizePolicy, QAbstractItemView
)
from PyQt5.QtCore import Qt, QThread, pyqtSignal, QSize
from PyQt5.QtGui import QFont, QIcon

# Visualization
import matplotlib
matplotlib.use("Qt5Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
import seaborn as sns

# Statistics & ML
from scipy import stats as scipy_stats
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

# ============================================================
# 1. CONSTANTS & CONFIGURATION
# ============================================================

class AppMetadata:
    NAME = "Elemental Society Analysis"
    VERSION = "3.1.0"
    CODE = "1010-PRO"
    DEVELOPER = "Mohamed Almoufty Abdoalmoutleb Abaker"

class Theme:
    """Design System Palette"""
    PRIMARY = "#1a73e8"
    PRIMARY_DARK = "#1557b0"
    PRIMARY_LIGHT = "#e8f0fe"
    SUCCESS = "#1e8e3e"
    WARNING = "#e37400"
    DANGER = "#d93025"
    BG = "#f0f2f5"
    SURFACE = "#ffffff"
    TEXT_PRIMARY = "#202124"
    TEXT_SECONDARY = "#5f6368"
    BORDER = "#dadce0"

class StyleSheets:
    """QSS Styles for Consistency"""
    TOOLBAR = f"""
        QToolBar {{ background-color: {Theme.SURFACE}; border-bottom: 1px solid {Theme.BORDER}; spacing: 6px; padding: 8px; }}
        QToolButton {{ border: none; padding: 8px 16px; border-radius: 4px; font-weight: 500; }}
        QToolButton:hover {{ background-color: {Theme.BG}; }}
        QToolButton#primary_btn {{ background-color: {Theme.PRIMARY}; color: white; }}
        QToolButton#primary_btn:hover {{ background-color: {Theme.PRIMARY_DARK}; }}
        QToolButton#success_btn {{ background-color: {Theme.SUCCESS}; color: white; }}
        QToolButton#success_btn:hover {{ background-color: #137333; }}
        QToolButton#warning_btn {{ background-color: {Theme.WARNING}; color: white; }}
    """
    
    MAIN_WINDOW = f"""
        QMainWindow {{ background-color: {Theme.BG}; }}
        QLabel {{ color: {Theme.TEXT_PRIMARY}; }}
        QTabWidget::pane {{ background: {Theme.SURFACE}; border: 1px solid {Theme.BORDER}; border-radius: 6px; top: -1px; }}
        QTabBar::tab {{ background: {Theme.BG}; padding: 12px 24px; margin-right: 2px; border-top-left-radius: 6px; border-top-right-radius: 6px; color: {Theme.TEXT_SECONDARY}; }}
        QTabBar::tab:selected {{ background: {Theme.SURFACE}; color: {Theme.PRIMARY}; border-bottom: 2px solid {Theme.PRIMARY}; font-weight: bold; }}
    """

# Element Standardization Map
ELEMENT_ALIAS_MAP = {
    'K': 'K_pct', 'K2O': 'K_pct', 'K_PCT': 'K_pct', 'K%': 'K_pct',
    'MG': 'Mg_pct', 'MGO': 'Mg_pct', 'MG_PCT': 'Mg_pct', 'MG%': 'Mg_pct',
    'CA': 'Ca_pct', 'CAO': 'Ca_pct', 'CA_PCT': 'Ca_pct', 'CA%': 'Ca_pct',
    'NA': 'Na_pct', 'NA2O': 'Na_pct', 'NA_PCT': 'Na_pct', 'NA%': 'Na_pct',
    'FE': 'Fe_pct', 'FE2O3': 'Fe_pct', 'FEO': 'Fe_pct', 'FE_PCT': 'Fe_pct', 'FE%': 'Fe_pct',
    'AU': 'Au_ppm', 'AU_PPM': 'Au_ppm', 'AU_PPB': 'Au_ppm',
    'CU': 'Cu_ppm', 'CU_PPM': 'Cu_ppm',
    'ZN': 'Zn_ppm', 'ZN_PPM': 'Zn_ppm',
    'PB': 'Pb_ppm', 'PB_PPM': 'Pb_ppm',
    'AG': 'Ag_ppm', 'AG_PPM': 'Ag_ppm',
    'AS': 'As_ppm', 'AS_PPM': 'As_ppm',
    'SB': 'Sb_ppm', 'SB_PPM': 'Sb_ppm',
    'MO': 'Mo_ppm', 'MO_PPM': 'Mo_ppm',
    'SIO2': 'SiO2_pct', 'SIO2_PCT': 'SiO2_pct', 'SIO2%': 'SiO2_pct',
    'AL': 'Al_pct', 'AL2O3': 'Al_pct', 'AL_PCT': 'Al_pct', 'AL%': 'Al_pct',
}

# ============================================================
# 2. MODEL: BACKEND LOGIC
# ============================================================

class DataProcessor:
    """Handles Data Loading and Cleaning."""
    
    @staticmethod
    def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
        renames = {}
        for col in df.columns:
            clean_col = re.sub(r'[^a-zA-Z0-9%_]', '', str(col)).upper()
            if clean_col in ELEMENT_ALIAS_MAP:
                renames[col] = ELEMENT_ALIAS_MAP[clean_col]
            else:
                # Partial matching
                for key, std in ELEMENT_ALIAS_MAP.items():
                    if key in clean_col:
                        renames[col] = std
                        break
        return df.rename(columns=renames)

    @staticmethod
    def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
        if df is None or df.empty: return df
        
        # 1. Clean Headers
        df.columns = [str(c).strip() for c in df.columns]
        df = df.loc[:, ~df.columns.duplicated()]
        
        # 2. Standardize
        df = DataProcessor.standardize_columns(df)
        
        # 3. Handle Missing Values
        missing_vals = ['', 'nan', 'NaN', 'null', 'None', 'NA', '-', '--', 'n/a', '?', '<nd>']
        df = df.replace(missing_vals, np.nan)
        
        # 4. Convert to Numeric
        text_cols = ['sample', 'id', 'point', 'lithology', 'comments', 'unit', 'rock_type']
        for col in df.columns:
            if not any(t in col.lower() for t in text_cols):
                # Strip detection limits (<, >)
                cleaned = df[col].astype(str).str.replace(r'^[<>]', '', regex=True)
                cleaned = cleaned.str.replace(',', '.')
                df[col] = pd.to_numeric(cleaned, errors='coerce')
        
        # 5. Cleanup
        df = df.dropna(how='all')
        return df

    @staticmethod
    def read_file(path: str) -> pd.DataFrame:
        """Robust file reader with encoding/delimiter sniffing."""
        ext = Path(path).suffix.lower()
        if ext in ['.xlsx', '.xls', '.xlsm']:
            return pd.read_excel(path, engine='openpyxl')

        encodings = ['utf-8-sig', 'utf-8', 'cp1252', 'latin-1', 'iso-8859-1']
        seps = [";", ",", "\t", "|"]

        for enc in encodings:
            for sep in seps:
                try:
                    df = pd.read_csv(path, sep=sep, encoding=enc, engine="python", on_bad_lines="skip")
                    if df.shape[1] > 1: return df
                except Exception:
                    continue
        raise ValueError("Could not determine file format or encoding.")

class GeoChemEngine:
    """Handles Geochemical Calculations."""
    
    @staticmethod
    def calculate_all(df: pd.DataFrame) -> Dict[str, Any]:
        results = {
            'alteration_indices': {},
            'element_ratios': {},
            'mineralization_indices': {},
            'grade_stats': {},
            'available': {'alteration': False, 'ratios': False, 'mineralization': False, 'grades': False}
        }

        cols = df.columns
        
        # --- Alteration Indices ---
        if {'K_pct', 'Mg_pct', 'Ca_pct', 'Na_pct'}.issubset(cols):
            try:
                denom = df['K_pct'] + df['Mg_pct'] + df['Ca_pct'] + df['Na_pct'] + 1e-6
                results['alteration_indices']['Ishikawa AI'] = 100 * (df['K_pct'] + df['Mg_pct']) / denom
                results['available']['alteration'] = True
            except Exception: pass

        if {'Fe_pct', 'Mg_pct', 'Na_pct', 'K_pct'}.issubset(cols):
            try:
                denom = df['Fe_pct'] + df['Mg_pct'] + df['Na_pct'] + df['K_pct'] + 1e-6
                results['alteration_indices']['CCPI'] = 100 * (df['Fe_pct'] + df['Mg_pct']) / denom
                results['available']['alteration'] = True
            except Exception: pass

        # --- Ratios ---
        ratios = [('Au_ppm', 'Cu_ppm', 'Au/Cu'), ('Zn_ppm', 'Pb_ppm', 'Zn/Pb'), ('Cu_ppm', 'Zn_ppm', 'Cu/Zn')]
        for c1, c2, name in ratios:
            if c1 in cols and c2 in cols:
                try:
                    results['element_ratios'][name] = df[c1] / (df[c2] + 1e-6)
                    results['available']['ratios'] = True
                except Exception: pass

        # --- Mineralization ---
        if 'Au_ppm' in cols:
            try:
                results['mineralization_indices']['Gold Index'] = df['Au_ppm'] * 100
                results['available']['mineralization'] = True
            except Exception: pass
            
        base_metals = [c for c in ['Cu_ppm', 'Zn_ppm', 'Pb_ppm'] if c in cols]
        if base_metals:
            try:
                results['mineralization_indices']['Base Metal Index'] = sum(df[c].fillna(0) for c in base_metals) / len(base_metals)
                results['available']['mineralization'] = True
            except Exception: pass

        # --- Grade Stats ---
        key_elements = ['Au_ppm', 'Cu_ppm', 'Zn_ppm', 'Pb_ppm', 'Ag_ppm', 'As_ppm']
        for elem in key_elements:
            if elem in cols:
                data = df[elem].dropna()
                if len(data) > 0:
                    threshold = data.quantile(0.75)
                    mean_val = data.mean()
                    
                    grade_class = 'HIGH' if mean_val > threshold * 2 else 'MEDIUM' if mean_val > threshold else 'LOW'
                    
                    results['grade_stats'][elem] = {
                        'Mean': mean_val, 'Median': data.median(), 'Std Dev': data.std(),
                        'Min': data.min(), 'Max': data.max(), 'N': len(data),
                        'Threshold': threshold, 'Grade': grade_class
                    }
                    results['available']['grades'] = True
                    
        return results

# ============================================================
# 3. WORKER THREADS (CONTROLLER)
# ============================================================

class FileLoaderWorker(QThread):
    """Background thread for file loading."""
    finished = pyqtSignal(object, str)  # dataframe, filename
    error = pyqtSignal(str)
    progress = pyqtSignal(str)

    def __init__(self, path: str):
        super().__init__()
        self.path = path

    def run(self):
        try:
            self.progress.emit("Reading file structure...")
            df = DataProcessor.read_file(self.path)
            self.progress.emit("Cleaning and standardizing data...")
            df = DataProcessor.clean_dataframe(df)
            self.progress.emit("Finalizing...")
            self.finished.emit(df, os.path.basename(self.path))
        except Exception as e:
            self.error.emit(f"Failed to load file: {str(e)}")

class AnalysisWorker(QThread):
    """Background thread for heavy computations."""
    finished = pyqtSignal(dict)
    error = pyqtSignal(str)
    progress = pyqtSignal(str)

    def __init__(self, df: pd.DataFrame):
        super().__init__()
        self.df = df

    def run(self):
        try:
            self.progress.emit("Computing Geochemistry...")
            geo_results = GeoChemEngine.calculate_all(self.df)
            self.progress.emit("Computing Statistics...")
            stats_results = self.df.describe().to_dict()
            self.progress.emit("Done.")
            self.finished.emit({'geochem': geo_results, 'stats': stats_results})
        except Exception as e:
            self.error.emit(str(e))

# ============================================================
# 4. VIEW: GUI COMPONENTS
# ============================================================

class MplCanvas(FigureCanvas):
    """Custom Matplotlib Canvas."""
    def __init__(self, parent=None, width=5, height=4, dpi=100):
        self.fig = Figure(figsize=(width, height), dpi=dpi, facecolor='white')
        self.axes = self.fig.add_subplot(111)
        super().__init__(self.fig)
        FigureCanvas.setSizePolicy(self, QSizePolicy.Expanding, QSizePolicy.Expanding)
        FigureCanvas.updateGeometry(self)

class WelcomeTab(QWidget):
    def __init__(self):
        super().__init__()
        layout = QVBoxLayout()
        layout.setAlignment(Qt.AlignCenter)
        
        lbl = QLabel(
            "<h1 style='color: #1a73e8;'>Elemental Society Analysis</h1>"
            "<h3 style='color: #5f6368;'>Professional Suite for Exploration Geochemistry</h3>"
            "<p><b>Project Code 1010-PRO</b><br>Upload data to begin analysis.</p>"
        )
        lbl.setAlignment(Qt.AlignCenter)
        layout.addWidget(lbl)
        self.setLayout(layout)

class DataTab(QWidget):
    def __init__(self):
        super().__init__()
        layout = QVBoxLayout(self)
        
        # Controls
        controls = QHBoxLayout()
        self.lbl_info = QLabel("No Data")
        self.lbl_info.setStyleSheet("font-weight: bold; color: #5f6368;")
        controls.addWidget(self.lbl_info)
        controls.addStretch()
        controls.addWidget(QLabel("Rows Limit:"))
        self.spin_limit = QSpinBox()
        self.spin_limit.setRange(10, 10000)
        self.spin_limit.setValue(500)
        self.spin_limit.valueChanged.connect(self.refresh_table) # Connected later
        controls.addWidget(self.spin_limit)
        
        layout.addLayout(controls)
        
        # Table
        self.table = QTableWidget()
        self.table.setAlternatingRowColors(True)
        self.table.setSortingEnabled(True)
        self.table.horizontalHeader().setStretchLastSection(True)
        self.table.setEditTriggers(QAbstractItemView.NoEditTriggers)
        layout.addWidget(self.table)

        self.df = None

    def set_data(self, df: pd.DataFrame):
        self.df = df
        self.lbl_info.setText(f"{len(df)} rows x {len(df.columns)} columns")
        self.refresh_table()

    def refresh_table(self):
        if self.df is None: return
        limit = min(self.spin_limit.value(), len(self.df))
        
        self.table.setRowCount(limit)
        self.table.setColumnCount(len(self.df.columns))
        self.table.setHorizontalHeaderLabels(self.df.columns)
        
        for r in range(limit):
            for c, col in enumerate(self.df.columns):
                val = self.df.iloc[r, c]
                item = QTableWidgetItem(str(val) if pd.notna(val) else "")
                self.table.setItem(r, c, item)

class AnalysisTab(QWidget):
    def __init__(self):
        super().__init__()
        layout = QVBoxLayout(self)
        
        # Controls
        controls = QHBoxLayout()
        self.combo_plot = QComboBox()
        self.combo_plot.addItems(["Correlation Matrix", "PCA Scores", "Scatter Plot", "Histogram"])
        self.combo_plot.currentTextChanged.connect(self.plot)
        
        self.combo_x = QComboBox()
        self.combo_y = QComboBox()
        
        controls.addWidget(QLabel("Type:"))
        controls.addWidget(self.combo_plot)
        controls.addWidget(QLabel("X:"))
        controls.addWidget(self.combo_x)
        controls.addWidget(QLabel("Y:"))
        controls.addWidget(self.combo_y)
        
        btn = QPushButton("Update Plot")
        btn.clicked.connect(self.plot)
        controls.addWidget(btn)
        
        layout.addLayout(controls)
        
        # Canvas
        self.canvas = MplCanvas(self, width=5, height=4, dpi=100)
        layout.addWidget(self.canvas)
        
        self.df = None

    def set_data(self, df: pd.DataFrame):
        self.df = df
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        self.combo_x.clear()
        self.combo_x.addItems(numeric_cols)
        self.combo_y.clear()
        self.combo_y.addItems(numeric_cols)
        if len(numeric_cols) > 1:
            self.combo_y.setCurrentIndex(1)

    def plot(self):
        if self.df is None: return
        
        self.canvas.fig.clear()
        ax = self.canvas.fig.add_subplot(111)
        plot_type = self.combo_plot.currentText()
        
        numeric_df = self.df.select_dtypes(include=[np.number])
        
        try:
            if plot_type == "Correlation Matrix":
                if len(numeric_df.columns) > 1:
                    sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", ax=ax)
            elif plot_type == "PCA Scores":
                if len(numeric_df.columns) >= 2:
                    scaler = StandardScaler()
                    scaled = scaler.fit_transform(numeric_df.dropna())
                    pca = SklearnPCA(n_components=2)
                    scores = pca.fit_transform(scaled)
                    ax.scatter(scores[:, 0], scores[:, 1], alpha=0.6, edgecolor='k')
                    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
                    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
            elif plot_type == "Scatter Plot":
                x, y = self.combo_x.currentText(), self.combo_y.currentText()
                if x in self.df.columns and y in self.df.columns:
                    ax.scatter(self.df[x], self.df[y], alpha=0.6, edgecolor='k', color='steelblue')
                    ax.set_xlabel(x)
                    ax.set_ylabel(y)
            elif plot_type == "Histogram":
                x = self.combo_x.currentText()
                if x in self.df.columns:
                    ax.hist(self.df[x].dropna(), bins=20, color='steelblue', edgecolor='white')
                    ax.set_xlabel(x)
            
            self.canvas.fig.tight_layout()
            self.canvas.draw()
        except Exception as e:
            print(f"Plot Error: {e}")

class GeochemTab(QWidget):
    def __init__(self):
        super().__init__()
        layout = QVBoxLayout(self)
        
        controls = QHBoxLayout()
        self.combo_type = QComboBox()
        self.combo_type.addItems(["All Results", "Alteration Indices", "Element Ratios", "Mineralization"])
        self.combo_type.currentTextChanged.connect(self.render)
        controls.addWidget(self.combo_type)
        controls.addStretch()
        
        layout.addLayout(controls)
        
        self.viewer = QTextEdit()
        self.viewer.setReadOnly(True)
        self.viewer.setFont(QFont("Segoe UI", 10))
        layout.addWidget(self.viewer)
        
        self.results = None

    def set_results(self, results: dict):
        self.results = results
        self.render()

    def render(self):
        if not self.results: return
        selection = self.combo_type.currentText()
        geochem = self.results.get('geochem', {})
        
        html = "<html><body style='font-family: sans-serif;'>"
        
        if selection == "All Results":
            html += "<h2>Comprehensive Report</h2>"
            # Simplified report for brevity
            idxs = geochem.get('alteration_indices', {})
            html += "<h3>Alteration Indices</h3><ul>"
            for k, v in idxs.items():
                html += f"<li><b>{k}</b>: Mean = {v.mean():.2f}</li>"
            html += "</ul>"
            
        elif selection == "Alteration Indices":
            html += "<h2>Alteration Indices</h2>"
            for k, v in geochem.get('alteration_indices', {}).items():
                html += f"<p><b>{k}</b>: Mean={v.mean():.4f}</p>"
                
        elif selection == "Element Ratios":
            html += "<h2>Element Ratios</h2>"
            for k, v in geochem.get('element_ratios', {}).items():
                html += f"<p><b>{k}</b>: Mean={v.mean():.4f}</p>"
                
        elif selection == "Mineralization":
            html += "<h2>Mineralization Indices</h2>"
            for k, v in geochem.get('mineralization_indices', {}).items():
                html += f"<p><b>{k}</b>: Mean={v.mean():.4f}</p>"

        html += "</body></html>"
        self.viewer.setHtml(html)

# ============================================================
# 5. CONTROLLER: MAIN WINDOW
# ============================================================

class ElementalSocietyPro(QMainWindow):
    def __init__(self):
        super().__init__()
        self.df: Optional[pd.DataFrame] = None
        self.worker: Optional[QThread] = None
        self.init_ui()

    def init_ui(self):
        self.setWindowTitle(f"{AppMetadata.NAME} v{AppMetadata.VERSION}")
        self.resize(1200, 800)
        self.setStyleSheet(StyleSheets.MAIN_WINDOW)

        # Central Widget
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout(central)
        main_layout.setContentsMargins(0, 0, 0, 0)

        # 1. Toolbar
        self.toolbar = QToolBar("Main Toolbar")
        self.toolbar.setStyleSheet(StyleSheets.TOOLBAR)
        self.toolbar.setMovable(False)
        self.addToolBar(self.toolbar)
        self.setup_toolbar()

        # 2. Status Bar
        self.status_bar = QStatusBar()
        self.setStatusBar(self.status_bar)
        
        self.progress_bar = QProgressBar()
        self.progress_bar.setVisible(False)
        self.progress_bar.setFixedWidth(200)
        self.status_bar.addPermanentWidget(self.progress_bar)
        
        self.status_label = QLabel("Ready")
        self.status_bar.addWidget(self.status_label)

        # 3. Main Content (Tabs)
        self.tabs = QTabWidget()
        main_layout.addWidget(self.tabs)

        # Initialize Tabs
        self.tab_welcome = WelcomeTab()
        self.tab_data = DataTab()
        self.tab_analysis = AnalysisTab()
        self.tab_geochem = GeochemTab()
        
        self.tabs.addTab(self.tab_welcome, "🏠 Welcome")
        self.tabs.addTab(self.tab_data, "📊 Data")
        self.tabs.addTab(self.tab_analysis, "📈 Analysis")
        self.tabs.addTab(self.tab_geochem, "🧪 Geochemistry")

    def setup_toolbar(self):
        # Upload
        btn_upload = QPushButton("Upload File")
        btn_upload.setObjectName("primary_btn")
        btn_upload.clicked.connect(self.on_upload_click)
        self.toolbar.addWidget(btn_upload)

        # Analyze
        self.btn_analyze = QPushButton("Run Analysis")
        self.btn_analyze.setObjectName("success_btn")
        self.btn_analyze.setEnabled(False)
        self.btn_analyze.clicked.connect(self.on_analyze_click)
        self.toolbar.addWidget(self.btn_analyze)
        
        self.toolbar.addSeparator()
        
        # Export
        btn_export = QPushButton("Export")
        btn_export.clicked.connect(self.on_export_click)
        self.toolbar.addWidget(btn_export)

        spacer = QWidget()
        spacer.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Preferred)
        self.toolbar.addWidget(spacer)
        
        self.file_label = QLabel("No File")
        self.toolbar.addWidget(self.file_label)

    # ==========================
    # LOGIC & SLOTS
    # ==========================

    def on_upload_click(self):
        path, _ = QFileDialog.getOpenFileName(self, "Open Data", "", "Files (*.csv *.xlsx *.xls);;All Files (*)")
        if not path: return

        self.set_busy(True, "Loading file...")
        
        self.worker = FileLoaderWorker(path)
        self.worker.progress.connect(self.status_label.setText)
        self.worker.finished.connect(self.on_data_loaded)
        self.worker.error.connect(self.on_worker_error)
        self.worker.start()

    def on_data_loaded(self, df: pd.DataFrame, filename: str):
        self.df = df
        self.set_busy(False)
        
        # Update UI
        self.file_label.setText(filename)
        self.btn_analyze.setEnabled(True)
        self.tab_data.set_data(df)
        self.tab_analysis.set_data(df)
        
        self.tabs.setCurrentWidget(self.tab_data)
        self.status_bar.showMessage(f"Loaded {len(df)} rows successfully.", 3000)

    def on_analyze_click(self):
        if self.df is None: return
        
        self.set_busy(True, "Running Analysis...")
        self.tabs.setCurrentWidget(self.tab_geochem)
        
        self.worker = AnalysisWorker(self.df)
        self.worker.progress.connect(self.status_label.setText)
        self.worker.finished.connect(self.on_analysis_done)
        self.worker.error.connect(self.on_worker_error)
        self.worker.start()

    def on_analysis_done(self, results: dict):
        self.set_busy(False)
        self.tab_geochem.set_results(results)
        self.status_bar.showMessage("Analysis Complete", 3000)

    def on_worker_error(self, msg: str):
        self.set_busy(False)
        QMessageBox.critical(self, "Error", msg)
        self.status_bar.showMessage("Error occurred", 5000)

    def on_export_click(self):
        if self.df is None:
            QMessageBox.warning(self, "Warning", "No data to export.")
            return
            
        path, _ = QFileDialog.getSaveFileName(self, "Save Data", "", "CSV (*.csv);;Excel (*.xlsx)")
        if path:
            try:
                if path.endswith('.csv'):
                    self.df.to_csv(path, index=False)
                elif path.endswith('.xlsx'):
                    self.df.to_excel(path, index=False)
                QMessageBox.information(self, "Success", "Data exported successfully.")
            except Exception as e:
                QMessageBox.critical(self, "Error", str(e))

    def set_busy(self, busy: bool, message: str = ""):
        self.progress_bar.setVisible(busy)
        self.progress_bar.setRange(0, 0) if busy else self.progress_bar.setRange(0, 1)
        self.status_label.setText(message if busy else "Ready")
        self.setEnabled(not busy)

    def closeEvent(self, event):
        if self.worker and self.worker.isRunning():
            self.worker.terminate()
            self.worker.wait()
        event.accept()

if __name__ == "__main__":
    app = QApplication(sys.argv)
    app.setStyle('Fusion') 
    window = ElementalSocietyPro()
    window.show()
    sys.exit(app.exec_())
import numpy as np
import pandas as pd
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *
import matplotlib
matplotlib.use("Qt5Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
import seaborn as sns
from scipy import stats as scipy_stats
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# APPLICATION METADATA
# ============================================================

APP_NAME = "Elemental Society Analysis"
APP_VERSION = "1.0.0"
APP_CODE = "1010"
DEVELOPER = "Mohamed Almoufty Abdoalmoutleb Abaker"
EMAIL = "almoufty121@gmail.com"
BACKGROUND = "Exploration Geologist | Msc Earth Science Engineering"

# ============================================================
# DESIGN SYSTEM - PROFESSIONAL CLEAN THEME
# ============================================================

class Palette:
    # Primary brand colors
    PRIMARY = "#1a73e8"
    PRIMARY_DARK = "#1557b0"
    PRIMARY_LIGHT = "#e8f0fe"
    
    # Status colors
    SUCCESS = "#1e8e3e"
    SUCCESS_LIGHT = "#e6f4ea"
    WARNING = "#e37400"
    WARNING_LIGHT = "#fef7e0"
    DANGER = "#d93025"
    DANGER_LIGHT = "#fce8e6"
    INFO = "#1a73e8"
    INFO_LIGHT = "#e8f0fe"
    
    # Professional light theme
    BG = "#f0f2f5"
    SURFACE = "#ffffff"
    SURFACE_LIGHT = "#f8f9fa"
    BORDER = "#dadce0"
    
    TEXT_PRIMARY = "#202124"
    TEXT_SECONDARY = "#5f6368"
    TEXT_MUTED = "#9aa0a6"
    
    SIDEBAR_BG = "#ffffff"
    SIDEBAR_BG_HOVER = "#f1f3f4"
    SIDEBAR_TEXT = "#5f6368"
    SIDEBAR_TEXT_ACTIVE = "#1a73e8"

# ============================================================
# ELEMENT ALIAS MAPPING
# ============================================================

ELEMENT_ALIAS_MAP = {
    'K': 'K_pct', 'K2O': 'K_pct', 'K_PCT': 'K_pct', 'K%': 'K_pct',
    'MG': 'Mg_pct', 'MGO': 'Mg_pct', 'MG_PCT': 'Mg_pct', 'MG%': 'Mg_pct',
    'CA': 'Ca_pct', 'CAO': 'Ca_pct', 'CA_PCT': 'Ca_pct', 'CA%': 'Ca_pct',
    'NA': 'Na_pct', 'NA2O': 'Na_pct', 'NA_PCT': 'Na_pct', 'NA%': 'Na_pct',
    'FE': 'Fe_pct', 'FE2O3': 'Fe_pct', 'FEO': 'Fe_pct', 'FE_PCT': 'Fe_pct', 'FE%': 'Fe_pct',
    'AU': 'Au_ppm', 'AU_PPM': 'Au_ppm', 'AU_PPB': 'Au_ppm',
    'CU': 'Cu_ppm', 'CU_PPM': 'Cu_ppm', 'CU_PCT': 'Cu_ppm',
    'ZN': 'Zn_ppm', 'ZN_PPM': 'Zn_ppm',
    'PB': 'Pb_ppm', 'PB_PPM': 'Pb_ppm',
    'AG': 'Ag_ppm', 'AG_PPM': 'Ag_ppm',
    'AS': 'As_ppm', 'AS_PPM': 'As_ppm',
    'SB': 'Sb_ppm', 'SB_PPM': 'Sb_ppm',
    'MO': 'Mo_ppm', 'MO_PPM': 'Mo_ppm',
    'SIO2': 'SiO2_pct', 'SIO2_PCT': 'SiO2_pct', 'SIO2%': 'SiO2_pct',
    'AL': 'Al_pct', 'AL2O3': 'Al_pct', 'AL_PCT': 'Al_pct', 'AL%': 'Al_pct',
}

# ============================================================
# DATA CLEANING ENGINE
# ============================================================

def standardize_element_names(df):
    """Normalize chemical headers to canonical standard names."""
    renames = {}
    for col in df.columns:
        clean_col = re.sub(r'[^a-zA-Z0-9%_]', '', str(col)).upper()
        if clean_col in ELEMENT_ALIAS_MAP:
            renames[col] = ELEMENT_ALIAS_MAP[clean_col]
        else:
            for key, std in ELEMENT_ALIAS_MAP.items():
                if key in clean_col:
                    renames[col] = std
                    break
    return df.rename(columns=renames)

def clean_dataframe(df):
    """Universal cleaning routine for geochemical datasets."""
    if df is None or df.empty:
        return df

    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]
    df = standardize_element_names(df)

    missing_vals = ['', 'nan', 'NaN', 'null', 'None', 'NA', '-', '--', 'n/a', '?']
    df = df.replace(missing_vals, np.nan)

    text_cols = ['sample', 'id', 'point', 'lithology', 'comments', 'unit', 'rock_type']
    for col in df.columns:
        is_text = any(t in col.lower() for t in text_cols)
        if not is_text:
            cleaned_series = df[col].astype(str).str.replace(r'^[<>]', '', regex=True)
            cleaned_series = pd.to_numeric(cleaned_series.str.replace(',', '.'), errors='coerce')
            if cleaned_series.notna().sum() > 0:
                df[col] = cleaned_series

    df = df.dropna(how='all')
    return df

def read_any_file(path):
    """Read any tabular file format with auto-detection."""
    ext = Path(path).suffix.lower()
    if ext in ['.xlsx', '.xls', '.xlsm']:
        return pd.read_excel(path)

    encodings = ["utf-8", "cp1252", "latin-1", "iso-8859-1", "utf-16"]
    seps = [";", ",", "\t", "|"]

    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, engine="python", on_bad_lines="skip")
                if df.shape[1] > 1:
                    return df
            except Exception:
                pass

    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read(3000)
        sep = max([";", ",", "\t", "|"], key=lambda d: txt.count(d))
        return pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")
    except Exception:
        return pd.read_csv(path, engine="python", on_bad_lines="skip")

# ============================================================
# GEOCHEMISTRY ENGINE
# ============================================================

class GeochemistryEngine:
    @staticmethod
    def calculate_all(df):
        results = {
            'alteration_indices': {},
            'element_ratios': {},
            'mineralization_indices': {},
            'grade_stats': {},
            'available': {'alteration': False, 'ratios': False, 'mineralization': False, 'grades': False}
        }

        # Alteration Indices
        if {'K_pct', 'Mg_pct', 'Ca_pct', 'Na_pct'}.issubset(df.columns):
            try:
                results['alteration_indices']['Ishikawa AI'] = 100 * (df['K_pct'] + df['Mg_pct']) / (
                    df['K_pct'] + df['Mg_pct'] + df['Ca_pct'] + df['Na_pct'] + 1e-6)
                results['available']['alteration'] = True
            except: pass

        if {'Fe_pct', 'Mg_pct', 'Na_pct', 'K_pct'}.issubset(df.columns):
            try:
                results['alteration_indices']['CCPI'] = 100 * (df['Fe_pct'] + df['Mg_pct']) / (
                    df['Fe_pct'] + df['Mg_pct'] + df['Na_pct'] + df['K_pct'] + 1e-6)
                results['available']['alteration'] = True
            except: pass

        if {'K_pct', 'Na_pct'}.issubset(df.columns):
            try:
                results['alteration_indices']['Sericite Index'] = df['K_pct'] / (df['K_pct'] + df['Na_pct'] + 1e-6)
                results['available']['alteration'] = True
            except: pass

        # Element Ratios
        ratio_pairs = [
            ('Au_ppm', 'Cu_ppm', 'Au/Cu'), ('Zn_ppm', 'Pb_ppm', 'Zn/Pb'),
            ('Cu_ppm', 'Zn_ppm', 'Cu/Zn'), ('Au_ppm', 'Ag_ppm', 'Au/Ag'),
            ('Au_ppm', 'As_ppm', 'Au/As'), ('Cu_ppm', 'Mo_ppm', 'Cu/Mo')
        ]
        for c1, c2, name in ratio_pairs:
            if c1 in df.columns and c2 in df.columns:
                try:
                    results['element_ratios'][name] = df[c1] / (df[c2] + 1e-6)
                    results['available']['ratios'] = True
                except: pass

        # Mineralization Indices
        if 'Au_ppm' in df.columns:
            try:
                results['mineralization_indices']['Gold Index'] = df['Au_ppm'] * 100
                results['available']['mineralization'] = True
            except: pass

        base_metals = [c for c in ['Cu_ppm', 'Zn_ppm', 'Pb_ppm'] if c in df.columns]
        if base_metals:
            try:
                results['mineralization_indices']['Base Metal Index'] = sum(df[c].fillna(0) for c in base_metals) / len(base_metals)
                results['available']['mineralization'] = True
            except: pass

        pathfinders = [c for c in ['As_ppm', 'Sb_ppm', 'Mo_ppm'] if c in df.columns]
        if pathfinders:
            try:
                results['mineralization_indices']['Pathfinder Index'] = sum(df[c].fillna(0) for c in pathfinders) / len(pathfinders)
                results['available']['mineralization'] = True
            except: pass

        # Grade Statistics
        key_elements = ['Au_ppm', 'Cu_ppm', 'Zn_ppm', 'Pb_ppm', 'Ag_ppm', 'As_ppm']
        for elem in key_elements:
            if elem in df.columns:
                data = df[elem].dropna()
                if len(data) > 0:
                    threshold = data.quantile(0.75)
                    mean_val = data.mean()
                    results['grade_stats'][elem] = {
                        'Mean': mean_val, 'Median': data.median(), 'Std Dev': data.std(),
                        'Minimum': data.min(), 'Maximum': data.max(),
                        'N': len(data), 'Threshold': threshold,
                        'Above Threshold': (data > threshold).sum(),
                        '% Above Threshold': (data > threshold).sum() / len(data) * 100,
                        'Grade Class': 'HIGH' if threshold and mean_val > threshold * 2
                                       else 'MEDIUM' if threshold and mean_val > threshold
                                       else 'LOW'
                    }
                    results['available']['grades'] = True

        return results

# ============================================================
# STATISTICS ENGINE
# ============================================================

class StatisticsEngine:
    @staticmethod
    def descriptive_table(df):
        numeric_df = df.select_dtypes(include=[np.number])
        if numeric_df.empty:
            return pd.DataFrame()

        rows = []
        for col in numeric_df.columns:
            data = numeric_df[col].dropna()
            if len(data) == 0:
                continue
            mean_val = data.mean()
            rows.append({
                'Variable': col,
                'N': len(data),
                'Mean': mean_val,
                'Median': data.median(),
                'Std Dev': data.std(),
                'Min': data.min(),
                'Max': data.max(),
                'Skewness': data.skew(),
                'Kurtosis': data.kurtosis(),
                'CV %': (data.std() / mean_val * 100) if mean_val not in (0, np.nan) else np.nan,
            })
        return pd.DataFrame(rows)

    @staticmethod
    def outlier_table(df, k=1.5):
        numeric_df = df.select_dtypes(include=[np.number])
        if numeric_df.empty:
            return pd.DataFrame()

        rows = []
        for col in numeric_df.columns:
            data = numeric_df[col].dropna()
            if len(data) < 4:
                continue
            q1 = data.quantile(0.25)
            q3 = data.quantile(0.75)
            iqr = q3 - q1
            lower = q1 - k * iqr
            upper = q3 + k * iqr
            outliers = data[(data < lower) | (data > upper)]
            rows.append({
                'Variable': col,
                'N': len(data),
                'Q1': q1,
                'Q3': q3,
                'IQR': iqr,
                'Lower Fence': lower,
                'Upper Fence': upper,
                'Outliers': len(outliers),
                'Outliers %': len(outliers) / len(data) * 100,
            })
        return pd.DataFrame(rows)

    @staticmethod
    def quality_table(df, before_df=None):
        if df is None:
            return pd.DataFrame()

        rows = []
        for col in df.columns:
            series = df[col]
            missing = series.isna().sum()
            before_missing = None
            if before_df is not None and col in before_df.columns:
                before_missing = before_df[col].isna().sum()
            rows.append({
                'Column': col,
                'Type': str(series.dtype),
                'Missing': int(missing),
                'Missing %': round(missing / len(df) * 100, 2) if len(df) else 0,
                'Missing (Before Clean)': int(before_missing) if before_missing is not None else '—',
                'Unique Values': int(series.nunique(dropna=True)),
            })
        return pd.DataFrame(rows)

# ============================================================
# ICON FACTORY
# ============================================================

class Icons:
    _cache = {}

    @classmethod
    def get(cls, name, color=Palette.TEXT_SECONDARY, size=20):
        key = (name, color, size)
        if key in cls._cache:
            return cls._cache[key]
        pix = QPixmap(size, size)
        pix.fill(Qt.transparent)
        p = QPainter(pix)
        p.setRenderHint(QPainter.Antialiasing)
        pen = QPen(QColor(color))
        pen.setWidthF(size / 11.0)
        pen.setCapStyle(Qt.RoundCap)
        pen.setJoinStyle(Qt.RoundJoin)
        p.setPen(pen)
        p.setBrush(Qt.NoBrush)
        m = size * 0.15
        s = size - 2 * m
        draw = getattr(cls, f"_draw_{name}", None)
        if draw:
            draw(p, m, s, size, color)
        p.end()
        icon = QIcon(pix)
        cls._cache[key] = icon
        return icon

    @staticmethod
    def _draw_home(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m, m + s * 0.45)
        path.lineTo(size / 2, m)
        path.lineTo(m + s, m + s * 0.45)
        p.drawPath(path)
        p.drawRect(QRectF(m + s * 0.18, m + s * 0.45, s * 0.64, s * 0.55))
        p.drawRect(QRectF(m + s * 0.4, m + s * 0.62, s * 0.2, s * 0.38))

    @staticmethod
    def _draw_upload(p, m, s, size, color):
        cx = size / 2
        p.drawLine(QPointF(cx, m), QPointF(cx, m + s * 0.6))
        path = QPainterPath()
        path.moveTo(cx - s * 0.25, m + s * 0.22)
        path.lineTo(cx, m)
        path.lineTo(cx + s * 0.25, m + s * 0.22)
        p.drawPath(path)
        p.drawLine(QPointF(m, m + s * 0.85), QPointF(m + s, m + s * 0.85))
        p.drawLine(QPointF(m, m + s * 0.85), QPointF(m, m + s * 0.6))
        p.drawLine(QPointF(m + s, m + s * 0.85), QPointF(m + s, m + s * 0.6))

    @staticmethod
    def _draw_broom(p, m, s, size, color):
        p.drawLine(QPointF(m + s * 0.15, m), QPointF(m + s * 0.65, m + s * 0.55))
        path = QPainterPath()
        path.moveTo(m + s * 0.65, m + s * 0.55)
        path.lineTo(m + s, m + s * 0.35)
        path.lineTo(m + s * 0.95, m + s * 0.75)
        path.lineTo(m + s * 0.45, m + s)
        path.closeSubpath()
        p.drawPath(path)

    @staticmethod
    def _draw_chart(p, m, s, size, color):
        base = m + s
        p.drawLine(QPointF(m, m), QPointF(m, base))
        p.drawLine(QPointF(m, base), QPointF(m + s, base))
        bars = [0.25, 0.55, 0.4, 0.8]
        w = s * 0.15
        for i, h in enumerate(bars):
            x = m + s * 0.15 + i * (w + s * 0.08)
            p.drawLine(QPointF(x, base), QPointF(x, base - s * h))

    @staticmethod
    def _draw_download(p, m, s, size, color):
        cx = size / 2
        p.drawLine(QPointF(cx, m), QPointF(cx, m + s * 0.6))
        path = QPainterPath()
        path.moveTo(cx - s * 0.25, m + s * 0.38)
        path.lineTo(cx, m + s * 0.6)
        path.lineTo(cx + s * 0.25, m + s * 0.38)
        p.drawPath(path)
        p.drawLine(QPointF(m, m + s * 0.85), QPointF(m + s, m + s * 0.85))

    @staticmethod
    def _draw_document(p, m, s, size, color):
        w = s * 0.7
        path = QPainterPath()
        path.moveTo(m, m)
        path.lineTo(m + w, m)
        path.lineTo(m + s, m + s * 0.3)
        path.lineTo(m + s, m + s)
        path.lineTo(m, m + s)
        path.closeSubpath()
        p.drawPath(path)
        for i in range(3):
            y = m + s * (0.45 + i * 0.18)
            p.drawLine(QPointF(m + s * 0.18, y), QPointF(m + s * 0.75, y))

    @staticmethod
    def _draw_help(p, m, s, size, color):
        p.drawEllipse(QRectF(m, m, s, s))
        f = QFont()
        f.setBold(True)
        f.setPixelSize(int(s * 0.55))
        p.setFont(f)
        p.drawText(QRectF(m, m, s, s), Qt.AlignCenter, "?")

    @staticmethod
    def _draw_info(p, m, s, size, color):
        p.drawEllipse(QRectF(m, m, s, s))
        cx = size / 2
        p.drawLine(QPointF(cx, m + s * 0.32), QPointF(cx, m + s * 0.33))
        pen = p.pen()
        pen.setWidthF(pen.widthF() * 1.6)
        p.setPen(pen)
        p.drawPoint(QPointF(cx, m + s * 0.28))
        pen.setWidthF(pen.widthF() / 1.6)
        p.setPen(pen)
        p.drawLine(QPointF(cx, m + s * 0.45), QPointF(cx, m + s * 0.78))

    @staticmethod
    def _draw_table(p, m, s, size, color):
        p.drawRect(QRectF(m, m, s, s))
        p.drawLine(QPointF(m, m + s / 3), QPointF(m + s, m + s / 3))
        p.drawLine(QPointF(m, m + 2 * s / 3), QPointF(m + s, m + 2 * s / 3))
        p.drawLine(QPointF(m + s / 2, m), QPointF(m + s / 2, m + s))

    @staticmethod
    def _draw_trend(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m, m + s * 0.8)
        path.lineTo(m + s * 0.3, m + s * 0.45)
        path.lineTo(m + s * 0.55, m + s * 0.65)
        path.lineTo(m + s, m + s * 0.1)
        p.drawPath(path)
        path2 = QPainterPath()
        path2.moveTo(m + s * 0.78, m + s * 0.1)
        path2.lineTo(m + s, m + s * 0.1)
        path2.lineTo(m + s, m + s * 0.32)
        p.drawPath(path2)

    @staticmethod
    def _draw_flask(p, m, s, size, color):
        p.drawLine(QPointF(m + s * 0.38, m), QPointF(m + s * 0.38, m + s * 0.4))
        p.drawLine(QPointF(m + s * 0.62, m), QPointF(m + s * 0.62, m + s * 0.4))
        p.drawLine(QPointF(m + s * 0.3, m), QPointF(m + s * 0.7, m))
        path = QPainterPath()
        path.moveTo(m + s * 0.38, m + s * 0.4)
        path.lineTo(m + s * 0.1, m + s * 0.92)
        path.lineTo(m + s * 0.9, m + s * 0.92)
        path.lineTo(m + s * 0.62, m + s * 0.4)
        p.drawPath(path)
        p.drawLine(QPointF(m + s * 0.22, m + s * 0.68), QPointF(m + s * 0.78, m + s * 0.68))

    @staticmethod
    def _draw_list(p, m, s, size, color):
        for i in range(3):
            y = m + s * (0.15 + i * 0.35)
            p.drawEllipse(QPointF(m + s * 0.05, y), 1.2, 1.2)
            p.drawLine(QPointF(m + s * 0.2, y), QPointF(m + s, y))

    @staticmethod
    def _draw_trash(p, m, s, size, color):
        p.drawLine(QPointF(m + s * 0.15, m + s * 0.22), QPointF(m + s * 0.85, m + s * 0.22))
        p.drawLine(QPointF(m + s * 0.35, m + s * 0.22), QPointF(m + s * 0.4, m))
        p.drawLine(QPointF(m + s * 0.65, m + s * 0.22), QPointF(m + s * 0.6, m))
        p.drawLine(QPointF(m + s * 0.4, m), QPointF(m + s * 0.6, m))
        path = QPainterPath()
        path.moveTo(m + s * 0.22, m + s * 0.3)
        path.lineTo(m + s * 0.3, m + s)
        path.lineTo(m + s * 0.7, m + s)
        path.lineTo(m + s * 0.78, m + s * 0.3)
        p.drawPath(path)

    @staticmethod
    def _draw_pencil(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m + s * 0.1, m + s * 0.9)
        path.lineTo(m + s * 0.15, m + s * 0.65)
        path.lineTo(m + s * 0.68, m + s * 0.12)
        path.lineTo(m + s * 0.88, m + s * 0.32)
        path.lineTo(m + s * 0.35, m + s * 0.85)
        path.closeSubpath()
        p.drawPath(path)
        p.drawLine(QPointF(m + s * 0.6, m + s * 0.2), QPointF(m + s * 0.8, m + s * 0.4))

    @staticmethod
    def _draw_dot(p, m, s, size, color):
        p.setBrush(QColor(color))
        p.drawEllipse(QRectF(m, m, s, s))

    @staticmethod
    def _draw_export(p, m, s, size, color):
        cx = size / 2
        p.drawLine(QPointF(cx, m), QPointF(cx, m + s * 0.7))
        path = QPainterPath()
        path.moveTo(cx - s * 0.3, m + s * 0.4)
        path.lineTo(cx, m + s * 0.7)
        path.lineTo(cx + s * 0.3, m + s * 0.4)
        p.drawPath(path)
        p.drawLine(QPointF(m, m + s * 0.85), QPointF(m + s, m + s * 0.85))

# ============================================================
# REUSABLE UI COMPONENTS
# ============================================================

class Card(QFrame):
    def __init__(self, parent=None):
        super().__init__(parent)
        self.setObjectName("Card")
        effect = QGraphicsDropShadowEffect(self)
        effect.setBlurRadius(12)
        effect.setOffset(0, 2)
        effect.setColor(QColor(0, 0, 0, 30))
        self.setGraphicsEffect(effect)

class SectionTitle(QLabel):
    def __init__(self, text, parent=None):
        super().__init__(text, parent)
        self.setObjectName("SectionTitle")

def styled_button(text, icon_name=None, kind="default", tooltip=None):
    btn = QPushButton(text)
    if icon_name:
        btn.setIcon(Icons.get(icon_name, color="#ffffff" if kind in ("primary", "success", "danger", "warning") else Palette.TEXT_SECONDARY, size=16))
        btn.setIconSize(QSize(16, 16))
    if tooltip:
        btn.setToolTip(tooltip)
    btn.setProperty("kind", kind)
    btn.setCursor(Qt.PointingHandCursor)
    btn.setFont(QFont("Segoe UI", 10, QFont.Bold))
    return btn

# ============================================================
# MAIN APPLICATION WINDOW
# ============================================================

NAV_ITEMS = [
    ("home", "Overview"),
    ("table", "Dataset"),
    ("trend", "Distributions"),
    ("chart", "Analysis"),
    ("flask", "Geochemistry"),
    ("list", "Activity Log"),
]

class ElementalSocietyAnalysis(QMainWindow):
    def __init__(self):
        super().__init__()
        self.df = None
        self.cleaned_df = None
        self.derived_df = None
        self.is_processing = False
        self.original_filename = None
        self.init_ui()

    def init_ui(self):
        self.setWindowTitle(f"{APP_NAME} — v{APP_VERSION}")
        self.setWindowIcon(Icons.get("flask", color=Palette.PRIMARY, size=32))
        self.setGeometry(50, 50, 1680, 980)
        self.setStyleSheet(self.get_stylesheet())

        central = QWidget()
        self.setCentralWidget(central)
        root = QHBoxLayout(central)
        root.setContentsMargins(0, 0, 0, 0)
        root.setSpacing(0)

        root.addWidget(self.build_sidebar())

        right = QWidget()
        right_layout = QVBoxLayout(right)
        right_layout.setContentsMargins(0, 0, 0, 0)
        right_layout.setSpacing(0)
        right_layout.addWidget(self.build_topbar())
        right_layout.addWidget(self.build_status_panel())
        self.stack = QStackedWidget()
        right_layout.addWidget(self.stack, 1)
        root.addWidget(right, 1)

        self.build_pages()

        self.statusBar = QStatusBar()
        self.statusBar.setStyleSheet("QStatusBar { background-color: #ffffff; color: #5f6368; border-top: 1px solid #dadce0; }")
        self.statusBar.showMessage("Ready")
        self.setStatusBar(self.statusBar)

        self.log_message(f"{APP_NAME} v{APP_VERSION} initialized")
        self.log_message(f"Project Code: {APP_CODE}")

        self.nav_list.setCurrentRow(0)

    def build_sidebar(self):
        sidebar = QWidget()
        sidebar.setObjectName("Sidebar")
        sidebar.setFixedWidth(240)
        layout = QVBoxLayout(sidebar)
        layout.setContentsMargins(0, 0, 0, 0)
        layout.setSpacing(0)

        brand = QWidget()
        brand.setObjectName("Brand")
        brand_layout = QHBoxLayout(brand)
        brand_layout.setContentsMargins(20, 22, 20, 22)
        icon_lbl = QLabel()
        icon_lbl.setPixmap(Icons.get("flask", color=Palette.PRIMARY, size=26).pixmap(26, 26))
        brand_layout.addWidget(icon_lbl)
        title_box = QVBoxLayout()
        title_box.setSpacing(0)
        name_lbl = QLabel("Elemental Society")
        name_lbl.setObjectName("BrandTitle")
        sub_lbl = QLabel(f"Analysis Suite · v{APP_VERSION}")
        sub_lbl.setObjectName("BrandSubtitle")
        title_box.addWidget(name_lbl)
        title_box.addWidget(sub_lbl)
        brand_layout.addLayout(title_box)
        brand_layout.addStretch()
        layout.addWidget(brand)

        self.nav_list = QListWidget()
        self.nav_list.setObjectName("NavList")
        self.nav_list.setFocusPolicy(Qt.NoFocus)
        self.nav_list.setIconSize(QSize(18, 18))
        for icon_name, label in NAV_ITEMS:
            item = QListWidgetItem(Icons.get(icon_name, color=Palette.SIDEBAR_TEXT, size=18), f"  {label}")
            item.setSizeHint(QSize(0, 44))
            self.nav_list.addItem(item)
        self.nav_list.currentRowChanged.connect(self.on_nav_changed)
        layout.addWidget(self.nav_list, 1)

        footer = QWidget()
        footer.setObjectName("SidebarFooter")
        footer_layout = QVBoxLayout(footer)
        footer_layout.setContentsMargins(20, 14, 20, 18)
        dev_lbl = QLabel(DEVELOPER)
        dev_lbl.setObjectName("SidebarFooterName")
        dev_lbl.setWordWrap(True)
        role_lbl = QLabel(BACKGROUND)
        role_lbl.setObjectName("SidebarFooterRole")
        role_lbl.setWordWrap(True)
        footer_layout.addWidget(dev_lbl)
        footer_layout.addWidget(role_lbl)
        layout.addWidget(footer)

        return sidebar

    def on_nav_changed(self, row):
        if row >= 0:
            self.stack.setCurrentIndex(row)
            self.page_title.setText(NAV_ITEMS[row][1])

    def build_topbar(self):
        bar = QWidget()
        bar.setObjectName("TopBar")
        layout = QHBoxLayout(bar)
        layout.setContentsMargins(24, 14, 24, 14)
        layout.setSpacing(10)

        self.page_title = QLabel("Overview")
        self.page_title.setObjectName("PageTitle")
        self.page_title.setFont(QFont("Segoe UI", 16, QFont.Bold))
        layout.addWidget(self.page_title)

        self.file_info_label = QLabel("No file loaded")
        self.file_info_label.setObjectName("FileInfo")
        self.file_info_label.setFont(QFont("Segoe UI", 10))
        layout.addWidget(self.file_info_label)

        layout.addStretch()

        upload_btn = styled_button("Upload", "upload", "primary", "Upload geochemical data file")
        upload_btn.clicked.connect(self.upload_file)
        clean_btn = styled_button("Clean", "broom", "warning", "Clean the loaded dataset")
        clean_btn.clicked.connect(self.clean_data)
        analyze_btn = styled_button("Analyze", "chart", "success", "Run comprehensive EDA")
        analyze_btn.clicked.connect(self.run_analysis)
        export_btn = styled_button("Export", "export", "default", "Export cleaned data")
        export_btn.clicked.connect(self.export_data)
        report_btn = styled_button("Report", "document", "default", "View full report")
        report_btn.clicked.connect(self.show_report)
        help_btn = styled_button("", "help", "ghost", "Help")
        help_btn.setFixedWidth(36)
        help_btn.clicked.connect(self.show_help)
        about_btn = styled_button("", "info", "ghost", "About")
        about_btn.setFixedWidth(36)
        about_btn.clicked.connect(self.show_about)

        for b in (upload_btn, clean_btn, analyze_btn, export_btn, report_btn, help_btn, about_btn):
            layout.addWidget(b)

        return bar

    def build_status_panel(self):
        widget = QWidget()
        widget.setObjectName("StatusPanel")
        layout = QHBoxLayout(widget)
        layout.setContentsMargins(24, 8, 24, 8)
        layout.setSpacing(10)

        self.status_dot = QLabel()
        self.status_dot.setPixmap(Icons.get("dot", color=Palette.TEXT_MUTED, size=12).pixmap(12, 12))
        layout.addWidget(self.status_dot)

        self.status_text = QLabel("Ready")
        self.status_text.setObjectName("StatusText")
        self.status_text.setFont(QFont("Segoe UI", 10, QFont.Bold))
        layout.addWidget(self.status_text)

        self.progress_bar = QProgressBar()
        self.progress_bar.setObjectName("StatusProgress")
        self.progress_bar.setVisible(False)
        self.progress_bar.setMaximumWidth(220)
        self.progress_bar.setTextVisible(False)
        layout.addWidget(self.progress_bar)

        layout.addStretch()

        self.process_info = QLabel("")
        self.process_info.setObjectName("ProcessInfo")
        self.process_info.setFont(QFont("Segoe UI", 9))
        layout.addWidget(self.process_info)

        return widget

    def set_status(self, status, message="", progress=False):
        status_map = {
            'idle': (Palette.TEXT_MUTED, 'Ready'),
            'processing': (Palette.WARNING, 'Processing…'),
            'success': (Palette.SUCCESS, 'Success'),
            'error': (Palette.DANGER, 'Error'),
        }
        color, default_text = status_map.get(status, status_map['idle'])
        self.status_dot.setPixmap(Icons.get("dot", color=color, size=12).pixmap(12, 12))
        self.status_text.setText(message if message else default_text)
        self.status_text.setStyleSheet(f"color: {color}; font-weight: bold;")

        show_progress = progress or status == 'processing'
        self.progress_bar.setVisible(show_progress)
        if show_progress:
            self.progress_bar.setRange(0, 0)
        if status == 'processing':
            QApplication.processEvents()

    def set_processing(self, active, message="", progress=False):
        self.is_processing = active
        if active:
            self.set_status('processing', message, progress)
            self.statusBar.showMessage(f"Processing: {message}")
        else:
            self.set_status('idle', 'Ready', False)
            self.process_info.setText("")
            self.statusBar.showMessage("Ready")

    def update_progress(self, message):
        self.process_info.setText(message)
        self.status_text.setText(message)
        self.log_message(message)
        QApplication.processEvents()

    def build_pages(self):
        self.stack.addWidget(self.create_welcome_page())
        self.stack.addWidget(self.create_data_page())
        self.stack.addWidget(self.create_eda_page())
        self.stack.addWidget(self.create_analyze_page())
        self.stack.addWidget(self.create_geochemistry_page())
        self.stack.addWidget(self.create_log_page())

    def create_welcome_page(self):
        page = QWidget()
        outer = QVBoxLayout(page)
        outer.setContentsMargins(24, 20, 24, 24)

        card = Card()
        card.setStyleSheet("QFrame#Card { background-color: #ffffff; border-radius: 12px; border: 1px solid #dadce0; }")
        card_layout = QVBoxLayout(card)
        card_layout.setContentsMargins(32, 28, 32, 28)
        card_layout.setSpacing(6)

        title = QLabel("Welcome to Elemental Society Analysis")
        title.setObjectName("WelcomeTitle")
        title.setFont(QFont("Segoe UI", 22, QFont.Bold))
        subtitle = QLabel("Professional geochemical data analysis suite for exploration geologists")
        subtitle.setObjectName("WelcomeSubtitle")
        subtitle.setFont(QFont("Segoe UI", 13))
        subtitle.setStyleSheet(f"color: {Palette.TEXT_SECONDARY};")
        card_layout.addWidget(title)
        card_layout.addWidget(subtitle)
        card_layout.addSpacing(18)

        steps = [
            ("upload", "1. Upload Data", "Load CSV or Excel geochemical dataset"),
            ("broom", "2. Clean Data", "Standardize headers, handle missing values"),
            ("chart", "3. Analyze Data", "Run comprehensive exploratory analysis"),
            ("trend", "4. Visualize", "Distributions, correlations and PCA"),
            ("flask", "5. Geochemistry", "Alteration indices, ratios, grade statistics"),
            ("document", "6. Generate Report", "Professional report ready for presentation"),
        ]
        grid = QGridLayout()
        grid.setHorizontalSpacing(16)
        grid.setVerticalSpacing(14)
        for i, (icon_name, step_title, desc) in enumerate(steps):
            step_widget = QWidget()
            sl = QHBoxLayout(step_widget)
            sl.setContentsMargins(0, 0, 0, 0)
            icon_badge = QLabel()
            icon_badge.setObjectName("StepIcon")
            icon_badge.setFixedSize(36, 36)
            icon_badge.setAlignment(Qt.AlignCenter)
            icon_badge.setPixmap(Icons.get(icon_name, color=Palette.PRIMARY, size=18).pixmap(18, 18))
            text_box = QVBoxLayout()
            text_box.setSpacing(0)
            t = QLabel(step_title)
            t.setObjectName("StepTitle")
            t.setFont(QFont("Segoe UI", 12, QFont.Bold))
            d = QLabel(desc)
            d.setObjectName("StepDesc")
            d.setFont(QFont("Segoe UI", 10))
            d.setStyleSheet(f"color: {Palette.TEXT_SECONDARY};")
            d.setWordWrap(True)
            text_box.addWidget(t)
            text_box.addWidget(d)
            sl.addWidget(icon_badge)
            sl.addLayout(text_box, 1)
            grid.addWidget(step_widget, i // 2, i % 2)
        card_layout.addLayout(grid)

        card_layout.addSpacing(10)
        tip = QLabel("Tip: Start by clicking <b>Upload</b> in the top bar.")
        tip.setObjectName("WelcomeTip")
        tip.setFont(QFont("Segoe UI", 11))
        tip.setStyleSheet(f"color: {Palette.TEXT_SECONDARY};")
        card_layout.addWidget(tip)

        outer.addWidget(card)
        outer.addStretch()
        return page

    def create_data_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        self.data_info = QLabel("No data loaded")
        self.data_info.setObjectName("MutedLabel")
        self.data_info.setFont(QFont("Segoe UI", 11, QFont.Bold))
        controls.addWidget(self.data_info)
        controls.addStretch()

        controls.addWidget(QLabel("Rows:"))
        self.row_limit = QSpinBox()
        self.row_limit.setRange(10, 5000)
        self.row_limit.setValue(500)
        self.row_limit.valueChanged.connect(self.refresh_data)
        controls.addWidget(self.row_limit)

        rename_btn = styled_button("Rename File", "pencil", "default")
        rename_btn.clicked.connect(self.rename_file)
        controls.addWidget(rename_btn)

        delete_btn = styled_button("Delete Selected", "trash", "danger")
        delete_btn.clicked.connect(self.delete_selected)
        controls.addWidget(delete_btn)

        layout.addLayout(controls)

        splitter = QSplitter(Qt.Vertical)
        splitter.setChildrenCollapsible(False)
        splitter.setHandleWidth(10)

        data_card = Card()
        data_card.setStyleSheet("QFrame#Card { background-color: #ffffff; border-radius: 12px; border: 1px solid #dadce0; }")
        data_card_layout = QVBoxLayout(data_card)
        data_card_layout.setContentsMargins(2, 2, 2, 2)
        self.data_table = QTableWidget()
        self.data_table.setAlternatingRowColors(True)
        self.data_table.setSortingEnabled(True)
        self.data_table.setSelectionBehavior(QTableWidget.SelectItems)
        self.data_table.setSelectionMode(QTableWidget.ExtendedSelection)
        self.data_table.setFont(QFont("Segoe UI", 10))
        self.data_table.setStyleSheet("""
            QTableWidget { 
                background-color: #ffffff; 
                gridline-color: #e8eaed; 
                border: none;
                border-radius: 10px;
                alternate-background-color: #f8f9fa;
            }
            QTableWidget::item { padding: 8px; }
            QTableWidget::item:selected { background-color: #e8f0fe; color: #1a73e8; }
            QHeaderView::section {
                background-color: #f8f9fa;
                padding: 10px;
                border: none;
                border-bottom: 2px solid #dadce0;
                font-weight: 700;
                color: #3c4043;
                font-size: 11px;
            }
        """)
        data_card_layout.addWidget(self.data_table)
        splitter.addWidget(data_card)

        quality_card = Card()
        quality_card.setStyleSheet("QFrame#Card { background-color: #ffffff; border-radius: 12px; border: 1px solid #dadce0; }")
        quality_layout = QVBoxLayout(quality_card)
        quality_layout.setContentsMargins(16, 14, 16, 14)
        quality_layout.setSpacing(8)
        quality_header = QHBoxLayout()
        quality_title = SectionTitle("Data Quality Summary")
        quality_title.setFont(QFont("Segoe UI", 13, QFont.Bold))
        quality_header.addWidget(quality_title)
        quality_header.addStretch()
        self.quality_status_label = QLabel("")
        self.quality_status_label.setObjectName("MutedLabel")
        self.quality_status_label.setFont(QFont("Segoe UI", 10))
        quality_header.addWidget(self.quality_status_label)
        quality_layout.addLayout(quality_header)
        self.quality_table = QTableWidget()
        self.quality_table.setObjectName("ReportTable")
        self.quality_table.setAlternatingRowColors(True)
        self.quality_table.setSortingEnabled(True)
        self.quality_table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.quality_table.verticalHeader().setVisible(False)
        self.quality_table.setFont(QFont("Segoe UI", 10))
        self.quality_table.setStyleSheet("""
            QTableWidget#ReportTable {
                border: 1px solid #dadce0;
                border-radius: 8px;
                background-color: #ffffff;
                alternate-background-color: #f8f9fa;
            }
            QTableWidget#ReportTable QHeaderView::section {
                background-color: #e8f0fe;
                color: #1a73e8;
                border-bottom: 2px solid #1a73e8;
                font-size: 10px;
                font-weight: 700;
            }
            QTableWidget#ReportTable::item { font-size: 12px; padding: 8px; }
        """)
        quality_layout.addWidget(self.quality_table)
        splitter.addWidget(quality_card)

        splitter.setStretchFactor(0, 3)
        splitter.setStretchFactor(1, 2)
        layout.addWidget(splitter, 1)

        return page

    def create_eda_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        controls.addWidget(QLabel("Variable:"))
        self.eda_var = QComboBox()
        self.eda_var.currentTextChanged.connect(self.plot_distribution)
        controls.addWidget(self.eda_var)

        controls.addWidget(QLabel("Type:"))
        self.eda_type = QComboBox()
        self.eda_type.addItems(["Histogram", "PDF", "CDF", "Box Plot"])
        self.eda_type.currentTextChanged.connect(self.plot_distribution)
        controls.addWidget(self.eda_type)

        self.show_combined = QCheckBox("Show combined view")
        self.show_combined.toggled.connect(self.plot_distribution)
        controls.addWidget(self.show_combined)
        controls.addStretch()
        layout.addLayout(controls)

        content = QHBoxLayout()
        content.setSpacing(12)

        plot_card = Card()
        plot_card.setStyleSheet("QFrame#Card { background-color: #ffffff; border-radius: 12px; border: 1px solid #dadce0; }")
        plot_card_layout = QVBoxLayout(plot_card)
        self.eda_figure = Figure(figsize=(10, 6), facecolor='#ffffff')
        self.eda_canvas = FigureCanvas(self.eda_figure)
        plot_card_layout.addWidget(self.eda_canvas)
        content.addWidget(plot_card, 3)

        stats_card = Card()
        stats_card.setStyleSheet("QFrame#Card { background-color: #ffffff; border-radius: 12px; border: 1px solid #dadce0; }")
        stats_card.setMaximumWidth(280)
        stats_layout = QVBoxLayout(stats_card)
        stats_layout.setContentsMargins(16, 14, 16, 14)
        stats_layout.setSpacing(10)
        stats_layout.addWidget(SectionTitle("Variable Summary"))
        self.eda_stats_table = QTableWidget()
        self.eda_stats_table.setObjectName("ReportTable")
        self.eda_stats_table.setColumnCount(2)
        self.eda_stats_table.setHorizontalHeaderLabels(["Metric", "Value"])
        self.eda_stats_table.horizontalHeader().setStretchLastSection(True)
        self.eda_stats_table.verticalHeader().setVisible(False)
        self.eda_stats_table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.eda_stats_table.setSelectionMode(QTableWidget.NoSelection)
        self.eda_stats_table.setFont(QFont("Segoe UI", 10))
        self.eda_stats_table.setStyleSheet("""
            QTableWidget#ReportTable {
                border: 1px solid #dadce0;
                border-radius: 8px;
                background-color: #ffffff;
                alternate-background-color: #f8f9fa;
            }
            QTableWidget#ReportTable QHeaderView::section {
                background-color: #e8f0fe;
                color: #1a73e8;
                border-bottom: 2px solid #1a73e8;
                font-size: 10px;
                font-weight: 700;
            }
        """)
        stats_layout.addWidget(self.eda_stats_table, 1)
        self.eda_outlier_badge = QLabel("")
        self.eda_outlier_badge.setObjectName("OutlierBadge")
        self.eda_outlier_badge.setWordWrap(True)
        self.eda_outlier_badge.setAlignment(Qt.AlignCenter)
        self.eda_outlier_badge.setFont(QFont("Segoe UI", 10, QFont.Bold))
        stats_layout.addWidget(self.eda_outlier_badge)
        content.addWidget(stats_card, 1)

        layout.addLayout(content, 1)

        return page

    def create_analyze_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        controls.addWidget(QLabel("Plot:"))
        self.plot_type = QComboBox()
        self.plot_type.addItems(["Correlation Matrix", "PCA Scores", "Scatter Plot", "All Plots Combined", "Outlier Visualization", "Threshold Analysis"])
        self.plot_type.currentTextChanged.connect(self.update_plot)
        controls.addWidget(self.plot_type)

        controls.addWidget(QLabel("X:"))
        self.analyze_x = QComboBox()
        controls.addWidget(self.analyze_x)

        controls.addWidget(QLabel("Y:"))
        self.analyze_y = QComboBox()
        controls.addWidget(self.analyze_y)

        plot_btn = styled_button("Generate", "trend", "primary")
        plot_btn.clicked.connect(self.update_plot)
        controls.addWidget(plot_btn)
        controls.addStretch()
        layout.addLayout(controls)

        splitter = QSplitter(Qt.Vertical)
        splitter.setChildrenCollapsible(False)
        splitter.setHandleWidth(10)

        plot_card = Card()
        plot_card.setStyleSheet("QFrame#Card { background-color: #ffffff; border-radius: 12px; border: 1px solid #dadce0; }")
        plot_card_layout = QVBoxLayout(plot_card)
        self.analyze_figure = Figure(figsize=(10, 6), facecolor='#ffffff')
        self.analyze_canvas = FigureCanvas(self.analyze_figure)
        plot_card_layout.addWidget(self.analyze_canvas)
        splitter.addWidget(plot_card)

        stats_card = Card()
        stats_card.setStyleSheet("QFrame#Card { background-color: #ffffff; border-radius: 12px; border: 1px solid #dadce0; }")
        stats_card_layout = QVBoxLayout(stats_card)
        stats_card_layout.setContentsMargins(16, 14, 16, 10)
        stats_card_layout.setSpacing(8)
        stats_card_layout.addWidget(SectionTitle("Statistics & Outlier Detection"))

        self.stats_tabs = QTabWidget()
        self.stats_tabs.setObjectName("ReportTabs")
        self.stats_tabs.setStyleSheet("""
            QTabWidget#ReportTabs::pane {
                border: 1px solid #dadce0;
                border-radius: 8px;
                top: -1px;
                background-color: #ffffff;
            }
            QTabWidget#ReportTabs QTabBar::tab {
                background-color: #f8f9fa;
                color: #5f6368;
                padding: 8px 16px;
                margin-right: 2px;
                border-top-left-radius: 6px;
                border-top-right-radius: 6px;
                font-weight: 600;
                font-size: 11px;
            }
            QTabWidget#ReportTabs QTabBar::tab:selected {
                background-color: #e8f0fe;
                color: #1a73e8;
                border-bottom: 2px solid #1a73e8;
            }
        """)

        stats_tab = QWidget()
        stats_tab_layout = QVBoxLayout(stats_tab)
        stats_tab_layout.setContentsMargins(0, 8, 0, 0)
        self.stats_table = QTableWidget()
        self.stats_table.setObjectName("ReportTable")
        self.stats_table.setAlternatingRowColors(True)
        self.stats_table.setSortingEnabled(True)
        self.stats_table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.stats_table.verticalHeader().setVisible(False)
        self.stats_table.setFont(QFont("Segoe UI", 10))
        self.stats_table.setStyleSheet("""
            QTableWidget#ReportTable {
                border: 1px solid #dadce0;
                border-radius: 8px;
                background-color: #ffffff;
                alternate-background-color: #f8f9fa;
            }
            QTableWidget#ReportTable QHeaderView::section {
                background-color: #e8f0fe;
                color: #1a73e8;
                border-bottom: 2px solid #1a73e8;
                font-size: 10px;
                font-weight: 700;
            }
            QTableWidget#ReportTable::item { font-size: 12px; padding: 8px; }
        """)
        stats_tab_layout.addWidget(self.stats_table)
        self.stats_tabs.addTab(stats_tab, "Descriptive Statistics")

        outlier_tab = QWidget()
        outlier_tab_layout = QVBoxLayout(outlier_tab)
        outlier_tab_layout.setContentsMargins(0, 8, 0, 0)
        self.outlier_table = QTableWidget()
        self.outlier_table.setObjectName("ReportTable")
        self.outlier_table.setAlternatingRowColors(True)
        self.outlier_table.setSortingEnabled(True)
        self.outlier_table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.outlier_table.verticalHeader().setVisible(False)
        self.outlier_table.setFont(QFont("Segoe UI", 10))
        self.outlier_table.setStyleSheet("""
            QTableWidget#ReportTable {
                border: 1px solid #dadce0;
                border-radius: 8px;
                background-color: #ffffff;
                alternate-background-color: #f8f9fa;
            }
            QTableWidget#ReportTable QHeaderView::section {
                background-color: #e8f0fe;
                color: #1a73e8;
                border-bottom: 2px solid #1a73e8;
                font-size: 10px;
                font-weight: 700;
            }
            QTableWidget#ReportTable::item { font-size: 12px; padding: 8px; }
        """)
        outlier_tab_layout.addWidget(self.outlier_table)
        self.stats_tabs.addTab(outlier_tab, "Outlier Detection (IQR)")

        stats_card_layout.addWidget(self.stats_tabs)
        splitter.addWidget(stats_card)

        splitter.setStretchFactor(0, 3)
        splitter.setStretchFactor(1, 2)
        layout.addWidget(splitter, 1)

        return page

    def create_geochemistry_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        splitter = QSplitter(Qt.Vertical)
        splitter.setChildrenCollapsible(False)
        splitter.setHandleWidth(10)

        narrative_card = Card()
        narrative_card.setStyleSheet("QFrame#Card { background-color: #ffffff; border-radius: 12px; border: 1px solid #dadce0; }")
        narrative_layout = QVBoxLayout(narrative_card)
        narrative_layout.setContentsMargins(2, 2, 2, 2)
        self.geochem_viewer = QTextEdit()
        self.geochem_viewer.setReadOnly(True)
        self.geochem_viewer.setFont(QFont("Segoe UI", 10))
        self.geochem_viewer.setFrameShape(QFrame.NoFrame)
        self.geochem_viewer.setStyleSheet("""
            QTextEdit {
                background-color: #ffffff;
                border: none;
                padding: 12px;
                color: #202124;
            }
        """)
        narrative_layout.addWidget(self.geochem_viewer)
        splitter.addWidget(narrative_card)

        grid_card = Card()
        grid_card.setStyleSheet("QFrame#Card { background-color: #ffffff; border-radius: 12px; border: 1px solid #dadce0; }")
        grid_layout = QVBoxLayout(grid_card)
        grid_layout.setContentsMargins(16, 14, 16, 10)
        grid_layout.setSpacing(8)
        grid_layout.addWidget(SectionTitle("Geochemistry Report — Data Tables"))

        self.geochem_tabs = QTabWidget()
        self.geochem_tabs.setObjectName("ReportTabs")
        self.geochem_tabs.setStyleSheet("""
            QTabWidget#ReportTabs::pane {
                border: 1px solid #dadce0;
                border-radius: 8px;
                top: -1px;
                background-color: #ffffff;
            }
            QTabWidget#ReportTabs QTabBar::tab {
                background-color: #f8f9fa;
                color: #5f6368;
                padding: 8px 16px;
                margin-right: 2px;
                border-top-left-radius: 6px;
                border-top-right-radius: 6px;
                font-weight: 600;
                font-size: 11px;
            }
            QTabWidget#ReportTabs QTabBar::tab:selected {
                background-color: #e8f0fe;
                color: #1a73e8;
                border-bottom: 2px solid #1a73e8;
            }
        """)

        self.geochem_alteration_table = self._make_report_table()
        self.geochem_ratio_table = self._make_report_table()
        self.geochem_mineralization_table = self._make_report_table()
        self.geochem_grade_table = self._make_report_table()

        self.geochem_tabs.addTab(self._wrap_table(self.geochem_alteration_table), "Alteration Indices")
        self.geochem_tabs.addTab(self._wrap_table(self.geochem_ratio_table), "Element Ratios")
        self.geochem_tabs.addTab(self._wrap_table(self.geochem_mineralization_table), "Mineralization Indices")
        self.geochem_tabs.addTab(self._wrap_table(self.geochem_grade_table), "Grade Statistics")

        grid_layout.addWidget(self.geochem_tabs)
        splitter.addWidget(grid_card)

        splitter.setStretchFactor(0, 1)
        splitter.setStretchFactor(1, 2)
        layout.addWidget(splitter, 1)
        return page

    def _make_report_table(self):
        table = QTableWidget()
        table.setObjectName("ReportTable")
        table.setAlternatingRowColors(True)
        table.setSortingEnabled(True)
        table.setEditTriggers(QTableWidget.NoEditTriggers)
        table.verticalHeader().setVisible(False)
        table.setFont(QFont("Segoe UI", 10))
        table.setStyleSheet("""
            QTableWidget#ReportTable {
                border: 1px solid #dadce0;
                border-radius: 8px;
                background-color: #ffffff;
                alternate-background-color: #f8f9fa;
            }
            QTableWidget#ReportTable QHeaderView::section {
                background-color: #e8f0fe;
                color: #1a73e8;
                border-bottom: 2px solid #1a73e8;
                font-size: 10px;
                font-weight: 700;
            }
            QTableWidget#ReportTable::item { font-size: 12px; padding: 8px; }
        """)
        return table

    @staticmethod
    def _wrap_table(table):
        wrapper = QWidget()
        wrapper_layout = QVBoxLayout(wrapper)
        wrapper_layout.setContentsMargins(0, 8, 0, 0)
        wrapper_layout.addWidget(table)
        return wrapper

    def create_log_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        controls.addStretch()
        clear_btn = styled_button("Clear", "trash", "danger")
        clear_btn.clicked.connect(self.clear_log)
        controls.addWidget(clear_btn)
        layout.addLayout(controls)

        card = Card()
        card.setStyleSheet("QFrame#Card { background-color: #ffffff; border-radius: 12px; border: 1px solid #dadce0; }")
        card_layout = QVBoxLayout(card)
        card_layout.setContentsMargins(2, 2, 2, 2)
        self.log_text = QTextEdit()
        self.log_text.setReadOnly(True)
        self.log_text.setFont(QFont("Consolas", 10))
        self.log_text.setFrameShape(QFrame.NoFrame)
        self.log_text.setStyleSheet("""
            QTextEdit {
                background-color: #ffffff;
                border: none;
                padding: 12px;
                color: #202124;
            }
        """)
        card_layout.addWidget(self.log_text)
        layout.addWidget(card, 1)

        return page

    # ============================================================
    # LOG FUNCTIONS
    # ============================================================

    def log_message(self, message):
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.append(f"[{timestamp}]  {message}")

    def clear_log(self):
        self.log_text.clear()
        self.log_message("Log cleared")

    # ============================================================
    # DATA FUNCTIONS
    # ============================================================

    def rename_file(self):
        if self.original_filename is None:
            QMessageBox.warning(self, "No File", "No file is currently loaded.")
            return

        new_name, ok = QInputDialog.getText(
            self, "Rename File", "Enter new filename:", QLineEdit.Normal, self.original_filename)

        if ok and new_name:
            self.original_filename = new_name
            self.file_info_label.setText(new_name)
            self.log_message(f"Renamed file to: {new_name}")
            self.set_status('success', f'Renamed to {new_name}')

    def delete_selected(self):
        if self.df is None:
            QMessageBox.warning(self, "No Data", "Please load data first.")
            return

        selected = self.data_table.selectedIndexes()
        if not selected:
            QMessageBox.warning(self, "Nothing Selected", "Select one or more cells first.")
            return

        rows = set(index.row() for index in selected)
        cols = set(index.column() for index in selected)

        reply = QMessageBox.question(
            self, "Delete Selection",
            f"Delete {len(rows)} row(s) and {len(cols)} column(s)?",
            QMessageBox.Yes | QMessageBox.No)

        if reply == QMessageBox.Yes:
            for col in sorted(cols, reverse=True):
                self.df = self.df.drop(self.df.columns[col], axis=1)
            for row in sorted(rows, reverse=True):
                self.df = self.df.drop(self.df.index[row])

            self.df = self.df.reset_index(drop=True)
            self.display_data()
            self.log_message(f"Deleted {len(rows)} row(s) and {len(cols)} column(s)")
            self.set_status('success', f'Deleted {len(rows)} rows, {len(cols)} columns')

    def upload_file(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open Geochemical Data", "",
            "All Files (*);;CSV (*.csv);;Excel (*.xlsx *.xls);;Text (*.txt)"
        )
        if not file_path:
            return

        self.original_filename = os.path.basename(file_path)
        self.set_processing(True, "Loading file…")
        self.log_message(f"Loading: {self.original_filename}")
        self.update_progress(f"Reading {self.original_filename}")

        try:
            self.df = read_any_file(file_path)

            if self.df is not None and not self.df.empty:
                self.df = clean_dataframe(self.df)
                self.cleaned_df = self.df.copy()
                self.derived_df = self.df.copy()
                self.display_data()
                self.update_quality_summary()
                self.file_info_label.setText(self.original_filename)

                numeric_cols = self.df.select_dtypes(include=[np.number]).columns
                self.eda_var.clear()
                self.eda_var.addItems(numeric_cols)
                self.analyze_x.clear()
                self.analyze_x.addItems(numeric_cols)
                self.analyze_y.clear()
                self.analyze_y.addItems(numeric_cols)

                self.log_message(f"Loaded {self.df.shape[0]} rows, {self.df.shape[1]} columns")
                self.set_status('success', f'Loaded {self.df.shape[0]} rows')

                QMessageBox.information(
                    self, "File Loaded",
                    f"{self.df.shape[0]} rows × {self.df.shape[1]} columns loaded successfully.")
            else:
                QMessageBox.warning(self, "Read Error", "Could not read the selected file.")

        except Exception as e:
            self.log_message(f"Error: {str(e)}")
            self.set_status('error', 'Load failed')
            QMessageBox.critical(self, "Error", f"Failed to load file:\n{str(e)}")
        finally:
            self.set_processing(False)

    def clean_data(self):
        if self.df is None or self.df.empty:
            QMessageBox.warning(self, "No Data", "Please load data first.")
            return

        reply = QMessageBox.question(self, "Clean Data", "Clean the current dataset?", QMessageBox.Yes | QMessageBox.No)
        if reply != QMessageBox.Yes:
            return

        self.set_processing(True, "Cleaning data…")
        self.log_message("Cleaning data...")
        self.update_progress("Cleaning data...")

        try:
            rows_before = len(self.df)
            df_before = self.df.copy()
            new_cleaned_df = clean_dataframe(self.df.copy())
            rows_after = len(new_cleaned_df)

            self.cleaned_df = new_cleaned_df
            self.derived_df = new_cleaned_df.copy()
            self.df = new_cleaned_df.copy()
            self.display_data()
            self.update_quality_summary(before_df=df_before)

            self.log_message(f"Cleaned dataset: {rows_after} rows remain")
            self.set_status('success', f'Cleaned: {rows_after} rows')

            QMessageBox.information(
                self, "Cleaning Complete",
                f"Before: {rows_before} rows\nAfter: {rows_after} rows")

        except Exception as e:
            self.log_message(f"Cleaning error: {str(e)}")
            self.set_status('error', 'Cleaning failed')
            QMessageBox.critical(self, "Error", f"Cleaning failed:\n{str(e)}")
        finally:
            self.set_processing(False)

    def run_analysis(self):
        if self.df is None:
            QMessageBox.warning(self, "No Data", "Please load data first.")
            return

        self.set_processing(True, "Running analysis…")
        self.log_message("Starting analysis...")
        self.update_progress("Running analysis...")

        try:
            self.update_progress("Generating statistics...")
            self.generate_statistics()

            self.update_progress("Creating visualizations...")
            self.plot_distribution()
            self.update_plot()

            self.update_progress("Calculating geochemistry...")
            self.calculate_geochemistry()

            self.log_message("Analysis complete")
            self.set_status('success', 'Analysis complete')

            QMessageBox.information(self, "Analysis Complete", "The analysis finished successfully.")

        except Exception as e:
            self.log_message(f"Analysis error: {str(e)}")
            self.set_status('error', 'Analysis failed')
            QMessageBox.critical(self, "Error", f"Analysis failed:\n{str(e)}")
        finally:
            self.set_processing(False)

    def display_data(self):
        if self.df is None:
            return

        display_limit = min(self.row_limit.value(), len(self.df))
        self.data_table.setRowCount(display_limit)
        self.data_table.setColumnCount(len(self.df.columns))
        self.data_table.setHorizontalHeaderLabels(self.df.columns.astype(str))

        for i in range(display_limit):
            for j, col in enumerate(self.df.columns):
                val = self.df.iloc[i, j]
                item = QTableWidgetItem("" if pd.isna(val) else str(val))
                item.setFont(QFont("Segoe UI", 10))
                self.data_table.setItem(i, j, item)

        self.data_table.resizeColumnsToContents()
        self.data_info.setText(f"{len(self.df)} rows × {len(self.df.columns)} columns")

    def refresh_data(self):
        self.display_data()

    def populate_table_from_df(self, table, df, precision=4, highlight_col=None, highlight_rule=None):
        table.setSortingEnabled(False)
        if df is None or df.empty:
            table.setRowCount(0)
            table.setColumnCount(0)
            table.setSortingEnabled(True)
            return

        table.setColumnCount(len(df.columns))
        table.setHorizontalHeaderLabels([str(c) for c in df.columns])
        table.setRowCount(len(df))

        for i in range(len(df)):
            row_color = None
            if highlight_col and highlight_col in df.columns and highlight_rule:
                row_color = highlight_rule(df.iloc[i][highlight_col])

            for j, col in enumerate(df.columns):
                val = df.iloc[i, j]
                if isinstance(val, (float, np.floating)):
                    text = "—" if pd.isna(val) else f"{val:,.{precision}f}"
                    align = Qt.AlignRight | Qt.AlignVCenter
                elif isinstance(val, (int, np.integer)):
                    text = f"{val:,}"
                    align = Qt.AlignRight | Qt.AlignVCenter
                else:
                    text = "—" if pd.isna(val) else str(val)
                    align = Qt.AlignLeft | Qt.AlignVCenter

                item = QTableWidgetItem(text)
                item.setTextAlignment(align)
                if row_color is not None:
                    item.setBackground(QColor(row_color))
                item.setFont(QFont("Segoe UI", 10))
                table.setItem(i, j, item)

        table.resizeColumnsToContents()
        table.setSortingEnabled(True)

    def update_quality_summary(self, before_df=None):
        if self.df is None:
            return
        quality_df = StatisticsEngine.quality_table(self.df, before_df=before_df)
        self.populate_table_from_df(
            self.quality_table, quality_df, precision=2,
            highlight_col='Missing %',
            highlight_rule=lambda v: Palette.WARNING_LIGHT if v and v > 0 else None,
        )
        total_missing = int(quality_df['Missing'].sum()) if not quality_df.empty else 0
        self.quality_status_label.setText(
            f"{len(self.df.columns)} columns · {total_missing} missing value(s) total")

    def update_eda_stats_panel(self, var, data):
        if not hasattr(self, 'eda_stats_table'):
            return
        outliers_df = StatisticsEngine.outlier_table(self.df[[var]])
        outlier_count, outlier_pct = 0, 0.0
        if not outliers_df.empty:
            outlier_count = int(outliers_df.iloc[0]['Outliers'])
            outlier_pct = float(outliers_df.iloc[0]['Outliers %'])

        metrics = [
            ("N", f"{len(data):,}"),
            ("Mean", f"{data.mean():.4f}"),
            ("Median", f"{data.median():.4f}"),
            ("Std Dev", f"{data.std():.4f}"),
            ("Min", f"{data.min():.4f}"),
            ("Max", f"{data.max():.4f}"),
            ("Skewness", f"{data.skew():.4f}"),
            ("Kurtosis", f"{data.kurtosis():.4f}"),
            ("Outliers (IQR)", f"{outlier_count} ({outlier_pct:.1f}%)"),
        ]
        self.eda_stats_table.setRowCount(len(metrics))
        for i, (label, value) in enumerate(metrics):
            label_item = QTableWidgetItem(label)
            label_item.setFont(QFont("Segoe UI", 10, QFont.Bold))
            value_item = QTableWidgetItem(value)
            value_item.setTextAlignment(Qt.AlignRight | Qt.AlignVCenter)
            value_item.setFont(QFont("Segoe UI", 10))
            self.eda_stats_table.setItem(i, 0, label_item)
            self.eda_stats_table.setItem(i, 1, value_item)
        self.eda_stats_table.resizeColumnsToContents()

        if outlier_count > 0:
            self.eda_outlier_badge.setText(f"⚠ {outlier_count} outlier value(s) flagged beyond the IQR fences")
            self.eda_outlier_badge.setStyleSheet(
                f"background-color: {Palette.WARNING_LIGHT}; color: {Palette.WARNING}; "
                f"border-radius: 6px; padding: 8px; font-size: 11px; font-weight: bold;")
        else:
            self.eda_outlier_badge.setText("✅ No outliers detected for this variable")
            self.eda_outlier_badge.setStyleSheet(
                f"background-color: {Palette.SUCCESS_LIGHT}; color: {Palette.SUCCESS}; "
                f"border-radius: 6px; padding: 8px; font-size: 11px; font-weight: bold;")

    # ============================================================
    # STATISTICS FUNCTIONS
    # ============================================================

    def generate_statistics(self):
        if self.df is None:
            return

        numeric_df = self.df.select_dtypes(include=[np.number])
        if numeric_df.empty:
            self.log_message("No numeric columns available for statistics")
            return

        self.stats_df = StatisticsEngine.descriptive_table(self.df)
        self.outliers_df = StatisticsEngine.outlier_table(self.df)

        self.populate_table_from_df(self.stats_table, self.stats_df, precision=4)
        self.populate_table_from_df(
            self.outlier_table, self.outliers_df, precision=4,
            highlight_col='Outliers %',
            highlight_rule=lambda v: Palette.DANGER_LIGHT if v and v > 10
                                      else (Palette.WARNING_LIGHT if v and v > 0 else None),
        )

        total_outliers = int(self.outliers_df['Outliers'].sum()) if not self.outliers_df.empty else 0
        self.log_message(
            f"Statistics generated for {len(numeric_df.columns)} variables "
            f"({total_outliers} outlier value(s) flagged)")

    # ============================================================
    # PLOTTING FUNCTIONS
    # ============================================================

    def plot_distribution(self):
        if self.df is None:
            return

        var = self.eda_var.currentText()
        plot_type = self.eda_type.currentText()
        show_combined = self.show_combined.isChecked()

        if not var or var not in self.df.columns:
            return

        data = self.df[var].dropna()
        if len(data) < 2:
            return

        self.update_eda_stats_panel(var, data)

        self.eda_figure.clear()

        if show_combined:
            axes = [self.eda_figure.add_subplot(2, 2, i + 1) for i in range(4)]
            plot_types = ["Histogram", "PDF", "CDF", "Box Plot"]
        else:
            axes = [self.eda_figure.add_subplot(111)]
            plot_types = [plot_type]

        accent = Palette.PRIMARY

        for ax, ptype in zip(axes, plot_types):
            if ptype == "Histogram":
                ax.hist(data, bins=20, density=True, alpha=0.75, color=accent, edgecolor='white')
                ax.axvline(data.mean(), color=Palette.DANGER, linestyle='--', linewidth=2, label=f'Mean: {data.mean():.2f}')
                ax.axvline(data.median(), color=Palette.SUCCESS, linestyle='--', linewidth=2, label=f'Median: {data.median():.2f}')
                ax.set_ylabel('Density', fontsize=10)
                ax.legend(frameon=False, fontsize=9)
                ax.tick_params(labelsize=9)

            elif ptype == "PDF":
                ax.hist(data, bins=20, density=True, alpha=0.35, color=accent)
                x = np.linspace(data.min(), data.max(), 100)
                mu, std = scipy_stats.norm.fit(data)
                pdf = scipy_stats.norm.pdf(x, mu, std)
                ax.plot(x, pdf, color=Palette.DANGER, linewidth=2, label='Normal fit')
                ax.set_ylabel('Density', fontsize=10)
                ax.legend(frameon=False, fontsize=9)
                ax.tick_params(labelsize=9)

            elif ptype == "CDF":
                sorted_data = np.sort(data)
                y = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
                ax.plot(sorted_data, y, color=accent, linewidth=2)
                ax.set_ylabel('Cumulative Probability', fontsize=10)
                ax.tick_params(labelsize=9)

            elif ptype == "Box Plot":
                bp = ax.boxplot(data, vert=True, patch_artist=True)
                for box in bp['boxes']:
                    box.set(facecolor=accent, alpha=0.6)

            ax.set_title(f'{var} — {ptype}', fontweight='bold', fontsize=11)
            ax.set_xlabel(var if ptype != "Box Plot" else '', fontsize=10)
            ax.grid(True, alpha=0.25)
            for spine in ('top', 'right'):
                ax.spines[spine].set_visible(False)

        self.eda_figure.tight_layout()
        self.eda_canvas.draw()

    def update_plot(self):
        if self.df is None:
            return

        plot_type = self.plot_type.currentText()
        self.analyze_figure.clear()
        numeric_df = self.df.select_dtypes(include=[np.number])

        def draw_correlation(ax):
            if len(numeric_df.columns) > 1:
                corr = numeric_df.corr()
                # Find most correlated elements
                upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
                strong_pairs = []
                for col1 in upper.columns:
                    for col2 in upper.index:
                        val = upper.loc[col2, col1]
                        if pd.notna(val) and abs(val) > 0.5:
                            strong_pairs.append((col1, col2, val))
                strong_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
                
                sns.heatmap(corr, annot=True, cmap='coolwarm', center=0,
                           fmt='.2f', square=True, linewidths=0.5, ax=ax, cbar=False)
                ax.set_title('Correlation Matrix', fontweight='bold', fontsize=11)
                
                # Add annotations for strongest correlations
                if strong_pairs:
                    msg = f"Strongest: {strong_pairs[0][0]} ↔ {strong_pairs[0][1]} = {strong_pairs[0][2]:.2f}"
                    ax.text(0.02, 0.98, msg, transform=ax.transAxes, fontsize=9, 
                           verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        def draw_pca(ax):
            if len(numeric_df.columns) >= 2:
                clean_data = numeric_df.dropna()
                if len(clean_data) > 1:
                    scaler = StandardScaler()
                    scaled = scaler.fit_transform(clean_data)
                    pca = SklearnPCA()
                    scores = pca.fit_transform(scaled)
                    explained = pca.explained_variance_ratio_
                    loadings = pca.components_.T
                    
                    # Plot scores with PC1 direction arrow
                    ax.scatter(scores[:, 0], scores[:, 1],
                               c=np.linalg.norm(scores[:, :2], axis=1),
                               cmap='viridis', alpha=0.7, edgecolors='white', linewidth=0.5)
                    
                    # Add PC1 direction arrow
                    center_x = scores[:, 0].mean()
                    center_y = scores[:, 1].mean()
                    ax.arrow(center_x, center_y, 1, 0, head_width=0.5, head_length=0.5, 
                            fc='red', ec='red', linewidth=2, label='PC1 Direction')
                    
                    ax.set_xlabel(f'PC1 ({explained[0] * 100:.1f}%)', fontsize=10)
                    ax.set_ylabel(f'PC2 ({explained[1] * 100:.1f}%)', fontsize=10)
                    ax.set_title('PCA Scores', fontweight='bold', fontsize=11)
                    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.4)
                    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.4)
                    ax.grid(True, alpha=0.25)
                    ax.tick_params(labelsize=9)
                    
                    # Show loadings
                    for i, col in enumerate(clean_data.columns):
                        ax.annotate(col, (loadings[i, 0]*3, loadings[i, 1]*3), fontsize=8)

        def draw_scatter(ax):
            x_col = self.analyze_x.currentText()
            y_col = self.analyze_y.currentText()
            if x_col and y_col and x_col in self.df.columns and y_col in self.df.columns:
                data = self.df[[x_col, y_col]].dropna()
                ax.scatter(data[x_col], data[y_col], alpha=0.7, color=Palette.PRIMARY, edgecolors='white')
                
                # Add trend line
                if len(data) > 2:
                    z = np.polyfit(data[x_col], data[y_col], 1)
                    p = np.poly1d(z)
                    x_line = np.linspace(data[x_col].min(), data[x_col].max(), 100)
                    ax.plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend Line')
                    ax.legend()
                
                ax.set_xlabel(x_col, fontsize=10)
                ax.set_ylabel(y_col, fontsize=10)
                ax.set_title('Scatter Plot', fontweight='bold', fontsize=11)
                ax.grid(True, alpha=0.25)
                ax.tick_params(labelsize=9)

        def draw_outliers(ax):
            if len(numeric_df.columns) > 0:
                # Show box plot with outlier highlighting
                data = numeric_df.iloc[:, 0].dropna()
                if len(data) > 1:
                    bp = ax.boxplot(data, vert=True, patch_artist=True, showfliers=True)
                    for box in bp['boxes']:
                        box.set(facecolor=Palette.PRIMARY, alpha=0.6)
                    
                    # Highlight outliers
                    q1 = data.quantile(0.25)
                    q3 = data.quantile(0.75)
                    iqr = q3 - q1
                    lower = q1 - 1.5 * iqr
                    upper = q3 + 1.5 * iqr
                    outliers = data[(data < lower) | (data > upper)]
                    if len(outliers) > 0:
                        for val in outliers:
                            ax.scatter(1, val, color='red', s=100, zorder=5, edgecolors='darkred', linewidth=2)
                    
                    ax.set_title(f'Outlier Detection: {numeric_df.columns[0]}', fontweight='bold', fontsize=11)
                    ax.set_ylabel(numeric_df.columns[0], fontsize=10)
                    ax.tick_params(labelsize=9)
                    ax.grid(True, alpha=0.25)

        def draw_threshold(ax):
            if len(numeric_df.columns) > 0:
                col = numeric_df.columns[0]
                data = numeric_df[col].dropna()
                if len(data) > 1:
                    threshold = data.quantile(0.75)
                    above = data[data > threshold]
                    
                    ax.hist(data, bins=20, alpha=0.6, color=Palette.PRIMARY, edgecolor='white')
                    ax.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold: {threshold:.2f}')
                    ax.fill_betweenx([0, ax.get_ylim()[1]], threshold, data.max(), alpha=0.3, color='red')
                    
                    ax.set_xlabel(col, fontsize=10)
                    ax.set_ylabel('Frequency', fontsize=10)
                    ax.set_title(f'Threshold Analysis: {col}', fontweight='bold', fontsize=11)
                    ax.legend()
                    ax.grid(True, alpha=0.25)
                    ax.tick_params(labelsize=9)
                    
                    # Add annotation
                    pct = len(above) / len(data) * 100
                    ax.text(0.02, 0.98, f'{len(above)} samples above threshold ({pct:.1f}%)', 
                           transform=ax.transAxes, fontsize=9, verticalalignment='top',
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        if plot_type == "All Plots Combined":
            ax1 = self.analyze_figure.add_subplot(2, 2, 1)
            ax2 = self.analyze_figure.add_subplot(2, 2, 2)
            ax3 = self.analyze_figure.add_subplot(2, 2, 3)
            ax4 = self.analyze_figure.add_subplot(2, 2, 4)
            draw_correlation(ax1)
            draw_pca(ax2)
            draw_scatter(ax3)
            draw_distribution(ax4)
            for ax in (ax2, ax3, ax4):
                ax.grid(True, alpha=0.25)
        elif plot_type == "Correlation Matrix":
            draw_correlation(self.analyze_figure.add_subplot(111))
        elif plot_type == "PCA Scores":
            ax = self.analyze_figure.add_subplot(111)
            draw_pca(ax)
            ax.grid(True, alpha=0.25)
        elif plot_type == "Scatter Plot":
            ax = self.analyze_figure.add_subplot(111)
            draw_scatter(ax)
            ax.grid(True, alpha=0.25)
        elif plot_type == "Outlier Visualization":
            ax = self.analyze_figure.add_subplot(111)
            draw_outliers(ax)
        elif plot_type == "Threshold Analysis":
            ax = self.analyze_figure.add_subplot(111)
            draw_threshold(ax)

        self.analyze_figure.tight_layout()
        self.analyze_canvas.draw()

    def draw_distribution(self, ax):
        if len(self.df.select_dtypes(include=[np.number]).columns) > 0:
            numeric_df = self.df.select_dtypes(include=[np.number])
            col = numeric_df.columns[0]
            data = numeric_df[col].dropna()
            if len(data) > 1:
                ax.hist(data, bins=20, density=True, alpha=0.75, color=Palette.PRIMARY, edgecolor='white')
                ax.set_title(f'Distribution: {col}', fontweight='bold', fontsize=11)
                ax.set_xlabel(col, fontsize=10)
                ax.set_ylabel('Density', fontsize=10)
                ax.grid(True, alpha=0.25)
                ax.tick_params(labelsize=9)

    # ============================================================
    # GEOCHEMISTRY FUNCTIONS
    # ============================================================

    def calculate_geochemistry(self):
        if self.df is None:
            return

        geochem = GeochemistryEngine.calculate_all(self.df)

        html = []
        html.append("<html><head><style>")
        html.append("""
            body { font-family: 'Segoe UI', Arial, sans-serif; margin: 4px; color: #202124; background: #ffffff; }
            h1 { color: #1a73e8; border-bottom: 3px solid #1a73e8; padding-bottom: 8px; font-size: 22px; font-weight: 700; }
            h2 { color: #202124; margin-top: 22px; font-size: 16px; font-weight: 700; border-bottom: 2px solid #e8eaed; padding-bottom: 6px; }
            .card { background-color: #f8f9fa; border-radius: 8px; padding: 15px; margin: 10px 0; border-left: 4px solid #1a73e8; }
            .stat-box { display: inline-block; margin: 3px; padding: 8px 15px; background: white; border-radius: 6px; border: 1px solid #dadce0; }
            .stat-value { font-weight: bold; color: #1a73e8; font-size: 16px; }
            .stat-label { font-size: 10px; color: #5f6368; }
            .badge { display: inline-block; padding: 2px 10px; border-radius: 10px; font-size: 11px; font-weight: 600; }
            .badge-high { background: #fce8e6; color: #d93025; }
            .badge-medium { background: #fef7e0; color: #e37400; }
            .badge-low { background: #e6f4ea; color: #1e8e3e; }
            .badge-available { background: #e6f4ea; color: #1e8e3e; }
            .badge-missing { background: #fce8e6; color: #d93025; }
            table { border-collapse: collapse; width: 100%; margin: 10px 0; font-size: 12px; }
            th { background-color: #e8f0fe; color: #1a73e8; padding: 8px; text-align: left; font-weight: 700; border-bottom: 2px solid #1a73e8; }
            td { padding: 8px; border-bottom: 1px solid #dadce0; }
            .report-table { border: 1px solid #dadce0; border-radius: 8px; overflow: hidden; }
        """)
        html.append("</style></head><body>")

        html.append("<h1>📊 Exploration Geochemistry Report</h1>")
        html.append(f"<p style='color: #5f6368;'><b>Generated:</b> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')} | ")
        html.append(f"<b>Dataset:</b> {len(self.df)} samples × {len(self.df.columns)} variables</p>")

        avail = geochem.get('available', {})
        html.append("<div class='card'>")
        html.append("<h3 style='margin-top: 0;'>Available Calculations</h3>")
        for key, val in avail.items():
            badge = "badge-available" if val else "badge-missing"
            status = "Available" if val else "Not Available"
            html.append(f"<span class='badge {badge}'>{key.replace('_', ' ').title()}: {status}</span> ")
        html.append("</div>")

        html.append("<h2>Alteration Indices</h2>")
        if geochem['alteration_indices']:
            html.append("<div class='card'>")
            for name, values in geochem['alteration_indices'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{values.min():.4f} - {values.max():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No alteration indices could be calculated.</p></div>")

        html.append("<h2>Element Ratios</h2>")
        if geochem['element_ratios']:
            html.append("<div class='card'>")
            for name, values in geochem['element_ratios'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No element ratios could be calculated.</p></div>")

        html.append("<h2>Mineralization Indices</h2>")
        if geochem['mineralization_indices']:
            html.append("<div class='card'>")
            for name, values in geochem['mineralization_indices'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No mineralization indices could be calculated.</p></div>")

        html.append("<h2>Grade Statistics</h2>")
        if geochem['grade_stats']:
            html.append("<div class='card'>")
            for element, elem_stats in geochem['grade_stats'].items():
                if elem_stats:
                    grade_class = elem_stats.get('Grade Class', 'LOW')
                    badge_class = 'badge-high' if grade_class == 'HIGH' else 'badge-medium' if grade_class == 'MEDIUM' else 'badge-low'
                    html.append(f"""
                    <div style='margin: 10px 0; padding: 10px; background: white; border-radius: 6px; border: 1px solid #dadce0;'>
                        <b>{element}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{elem_stats.get('Mean', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{elem_stats.get('Median', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Std Dev</div><div class='stat-value'>{elem_stats.get('Std Dev', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{elem_stats.get('Minimum', 0):.4f} - {elem_stats.get('Maximum', 0):.4f}</div></div>
                        <div style='margin-top: 5px;'>
                            <span class='badge {badge_class}'>Grade: {grade_class}</span>
                            <span style='margin-left: 10px;'>Threshold: {elem_stats.get('Threshold', 0):.4f}</span>
                            <span style='margin-left: 10px;'>Above: {elem_stats.get('Above Threshold', 0)} ({elem_stats.get('% Above Threshold', 0):.1f}%)</span>
                        </div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No grade statistics could be calculated.</p></div>")

        html.append(f"""
        <div style='margin-top: 20px; padding: 15px; background-color: {Palette.PRIMARY_LIGHT}; border-radius: 8px; border-left: 4px solid {Palette.PRIMARY};'>
            <p style='font-weight: 700; margin-top: 0;'>📌 Summary</p>
            <table style='width: auto;'>
        """)
        html.append(f"<tr><td>Total Samples</td><td><b>{len(self.df)}</b></td></tr>")
        html.append(f"<tr><td>Total Variables</td><td><b>{len(self.df.columns)}</b></td></tr>")
        html.append(f"<tr><td>Alteration Indices</td><td><b>{len(geochem['alteration_indices'])}</b></td></tr>")
        html.append(f"<tr><td>Element Ratios</td><td><b>{len(geochem['element_ratios'])}</b></td></tr>")
        html.append(f"<tr><td>Mineralization Indices</td><td><b>{len(geochem['mineralization_indices'])}</b></td></tr>")
        html.append(f"<tr><td>Grade Statistics</td><td><b>{len(geochem['grade_stats'])}</b></td></tr>")
        html.append("</table></div></body></html>")

        self.geochem_viewer.setHtml("\n".join(html))

        def series_dict_to_df(series_dict):
            rows = []
            for name, values in series_dict.items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    rows.append({
                        'Name': name, 'N': int(values.notna().sum()),
                        'Mean': values.mean(), 'Median': values.median(),
                        'Std Dev': values.std(), 'Min': values.min(), 'Max': values.max(),
                    })
            return pd.DataFrame(rows)

        self.populate_table_from_df(self.geochem_alteration_table,
                                     series_dict_to_df(geochem['alteration_indices']))
        self.populate_table_from_df(self.geochem_ratio_table,
                                     series_dict_to_df(geochem['element_ratios']))
        self.populate_table_from_df(self.geochem_mineralization_table,
                                     series_dict_to_df(geochem['mineralization_indices']))

        if geochem['grade_stats']:
            grade_df = pd.DataFrame(geochem['grade_stats']).T.reset_index().rename(columns={'index': 'Element'})
            grade_df = grade_df[['Element', 'N', 'Mean', 'Median', 'Std Dev', 'Minimum', 'Maximum',
                                  'Threshold', 'Above Threshold', '% Above Threshold', 'Grade Class']]
        else:
            grade_df = pd.DataFrame()

        self.populate_table_from_df(
            self.geochem_grade_table, grade_df, precision=4,
            highlight_col='Grade Class',
            highlight_rule=lambda v: Palette.DANGER_LIGHT if v == 'HIGH'
                                      else (Palette.WARNING_LIGHT if v == 'MEDIUM' else Palette.SUCCESS_LIGHT),
        )

    def show_report(self):
        self.nav_list.setCurrentRow(4)

    # ============================================================
    # EXPORT FUNCTIONS
    # ============================================================

    def export_data(self):
        if self.df is None:
            QMessageBox.warning(self, "No Data", "There is no data to export.")
            return

        df = self.df

        formats = {
            "CSV (;)": "csv_semicolon",
            "CSV (,)": "csv_comma",
            "Excel (.xlsx)": "excel",
            "JSON": "json"
        }

        format_choice, ok = QInputDialog.getItem(
            self, "Export Format", "Select format:", list(formats.keys()), 0, False)
        if not ok:
            return

        file_ext_map = {
            "csv_semicolon": "csv",
            "csv_comma": "csv",
            "excel": "xlsx",
            "json": "json"
        }

        file_path, _ = QFileDialog.getSaveFileName(
            self, "Export Data", f"cleaned_data.{file_ext_map[formats[format_choice]]}",
            f"{format_choice} (*.{file_ext_map[formats[format_choice]]})")

        if file_path:
            try:
                self.set_processing(True, "Exporting…")
                self.update_progress(f"Exporting to {format_choice}...")

                fmt = formats[format_choice]
                if fmt == "csv_semicolon":
                    df.to_csv(file_path, sep=';', index=False, encoding='utf-8-sig')
                elif fmt == "csv_comma":
                    df.to_csv(file_path, sep=',', index=False, encoding='utf-8-sig')
                elif fmt == "excel":
                    df.to_excel(file_path, index=False)
                elif fmt == "json":
                    df.to_json(file_path, orient='records', indent=2)

                self.log_message(f"Exported: {file_path}")
                self.set_status('success', 'Export complete')
                QMessageBox.information(self, "Export Complete", f"Data exported to:\n{file_path}")
            except Exception as e:
                self.set_status('error', 'Export failed')
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")
            finally:
                self.set_processing(False)

    # ============================================================
    # HELP AND ABOUT
    # ============================================================

    def show_help(self):
        help_text = f"""{APP_NAME} v{APP_VERSION}

Project Code: {APP_CODE}
Developer: {DEVELOPER}

QUICK START
1. Upload your data
2. Clean the data
3. Run analysis
4. Review visualizations
5. Check geochemistry
6. Generate report

FEATURES

Data Management
• Upload CSV/Excel files
• Clean data (missing values, outliers)
• Rename file
• Delete selected rows/columns

Statistics
• Mean, Median, Std Dev
• Variance, Skewness, Kurtosis
• Correlation Matrix

Visualization
• Correlation Heatmap
• PCA Scores with PC1 direction
• Scatter Plot with Trend Line
• Outlier Visualization
• Threshold Analysis
• Combined view
• EDA: Histogram, PDF, CDF, Box Plot

Geochemistry
• Alteration Indices
• Element Ratios
• Mineralization Indices
• Grade Statistics

Export
• CSV, Excel, JSON

Copyright © 2026 {DEVELOPER}
"""
        QMessageBox.information(self, "Help", help_text)

    def show_about(self):
        about_text = f"""{APP_NAME}

Version: {APP_VERSION}
Project Code: {APP_CODE}

Developer: {DEVELOPER}
Email: {EMAIL}
{BACKGROUND}

Professional geochemical data cleaning and
exploratory data analysis suite for mineral
exploration and geochemistry.

Copyright © 2026 {DEVELOPER}
All rights reserved.
"""
        QMessageBox.about(self, "About", about_text)

    # --------------------------------------------------------
    # STYLESHEET
    # --------------------------------------------------------

    def get_stylesheet(self):
        P = Palette
        return f"""
            QMainWindow {{ background-color: {P.BG}; }}
            QWidget {{ font-family: 'Segoe UI', Arial, sans-serif; font-size: 13px; color: {P.TEXT_PRIMARY}; }}

            /* Sidebar */
            #Sidebar {{ background-color: {P.SIDEBAR_BG}; border-right: 1px solid {P.BORDER}; }}
            #Brand {{ border-bottom: 1px solid {P.BORDER}; }}
            #BrandTitle {{ color: {P.TEXT_PRIMARY}; font-size: 15px; font-weight: 700; }}
            #BrandSubtitle {{ color: {P.TEXT_SECONDARY}; font-size: 11px; }}
            #NavList {{
                background-color: transparent;
                border: none;
                outline: none;
                padding: 8px;
            }}
            #NavList::item {{
                color: {P.SIDEBAR_TEXT};
                border-radius: 8px;
                padding-left: 10px;
                margin: 2px 4px;
                font-weight: 500;
            }}
            #NavList::item:hover {{ background-color: {P.SIDEBAR_BG_HOVER}; color: {P.TEXT_PRIMARY}; }}
            #NavList::item:selected {{ background-color: {P.PRIMARY_LIGHT}; color: {P.PRIMARY}; font-weight: 700; }}
            #SidebarFooter {{ border-top: 1px solid {P.BORDER}; }}
            #SidebarFooterName {{ color: {P.TEXT_PRIMARY}; font-size: 11px; font-weight: 600; }}
            #SidebarFooterRole {{ color: {P.TEXT_SECONDARY}; font-size: 10px; }}

            /* Top bar / status */
            #TopBar {{ background-color: {P.SURFACE}; border-bottom: 1px solid {P.BORDER}; }}
            #PageTitle {{ font-size: 17px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}
            #FileInfo {{ color: {P.TEXT_SECONDARY}; font-size: 12px; padding-left: 6px; }}
            #StatusPanel {{ background-color: {P.SURFACE}; border-bottom: 1px solid {P.BORDER}; }}
            #StatusText {{ color: {P.TEXT_SECONDARY}; font-size: 12px; }}
            #ProcessInfo {{ color: {P.TEXT_MUTED}; font-size: 11px; }}
            #StatusProgress {{
                border: none; border-radius: 3px; background-color: {P.BORDER}; height: 6px;
            }}
            #StatusProgress::chunk {{ background-color: {P.PRIMARY}; border-radius: 3px; }}

            /* Cards */
            #Card {{ background-color: {P.SURFACE}; border-radius: 12px; border: 1px solid {P.BORDER}; }}
            #StatBadge {{ background-color: {P.SURFACE_LIGHT}; border-radius: 6px; }}
            #StatBadgeLabel {{ color: {P.TEXT_MUTED}; font-size: 9px; font-weight: 700; letter-spacing: 0.5px; }}
            #StatBadgeValue {{ color: {P.PRIMARY}; font-size: 15px; font-weight: 700; }}
            #SectionTitle {{ font-size: 14px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}

            /* Welcome */
            #WelcomeTitle {{ font-size: 22px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}
            #WelcomeSubtitle {{ font-size: 13px; color: {P.TEXT_SECONDARY}; }}
            #StepIcon {{ background-color: {P.PRIMARY_LIGHT}; border-radius: 8px; }}
            #StepTitle {{ font-size: 13px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}
            #StepDesc {{ font-size: 11px; color: {P.TEXT_SECONDARY}; }}
            #WelcomeTip {{ color: {P.TEXT_SECONDARY}; font-size: 12px; }}
            #MutedLabel {{ color: {P.TEXT_SECONDARY}; }}

            /* Buttons */
            QPushButton {{
                background-color: {P.SURFACE};
                color: {P.TEXT_PRIMARY};
                padding: 8px 16px;
                border: 1px solid {P.BORDER};
                border-radius: 6px;
                font-weight: 600;
                font-size: 12px;
            }}
            QPushButton:hover {{ background-color: {P.BG}; }}
            QPushButton[kind="primary"] {{ background-color: {P.PRIMARY}; color: white; border: none; }}
            QPushButton[kind="primary"]:hover {{ background-color: {P.PRIMARY_DARK}; }}
            QPushButton[kind="success"] {{ background-color: {P.SUCCESS}; color: white; border: none; }}
            QPushButton[kind="success"]:hover {{ background-color: #047857; }}
            QPushButton[kind="warning"] {{ background-color: {P.WARNING}; color: white; border: none; }}
            QPushButton[kind="warning"]:hover {{ background-color: #B45309; }}
            QPushButton[kind="danger"] {{ background-color: {P.DANGER}; color: white; border: none; }}
            QPushButton[kind="danger"]:hover {{ background-color: #B91C1C; }}
            QPushButton[kind="ghost"] {{ background-color: transparent; border: none; }}
            QPushButton[kind="ghost"]:hover {{ background-color: {P.BG}; }}

            /* Inputs */
            QComboBox, QSpinBox, QLineEdit {{
                padding: 6px 10px;
                border: 1px solid {P.BORDER};
                border-radius: 6px;
                background-color: {P.SURFACE};
                color: {P.TEXT_PRIMARY};
                min-width: 110px;
            }}
            QComboBox:hover, QSpinBox:hover, QLineEdit:hover {{
                border-color: {P.PRIMARY};
            }}
            QCheckBox {{ spacing: 6px; color: {P.TEXT_PRIMARY}; }}

            QStatusBar {{ background-color: {P.SURFACE}; color: {P.TEXT_SECONDARY}; border-top: 1px solid {P.BORDER}; }}
        """


# ============================================================
# MAIN APPLICATION
# ============================================================

def main():
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    app.setFont(QFont("Segoe UI", 10))
    window = ElementalSocietyAnalysis()
    window.show()
    sys.exit(app.exec_())


if __name__ == '__main__':
    main()

In [ ]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - PROFESSIONAL EDITION v2.1
# Project Code: 1010
# Developer: Mohamed Almoufty Abdoalmoutleb Abaker
# ============================================================
#
# v2.0 changes:
#   - Full visual redesign: dark sidebar navigation, card-based
#     content, consistent spacing/typography, subtle shadows.
#   - Custom vector icon set (drawn with QPainter, no external
#     icon packages required) replacing emoji glyphs everywhere:
#     toolbar, navigation, status, dialogs, window icon.
#   - Status panel now uses an icon + animated-style dot instead
#     of plain text, and a proper busy progress bar.
#   - Minor logic fixes: "before/after" counts in Clean no longer
#     collide because df is only swapped after the message is
#     built; `stats` dict no longer shadows the imported
#     scipy.stats module; safer division guards.
#
# v2.1 changes (ioGAS-style data-grid workflow):
#   - New StatisticsEngine: descriptive stats, IQR outlier
#     detection, and a before/after data-quality table, all
#     returned as tidy DataFrames.
#   - Dataset page now shows a live "Data Quality Summary" grid
#     under the raw table (per-column type, missing %, missing
#     before/after cleaning, unique count).
#   - Distributions (EDA) page gained a variable-summary side
#     panel (mean/median/std/skew/kurtosis + IQR outlier count
#     and a colored flag) that updates with the plot.
#   - Analysis page gained a "Statistics & Outlier Detection"
#     tab set with full descriptive-statistics and outlier
#     grids for every numeric column — previously these numbers
#     were computed and thrown away.
#   - Geochemistry report now pairs the narrative summary with
#     proper DataFrame-style grids (alteration indices, ratios,
#     mineralization indices, grade statistics), sortable and
#     colored by severity.
#   - One shared populate_table_from_df() renderer + a single
#     "ReportTable"/"ReportTabs" QSS style keeps every grid's
#     colors, alignment, and header treatment consistent.
# ============================================================

import sys
import os
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *
import matplotlib
matplotlib.use("Qt5Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
import seaborn as sns
from scipy import stats as scipy_stats
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# APPLICATION METADATA
# ============================================================

APP_NAME = "Elemental Society Analysis"
APP_VERSION = "2.1.0"
APP_CODE = "1010"
DEVELOPER = "Mohamed Almoufty Abdoalmoutleb Abaker"
EMAIL = "almoufty121@gmail.com"
BACKGROUND = "Exploration Geologist | Msc Earth Science Engineering"

# ============================================================
# DESIGN SYSTEM — colors, spacing, typography live here so the
# whole app can be re-themed from one place.
# ============================================================

class Palette:
    PRIMARY = "#4F46E5"       # Indigo — primary brand / actions
    PRIMARY_DARK = "#4338CA"
    PRIMARY_LIGHT = "#EEF2FF"
    SUCCESS = "#059669"
    SUCCESS_LIGHT = "#ECFDF5"
    WARNING = "#D97706"
    WARNING_LIGHT = "#FFFBEB"
    DANGER = "#DC2626"
    DANGER_LIGHT = "#FEF2F2"
    INFO = "#0284C7"

    SIDEBAR_BG = "#111827"
    SIDEBAR_BG_HOVER = "#1F2937"
    SIDEBAR_TEXT = "#9CA3AF"
    SIDEBAR_TEXT_ACTIVE = "#FFFFFF"

    BG = "#F3F4F6"
    SURFACE = "#FFFFFF"
    BORDER = "#E5E7EB"
    TEXT_PRIMARY = "#111827"
    TEXT_SECONDARY = "#6B7280"
    TEXT_MUTED = "#9CA3AF"


# ============================================================
# VECTOR ICON FACTORY
# Simple line-icon set drawn on the fly with QPainter so the
# app has a consistent, crisp icon language without bundling
# image assets or third-party icon fonts.
# ============================================================

class Icons:
    """Generates QIcon objects for a small line-icon set."""

    _cache = {}

    @classmethod
    def get(cls, name, color=Palette.TEXT_PRIMARY, size=20):
        key = (name, color, size)
        if key in cls._cache:
            return cls._cache[key]
        pix = QPixmap(size, size)
        pix.fill(Qt.transparent)
        p = QPainter(pix)
        p.setRenderHint(QPainter.Antialiasing)
        pen = QPen(QColor(color))
        pen.setWidthF(size / 11.0)
        pen.setCapStyle(Qt.RoundCap)
        pen.setJoinStyle(Qt.RoundJoin)
        p.setPen(pen)
        p.setBrush(Qt.NoBrush)
        m = size * 0.15  # margin
        s = size - 2 * m  # usable size
        draw = getattr(cls, f"_draw_{name}", None)
        if draw:
            draw(p, m, s, size, color)
        p.end()
        icon = QIcon(pix)
        cls._cache[key] = icon
        return icon

    # -- individual glyphs -----------------------------------
    @staticmethod
    def _draw_home(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m, m + s * 0.45)
        path.lineTo(size / 2, m)
        path.lineTo(m + s, m + s * 0.45)
        p.drawPath(path)
        p.drawRect(QRectF(m + s * 0.18, m + s * 0.45, s * 0.64, s * 0.55))
        p.drawRect(QRectF(m + s * 0.4, m + s * 0.62, s * 0.2, s * 0.38))

    @staticmethod
    def _draw_upload(p, m, s, size, center=None, color=None):
        cx = size / 2
        p.drawLine(QPointF(cx, m), QPointF(cx, m + s * 0.6))
        path = QPainterPath()
        path.moveTo(cx - s * 0.25, m + s * 0.22)
        path.lineTo(cx, m)
        path.lineTo(cx + s * 0.25, m + s * 0.22)
        p.drawPath(path)
        p.drawLine(QPointF(m, m + s * 0.85), QPointF(m + s, m + s * 0.85))
        p.drawLine(QPointF(m, m + s * 0.85), QPointF(m, m + s * 0.6))
        p.drawLine(QPointF(m + s, m + s * 0.85), QPointF(m + s, m + s * 0.6))

    @staticmethod
    def _draw_broom(p, m, s, size, color):
        p.drawLine(QPointF(m + s * 0.15, m), QPointF(m + s * 0.65, m + s * 0.55))
        path = QPainterPath()
        path.moveTo(m + s * 0.65, m + s * 0.55)
        path.lineTo(m + s, m + s * 0.35)
        path.lineTo(m + s * 0.95, m + s * 0.75)
        path.lineTo(m + s * 0.45, m + s)
        path.closeSubpath()
        p.drawPath(path)

    @staticmethod
    def _draw_chart(p, m, s, size, color):
        base = m + s
        p.drawLine(QPointF(m, m), QPointF(m, base))
        p.drawLine(QPointF(m, base), QPointF(m + s, base))
        bars = [0.25, 0.55, 0.4, 0.8]
        w = s * 0.15
        for i, h in enumerate(bars):
            x = m + s * 0.15 + i * (w + s * 0.08)
            p.drawLine(QPointF(x, base), QPointF(x, base - s * h))

    @staticmethod
    def _draw_download(p, m, s, size, color):
        cx = size / 2
        p.drawLine(QPointF(cx, m), QPointF(cx, m + s * 0.6))
        path = QPainterPath()
        path.moveTo(cx - s * 0.25, m + s * 0.38)
        path.lineTo(cx, m + s * 0.6)
        path.lineTo(cx + s * 0.25, m + s * 0.38)
        p.drawPath(path)
        p.drawLine(QPointF(m, m + s * 0.85), QPointF(m + s, m + s * 0.85))

    @staticmethod
    def _draw_document(p, m, s, size, color):
        w = s * 0.7
        path = QPainterPath()
        path.moveTo(m, m)
        path.lineTo(m + w, m)
        path.lineTo(m + s, m + s * 0.3)
        path.lineTo(m + s, m + s)
        path.lineTo(m, m + s)
        path.closeSubpath()
        p.drawPath(path)
        for i in range(3):
            y = m + s * (0.45 + i * 0.18)
            p.drawLine(QPointF(m + s * 0.18, y), QPointF(m + s * 0.75, y))

    @staticmethod
    def _draw_help(p, m, s, size, color):
        p.drawEllipse(QRectF(m, m, s, s))
        f = QFont()
        f.setBold(True)
        f.setPixelSize(int(s * 0.55))
        p.setFont(f)
        p.drawText(QRectF(m, m, s, s), Qt.AlignCenter, "?")

    @staticmethod
    def _draw_info(p, m, s, size, color):
        p.drawEllipse(QRectF(m, m, s, s))
        cx = size / 2
        p.drawLine(QPointF(cx, m + s * 0.32), QPointF(cx, m + s * 0.33))
        pen = p.pen()
        pen.setWidthF(pen.widthF() * 1.6)
        p.setPen(pen)
        p.drawPoint(QPointF(cx, m + s * 0.28))
        pen.setWidthF(pen.widthF() / 1.6)
        p.setPen(pen)
        p.drawLine(QPointF(cx, m + s * 0.45), QPointF(cx, m + s * 0.78))

    @staticmethod
    def _draw_table(p, m, s, size, color):
        p.drawRect(QRectF(m, m, s, s))
        p.drawLine(QPointF(m, m + s / 3), QPointF(m + s, m + s / 3))
        p.drawLine(QPointF(m, m + 2 * s / 3), QPointF(m + s, m + 2 * s / 3))
        p.drawLine(QPointF(m + s / 2, m), QPointF(m + s / 2, m + s))

    @staticmethod
    def _draw_trend(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m, m + s * 0.8)
        path.lineTo(m + s * 0.3, m + s * 0.45)
        path.lineTo(m + s * 0.55, m + s * 0.65)
        path.lineTo(m + s, m + s * 0.1)
        p.drawPath(path)
        path2 = QPainterPath()
        path2.moveTo(m + s * 0.78, m + s * 0.1)
        path2.lineTo(m + s, m + s * 0.1)
        path2.lineTo(m + s, m + s * 0.32)
        p.drawPath(path2)

    @staticmethod
    def _draw_flask(p, m, s, size, color):
        p.drawLine(QPointF(m + s * 0.38, m), QPointF(m + s * 0.38, m + s * 0.4))
        p.drawLine(QPointF(m + s * 0.62, m), QPointF(m + s * 0.62, m + s * 0.4))
        p.drawLine(QPointF(m + s * 0.3, m), QPointF(m + s * 0.7, m))
        path = QPainterPath()
        path.moveTo(m + s * 0.38, m + s * 0.4)
        path.lineTo(m + s * 0.1, m + s * 0.92)
        path.lineTo(m + s * 0.9, m + s * 0.92)
        path.lineTo(m + s * 0.62, m + s * 0.4)
        p.drawPath(path)
        p.drawLine(QPointF(m + s * 0.22, m + s * 0.68), QPointF(m + s * 0.78, m + s * 0.68))

    @staticmethod
    def _draw_list(p, m, s, size, color):
        for i in range(3):
            y = m + s * (0.15 + i * 0.35)
            p.drawEllipse(QPointF(m + s * 0.05, y), 1.2, 1.2)
            p.drawLine(QPointF(m + s * 0.2, y), QPointF(m + s, y))

    @staticmethod
    def _draw_trash(p, m, s, size, color):
        p.drawLine(QPointF(m + s * 0.15, m + s * 0.22), QPointF(m + s * 0.85, m + s * 0.22))
        p.drawLine(QPointF(m + s * 0.35, m + s * 0.22), QPointF(m + s * 0.4, m))
        p.drawLine(QPointF(m + s * 0.65, m + s * 0.22), QPointF(m + s * 0.6, m))
        p.drawLine(QPointF(m + s * 0.4, m), QPointF(m + s * 0.6, m))
        path = QPainterPath()
        path.moveTo(m + s * 0.22, m + s * 0.3)
        path.lineTo(m + s * 0.3, m + s)
        path.lineTo(m + s * 0.7, m + s)
        path.lineTo(m + s * 0.78, m + s * 0.3)
        p.drawPath(path)

    @staticmethod
    def _draw_pencil(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m + s * 0.1, m + s * 0.9)
        path.lineTo(m + s * 0.15, m + s * 0.65)
        path.lineTo(m + s * 0.68, m + s * 0.12)
        path.lineTo(m + s * 0.88, m + s * 0.32)
        path.lineTo(m + s * 0.35, m + s * 0.85)
        path.closeSubpath()
        p.drawPath(path)
        p.drawLine(QPointF(m + s * 0.6, m + s * 0.2), QPointF(m + s * 0.8, m + s * 0.4))

    @staticmethod
    def _draw_check(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m, m + s * 0.55)
        path.lineTo(m + s * 0.4, m + s * 0.9)
        path.lineTo(m + s, m + s * 0.15)
        p.drawPath(path)

    @staticmethod
    def _draw_dot(p, m, s, size, color):
        p.setBrush(QColor(color))
        p.drawEllipse(QRectF(m, m, s, s))


def styled_button(text, icon_name=None, kind="default", tooltip=None, icon_color=None):
    """Factory for a consistently styled QPushButton with an optional icon."""
    btn = QPushButton(text)
    if icon_name:
        col = icon_color or ("#FFFFFF" if kind in ("primary", "success", "danger", "warning") else Palette.TEXT_SECONDARY)
        btn.setIcon(Icons.get(icon_name, color=col, size=16))
        btn.setIconSize(QSize(16, 16))
    if tooltip:
        btn.setToolTip(tooltip)
    btn.setProperty("kind", kind)
    btn.setCursor(Qt.PointingHandCursor)
    return btn


# ============================================================
# ELEMENT ALIAS MAPPING
# ============================================================

ELEMENT_ALIAS_MAP = {
    'K': 'K_pct', 'K2O': 'K_pct', 'K_PCT': 'K_pct', 'K%': 'K_pct',
    'MG': 'Mg_pct', 'MGO': 'Mg_pct', 'MG_PCT': 'Mg_pct', 'MG%': 'Mg_pct',
    'CA': 'Ca_pct', 'CAO': 'Ca_pct', 'CA_PCT': 'Ca_pct', 'CA%': 'Ca_pct',
    'NA': 'Na_pct', 'NA2O': 'Na_pct', 'NA_PCT': 'Na_pct', 'NA%': 'Na_pct',
    'FE': 'Fe_pct', 'FE2O3': 'Fe_pct', 'FEO': 'Fe_pct', 'FE_PCT': 'Fe_pct', 'FE%': 'Fe_pct',
    'AU': 'Au_ppm', 'AU_PPM': 'Au_ppm', 'AU_PPB': 'Au_ppm',
    'CU': 'Cu_ppm', 'CU_PPM': 'Cu_ppm', 'CU_PCT': 'Cu_ppm',
    'ZN': 'Zn_ppm', 'ZN_PPM': 'Zn_ppm',
    'PB': 'Pb_ppm', 'PB_PPM': 'Pb_ppm',
    'AG': 'Ag_ppm', 'AG_PPM': 'Ag_ppm',
    'AS': 'As_ppm', 'AS_PPM': 'As_ppm',
    'SB': 'Sb_ppm', 'SB_PPM': 'Sb_ppm',
    'MO': 'Mo_ppm', 'MO_PPM': 'Mo_ppm',
    'SIO2': 'SiO2_pct', 'SIO2_PCT': 'SiO2_pct', 'SIO2%': 'SiO2_pct',
    'AL': 'Al_pct', 'AL2O3': 'Al_pct', 'AL_PCT': 'Al_pct', 'AL%': 'Al_pct',
}

# ============================================================
# DATA CLEANING ENGINE
# ============================================================

def standardize_element_names(df):
    """Normalize chemical headers to canonical standard names."""
    renames = {}
    for col in df.columns:
        clean_col = re.sub(r'[^a-zA-Z0-9%_]', '', str(col)).upper()
        if clean_col in ELEMENT_ALIAS_MAP:
            renames[col] = ELEMENT_ALIAS_MAP[clean_col]
        else:
            for key, std in ELEMENT_ALIAS_MAP.items():
                if key in clean_col:
                    renames[col] = std
                    break
    return df.rename(columns=renames)


def clean_dataframe(df):
    """Universal cleaning routine for geochemical datasets."""
    if df is None or df.empty:
        return df

    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]
    df = standardize_element_names(df)

    missing_vals = ['', 'nan', 'NaN', 'null', 'None', 'NA', '-', '--', 'n/a', '?']
    df = df.replace(missing_vals, np.nan)

    text_cols = ['sample', 'id', 'point', 'lithology', 'comments', 'unit', 'rock_type']
    for col in df.columns:
        is_text = any(t in col.lower() for t in text_cols)
        if not is_text:
            cleaned_series = df[col].astype(str).str.replace(r'^[<>]', '', regex=True)
            cleaned_series = pd.to_numeric(cleaned_series.str.replace(',', '.'), errors='coerce')
            if cleaned_series.notna().sum() > 0:
                df[col] = cleaned_series

    df = df.dropna(how='all')
    return df


def read_any_file(path):
    """Read any tabular file format with auto-detection of encoding/separator."""
    ext = Path(path).suffix.lower()
    if ext in ['.xlsx', '.xls', '.xlsm']:
        return pd.read_excel(path)

    encodings = ["utf-8", "cp1252", "latin-1", "iso-8859-1", "utf-16"]
    seps = [";", ",", "\t", "|"]

    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, engine="python", on_bad_lines="skip")
                if df.shape[1] > 1:
                    return df
            except Exception:
                pass

    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read(3000)
        sep = max([";", ",", "\t", "|"], key=lambda d: txt.count(d))
        return pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")
    except Exception:
        return pd.read_csv(path, engine="python", on_bad_lines="skip")


# ============================================================
# GEOCHEMISTRY ENGINE
# ============================================================

class GeochemistryEngine:
    """Smart geochemistry calculations with auto-detection of available fields."""

    @staticmethod
    def calculate_all(df):
        results = {
            'alteration_indices': {},
            'element_ratios': {},
            'mineralization_indices': {},
            'grade_stats': {},
            'available': {'alteration': False, 'ratios': False, 'mineralization': False, 'grades': False}
        }

        # 1. Alteration Indices
        if {'K_pct', 'Mg_pct', 'Ca_pct', 'Na_pct'}.issubset(df.columns):
            try:
                results['alteration_indices']['Ishikawa AI'] = 100 * (df['K_pct'] + df['Mg_pct']) / (
                    df['K_pct'] + df['Mg_pct'] + df['Ca_pct'] + df['Na_pct'] + 1e-6)
                results['available']['alteration'] = True
            except Exception:
                pass

        if {'Fe_pct', 'Mg_pct', 'Na_pct', 'K_pct'}.issubset(df.columns):
            try:
                results['alteration_indices']['CCPI'] = 100 * (df['Fe_pct'] + df['Mg_pct']) / (
                    df['Fe_pct'] + df['Mg_pct'] + df['Na_pct'] + df['K_pct'] + 1e-6)
                results['available']['alteration'] = True
            except Exception:
                pass

        if {'K_pct', 'Na_pct'}.issubset(df.columns):
            try:
                results['alteration_indices']['Sericite Index'] = df['K_pct'] / (df['K_pct'] + df['Na_pct'] + 1e-6)
                results['available']['alteration'] = True
            except Exception:
                pass

        # 2. Element Ratios
        ratio_pairs = [
            ('Au_ppm', 'Cu_ppm', 'Au/Cu'), ('Zn_ppm', 'Pb_ppm', 'Zn/Pb'),
            ('Cu_ppm', 'Zn_ppm', 'Cu/Zn'), ('Au_ppm', 'Ag_ppm', 'Au/Ag'),
            ('Au_ppm', 'As_ppm', 'Au/As'), ('Cu_ppm', 'Mo_ppm', 'Cu/Mo')
        ]
        for c1, c2, name in ratio_pairs:
            if c1 in df.columns and c2 in df.columns:
                try:
                    results['element_ratios'][name] = df[c1] / (df[c2] + 1e-6)
                    results['available']['ratios'] = True
                except Exception:
                    pass

        # 3. Mineralization Indices
        if 'Au_ppm' in df.columns:
            try:
                results['mineralization_indices']['Gold Index'] = df['Au_ppm'] * 100
                results['available']['mineralization'] = True
            except Exception:
                pass

        base_metals = [c for c in ['Cu_ppm', 'Zn_ppm', 'Pb_ppm'] if c in df.columns]
        if base_metals:
            try:
                results['mineralization_indices']['Base Metal Index'] = sum(df[c].fillna(0) for c in base_metals) / len(base_metals)
                results['available']['mineralization'] = True
            except Exception:
                pass

        pathfinders = [c for c in ['As_ppm', 'Sb_ppm', 'Mo_ppm'] if c in df.columns]
        if pathfinders:
            try:
                results['mineralization_indices']['Pathfinder Index'] = sum(df[c].fillna(0) for c in pathfinders) / len(pathfinders)
                results['available']['mineralization'] = True
            except Exception:
                pass

        # 4. Grade Statistics
        key_elements = ['Au_ppm', 'Cu_ppm', 'Zn_ppm', 'Pb_ppm', 'Ag_ppm', 'As_ppm']
        for elem in key_elements:
            if elem in df.columns:
                data = df[elem].dropna()
                if len(data) > 0:
                    threshold = data.quantile(0.75)
                    mean_val = data.mean()
                    results['grade_stats'][elem] = {
                        'Mean': mean_val, 'Median': data.median(), 'Std Dev': data.std(),
                        'Minimum': data.min(), 'Maximum': data.max(),
                        'N': len(data), 'Threshold': threshold,
                        'Above Threshold': (data > threshold).sum(),
                        '% Above Threshold': (data > threshold).sum() / len(data) * 100,
                        'Grade Class': 'HIGH' if threshold and mean_val > threshold * 2
                                       else 'MEDIUM' if threshold and mean_val > threshold
                                       else 'LOW'
                    }
                    results['available']['grades'] = True

        return results


# ============================================================
# STATISTICS ENGINE
# Produces tidy, ready-to-display DataFrames for the data
# quality, descriptive-statistics, and outlier-detection panels.
# ============================================================

class StatisticsEngine:
    """Descriptive statistics, outlier detection, and data-quality summaries."""

    @staticmethod
    def descriptive_table(df):
        """One row per numeric variable: count, central tendency, spread, shape."""
        numeric_df = df.select_dtypes(include=[np.number])
        if numeric_df.empty:
            return pd.DataFrame()

        rows = []
        for col in numeric_df.columns:
            data = numeric_df[col].dropna()
            if len(data) == 0:
                continue
            mean_val = data.mean()
            rows.append({
                'Variable': col,
                'N': len(data),
                'Mean': mean_val,
                'Median': data.median(),
                'Std Dev': data.std(),
                'Min': data.min(),
                'Max': data.max(),
                'Skewness': data.skew(),
                'Kurtosis': data.kurtosis(),
                'CV %': (data.std() / mean_val * 100) if mean_val not in (0, np.nan) else np.nan,
            })
        return pd.DataFrame(rows)

    @staticmethod
    def outlier_table(df, k=1.5):
        """One row per numeric variable using the Tukey IQR fence method."""
        numeric_df = df.select_dtypes(include=[np.number])
        if numeric_df.empty:
            return pd.DataFrame()

        rows = []
        for col in numeric_df.columns:
            data = numeric_df[col].dropna()
            if len(data) < 4:
                continue
            q1 = data.quantile(0.25)
            q3 = data.quantile(0.75)
            iqr = q3 - q1
            lower = q1 - k * iqr
            upper = q3 + k * iqr
            outliers = data[(data < lower) | (data > upper)]
            rows.append({
                'Variable': col,
                'N': len(data),
                'Q1': q1,
                'Q3': q3,
                'IQR': iqr,
                'Lower Fence': lower,
                'Upper Fence': upper,
                'Outliers': len(outliers),
                'Outliers %': len(outliers) / len(data) * 100,
            })
        return pd.DataFrame(rows)

    @staticmethod
    def quality_table(df, before_df=None):
        """One row per column: dtype, missing values, uniqueness — the cleaning summary."""
        if df is None:
            return pd.DataFrame()

        rows = []
        for col in df.columns:
            series = df[col]
            missing = series.isna().sum()
            before_missing = None
            if before_df is not None and col in before_df.columns:
                before_missing = before_df[col].isna().sum()
            rows.append({
                'Column': col,
                'Type': str(series.dtype),
                'Missing': int(missing),
                'Missing %': round(missing / len(df) * 100, 2) if len(df) else 0,
                'Missing (Before Clean)': int(before_missing) if before_missing is not None else '—',
                'Unique Values': int(series.nunique(dropna=True)),
            })
        return pd.DataFrame(rows)


# ============================================================
# REUSABLE UI COMPONENTS
# ============================================================

class Card(QFrame):
    """A elevated white panel used as the base content container."""

    def __init__(self, parent=None):
        super().__init__(parent)
        self.setObjectName("Card")
        effect = QGraphicsDropShadowEffect(self)
        effect.setBlurRadius(18)
        effect.setOffset(0, 2)
        effect.setColor(QColor(17, 24, 39, 35))
        self.setGraphicsEffect(effect)


class StatBadge(QFrame):
    """Small metric chip used for stat_boxes (mean/median/etc.)."""

    def __init__(self, label, value, parent=None):
        super().__init__(parent)
        self.setObjectName("StatBadge")
        layout = QVBoxLayout(self)
        layout.setContentsMargins(12, 6, 12, 6)
        layout.setSpacing(0)
        lbl = QLabel(label.upper())
        lbl.setObjectName("StatBadgeLabel")
        val = QLabel(str(value))
        val.setObjectName("StatBadgeValue")
        layout.addWidget(lbl)
        layout.addWidget(val)


class SectionTitle(QLabel):
    def __init__(self, text, parent=None):
        super().__init__(text, parent)
        self.setObjectName("SectionTitle")


# ============================================================
# MAIN APPLICATION WINDOW
# ============================================================

NAV_ITEMS = [
    ("home", "Overview"),
    ("table", "Dataset"),
    ("trend", "Distributions"),
    ("chart", "Analysis"),
    ("flask", "Geochemistry"),
    ("list", "Activity Log"),
]


class ElementalSocietyAnalysis(QMainWindow):
    def __init__(self):
        super().__init__()
        self.df = None
        self.cleaned_df = None
        self.derived_df = None
        self.is_processing = False
        self.original_filename = None
        self.init_ui()

    # --------------------------------------------------------
    # UI CONSTRUCTION
    # --------------------------------------------------------

    def init_ui(self):
        self.setWindowTitle(f"{APP_NAME} — v{APP_VERSION}")
        self.setWindowIcon(Icons.get("flask", color=Palette.PRIMARY, size=32))
        self.setGeometry(50, 50, 1680, 980)
        self.setStyleSheet(self.get_stylesheet())

        central = QWidget()
        self.setCentralWidget(central)
        root = QHBoxLayout(central)
        root.setContentsMargins(0, 0, 0, 0)
        root.setSpacing(0)

        root.addWidget(self.build_sidebar())

        right = QWidget()
        right_layout = QVBoxLayout(right)
        right_layout.setContentsMargins(0, 0, 0, 0)
        right_layout.setSpacing(0)
        right_layout.addWidget(self.build_topbar())
        right_layout.addWidget(self.build_status_panel())
        self.stack = QStackedWidget()
        right_layout.addWidget(self.stack, 1)
        root.addWidget(right, 1)

        self.build_pages()

        self.statusBar = QStatusBar()
        self.statusBar.showMessage("Ready")
        self.setStatusBar(self.statusBar)

        self.log_message(f"{APP_NAME} v{APP_VERSION} initialized")
        self.log_message(f"Project Code: {APP_CODE}")

        self.nav_list.setCurrentRow(0)

    # -- sidebar ----------------------------------------------

    def build_sidebar(self):
        sidebar = QWidget()
        sidebar.setObjectName("Sidebar")
        sidebar.setFixedWidth(240)
        layout = QVBoxLayout(sidebar)
        layout.setContentsMargins(0, 0, 0, 0)
        layout.setSpacing(0)

        brand = QWidget()
        brand.setObjectName("Brand")
        brand_layout = QHBoxLayout(brand)
        brand_layout.setContentsMargins(20, 22, 20, 22)
        icon_lbl = QLabel()
        icon_lbl.setPixmap(Icons.get("flask", color="#FFFFFF", size=26).pixmap(26, 26))
        brand_layout.addWidget(icon_lbl)
        title_box = QVBoxLayout()
        title_box.setSpacing(0)
        name_lbl = QLabel("Elemental Society")
        name_lbl.setObjectName("BrandTitle")
        sub_lbl = QLabel(f"Analysis Suite · v{APP_VERSION}")
        sub_lbl.setObjectName("BrandSubtitle")
        title_box.addWidget(name_lbl)
        title_box.addWidget(sub_lbl)
        brand_layout.addLayout(title_box)
        brand_layout.addStretch()
        layout.addWidget(brand)

        self.nav_list = QListWidget()
        self.nav_list.setObjectName("NavList")
        self.nav_list.setFocusPolicy(Qt.NoFocus)
        self.nav_list.setIconSize(QSize(18, 18))
        for icon_name, label in NAV_ITEMS:
            item = QListWidgetItem(Icons.get(icon_name, color=Palette.SIDEBAR_TEXT, size=18), f"  {label}")
            item.setSizeHint(QSize(0, 44))
            self.nav_list.addItem(item)
        self.nav_list.currentRowChanged.connect(self.on_nav_changed)
        layout.addWidget(self.nav_list, 1)

        footer = QWidget()
        footer.setObjectName("SidebarFooter")
        footer_layout = QVBoxLayout(footer)
        footer_layout.setContentsMargins(20, 14, 20, 18)
        dev_lbl = QLabel(DEVELOPER)
        dev_lbl.setObjectName("SidebarFooterName")
        dev_lbl.setWordWrap(True)
        role_lbl = QLabel(BACKGROUND)
        role_lbl.setObjectName("SidebarFooterRole")
        role_lbl.setWordWrap(True)
        footer_layout.addWidget(dev_lbl)
        footer_layout.addWidget(role_lbl)
        layout.addWidget(footer)

        return sidebar

    def on_nav_changed(self, row):
        if row >= 0:
            self.stack.setCurrentIndex(row)
            self.page_title.setText(NAV_ITEMS[row][1])

    # -- top bar ------------------------------------------------

    def build_topbar(self):
        bar = QWidget()
        bar.setObjectName("TopBar")
        layout = QHBoxLayout(bar)
        layout.setContentsMargins(24, 14, 24, 14)
        layout.setSpacing(10)

        self.page_title = QLabel("Overview")
        self.page_title.setObjectName("PageTitle")
        layout.addWidget(self.page_title)

        self.file_info_label = QLabel("No file loaded")
        self.file_info_label.setObjectName("FileInfo")
        layout.addWidget(self.file_info_label)

        layout.addStretch()

        upload_btn = styled_button("Upload", "upload", "primary", "Upload a geochemical data file")
        upload_btn.clicked.connect(self.upload_file)
        clean_btn = styled_button("Clean", "broom", "warning", "Clean the loaded dataset")
        clean_btn.clicked.connect(self.clean_data)
        analyze_btn = styled_button("Analyze", "chart", "success", "Run comprehensive EDA")
        analyze_btn.clicked.connect(self.run_analysis)
        export_btn = styled_button("Export", "download", "default", "Export cleaned data")
        export_btn.clicked.connect(self.export_data)
        report_btn = styled_button("Report", "document", "default", "Jump to the geochemistry report")
        report_btn.clicked.connect(self.show_report)
        help_btn = styled_button("", "help", "ghost", "Help")
        help_btn.setFixedWidth(36)
        help_btn.clicked.connect(self.show_help)
        about_btn = styled_button("", "info", "ghost", "About")
        about_btn.setFixedWidth(36)
        about_btn.clicked.connect(self.show_about)

        for b in (upload_btn, clean_btn, analyze_btn, export_btn, report_btn, help_btn, about_btn):
            layout.addWidget(b)

        return bar

    # -- status panel -------------------------------------------

    def build_status_panel(self):
        widget = QWidget()
        widget.setObjectName("StatusPanel")
        layout = QHBoxLayout(widget)
        layout.setContentsMargins(24, 8, 24, 8)
        layout.setSpacing(10)

        self.status_dot = QLabel()
        self.status_dot.setPixmap(Icons.get("dot", color=Palette.TEXT_MUTED, size=12).pixmap(12, 12))
        layout.addWidget(self.status_dot)

        self.status_text = QLabel("Ready")
        self.status_text.setObjectName("StatusText")
        layout.addWidget(self.status_text)

        self.progress_bar = QProgressBar()
        self.progress_bar.setObjectName("StatusProgress")
        self.progress_bar.setVisible(False)
        self.progress_bar.setMaximumWidth(220)
        self.progress_bar.setTextVisible(False)
        layout.addWidget(self.progress_bar)

        layout.addStretch()

        self.process_info = QLabel("")
        self.process_info.setObjectName("ProcessInfo")
        layout.addWidget(self.process_info)

        return widget

    def set_status(self, status, message="", progress=False):
        status_map = {
            'idle': (Palette.TEXT_MUTED, 'Ready'),
            'processing': (Palette.WARNING, 'Processing…'),
            'success': (Palette.SUCCESS, 'Success'),
            'error': (Palette.DANGER, 'Error'),
        }
        color, default_text = status_map.get(status, status_map['idle'])
        self.status_dot.setPixmap(Icons.get("dot", color=color, size=12).pixmap(12, 12))
        self.status_text.setText(message if message else default_text)
        self.status_text.setStyleSheet(f"color: {color}; font-weight: 600;")

        show_progress = progress or status == 'processing'
        self.progress_bar.setVisible(show_progress)
        if show_progress:
            self.progress_bar.setRange(0, 0)
        if status == 'processing':
            QApplication.processEvents()

    def set_processing(self, active, message="", progress=False):
        self.is_processing = active
        if active:
            self.set_status('processing', message, progress)
            self.statusBar.showMessage(f"Processing: {message}")
        else:
            self.set_status('idle', 'Ready', False)
            self.process_info.setText("")
            self.statusBar.showMessage("Ready")

    def update_progress(self, message):
        self.process_info.setText(message)
        self.status_text.setText(message)
        self.log_message(message)
        QApplication.processEvents()

    # --------------------------------------------------------
    # PAGES
    # --------------------------------------------------------

    def build_pages(self):
        self.stack.addWidget(self.create_welcome_page())
        self.stack.addWidget(self.create_data_page())
        self.stack.addWidget(self.create_eda_page())
        self.stack.addWidget(self.create_analyze_page())
        self.stack.addWidget(self.create_geochemistry_page())
        self.stack.addWidget(self.create_log_page())

    def create_welcome_page(self):
        page = QWidget()
        outer = QVBoxLayout(page)
        outer.setContentsMargins(24, 20, 24, 24)

        card = Card()
        card_layout = QVBoxLayout(card)
        card_layout.setContentsMargins(32, 28, 32, 28)
        card_layout.setSpacing(6)

        title = QLabel("Welcome to Elemental Society Analysis")
        title.setObjectName("WelcomeTitle")
        subtitle = QLabel("A professional workflow for exploration geochemistry data")
        subtitle.setObjectName("WelcomeSubtitle")
        card_layout.addWidget(title)
        card_layout.addWidget(subtitle)
        card_layout.addSpacing(18)

        steps = [
            ("upload", "1. Upload", "Load a CSV or Excel dataset"),
            ("broom", "2. Clean", "Standardize headers and handle missing values"),
            ("chart", "3. Analyze", "Run a full exploratory statistical pass"),
            ("trend", "4. Visualize", "Distributions, correlations and PCA"),
            ("flask", "5. Geochemistry", "Alteration indices, ratios, grade statistics"),
            ("document", "6. Report", "Review and export the findings"),
        ]
        grid = QGridLayout()
        grid.setHorizontalSpacing(16)
        grid.setVerticalSpacing(14)
        for i, (icon_name, step_title, desc) in enumerate(steps):
            step_widget = QWidget()
            sl = QHBoxLayout(step_widget)
            sl.setContentsMargins(0, 0, 0, 0)
            icon_badge = QLabel()
            icon_badge.setObjectName("StepIcon")
            icon_badge.setFixedSize(36, 36)
            icon_badge.setAlignment(Qt.AlignCenter)
            icon_badge.setPixmap(Icons.get(icon_name, color=Palette.PRIMARY, size=18).pixmap(18, 18))
            text_box = QVBoxLayout()
            text_box.setSpacing(0)
            t = QLabel(step_title)
            t.setObjectName("StepTitle")
            d = QLabel(desc)
            d.setObjectName("StepDesc")
            d.setWordWrap(True)
            text_box.addWidget(t)
            text_box.addWidget(d)
            sl.addWidget(icon_badge)
            sl.addLayout(text_box, 1)
            grid.addWidget(step_widget, i // 2, i % 2)
        card_layout.addLayout(grid)

        card_layout.addSpacing(10)
        tip = QLabel("Tip: start by clicking <b>Upload</b> in the top bar.")
        tip.setObjectName("WelcomeTip")
        card_layout.addWidget(tip)

        outer.addWidget(card)
        outer.addStretch()
        return page

    def create_data_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        self.data_info = QLabel("No data loaded")
        self.data_info.setObjectName("MutedLabel")
        controls.addWidget(self.data_info)
        controls.addStretch()

        controls.addWidget(QLabel("Rows:"))
        self.row_limit = QSpinBox()
        self.row_limit.setRange(10, 5000)
        self.row_limit.setValue(500)
        self.row_limit.valueChanged.connect(self.refresh_data)
        controls.addWidget(self.row_limit)

        rename_btn = styled_button("Rename File", "pencil", "default")
        rename_btn.clicked.connect(self.rename_file)
        controls.addWidget(rename_btn)

        delete_btn = styled_button("Delete Selected", "trash", "danger")
        delete_btn.clicked.connect(self.delete_selected)
        controls.addWidget(delete_btn)

        layout.addLayout(controls)

        splitter = QSplitter(Qt.Vertical)
        splitter.setChildrenCollapsible(False)
        splitter.setHandleWidth(10)

        data_card = Card()
        data_card_layout = QVBoxLayout(data_card)
        data_card_layout.setContentsMargins(2, 2, 2, 2)
        self.data_table = QTableWidget()
        self.data_table.setAlternatingRowColors(True)
        self.data_table.setSortingEnabled(True)
        self.data_table.setSelectionBehavior(QTableWidget.SelectItems)
        self.data_table.setSelectionMode(QTableWidget.ExtendedSelection)
        data_card_layout.addWidget(self.data_table)
        splitter.addWidget(data_card)

        quality_card = Card()
        quality_layout = QVBoxLayout(quality_card)
        quality_layout.setContentsMargins(16, 14, 16, 14)
        quality_layout.setSpacing(8)
        quality_header = QHBoxLayout()
        quality_title = SectionTitle("Data Quality Summary")
        quality_header.addWidget(quality_title)
        quality_header.addStretch()
        self.quality_status_label = QLabel("")
        self.quality_status_label.setObjectName("MutedLabel")
        quality_header.addWidget(self.quality_status_label)
        quality_layout.addLayout(quality_header)
        self.quality_table = QTableWidget()
        self.quality_table.setObjectName("ReportTable")
        self.quality_table.setAlternatingRowColors(True)
        self.quality_table.setSortingEnabled(True)
        self.quality_table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.quality_table.verticalHeader().setVisible(False)
        quality_layout.addWidget(self.quality_table)
        splitter.addWidget(quality_card)

        splitter.setStretchFactor(0, 3)
        splitter.setStretchFactor(1, 2)
        layout.addWidget(splitter, 1)

        return page

    def create_eda_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        controls.addWidget(QLabel("Variable:"))
        self.eda_var = QComboBox()
        self.eda_var.currentTextChanged.connect(self.plot_distribution)
        controls.addWidget(self.eda_var)

        controls.addWidget(QLabel("Type:"))
        self.eda_type = QComboBox()
        self.eda_type.addItems(["Histogram", "PDF", "CDF", "Box Plot"])
        self.eda_type.currentTextChanged.connect(self.plot_distribution)
        controls.addWidget(self.eda_type)

        self.show_combined = QCheckBox("Show combined view")
        self.show_combined.toggled.connect(self.plot_distribution)
        controls.addWidget(self.show_combined)
        controls.addStretch()
        layout.addLayout(controls)

        content = QHBoxLayout()
        content.setSpacing(12)

        plot_card = Card()
        plot_card_layout = QVBoxLayout(plot_card)
        self.eda_figure = Figure(figsize=(10, 6))
        self.eda_canvas = FigureCanvas(self.eda_figure)
        plot_card_layout.addWidget(self.eda_canvas)
        content.addWidget(plot_card, 3)

        stats_card = Card()
        stats_card.setMaximumWidth(260)
        stats_layout = QVBoxLayout(stats_card)
        stats_layout.setContentsMargins(16, 14, 16, 14)
        stats_layout.setSpacing(10)
        stats_layout.addWidget(SectionTitle("Variable Summary"))
        self.eda_stats_table = QTableWidget()
        self.eda_stats_table.setObjectName("ReportTable")
        self.eda_stats_table.setColumnCount(2)
        self.eda_stats_table.setHorizontalHeaderLabels(["Metric", "Value"])
        self.eda_stats_table.horizontalHeader().setStretchLastSection(True)
        self.eda_stats_table.verticalHeader().setVisible(False)
        self.eda_stats_table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.eda_stats_table.setSelectionMode(QTableWidget.NoSelection)
        stats_layout.addWidget(self.eda_stats_table, 1)
        self.eda_outlier_badge = QLabel("")
        self.eda_outlier_badge.setObjectName("OutlierBadge")
        self.eda_outlier_badge.setWordWrap(True)
        self.eda_outlier_badge.setAlignment(Qt.AlignCenter)
        stats_layout.addWidget(self.eda_outlier_badge)
        content.addWidget(stats_card, 1)

        layout.addLayout(content, 1)

        return page

    def create_analyze_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        controls.addWidget(QLabel("Plot:"))
        self.plot_type = QComboBox()
        self.plot_type.addItems(["Correlation Matrix", "PCA Scores", "Scatter Plot", "All Plots Combined"])
        self.plot_type.currentTextChanged.connect(self.update_plot)
        controls.addWidget(self.plot_type)

        controls.addWidget(QLabel("X:"))
        self.analyze_x = QComboBox()
        controls.addWidget(self.analyze_x)

        controls.addWidget(QLabel("Y:"))
        self.analyze_y = QComboBox()
        controls.addWidget(self.analyze_y)

        plot_btn = styled_button("Generate", "trend", "primary")
        plot_btn.clicked.connect(self.update_plot)
        controls.addWidget(plot_btn)
        controls.addStretch()
        layout.addLayout(controls)

        splitter = QSplitter(Qt.Vertical)
        splitter.setChildrenCollapsible(False)
        splitter.setHandleWidth(10)

        plot_card = Card()
        plot_card_layout = QVBoxLayout(plot_card)
        self.analyze_figure = Figure(figsize=(10, 6))
        self.analyze_canvas = FigureCanvas(self.analyze_figure)
        plot_card_layout.addWidget(self.analyze_canvas)
        splitter.addWidget(plot_card)

        stats_card = Card()
        stats_card_layout = QVBoxLayout(stats_card)
        stats_card_layout.setContentsMargins(16, 14, 16, 10)
        stats_card_layout.setSpacing(8)
        stats_card_layout.addWidget(SectionTitle("Statistics & Outlier Detection"))

        self.stats_tabs = QTabWidget()
        self.stats_tabs.setObjectName("ReportTabs")

        stats_tab = QWidget()
        stats_tab_layout = QVBoxLayout(stats_tab)
        stats_tab_layout.setContentsMargins(0, 8, 0, 0)
        self.stats_table = QTableWidget()
        self.stats_table.setObjectName("ReportTable")
        self.stats_table.setAlternatingRowColors(True)
        self.stats_table.setSortingEnabled(True)
        self.stats_table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.stats_table.verticalHeader().setVisible(False)
        stats_tab_layout.addWidget(self.stats_table)
        self.stats_tabs.addTab(stats_tab, "Descriptive Statistics")

        outlier_tab = QWidget()
        outlier_tab_layout = QVBoxLayout(outlier_tab)
        outlier_tab_layout.setContentsMargins(0, 8, 0, 0)
        self.outlier_table = QTableWidget()
        self.outlier_table.setObjectName("ReportTable")
        self.outlier_table.setAlternatingRowColors(True)
        self.outlier_table.setSortingEnabled(True)
        self.outlier_table.setEditTriggers(QTableWidget.NoEditTriggers)
        self.outlier_table.verticalHeader().setVisible(False)
        outlier_tab_layout.addWidget(self.outlier_table)
        self.stats_tabs.addTab(outlier_tab, "Outlier Detection (IQR)")

        stats_card_layout.addWidget(self.stats_tabs)
        splitter.addWidget(stats_card)

        splitter.setStretchFactor(0, 3)
        splitter.setStretchFactor(1, 2)
        layout.addWidget(splitter, 1)

        return page

    def create_geochemistry_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        splitter = QSplitter(Qt.Vertical)
        splitter.setChildrenCollapsible(False)
        splitter.setHandleWidth(10)

        narrative_card = Card()
        narrative_layout = QVBoxLayout(narrative_card)
        narrative_layout.setContentsMargins(2, 2, 2, 2)
        self.geochem_viewer = QTextEdit()
        self.geochem_viewer.setReadOnly(True)
        self.geochem_viewer.setFont(QFont("Segoe UI", 10))
        self.geochem_viewer.setFrameShape(QFrame.NoFrame)
        narrative_layout.addWidget(self.geochem_viewer)
        splitter.addWidget(narrative_card)

        grid_card = Card()
        grid_layout = QVBoxLayout(grid_card)
        grid_layout.setContentsMargins(16, 14, 16, 10)
        grid_layout.setSpacing(8)
        grid_layout.addWidget(SectionTitle("Geochemistry Report — Data Tables"))

        self.geochem_tabs = QTabWidget()
        self.geochem_tabs.setObjectName("ReportTabs")

        self.geochem_alteration_table = self._make_report_table()
        self.geochem_ratio_table = self._make_report_table()
        self.geochem_mineralization_table = self._make_report_table()
        self.geochem_grade_table = self._make_report_table()

        self.geochem_tabs.addTab(self._wrap_table(self.geochem_alteration_table), "Alteration Indices")
        self.geochem_tabs.addTab(self._wrap_table(self.geochem_ratio_table), "Element Ratios")
        self.geochem_tabs.addTab(self._wrap_table(self.geochem_mineralization_table), "Mineralization Indices")
        self.geochem_tabs.addTab(self._wrap_table(self.geochem_grade_table), "Grade Statistics")

        grid_layout.addWidget(self.geochem_tabs)
        splitter.addWidget(grid_card)

        splitter.setStretchFactor(0, 1)
        splitter.setStretchFactor(1, 2)
        layout.addWidget(splitter, 1)
        return page

    def _make_report_table(self):
        """Create a QTableWidget pre-styled to match the report/data-grid look."""
        table = QTableWidget()
        table.setObjectName("ReportTable")
        table.setAlternatingRowColors(True)
        table.setSortingEnabled(True)
        table.setEditTriggers(QTableWidget.NoEditTriggers)
        table.verticalHeader().setVisible(False)
        return table

    @staticmethod
    def _wrap_table(table):
        wrapper = QWidget()
        wrapper_layout = QVBoxLayout(wrapper)
        wrapper_layout.setContentsMargins(0, 8, 0, 0)
        wrapper_layout.addWidget(table)
        return wrapper

    def create_log_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        controls.addStretch()
        clear_btn = styled_button("Clear", "trash", "danger")
        clear_btn.clicked.connect(self.clear_log)
        controls.addWidget(clear_btn)
        layout.addLayout(controls)

        card = Card()
        card_layout = QVBoxLayout(card)
        card_layout.setContentsMargins(2, 2, 2, 2)
        self.log_text = QTextEdit()
        self.log_text.setReadOnly(True)
        self.log_text.setFont(QFont("Consolas", 10))
        self.log_text.setFrameShape(QFrame.NoFrame)
        card_layout.addWidget(self.log_text)
        layout.addWidget(card, 1)

        return page

    # ============================================================
    # LOG FUNCTIONS
    # ============================================================

    def log_message(self, message):
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.append(f"[{timestamp}]  {message}")

    def clear_log(self):
        self.log_text.clear()
        self.log_message("Log cleared")

    # ============================================================
    # DATA FUNCTIONS
    # ============================================================

    def rename_file(self):
        if self.original_filename is None:
            QMessageBox.warning(self, "No File", "No file is currently loaded.")
            return

        new_name, ok = QInputDialog.getText(
            self, "Rename File", "Enter new filename:", QLineEdit.Normal, self.original_filename)

        if ok and new_name:
            self.original_filename = new_name
            self.file_info_label.setText(new_name)
            self.log_message(f"Renamed file to: {new_name}")
            self.set_status('success', f'Renamed to {new_name}')

    def delete_selected(self):
        if self.df is None:
            QMessageBox.warning(self, "No Data", "Please load data first.")
            return

        selected = self.data_table.selectedIndexes()
        if not selected:
            QMessageBox.warning(self, "Nothing Selected", "Select one or more cells first.")
            return

        rows = set(index.row() for index in selected)
        cols = set(index.column() for index in selected)

        reply = QMessageBox.question(
            self, "Delete Selection",
            f"Delete {len(rows)} row(s) and {len(cols)} column(s)?",
            QMessageBox.Yes | QMessageBox.No)

        if reply == QMessageBox.Yes:
            for col in sorted(cols, reverse=True):
                self.df = self.df.drop(self.df.columns[col], axis=1)
            for row in sorted(rows, reverse=True):
                self.df = self.df.drop(self.df.index[row])

            self.df = self.df.reset_index(drop=True)
            self.display_data()
            self.log_message(f"Deleted {len(rows)} row(s) and {len(cols)} column(s)")
            self.set_status('success', f'Deleted {len(rows)} rows, {len(cols)} columns')

    def upload_file(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open Geochemical Data", "",
            "All Files (*);;CSV (*.csv);;Excel (*.xlsx *.xls);;Text (*.txt)"
        )
        if not file_path:
            return

        self.original_filename = os.path.basename(file_path)
        self.set_processing(True, "Loading file…")
        self.log_message(f"Loading: {self.original_filename}")
        self.update_progress(f"Reading {self.original_filename}")

        try:
            self.df = read_any_file(file_path)

            if self.df is not None and not self.df.empty:
                self.df = clean_dataframe(self.df)
                self.cleaned_df = self.df.copy()
                self.derived_df = self.df.copy()
                self.display_data()
                self.update_quality_summary()
                self.file_info_label.setText(self.original_filename)

                numeric_cols = self.df.select_dtypes(include=[np.number]).columns
                self.eda_var.clear()
                self.eda_var.addItems(numeric_cols)
                self.analyze_x.clear()
                self.analyze_x.addItems(numeric_cols)
                self.analyze_y.clear()
                self.analyze_y.addItems(numeric_cols)

                self.log_message(f"Loaded {self.df.shape[0]} rows, {self.df.shape[1]} columns")
                self.set_status('success', f'Loaded {self.df.shape[0]} rows')

                QMessageBox.information(
                    self, "File Loaded",
                    f"{self.df.shape[0]} rows × {self.df.shape[1]} columns loaded successfully.")
            else:
                QMessageBox.warning(self, "Read Error", "Could not read the selected file.")

        except Exception as e:
            self.log_message(f"Error: {str(e)}")
            self.set_status('error', 'Load failed')
            QMessageBox.critical(self, "Error", f"Failed to load file:\n{str(e)}")
        finally:
            self.set_processing(False)

    def clean_data(self):
        if self.df is None or self.df.empty:
            QMessageBox.warning(self, "No Data", "Please load data first.")
            return

        reply = QMessageBox.question(self, "Clean Data", "Clean the current dataset?", QMessageBox.Yes | QMessageBox.No)
        if reply != QMessageBox.Yes:
            return

        self.set_processing(True, "Cleaning data…")
        self.log_message("Cleaning data...")
        self.update_progress("Cleaning data...")

        try:
            rows_before = len(self.df)
            df_before = self.df.copy()
            new_cleaned_df = clean_dataframe(self.df.copy())
            rows_after = len(new_cleaned_df)

            self.cleaned_df = new_cleaned_df
            self.derived_df = new_cleaned_df.copy()
            self.df = new_cleaned_df.copy()
            self.display_data()
            self.update_quality_summary(before_df=df_before)

            self.log_message(f"Cleaned dataset: {rows_after} rows remain")
            self.set_status('success', f'Cleaned: {rows_after} rows')

            QMessageBox.information(
                self, "Cleaning Complete",
                f"Before: {rows_before} rows\nAfter: {rows_after} rows")

        except Exception as e:
            self.log_message(f"Cleaning error: {str(e)}")
            self.set_status('error', 'Cleaning failed')
            QMessageBox.critical(self, "Error", f"Cleaning failed:\n{str(e)}")
        finally:
            self.set_processing(False)

    def run_analysis(self):
        if self.df is None:
            QMessageBox.warning(self, "No Data", "Please load data first.")
            return

        self.set_processing(True, "Running analysis…")
        self.log_message("Starting analysis...")
        self.update_progress("Running analysis...")

        try:
            self.update_progress("Generating statistics...")
            self.generate_statistics()

            self.update_progress("Creating visualizations...")
            self.plot_distribution()
            self.update_plot()

            self.update_progress("Calculating geochemistry...")
            self.calculate_geochemistry()

            self.log_message("Analysis complete")
            self.set_status('success', 'Analysis complete')

            QMessageBox.information(self, "Analysis Complete", "The analysis finished successfully.")

        except Exception as e:
            self.log_message(f"Analysis error: {str(e)}")
            self.set_status('error', 'Analysis failed')
            QMessageBox.critical(self, "Error", f"Analysis failed:\n{str(e)}")
        finally:
            self.set_processing(False)

    def display_data(self):
        if self.df is None:
            return

        display_limit = min(self.row_limit.value(), len(self.df))
        self.data_table.setRowCount(display_limit)
        self.data_table.setColumnCount(len(self.df.columns))
        self.data_table.setHorizontalHeaderLabels(self.df.columns.astype(str))

        for i in range(display_limit):
            for j, col in enumerate(self.df.columns):
                val = self.df.iloc[i, j]
                item = QTableWidgetItem("" if pd.isna(val) else str(val))
                self.data_table.setItem(i, j, item)

        self.data_table.resizeColumnsToContents()
        self.data_info.setText(f"{len(self.df)} rows × {len(self.df.columns)} columns")

    def refresh_data(self):
        self.display_data()

    # ============================================================
    # DATAFRAME → TABLE RENDERING
    # Shared renderer so every grid (data quality, statistics,
    # outliers, geochemistry report) looks and behaves the same.
    # ============================================================

    def populate_table_from_df(self, table, df, precision=4, highlight_col=None, highlight_rule=None):
        """Render a DataFrame into a QTableWidget with unified, right-aligned
        numeric formatting. If highlight_col/highlight_rule are given, rows
        matching the rule get a soft background tint (used for outlier %
        and grade-class flags) so problems are visible at a glance."""
        table.setSortingEnabled(False)
        if df is None or df.empty:
            table.setRowCount(0)
            table.setColumnCount(0)
            table.setSortingEnabled(True)
            return

        table.setColumnCount(len(df.columns))
        table.setHorizontalHeaderLabels([str(c) for c in df.columns])
        table.setRowCount(len(df))

        for i in range(len(df)):
            row_color = None
            if highlight_col and highlight_col in df.columns and highlight_rule:
                row_color = highlight_rule(df.iloc[i][highlight_col])

            for j, col in enumerate(df.columns):
                val = df.iloc[i, j]
                if isinstance(val, (float, np.floating)):
                    text = "—" if pd.isna(val) else f"{val:,.{precision}f}"
                    align = Qt.AlignRight | Qt.AlignVCenter
                elif isinstance(val, (int, np.integer)):
                    text = f"{val:,}"
                    align = Qt.AlignRight | Qt.AlignVCenter
                else:
                    text = "—" if pd.isna(val) else str(val)
                    align = Qt.AlignLeft | Qt.AlignVCenter

                item = QTableWidgetItem(text)
                item.setTextAlignment(align)
                if row_color is not None:
                    item.setBackground(QColor(row_color))
                table.setItem(i, j, item)

        table.resizeColumnsToContents()
        table.setSortingEnabled(True)

    def update_quality_summary(self, before_df=None):
        """Refresh the Data Quality Summary grid on the Dataset page."""
        if self.df is None:
            return
        quality_df = StatisticsEngine.quality_table(self.df, before_df=before_df)
        self.populate_table_from_df(
            self.quality_table, quality_df, precision=2,
            highlight_col='Missing %',
            highlight_rule=lambda v: Palette.WARNING_LIGHT if v and v > 0 else None,
        )
        total_missing = int(quality_df['Missing'].sum()) if not quality_df.empty else 0
        self.quality_status_label.setText(
            f"{len(self.df.columns)} columns · {total_missing} missing value(s) total")

    def update_eda_stats_panel(self, var, data):
        """Refresh the small variable-summary table on the EDA page."""
        if not hasattr(self, 'eda_stats_table'):
            return
        outliers_df = StatisticsEngine.outlier_table(self.df[[var]])
        outlier_count, outlier_pct = 0, 0.0
        if not outliers_df.empty:
            outlier_count = int(outliers_df.iloc[0]['Outliers'])
            outlier_pct = float(outliers_df.iloc[0]['Outliers %'])

        metrics = [
            ("N", f"{len(data):,}"),
            ("Mean", f"{data.mean():.4f}"),
            ("Median", f"{data.median():.4f}"),
            ("Std Dev", f"{data.std():.4f}"),
            ("Min", f"{data.min():.4f}"),
            ("Max", f"{data.max():.4f}"),
            ("Skewness", f"{data.skew():.4f}"),
            ("Kurtosis", f"{data.kurtosis():.4f}"),
            ("Outliers (IQR)", f"{outlier_count} ({outlier_pct:.1f}%)"),
        ]
        self.eda_stats_table.setRowCount(len(metrics))
        for i, (label, value) in enumerate(metrics):
            label_item = QTableWidgetItem(label)
            value_item = QTableWidgetItem(value)
            value_item.setTextAlignment(Qt.AlignRight | Qt.AlignVCenter)
            self.eda_stats_table.setItem(i, 0, label_item)
            self.eda_stats_table.setItem(i, 1, value_item)
        self.eda_stats_table.resizeColumnsToContents()

        if outlier_count > 0:
            self.eda_outlier_badge.setText(f"⚠ {outlier_count} outlier value(s) flagged beyond the IQR fences")
            self.eda_outlier_badge.setStyleSheet(
                f"background-color: {Palette.WARNING_LIGHT}; color: {Palette.WARNING}; "
                f"border-radius: 6px; padding: 8px; font-size: 11px; font-weight: 600;")
        else:
            self.eda_outlier_badge.setText("No outliers detected for this variable")
            self.eda_outlier_badge.setStyleSheet(
                f"background-color: {Palette.SUCCESS_LIGHT}; color: {Palette.SUCCESS}; "
                f"border-radius: 6px; padding: 8px; font-size: 11px; font-weight: 600;")

    # ============================================================
    # STATISTICS FUNCTIONS
    # ============================================================

    def generate_statistics(self):
        """Build and display the descriptive-statistics and outlier-detection
        grids on the Analysis page (Statistics & Outlier Detection tabs)."""
        if self.df is None:
            return

        numeric_df = self.df.select_dtypes(include=[np.number])
        if numeric_df.empty:
            self.log_message("No numeric columns available for statistics")
            return

        self.stats_df = StatisticsEngine.descriptive_table(self.df)
        self.outliers_df = StatisticsEngine.outlier_table(self.df)

        self.populate_table_from_df(self.stats_table, self.stats_df, precision=4)
        self.populate_table_from_df(
            self.outlier_table, self.outliers_df, precision=4,
            highlight_col='Outliers %',
            highlight_rule=lambda v: Palette.DANGER_LIGHT if v and v > 10
                                      else (Palette.WARNING_LIGHT if v and v > 0 else None),
        )

        total_outliers = int(self.outliers_df['Outliers'].sum()) if not self.outliers_df.empty else 0
        self.log_message(
            f"Statistics generated for {len(numeric_df.columns)} variables "
            f"({total_outliers} outlier value(s) flagged)")

    # ============================================================
    # PLOTTING FUNCTIONS
    # ============================================================

    def plot_distribution(self):
        if self.df is None:
            return

        var = self.eda_var.currentText()
        plot_type = self.eda_type.currentText()
        show_combined = self.show_combined.isChecked()

        if not var or var not in self.df.columns:
            return

        data = self.df[var].dropna()
        if len(data) < 2:
            return

        self.update_eda_stats_panel(var, data)

        self.eda_figure.clear()

        if show_combined:
            axes = [self.eda_figure.add_subplot(2, 2, i + 1) for i in range(4)]
            plot_types = ["Histogram", "PDF", "CDF", "Box Plot"]
        else:
            axes = [self.eda_figure.add_subplot(111)]
            plot_types = [plot_type]

        accent = Palette.PRIMARY

        for ax, ptype in zip(axes, plot_types):
            if ptype == "Histogram":
                ax.hist(data, bins=20, density=True, alpha=0.75, color=accent, edgecolor='white')
                ax.axvline(data.mean(), color=Palette.DANGER, linestyle='--', label=f'Mean: {data.mean():.2f}')
                ax.axvline(data.median(), color=Palette.SUCCESS, linestyle='--', label=f'Median: {data.median():.2f}')
                ax.set_ylabel('Density')
                ax.legend(frameon=False, fontsize=8)

            elif ptype == "PDF":
                ax.hist(data, bins=20, density=True, alpha=0.35, color=accent)
                x = np.linspace(data.min(), data.max(), 100)
                mu, std = scipy_stats.norm.fit(data)
                pdf = scipy_stats.norm.pdf(x, mu, std)
                ax.plot(x, pdf, color=Palette.DANGER, linewidth=2, label='Normal fit')
                ax.set_ylabel('Density')
                ax.legend(frameon=False, fontsize=8)

            elif ptype == "CDF":
                sorted_data = np.sort(data)
                y = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
                ax.plot(sorted_data, y, color=accent, linewidth=2)
                ax.set_ylabel('Cumulative Probability')

            elif ptype == "Box Plot":
                bp = ax.boxplot(data, vert=True, patch_artist=True)
                for box in bp['boxes']:
                    box.set(facecolor=accent, alpha=0.6)

            ax.set_title(f'{var} — {ptype}', fontweight='bold', fontsize=10)
            ax.set_xlabel(var if ptype != "Box Plot" else '')
            ax.grid(True, alpha=0.25)
            for spine in ('top', 'right'):
                ax.spines[spine].set_visible(False)

        self.eda_figure.tight_layout()
        self.eda_canvas.draw()

    def update_plot(self):
        if self.df is None:
            return

        plot_type = self.plot_type.currentText()
        self.analyze_figure.clear()
        numeric_df = self.df.select_dtypes(include=[np.number])

        def draw_correlation(ax):
            if len(numeric_df.columns) > 1:
                corr = numeric_df.corr()
                sns.heatmap(corr, annot=True, cmap='coolwarm', center=0,
                            fmt='.2f', square=True, linewidths=0.5, ax=ax, cbar=False)
                ax.set_title('Correlation Matrix', fontweight='bold', fontsize=10)

        def draw_pca(ax):
            if len(numeric_df.columns) >= 2:
                clean_data = numeric_df.dropna()
                if len(clean_data) > 1:
                    scaler = StandardScaler()
                    scaled = scaler.fit_transform(clean_data)
                    pca = SklearnPCA()
                    scores = pca.fit_transform(scaled)
                    explained = pca.explained_variance_ratio_
                    ax.scatter(scores[:, 0], scores[:, 1],
                               c=np.linalg.norm(scores[:, :2], axis=1),
                               cmap='viridis', alpha=0.7, edgecolors='white', linewidth=0.5)
                    ax.set_xlabel(f'PC1 ({explained[0] * 100:.1f}%)')
                    ax.set_ylabel(f'PC2 ({explained[1] * 100:.1f}%)')
                    ax.set_title('PCA Scores', fontweight='bold', fontsize=10)
                    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.4)
                    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.4)

        def draw_scatter(ax):
            x_col = self.analyze_x.currentText()
            y_col = self.analyze_y.currentText()
            if x_col and y_col and x_col in self.df.columns and y_col in self.df.columns:
                data = self.df[[x_col, y_col]].dropna()
                ax.scatter(data[x_col], data[y_col], alpha=0.7, color=Palette.PRIMARY, edgecolors='white')
                ax.set_xlabel(x_col)
                ax.set_ylabel(y_col)
                ax.set_title('Scatter Plot', fontweight='bold', fontsize=10)

        def draw_distribution(ax):
            if len(numeric_df.columns) > 0:
                col = numeric_df.columns[0]
                data = numeric_df[col].dropna()
                if len(data) > 1:
                    ax.hist(data, bins=20, density=True, alpha=0.75, color=Palette.PRIMARY, edgecolor='white')
                    ax.set_title(f'Distribution: {col}', fontweight='bold', fontsize=10)
                    ax.set_xlabel(col)
                    ax.set_ylabel('Density')

        if plot_type == "All Plots Combined":
            ax1 = self.analyze_figure.add_subplot(2, 2, 1)
            ax2 = self.analyze_figure.add_subplot(2, 2, 2)
            ax3 = self.analyze_figure.add_subplot(2, 2, 3)
            ax4 = self.analyze_figure.add_subplot(2, 2, 4)
            draw_correlation(ax1)
            draw_pca(ax2)
            draw_scatter(ax3)
            draw_distribution(ax4)
            for ax in (ax2, ax3, ax4):
                ax.grid(True, alpha=0.25)
        elif plot_type == "Correlation Matrix":
            draw_correlation(self.analyze_figure.add_subplot(111))
        elif plot_type == "PCA Scores":
            ax = self.analyze_figure.add_subplot(111)
            draw_pca(ax)
            ax.grid(True, alpha=0.25)
        elif plot_type == "Scatter Plot":
            ax = self.analyze_figure.add_subplot(111)
            draw_scatter(ax)
            ax.grid(True, alpha=0.25)

        self.analyze_figure.tight_layout()
        self.analyze_canvas.draw()

    # ============================================================
    # GEOCHEMISTRY FUNCTIONS
    # ============================================================

    def calculate_geochemistry(self):
        if self.df is None:
            return

        geochem = GeochemistryEngine.calculate_all(self.df)

        html = []
        html.append("<html><head><style>")
        html.append(f"""
            body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 4px; color: {Palette.TEXT_PRIMARY}; }}
            h1 {{ color: {Palette.PRIMARY}; border-bottom: 2px solid {Palette.PRIMARY}; padding-bottom: 8px; font-size: 20px; }}
            h2 {{ color: {Palette.TEXT_PRIMARY}; margin-top: 22px; font-size: 15px; }}
            .card {{ background-color: {Palette.BG}; border-radius: 8px; padding: 15px; margin: 10px 0; border-left: 4px solid {Palette.PRIMARY}; }}
            .stat-box {{ display: inline-block; margin: 3px; padding: 8px 15px; background: white; border-radius: 6px; border: 1px solid {Palette.BORDER}; }}
            .stat-value {{ font-weight: bold; color: {Palette.PRIMARY}; font-size: 16px; }}
            .stat-label {{ font-size: 10px; color: {Palette.TEXT_SECONDARY}; }}
            .badge {{ display: inline-block; padding: 2px 10px; border-radius: 10px; font-size: 11px; font-weight: 600; }}
            .badge-high {{ background: {Palette.DANGER_LIGHT}; color: {Palette.DANGER}; }}
            .badge-medium {{ background: {Palette.WARNING_LIGHT}; color: {Palette.WARNING}; }}
            .badge-low {{ background: {Palette.SUCCESS_LIGHT}; color: {Palette.SUCCESS}; }}
            .badge-available {{ background: {Palette.SUCCESS_LIGHT}; color: {Palette.SUCCESS}; }}
            .badge-missing {{ background: {Palette.DANGER_LIGHT}; color: {Palette.DANGER}; }}
        """)
        html.append("</style></head><body>")

        html.append("<h1>Exploration Geochemistry Report</h1>")
        html.append(f"<p><b>Generated:</b> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>")
        html.append(f"<p><b>Dataset:</b> {len(self.df)} samples × {len(self.df.columns)} variables</p>")

        avail = geochem.get('available', {})
        html.append("<div class='card'>")
        html.append("<h3>Available Calculations</h3>")
        for key, val in avail.items():
            badge = "badge-available" if val else "badge-missing"
            status = "Available" if val else "Not Available"
            html.append(f"<span class='badge {badge}'>{key.replace('_', ' ').title()}: {status}</span> ")
        html.append("</div>")

        html.append("<h2>Alteration Indices</h2>")
        if geochem['alteration_indices']:
            html.append("<div class='card'>")
            for name, values in geochem['alteration_indices'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{values.min():.4f} - {values.max():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No alteration indices could be calculated.</p></div>")

        html.append("<h2>Element Ratios</h2>")
        if geochem['element_ratios']:
            html.append("<div class='card'>")
            for name, values in geochem['element_ratios'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No element ratios could be calculated.</p></div>")

        html.append("<h2>Mineralization Indices</h2>")
        if geochem['mineralization_indices']:
            html.append("<div class='card'>")
            for name, values in geochem['mineralization_indices'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No mineralization indices could be calculated.</p></div>")

        html.append("<h2>Grade Statistics</h2>")
        if geochem['grade_stats']:
            html.append("<div class='card'>")
            for element, elem_stats in geochem['grade_stats'].items():
                if elem_stats:
                    grade_class = elem_stats.get('Grade Class', 'LOW')
                    badge_class = 'badge-high' if grade_class == 'HIGH' else 'badge-medium' if grade_class == 'MEDIUM' else 'badge-low'
                    html.append(f"""
                    <div style='margin: 10px 0; padding: 10px; background: white; border-radius: 6px; border: 1px solid {Palette.BORDER};'>
                        <b>{element}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{elem_stats.get('Mean', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{elem_stats.get('Median', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Std Dev</div><div class='stat-value'>{elem_stats.get('Std Dev', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{elem_stats.get('Minimum', 0):.4f} - {elem_stats.get('Maximum', 0):.4f}</div></div>
                        <div style='margin-top: 5px;'>
                            <span class='badge {badge_class}'>Grade: {grade_class}</span>
                            <span style='margin-left: 10px;'>Threshold: {elem_stats.get('Threshold', 0):.4f}</span>
                            <span style='margin-left: 10px;'>Above: {elem_stats.get('Above Threshold', 0)} ({elem_stats.get('% Above Threshold', 0):.1f}%)</span>
                        </div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No grade statistics could be calculated.</p></div>")

        html.append(f"""
        <div style='margin-top: 20px; padding: 15px; background-color: {Palette.PRIMARY_LIGHT}; border-radius: 8px;'>
            <p><b>Summary</b></p>
            <table style='width: auto;'>
        """)
        html.append(f"<tr><td>Total Samples</td><td><b>{len(self.df)}</b></td></tr>")
        html.append(f"<tr><td>Total Variables</td><td><b>{len(self.df.columns)}</b></td></tr>")
        html.append(f"<tr><td>Alteration Indices</td><td><b>{len(geochem['alteration_indices'])}</b></td></tr>")
        html.append(f"<tr><td>Element Ratios</td><td><b>{len(geochem['element_ratios'])}</b></td></tr>")
        html.append(f"<tr><td>Mineralization Indices</td><td><b>{len(geochem['mineralization_indices'])}</b></td></tr>")
        html.append(f"<tr><td>Grade Statistics</td><td><b>{len(geochem['grade_stats'])}</b></td></tr>")
        html.append("</table></div></body></html>")

        self.geochem_viewer.setHtml("\n".join(html))

        # -- Populate the data-grid tables (DataFrame view of the same results) --
        def series_dict_to_df(series_dict):
            rows = []
            for name, values in series_dict.items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    rows.append({
                        'Name': name, 'N': int(values.notna().sum()),
                        'Mean': values.mean(), 'Median': values.median(),
                        'Std Dev': values.std(), 'Min': values.min(), 'Max': values.max(),
                    })
            return pd.DataFrame(rows)

        self.populate_table_from_df(self.geochem_alteration_table,
                                     series_dict_to_df(geochem['alteration_indices']))
        self.populate_table_from_df(self.geochem_ratio_table,
                                     series_dict_to_df(geochem['element_ratios']))
        self.populate_table_from_df(self.geochem_mineralization_table,
                                     series_dict_to_df(geochem['mineralization_indices']))

        if geochem['grade_stats']:
            grade_df = pd.DataFrame(geochem['grade_stats']).T.reset_index().rename(columns={'index': 'Element'})
            grade_df = grade_df[['Element', 'N', 'Mean', 'Median', 'Std Dev', 'Minimum', 'Maximum',
                                  'Threshold', 'Above Threshold', '% Above Threshold', 'Grade Class']]
        else:
            grade_df = pd.DataFrame()

        self.populate_table_from_df(
            self.geochem_grade_table, grade_df, precision=4,
            highlight_col='Grade Class',
            highlight_rule=lambda v: Palette.DANGER_LIGHT if v == 'HIGH'
                                      else (Palette.WARNING_LIGHT if v == 'MEDIUM' else Palette.SUCCESS_LIGHT),
        )

    def show_report(self):
        self.nav_list.setCurrentRow(4)  # Geochemistry page

    # ============================================================
    # EXPORT FUNCTIONS
    # ============================================================

    def export_data(self):
        if self.df is None:
            QMessageBox.warning(self, "No Data", "There is no data to export.")
            return

        df = self.df

        formats = {
            "CSV (;)": "csv_semicolon",
            "CSV (,)": "csv_comma",
            "Excel (.xlsx)": "excel",
            "JSON": "json"
        }

        format_choice, ok = QInputDialog.getItem(
            self, "Export Format", "Select format:", list(formats.keys()), 0, False)
        if not ok:
            return

        file_ext_map = {
            "csv_semicolon": "csv",
            "csv_comma": "csv",
            "excel": "xlsx",
            "json": "json"
        }

        file_path, _ = QFileDialog.getSaveFileName(
            self, "Export Data", f"cleaned_data.{file_ext_map[formats[format_choice]]}",
            f"{format_choice} (*.{file_ext_map[formats[format_choice]]})")

        if file_path:
            try:
                self.set_processing(True, "Exporting…")
                self.update_progress(f"Exporting to {format_choice}...")

                fmt = formats[format_choice]
                if fmt == "csv_semicolon":
                    df.to_csv(file_path, sep=';', index=False, encoding='utf-8-sig')
                elif fmt == "csv_comma":
                    df.to_csv(file_path, sep=',', index=False, encoding='utf-8-sig')
                elif fmt == "excel":
                    df.to_excel(file_path, index=False)
                elif fmt == "json":
                    df.to_json(file_path, orient='records', indent=2)

                self.log_message(f"Exported: {file_path}")
                self.set_status('success', 'Export complete')
                QMessageBox.information(self, "Export Complete", f"Data exported to:\n{file_path}")
            except Exception as e:
                self.set_status('error', 'Export failed')
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")
            finally:
                self.set_processing(False)

    # ============================================================
    # HELP AND ABOUT
    # ============================================================

    def show_help(self):
        help_text = f"""{APP_NAME} v{APP_VERSION}

Project Code: {APP_CODE}
Developer: {DEVELOPER}

QUICK START
1. Upload your data
2. Clean the data
3. Run analysis
4. Review visualizations
5. Check geochemistry
6. Generate report

FEATURES

Data Management
• Upload CSV/Excel files
• Clean data (missing values, outliers)
• Rename file
• Delete selected rows/columns

Statistics
• Mean, Median, Std Dev
• Variance, Skewness, Kurtosis
• Correlation Matrix

Visualization
• Correlation Heatmap
• PCA Scores
• Scatter Plot
• Combined view
• EDA: Histogram, PDF, CDF, Box Plot

Geochemistry
• Alteration Indices
• Element Ratios
• Mineralization Indices
• Grade Statistics

Export
• CSV, Excel, JSON

Copyright © 2026 {DEVELOPER}
"""
        QMessageBox.information(self, "Help", help_text)

    def show_about(self):
        about_text = f"""{APP_NAME}

Version: {APP_VERSION}
Project Code: {APP_CODE}

Developer: {DEVELOPER}
Email: {EMAIL}
{BACKGROUND}

Professional geochemical data cleaning and
exploratory data analysis suite for mineral
exploration and geochemistry.

Copyright © 2026 {DEVELOPER}
All rights reserved.
"""
        QMessageBox.about(self, "About", about_text)

    # --------------------------------------------------------
    # STYLESHEET
    # --------------------------------------------------------

    def get_stylesheet(self):
        P = Palette
        return f"""
            QMainWindow {{ background-color: {P.BG}; }}
            QWidget {{ font-family: 'Segoe UI', Arial, sans-serif; font-size: 13px; color: {P.TEXT_PRIMARY}; }}

            /* -- Sidebar -- */
            #Sidebar {{ background-color: {P.SIDEBAR_BG}; }}
            #Brand {{ border-bottom: 1px solid #1F2937; }}
            #BrandTitle {{ color: {P.SIDEBAR_TEXT_ACTIVE}; font-size: 15px; font-weight: 700; }}
            #BrandSubtitle {{ color: {P.SIDEBAR_TEXT}; font-size: 11px; }}
            #NavList {{
                background-color: transparent;
                border: none;
                outline: none;
                padding: 8px;
            }}
            #NavList::item {{
                color: {P.SIDEBAR_TEXT};
                border-radius: 8px;
                padding-left: 10px;
                margin: 2px 4px;
            }}
            #NavList::item:hover {{ background-color: {P.SIDEBAR_BG_HOVER}; color: {P.SIDEBAR_TEXT_ACTIVE}; }}
            #NavList::item:selected {{ background-color: {P.PRIMARY}; color: {P.SIDEBAR_TEXT_ACTIVE}; }}
            #SidebarFooter {{ border-top: 1px solid #1F2937; }}
            #SidebarFooterName {{ color: {P.SIDEBAR_TEXT_ACTIVE}; font-size: 11px; font-weight: 600; }}
            #SidebarFooterRole {{ color: {P.SIDEBAR_TEXT}; font-size: 10px; }}

            /* -- Top bar / status -- */
            #TopBar {{ background-color: {P.SURFACE}; border-bottom: 1px solid {P.BORDER}; }}
            #PageTitle {{ font-size: 17px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}
            #FileInfo {{ color: {P.TEXT_SECONDARY}; font-size: 12px; padding-left: 6px; }}
            #StatusPanel {{ background-color: {P.SURFACE}; border-bottom: 1px solid {P.BORDER}; }}
            #StatusText {{ color: {P.TEXT_SECONDARY}; font-size: 12px; }}
            #ProcessInfo {{ color: {P.TEXT_MUTED}; font-size: 11px; }}
            #StatusProgress {{
                border: none; border-radius: 3px; background-color: {P.BORDER}; height: 6px;
            }}
            #StatusProgress::chunk {{ background-color: {P.PRIMARY}; border-radius: 3px; }}

            /* -- Cards -- */
            #Card {{ background-color: {P.SURFACE}; border-radius: 12px; border: 1px solid {P.BORDER}; }}
            #StatBadge {{ background-color: {P.BG}; border-radius: 6px; }}
            #StatBadgeLabel {{ color: {P.TEXT_MUTED}; font-size: 9px; font-weight: 700; letter-spacing: 0.5px; }}
            #StatBadgeValue {{ color: {P.PRIMARY}; font-size: 15px; font-weight: 700; }}
            #SectionTitle {{ font-size: 14px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}

            /* -- Welcome page -- */
            #WelcomeTitle {{ font-size: 22px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}
            #WelcomeSubtitle {{ font-size: 13px; color: {P.TEXT_SECONDARY}; }}
            #StepIcon {{ background-color: {P.PRIMARY_LIGHT}; border-radius: 8px; }}
            #StepTitle {{ font-size: 13px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}
            #StepDesc {{ font-size: 11px; color: {P.TEXT_SECONDARY}; }}
            #WelcomeTip {{ color: {P.TEXT_SECONDARY}; font-size: 12px; }}
            #MutedLabel {{ color: {P.TEXT_SECONDARY}; }}

            /* -- Buttons -- */
            QPushButton {{
                background-color: {P.SURFACE};
                color: {P.TEXT_PRIMARY};
                padding: 8px 16px;
                border: 1px solid {P.BORDER};
                border-radius: 6px;
                font-weight: 600;
                font-size: 12px;
            }}
            QPushButton:hover {{ background-color: {P.BG}; }}
            QPushButton[kind="primary"] {{ background-color: {P.PRIMARY}; color: white; border: none; }}
            QPushButton[kind="primary"]:hover {{ background-color: {P.PRIMARY_DARK}; }}
            QPushButton[kind="success"] {{ background-color: {P.SUCCESS}; color: white; border: none; }}
            QPushButton[kind="success"]:hover {{ background-color: #047857; }}
            QPushButton[kind="warning"] {{ background-color: {P.WARNING}; color: white; border: none; }}
            QPushButton[kind="warning"]:hover {{ background-color: #B45309; }}
            QPushButton[kind="danger"] {{ background-color: {P.DANGER}; color: white; border: none; }}
            QPushButton[kind="danger"]:hover {{ background-color: #B91C1C; }}
            QPushButton[kind="ghost"] {{ background-color: transparent; border: none; }}
            QPushButton[kind="ghost"]:hover {{ background-color: {P.BG}; }}

            /* -- Inputs -- */
            QComboBox, QSpinBox, QLineEdit {{
                padding: 6px 10px;
                border: 1px solid {P.BORDER};
                border-radius: 6px;
                background-color: {P.SURFACE};
                min-width: 110px;
            }}
            QCheckBox {{ spacing: 6px; }}

            /* -- Tables -- */
            QTableWidget {{
                background-color: {P.SURFACE};
                gridline-color: {P.BORDER};
                border: none;
                border-radius: 10px;
                alternate-background-color: {P.BG};
            }}
            QTableWidget::item {{ padding: 6px; }}
            QTableWidget::item:selected {{ background-color: {P.PRIMARY_LIGHT}; color: {P.PRIMARY}; }}
            QHeaderView::section {{
                background-color: {P.BG};
                padding: 10px;
                border: none;
                border-bottom: 2px solid {P.BORDER};
                font-weight: 700;
                color: {P.TEXT_SECONDARY};
                font-size: 11px;
            }}

            /* Report / statistics data-grids share one accent language:
               indigo headers so every derived table (quality, stats,
               outliers, geochemistry) reads as part of the same system. */
            QTableWidget#ReportTable {{
                border: 1px solid {P.BORDER};
                border-radius: 8px;
            }}
            QTableWidget#ReportTable QHeaderView::section {{
                background-color: {P.PRIMARY_LIGHT};
                color: {P.PRIMARY_DARK};
                border-bottom: 2px solid {P.PRIMARY};
                font-size: 10px;
                letter-spacing: 0.3px;
            }}
            QTableWidget#ReportTable::item {{ font-size: 12px; }}

            QTabWidget#ReportTabs::pane {{
                border: 1px solid {P.BORDER};
                border-radius: 8px;
                top: -1px;
            }}
            QTabWidget#ReportTabs QTabBar::tab {{
                background-color: {P.BG};
                color: {P.TEXT_SECONDARY};
                padding: 7px 16px;
                margin-right: 2px;
                border-top-left-radius: 6px;
                border-top-right-radius: 6px;
                font-weight: 600;
                font-size: 11px;
            }}
            QTabWidget#ReportTabs QTabBar::tab:selected {{
                background-color: {P.PRIMARY_LIGHT};
                color: {P.PRIMARY_DARK};
                border-bottom: 2px solid {P.PRIMARY};
            }}

            QTextEdit {{
                background-color: {P.SURFACE};
                border: none;
                padding: 12px;
            }}

            QStatusBar {{ background-color: {P.SURFACE}; color: {P.TEXT_SECONDARY}; border-top: 1px solid {P.BORDER}; }}
        """


# ============================================================
# MAIN APPLICATION
# ============================================================

def main():
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    app.setFont(QFont("Segoe UI", 10))
    window = ElementalSocietyAnalysis()
    window.show()
    sys.exit(app.exec_())


if __name__ == '__main__':
    main()

In [ ]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - PROFESSIONAL EDITION v2.0
# Project Code: 1010
# Developer: Mohamed Almoufty Abdoalmoutleb Abaker
# ============================================================
#
# v2.0 changes:
#   - Full visual redesign: dark sidebar navigation, card-based
#     content, consistent spacing/typography, subtle shadows.
#   - Custom vector icon set (drawn with QPainter, no external
#     icon packages required) replacing emoji glyphs everywhere:
#     toolbar, navigation, status, dialogs, window icon.
#   - Status panel now uses an icon + animated-style dot instead
#     of plain text, and a proper busy progress bar.
#   - Minor logic fixes: "before/after" counts in Clean no longer
#     collide because df is only swapped after the message is
#     built; `stats` dict no longer shadows the imported
#     scipy.stats module; safer division guards.
# ============================================================

import sys
import os
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *
import matplotlib
matplotlib.use("Qt5Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
import seaborn as sns
from scipy import stats as scipy_stats
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# APPLICATION METADATA
# ============================================================

APP_NAME = "Elemental Society Analysis"
APP_VERSION = "2.0.0"
APP_CODE = "1010"
DEVELOPER = "Mohamed Almoufty Abdoalmoutleb Abaker"
EMAIL = "almoufty121@gmail.com"
BACKGROUND = "Exploration Geologist | Msc Earth Science Engineering"

# ============================================================
# DESIGN SYSTEM — colors, spacing, typography live here so the
# whole app can be re-themed from one place.
# ============================================================

class Palette:
    PRIMARY = "#4F46E5"       # Indigo — primary brand / actions
    PRIMARY_DARK = "#4338CA"
    PRIMARY_LIGHT = "#EEF2FF"
    SUCCESS = "#059669"
    SUCCESS_LIGHT = "#ECFDF5"
    WARNING = "#D97706"
    WARNING_LIGHT = "#FFFBEB"
    DANGER = "#DC2626"
    DANGER_LIGHT = "#FEF2F2"
    INFO = "#0284C7"

    SIDEBAR_BG = "#111827"
    SIDEBAR_BG_HOVER = "#1F2937"
    SIDEBAR_TEXT = "#9CA3AF"
    SIDEBAR_TEXT_ACTIVE = "#FFFFFF"

    BG = "#F3F4F6"
    SURFACE = "#FFFFFF"
    BORDER = "#E5E7EB"
    TEXT_PRIMARY = "#111827"
    TEXT_SECONDARY = "#6B7280"
    TEXT_MUTED = "#9CA3AF"


# ============================================================
# VECTOR ICON FACTORY
# Simple line-icon set drawn on the fly with QPainter so the
# app has a consistent, crisp icon language without bundling
# image assets or third-party icon fonts.
# ============================================================

class Icons:
    """Generates QIcon objects for a small line-icon set."""

    _cache = {}

    @classmethod
    def get(cls, name, color=Palette.TEXT_PRIMARY, size=20):
        key = (name, color, size)
        if key in cls._cache:
            return cls._cache[key]
        pix = QPixmap(size, size)
        pix.fill(Qt.transparent)
        p = QPainter(pix)
        p.setRenderHint(QPainter.Antialiasing)
        pen = QPen(QColor(color))
        pen.setWidthF(size / 11.0)
        pen.setCapStyle(Qt.RoundCap)
        pen.setJoinStyle(Qt.RoundJoin)
        p.setPen(pen)
        p.setBrush(Qt.NoBrush)
        m = size * 0.15  # margin
        s = size - 2 * m  # usable size
        draw = getattr(cls, f"_draw_{name}", None)
        if draw:
            draw(p, m, s, size, color)
        p.end()
        icon = QIcon(pix)
        cls._cache[key] = icon
        return icon

    # -- individual glyphs -----------------------------------
    @staticmethod
    def _draw_home(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m, m + s * 0.45)
        path.lineTo(size / 2, m)
        path.lineTo(m + s, m + s * 0.45)
        p.drawPath(path)
        p.drawRect(QRectF(m + s * 0.18, m + s * 0.45, s * 0.64, s * 0.55))
        p.drawRect(QRectF(m + s * 0.4, m + s * 0.62, s * 0.2, s * 0.38))

    @staticmethod
    def _draw_upload(p, m, s, size, center=None, color=None):
        cx = size / 2
        p.drawLine(QPointF(cx, m), QPointF(cx, m + s * 0.6))
        path = QPainterPath()
        path.moveTo(cx - s * 0.25, m + s * 0.22)
        path.lineTo(cx, m)
        path.lineTo(cx + s * 0.25, m + s * 0.22)
        p.drawPath(path)
        p.drawLine(QPointF(m, m + s * 0.85), QPointF(m + s, m + s * 0.85))
        p.drawLine(QPointF(m, m + s * 0.85), QPointF(m, m + s * 0.6))
        p.drawLine(QPointF(m + s, m + s * 0.85), QPointF(m + s, m + s * 0.6))

    @staticmethod
    def _draw_broom(p, m, s, size, color):
        p.drawLine(QPointF(m + s * 0.15, m), QPointF(m + s * 0.65, m + s * 0.55))
        path = QPainterPath()
        path.moveTo(m + s * 0.65, m + s * 0.55)
        path.lineTo(m + s, m + s * 0.35)
        path.lineTo(m + s * 0.95, m + s * 0.75)
        path.lineTo(m + s * 0.45, m + s)
        path.closeSubpath()
        p.drawPath(path)

    @staticmethod
    def _draw_chart(p, m, s, size, color):
        base = m + s
        p.drawLine(QPointF(m, m), QPointF(m, base))
        p.drawLine(QPointF(m, base), QPointF(m + s, base))
        bars = [0.25, 0.55, 0.4, 0.8]
        w = s * 0.15
        for i, h in enumerate(bars):
            x = m + s * 0.15 + i * (w + s * 0.08)
            p.drawLine(QPointF(x, base), QPointF(x, base - s * h))

    @staticmethod
    def _draw_download(p, m, s, size, color):
        cx = size / 2
        p.drawLine(QPointF(cx, m), QPointF(cx, m + s * 0.6))
        path = QPainterPath()
        path.moveTo(cx - s * 0.25, m + s * 0.38)
        path.lineTo(cx, m + s * 0.6)
        path.lineTo(cx + s * 0.25, m + s * 0.38)
        p.drawPath(path)
        p.drawLine(QPointF(m, m + s * 0.85), QPointF(m + s, m + s * 0.85))

    @staticmethod
    def _draw_document(p, m, s, size, color):
        w = s * 0.7
        path = QPainterPath()
        path.moveTo(m, m)
        path.lineTo(m + w, m)
        path.lineTo(m + s, m + s * 0.3)
        path.lineTo(m + s, m + s)
        path.lineTo(m, m + s)
        path.closeSubpath()
        p.drawPath(path)
        for i in range(3):
            y = m + s * (0.45 + i * 0.18)
            p.drawLine(QPointF(m + s * 0.18, y), QPointF(m + s * 0.75, y))

    @staticmethod
    def _draw_help(p, m, s, size, color):
        p.drawEllipse(QRectF(m, m, s, s))
        f = QFont()
        f.setBold(True)
        f.setPixelSize(int(s * 0.55))
        p.setFont(f)
        p.drawText(QRectF(m, m, s, s), Qt.AlignCenter, "?")

    @staticmethod
    def _draw_info(p, m, s, size, color):
        p.drawEllipse(QRectF(m, m, s, s))
        cx = size / 2
        p.drawLine(QPointF(cx, m + s * 0.32), QPointF(cx, m + s * 0.33))
        pen = p.pen()
        pen.setWidthF(pen.widthF() * 1.6)
        p.setPen(pen)
        p.drawPoint(QPointF(cx, m + s * 0.28))
        pen.setWidthF(pen.widthF() / 1.6)
        p.setPen(pen)
        p.drawLine(QPointF(cx, m + s * 0.45), QPointF(cx, m + s * 0.78))

    @staticmethod
    def _draw_table(p, m, s, size, color):
        p.drawRect(QRectF(m, m, s, s))
        p.drawLine(QPointF(m, m + s / 3), QPointF(m + s, m + s / 3))
        p.drawLine(QPointF(m, m + 2 * s / 3), QPointF(m + s, m + 2 * s / 3))
        p.drawLine(QPointF(m + s / 2, m), QPointF(m + s / 2, m + s))

    @staticmethod
    def _draw_trend(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m, m + s * 0.8)
        path.lineTo(m + s * 0.3, m + s * 0.45)
        path.lineTo(m + s * 0.55, m + s * 0.65)
        path.lineTo(m + s, m + s * 0.1)
        p.drawPath(path)
        path2 = QPainterPath()
        path2.moveTo(m + s * 0.78, m + s * 0.1)
        path2.lineTo(m + s, m + s * 0.1)
        path2.lineTo(m + s, m + s * 0.32)
        p.drawPath(path2)

    @staticmethod
    def _draw_flask(p, m, s, size, color):
        p.drawLine(QPointF(m + s * 0.38, m), QPointF(m + s * 0.38, m + s * 0.4))
        p.drawLine(QPointF(m + s * 0.62, m), QPointF(m + s * 0.62, m + s * 0.4))
        p.drawLine(QPointF(m + s * 0.3, m), QPointF(m + s * 0.7, m))
        path = QPainterPath()
        path.moveTo(m + s * 0.38, m + s * 0.4)
        path.lineTo(m + s * 0.1, m + s * 0.92)
        path.lineTo(m + s * 0.9, m + s * 0.92)
        path.lineTo(m + s * 0.62, m + s * 0.4)
        p.drawPath(path)
        p.drawLine(QPointF(m + s * 0.22, m + s * 0.68), QPointF(m + s * 0.78, m + s * 0.68))

    @staticmethod
    def _draw_list(p, m, s, size, color):
        for i in range(3):
            y = m + s * (0.15 + i * 0.35)
            p.drawEllipse(QPointF(m + s * 0.05, y), 1.2, 1.2)
            p.drawLine(QPointF(m + s * 0.2, y), QPointF(m + s, y))

    @staticmethod
    def _draw_trash(p, m, s, size, color):
        p.drawLine(QPointF(m + s * 0.15, m + s * 0.22), QPointF(m + s * 0.85, m + s * 0.22))
        p.drawLine(QPointF(m + s * 0.35, m + s * 0.22), QPointF(m + s * 0.4, m))
        p.drawLine(QPointF(m + s * 0.65, m + s * 0.22), QPointF(m + s * 0.6, m))
        p.drawLine(QPointF(m + s * 0.4, m), QPointF(m + s * 0.6, m))
        path = QPainterPath()
        path.moveTo(m + s * 0.22, m + s * 0.3)
        path.lineTo(m + s * 0.3, m + s)
        path.lineTo(m + s * 0.7, m + s)
        path.lineTo(m + s * 0.78, m + s * 0.3)
        p.drawPath(path)

    @staticmethod
    def _draw_pencil(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m + s * 0.1, m + s * 0.9)
        path.lineTo(m + s * 0.15, m + s * 0.65)
        path.lineTo(m + s * 0.68, m + s * 0.12)
        path.lineTo(m + s * 0.88, m + s * 0.32)
        path.lineTo(m + s * 0.35, m + s * 0.85)
        path.closeSubpath()
        p.drawPath(path)
        p.drawLine(QPointF(m + s * 0.6, m + s * 0.2), QPointF(m + s * 0.8, m + s * 0.4))

    @staticmethod
    def _draw_check(p, m, s, size, color):
        path = QPainterPath()
        path.moveTo(m, m + s * 0.55)
        path.lineTo(m + s * 0.4, m + s * 0.9)
        path.lineTo(m + s, m + s * 0.15)
        p.drawPath(path)

    @staticmethod
    def _draw_dot(p, m, s, size, color):
        p.setBrush(QColor(color))
        p.drawEllipse(QRectF(m, m, s, s))


def styled_button(text, icon_name=None, kind="default", tooltip=None, icon_color=None):
    """Factory for a consistently styled QPushButton with an optional icon."""
    btn = QPushButton(text)
    if icon_name:
        col = icon_color or ("#FFFFFF" if kind in ("primary", "success", "danger", "warning") else Palette.TEXT_SECONDARY)
        btn.setIcon(Icons.get(icon_name, color=col, size=16))
        btn.setIconSize(QSize(16, 16))
    if tooltip:
        btn.setToolTip(tooltip)
    btn.setProperty("kind", kind)
    btn.setCursor(Qt.PointingHandCursor)
    return btn


# ============================================================
# ELEMENT ALIAS MAPPING
# ============================================================

ELEMENT_ALIAS_MAP = {
    'K': 'K_pct', 'K2O': 'K_pct', 'K_PCT': 'K_pct', 'K%': 'K_pct',
    'MG': 'Mg_pct', 'MGO': 'Mg_pct', 'MG_PCT': 'Mg_pct', 'MG%': 'Mg_pct',
    'CA': 'Ca_pct', 'CAO': 'Ca_pct', 'CA_PCT': 'Ca_pct', 'CA%': 'Ca_pct',
    'NA': 'Na_pct', 'NA2O': 'Na_pct', 'NA_PCT': 'Na_pct', 'NA%': 'Na_pct',
    'FE': 'Fe_pct', 'FE2O3': 'Fe_pct', 'FEO': 'Fe_pct', 'FE_PCT': 'Fe_pct', 'FE%': 'Fe_pct',
    'AU': 'Au_ppm', 'AU_PPM': 'Au_ppm', 'AU_PPB': 'Au_ppm',
    'CU': 'Cu_ppm', 'CU_PPM': 'Cu_ppm', 'CU_PCT': 'Cu_ppm',
    'ZN': 'Zn_ppm', 'ZN_PPM': 'Zn_ppm',
    'PB': 'Pb_ppm', 'PB_PPM': 'Pb_ppm',
    'AG': 'Ag_ppm', 'AG_PPM': 'Ag_ppm',
    'AS': 'As_ppm', 'AS_PPM': 'As_ppm',
    'SB': 'Sb_ppm', 'SB_PPM': 'Sb_ppm',
    'MO': 'Mo_ppm', 'MO_PPM': 'Mo_ppm',
    'SIO2': 'SiO2_pct', 'SIO2_PCT': 'SiO2_pct', 'SIO2%': 'SiO2_pct',
    'AL': 'Al_pct', 'AL2O3': 'Al_pct', 'AL_PCT': 'Al_pct', 'AL%': 'Al_pct',
}

# ============================================================
# DATA CLEANING ENGINE
# ============================================================

def standardize_element_names(df):
    """Normalize chemical headers to canonical standard names."""
    renames = {}
    for col in df.columns:
        clean_col = re.sub(r'[^a-zA-Z0-9%_]', '', str(col)).upper()
        if clean_col in ELEMENT_ALIAS_MAP:
            renames[col] = ELEMENT_ALIAS_MAP[clean_col]
        else:
            for key, std in ELEMENT_ALIAS_MAP.items():
                if key in clean_col:
                    renames[col] = std
                    break
    return df.rename(columns=renames)


def clean_dataframe(df):
    """Universal cleaning routine for geochemical datasets."""
    if df is None or df.empty:
        return df

    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]
    df = standardize_element_names(df)

    missing_vals = ['', 'nan', 'NaN', 'null', 'None', 'NA', '-', '--', 'n/a', '?']
    df = df.replace(missing_vals, np.nan)

    text_cols = ['sample', 'id', 'point', 'lithology', 'comments', 'unit', 'rock_type']
    for col in df.columns:
        is_text = any(t in col.lower() for t in text_cols)
        if not is_text:
            cleaned_series = df[col].astype(str).str.replace(r'^[<>]', '', regex=True)
            cleaned_series = pd.to_numeric(cleaned_series.str.replace(',', '.'), errors='coerce')
            if cleaned_series.notna().sum() > 0:
                df[col] = cleaned_series

    df = df.dropna(how='all')
    return df


def read_any_file(path):
    """Read any tabular file format with auto-detection of encoding/separator."""
    ext = Path(path).suffix.lower()
    if ext in ['.xlsx', '.xls', '.xlsm']:
        return pd.read_excel(path)

    encodings = ["utf-8", "cp1252", "latin-1", "iso-8859-1", "utf-16"]
    seps = [";", ",", "\t", "|"]

    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, engine="python", on_bad_lines="skip")
                if df.shape[1] > 1:
                    return df
            except Exception:
                pass

    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read(3000)
        sep = max([";", ",", "\t", "|"], key=lambda d: txt.count(d))
        return pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")
    except Exception:
        return pd.read_csv(path, engine="python", on_bad_lines="skip")


# ============================================================
# GEOCHEMISTRY ENGINE
# ============================================================

class GeochemistryEngine:
    """Smart geochemistry calculations with auto-detection of available fields."""

    @staticmethod
    def calculate_all(df):
        results = {
            'alteration_indices': {},
            'element_ratios': {},
            'mineralization_indices': {},
            'grade_stats': {},
            'available': {'alteration': False, 'ratios': False, 'mineralization': False, 'grades': False}
        }

        # 1. Alteration Indices
        if {'K_pct', 'Mg_pct', 'Ca_pct', 'Na_pct'}.issubset(df.columns):
            try:
                results['alteration_indices']['Ishikawa AI'] = 100 * (df['K_pct'] + df['Mg_pct']) / (
                    df['K_pct'] + df['Mg_pct'] + df['Ca_pct'] + df['Na_pct'] + 1e-6)
                results['available']['alteration'] = True
            except Exception:
                pass

        if {'Fe_pct', 'Mg_pct', 'Na_pct', 'K_pct'}.issubset(df.columns):
            try:
                results['alteration_indices']['CCPI'] = 100 * (df['Fe_pct'] + df['Mg_pct']) / (
                    df['Fe_pct'] + df['Mg_pct'] + df['Na_pct'] + df['K_pct'] + 1e-6)
                results['available']['alteration'] = True
            except Exception:
                pass

        if {'K_pct', 'Na_pct'}.issubset(df.columns):
            try:
                results['alteration_indices']['Sericite Index'] = df['K_pct'] / (df['K_pct'] + df['Na_pct'] + 1e-6)
                results['available']['alteration'] = True
            except Exception:
                pass

        # 2. Element Ratios
        ratio_pairs = [
            ('Au_ppm', 'Cu_ppm', 'Au/Cu'), ('Zn_ppm', 'Pb_ppm', 'Zn/Pb'),
            ('Cu_ppm', 'Zn_ppm', 'Cu/Zn'), ('Au_ppm', 'Ag_ppm', 'Au/Ag'),
            ('Au_ppm', 'As_ppm', 'Au/As'), ('Cu_ppm', 'Mo_ppm', 'Cu/Mo')
        ]
        for c1, c2, name in ratio_pairs:
            if c1 in df.columns and c2 in df.columns:
                try:
                    results['element_ratios'][name] = df[c1] / (df[c2] + 1e-6)
                    results['available']['ratios'] = True
                except Exception:
                    pass

        # 3. Mineralization Indices
        if 'Au_ppm' in df.columns:
            try:
                results['mineralization_indices']['Gold Index'] = df['Au_ppm'] * 100
                results['available']['mineralization'] = True
            except Exception:
                pass

        base_metals = [c for c in ['Cu_ppm', 'Zn_ppm', 'Pb_ppm'] if c in df.columns]
        if base_metals:
            try:
                results['mineralization_indices']['Base Metal Index'] = sum(df[c].fillna(0) for c in base_metals) / len(base_metals)
                results['available']['mineralization'] = True
            except Exception:
                pass

        pathfinders = [c for c in ['As_ppm', 'Sb_ppm', 'Mo_ppm'] if c in df.columns]
        if pathfinders:
            try:
                results['mineralization_indices']['Pathfinder Index'] = sum(df[c].fillna(0) for c in pathfinders) / len(pathfinders)
                results['available']['mineralization'] = True
            except Exception:
                pass

        # 4. Grade Statistics
        key_elements = ['Au_ppm', 'Cu_ppm', 'Zn_ppm', 'Pb_ppm', 'Ag_ppm', 'As_ppm']
        for elem in key_elements:
            if elem in df.columns:
                data = df[elem].dropna()
                if len(data) > 0:
                    threshold = data.quantile(0.75)
                    mean_val = data.mean()
                    results['grade_stats'][elem] = {
                        'Mean': mean_val, 'Median': data.median(), 'Std Dev': data.std(),
                        'Minimum': data.min(), 'Maximum': data.max(),
                        'N': len(data), 'Threshold': threshold,
                        'Above Threshold': (data > threshold).sum(),
                        '% Above Threshold': (data > threshold).sum() / len(data) * 100,
                        'Grade Class': 'HIGH' if threshold and mean_val > threshold * 2
                                       else 'MEDIUM' if threshold and mean_val > threshold
                                       else 'LOW'
                    }
                    results['available']['grades'] = True

        return results


# ============================================================
# REUSABLE UI COMPONENTS
# ============================================================

class Card(QFrame):
    """A elevated white panel used as the base content container."""

    def __init__(self, parent=None):
        super().__init__(parent)
        self.setObjectName("Card")
        effect = QGraphicsDropShadowEffect(self)
        effect.setBlurRadius(18)
        effect.setOffset(0, 2)
        effect.setColor(QColor(17, 24, 39, 35))
        self.setGraphicsEffect(effect)


class StatBadge(QFrame):
    """Small metric chip used for stat_boxes (mean/median/etc.)."""

    def __init__(self, label, value, parent=None):
        super().__init__(parent)
        self.setObjectName("StatBadge")
        layout = QVBoxLayout(self)
        layout.setContentsMargins(12, 6, 12, 6)
        layout.setSpacing(0)
        lbl = QLabel(label.upper())
        lbl.setObjectName("StatBadgeLabel")
        val = QLabel(str(value))
        val.setObjectName("StatBadgeValue")
        layout.addWidget(lbl)
        layout.addWidget(val)


class SectionTitle(QLabel):
    def __init__(self, text, parent=None):
        super().__init__(text, parent)
        self.setObjectName("SectionTitle")


# ============================================================
# MAIN APPLICATION WINDOW
# ============================================================

NAV_ITEMS = [
    ("home", "Overview"),
    ("table", "Dataset"),
    ("trend", "Distributions"),
    ("chart", "Analysis"),
    ("flask", "Geochemistry"),
    ("list", "Activity Log"),
]


class ElementalSocietyAnalysis(QMainWindow):
    def __init__(self):
        super().__init__()
        self.df = None
        self.cleaned_df = None
        self.derived_df = None
        self.is_processing = False
        self.original_filename = None
        self.init_ui()

    # --------------------------------------------------------
    # UI CONSTRUCTION
    # --------------------------------------------------------

    def init_ui(self):
        self.setWindowTitle(f"{APP_NAME} — v{APP_VERSION}")
        self.setWindowIcon(Icons.get("flask", color=Palette.PRIMARY, size=32))
        self.setGeometry(50, 50, 1680, 980)
        self.setStyleSheet(self.get_stylesheet())

        central = QWidget()
        self.setCentralWidget(central)
        root = QHBoxLayout(central)
        root.setContentsMargins(0, 0, 0, 0)
        root.setSpacing(0)

        root.addWidget(self.build_sidebar())

        right = QWidget()
        right_layout = QVBoxLayout(right)
        right_layout.setContentsMargins(0, 0, 0, 0)
        right_layout.setSpacing(0)
        right_layout.addWidget(self.build_topbar())
        right_layout.addWidget(self.build_status_panel())
        self.stack = QStackedWidget()
        right_layout.addWidget(self.stack, 1)
        root.addWidget(right, 1)

        self.build_pages()

        self.statusBar = QStatusBar()
        self.statusBar.showMessage("Ready")
        self.setStatusBar(self.statusBar)

        self.log_message(f"{APP_NAME} v{APP_VERSION} initialized")
        self.log_message(f"Project Code: {APP_CODE}")

        self.nav_list.setCurrentRow(0)

    # -- sidebar ----------------------------------------------

    def build_sidebar(self):
        sidebar = QWidget()
        sidebar.setObjectName("Sidebar")
        sidebar.setFixedWidth(240)
        layout = QVBoxLayout(sidebar)
        layout.setContentsMargins(0, 0, 0, 0)
        layout.setSpacing(0)

        brand = QWidget()
        brand.setObjectName("Brand")
        brand_layout = QHBoxLayout(brand)
        brand_layout.setContentsMargins(20, 22, 20, 22)
        icon_lbl = QLabel()
        icon_lbl.setPixmap(Icons.get("flask", color="#FFFFFF", size=26).pixmap(26, 26))
        brand_layout.addWidget(icon_lbl)
        title_box = QVBoxLayout()
        title_box.setSpacing(0)
        name_lbl = QLabel("Elemental Society")
        name_lbl.setObjectName("BrandTitle")
        sub_lbl = QLabel(f"Analysis Suite · v{APP_VERSION}")
        sub_lbl.setObjectName("BrandSubtitle")
        title_box.addWidget(name_lbl)
        title_box.addWidget(sub_lbl)
        brand_layout.addLayout(title_box)
        brand_layout.addStretch()
        layout.addWidget(brand)

        self.nav_list = QListWidget()
        self.nav_list.setObjectName("NavList")
        self.nav_list.setFocusPolicy(Qt.NoFocus)
        self.nav_list.setIconSize(QSize(18, 18))
        for icon_name, label in NAV_ITEMS:
            item = QListWidgetItem(Icons.get(icon_name, color=Palette.SIDEBAR_TEXT, size=18), f"  {label}")
            item.setSizeHint(QSize(0, 44))
            self.nav_list.addItem(item)
        self.nav_list.currentRowChanged.connect(self.on_nav_changed)
        layout.addWidget(self.nav_list, 1)

        footer = QWidget()
        footer.setObjectName("SidebarFooter")
        footer_layout = QVBoxLayout(footer)
        footer_layout.setContentsMargins(20, 14, 20, 18)
        dev_lbl = QLabel(DEVELOPER)
        dev_lbl.setObjectName("SidebarFooterName")
        dev_lbl.setWordWrap(True)
        role_lbl = QLabel(BACKGROUND)
        role_lbl.setObjectName("SidebarFooterRole")
        role_lbl.setWordWrap(True)
        footer_layout.addWidget(dev_lbl)
        footer_layout.addWidget(role_lbl)
        layout.addWidget(footer)

        return sidebar

    def on_nav_changed(self, row):
        if row >= 0:
            self.stack.setCurrentIndex(row)
            self.page_title.setText(NAV_ITEMS[row][1])

    # -- top bar ------------------------------------------------

    def build_topbar(self):
        bar = QWidget()
        bar.setObjectName("TopBar")
        layout = QHBoxLayout(bar)
        layout.setContentsMargins(24, 14, 24, 14)
        layout.setSpacing(10)

        self.page_title = QLabel("Overview")
        self.page_title.setObjectName("PageTitle")
        layout.addWidget(self.page_title)

        self.file_info_label = QLabel("No file loaded")
        self.file_info_label.setObjectName("FileInfo")
        layout.addWidget(self.file_info_label)

        layout.addStretch()

        upload_btn = styled_button("Upload", "upload", "primary", "Upload a geochemical data file")
        upload_btn.clicked.connect(self.upload_file)
        clean_btn = styled_button("Clean", "broom", "warning", "Clean the loaded dataset")
        clean_btn.clicked.connect(self.clean_data)
        analyze_btn = styled_button("Analyze", "chart", "success", "Run comprehensive EDA")
        analyze_btn.clicked.connect(self.run_analysis)
        export_btn = styled_button("Export", "download", "default", "Export cleaned data")
        export_btn.clicked.connect(self.export_data)
        report_btn = styled_button("Report", "document", "default", "Jump to the geochemistry report")
        report_btn.clicked.connect(self.show_report)
        help_btn = styled_button("", "help", "ghost", "Help")
        help_btn.setFixedWidth(36)
        help_btn.clicked.connect(self.show_help)
        about_btn = styled_button("", "info", "ghost", "About")
        about_btn.setFixedWidth(36)
        about_btn.clicked.connect(self.show_about)

        for b in (upload_btn, clean_btn, analyze_btn, export_btn, report_btn, help_btn, about_btn):
            layout.addWidget(b)

        return bar

    # -- status panel -------------------------------------------

    def build_status_panel(self):
        widget = QWidget()
        widget.setObjectName("StatusPanel")
        layout = QHBoxLayout(widget)
        layout.setContentsMargins(24, 8, 24, 8)
        layout.setSpacing(10)

        self.status_dot = QLabel()
        self.status_dot.setPixmap(Icons.get("dot", color=Palette.TEXT_MUTED, size=12).pixmap(12, 12))
        layout.addWidget(self.status_dot)

        self.status_text = QLabel("Ready")
        self.status_text.setObjectName("StatusText")
        layout.addWidget(self.status_text)

        self.progress_bar = QProgressBar()
        self.progress_bar.setObjectName("StatusProgress")
        self.progress_bar.setVisible(False)
        self.progress_bar.setMaximumWidth(220)
        self.progress_bar.setTextVisible(False)
        layout.addWidget(self.progress_bar)

        layout.addStretch()

        self.process_info = QLabel("")
        self.process_info.setObjectName("ProcessInfo")
        layout.addWidget(self.process_info)

        return widget

    def set_status(self, status, message="", progress=False):
        status_map = {
            'idle': (Palette.TEXT_MUTED, 'Ready'),
            'processing': (Palette.WARNING, 'Processing…'),
            'success': (Palette.SUCCESS, 'Success'),
            'error': (Palette.DANGER, 'Error'),
        }
        color, default_text = status_map.get(status, status_map['idle'])
        self.status_dot.setPixmap(Icons.get("dot", color=color, size=12).pixmap(12, 12))
        self.status_text.setText(message if message else default_text)
        self.status_text.setStyleSheet(f"color: {color}; font-weight: 600;")

        show_progress = progress or status == 'processing'
        self.progress_bar.setVisible(show_progress)
        if show_progress:
            self.progress_bar.setRange(0, 0)
        if status == 'processing':
            QApplication.processEvents()

    def set_processing(self, active, message="", progress=False):
        self.is_processing = active
        if active:
            self.set_status('processing', message, progress)
            self.statusBar.showMessage(f"Processing: {message}")
        else:
            self.set_status('idle', 'Ready', False)
            self.process_info.setText("")
            self.statusBar.showMessage("Ready")

    def update_progress(self, message):
        self.process_info.setText(message)
        self.status_text.setText(message)
        self.log_message(message)
        QApplication.processEvents()

    # --------------------------------------------------------
    # PAGES
    # --------------------------------------------------------

    def build_pages(self):
        self.stack.addWidget(self.create_welcome_page())
        self.stack.addWidget(self.create_data_page())
        self.stack.addWidget(self.create_eda_page())
        self.stack.addWidget(self.create_analyze_page())
        self.stack.addWidget(self.create_geochemistry_page())
        self.stack.addWidget(self.create_log_page())

    def create_welcome_page(self):
        page = QWidget()
        outer = QVBoxLayout(page)
        outer.setContentsMargins(24, 20, 24, 24)

        card = Card()
        card_layout = QVBoxLayout(card)
        card_layout.setContentsMargins(32, 28, 32, 28)
        card_layout.setSpacing(6)

        title = QLabel("Welcome to Elemental Society Analysis")
        title.setObjectName("WelcomeTitle")
        subtitle = QLabel("A professional workflow for exploration geochemistry data")
        subtitle.setObjectName("WelcomeSubtitle")
        card_layout.addWidget(title)
        card_layout.addWidget(subtitle)
        card_layout.addSpacing(18)

        steps = [
            ("upload", "1. Upload", "Load a CSV or Excel dataset"),
            ("broom", "2. Clean", "Standardize headers and handle missing values"),
            ("chart", "3. Analyze", "Run a full exploratory statistical pass"),
            ("trend", "4. Visualize", "Distributions, correlations and PCA"),
            ("flask", "5. Geochemistry", "Alteration indices, ratios, grade statistics"),
            ("document", "6. Report", "Review and export the findings"),
        ]
        grid = QGridLayout()
        grid.setHorizontalSpacing(16)
        grid.setVerticalSpacing(14)
        for i, (icon_name, step_title, desc) in enumerate(steps):
            step_widget = QWidget()
            sl = QHBoxLayout(step_widget)
            sl.setContentsMargins(0, 0, 0, 0)
            icon_badge = QLabel()
            icon_badge.setObjectName("StepIcon")
            icon_badge.setFixedSize(36, 36)
            icon_badge.setAlignment(Qt.AlignCenter)
            icon_badge.setPixmap(Icons.get(icon_name, color=Palette.PRIMARY, size=18).pixmap(18, 18))
            text_box = QVBoxLayout()
            text_box.setSpacing(0)
            t = QLabel(step_title)
            t.setObjectName("StepTitle")
            d = QLabel(desc)
            d.setObjectName("StepDesc")
            d.setWordWrap(True)
            text_box.addWidget(t)
            text_box.addWidget(d)
            sl.addWidget(icon_badge)
            sl.addLayout(text_box, 1)
            grid.addWidget(step_widget, i // 2, i % 2)
        card_layout.addLayout(grid)

        card_layout.addSpacing(10)
        tip = QLabel("Tip: start by clicking <b>Upload</b> in the top bar.")
        tip.setObjectName("WelcomeTip")
        card_layout.addWidget(tip)

        outer.addWidget(card)
        outer.addStretch()
        return page

    def create_data_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        self.data_info = QLabel("No data loaded")
        self.data_info.setObjectName("MutedLabel")
        controls.addWidget(self.data_info)
        controls.addStretch()

        controls.addWidget(QLabel("Rows:"))
        self.row_limit = QSpinBox()
        self.row_limit.setRange(10, 5000)
        self.row_limit.setValue(500)
        self.row_limit.valueChanged.connect(self.refresh_data)
        controls.addWidget(self.row_limit)

        rename_btn = styled_button("Rename File", "pencil", "default")
        rename_btn.clicked.connect(self.rename_file)
        controls.addWidget(rename_btn)

        delete_btn = styled_button("Delete Selected", "trash", "danger")
        delete_btn.clicked.connect(self.delete_selected)
        controls.addWidget(delete_btn)

        layout.addLayout(controls)

        card = Card()
        card_layout = QVBoxLayout(card)
        card_layout.setContentsMargins(2, 2, 2, 2)
        self.data_table = QTableWidget()
        self.data_table.setAlternatingRowColors(True)
        self.data_table.setSortingEnabled(True)
        self.data_table.setSelectionBehavior(QTableWidget.SelectItems)
        self.data_table.setSelectionMode(QTableWidget.ExtendedSelection)
        card_layout.addWidget(self.data_table)
        layout.addWidget(card, 1)

        return page

    def create_eda_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        controls.addWidget(QLabel("Variable:"))
        self.eda_var = QComboBox()
        self.eda_var.currentTextChanged.connect(self.plot_distribution)
        controls.addWidget(self.eda_var)

        controls.addWidget(QLabel("Type:"))
        self.eda_type = QComboBox()
        self.eda_type.addItems(["Histogram", "PDF", "CDF", "Box Plot"])
        self.eda_type.currentTextChanged.connect(self.plot_distribution)
        controls.addWidget(self.eda_type)

        self.show_combined = QCheckBox("Show combined view")
        self.show_combined.toggled.connect(self.plot_distribution)
        controls.addWidget(self.show_combined)
        controls.addStretch()
        layout.addLayout(controls)

        card = Card()
        card_layout = QVBoxLayout(card)
        self.eda_figure = Figure(figsize=(10, 6))
        self.eda_canvas = FigureCanvas(self.eda_figure)
        card_layout.addWidget(self.eda_canvas)
        layout.addWidget(card, 1)

        return page

    def create_analyze_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        controls.addWidget(QLabel("Plot:"))
        self.plot_type = QComboBox()
        self.plot_type.addItems(["Correlation Matrix", "PCA Scores", "Scatter Plot", "All Plots Combined"])
        self.plot_type.currentTextChanged.connect(self.update_plot)
        controls.addWidget(self.plot_type)

        controls.addWidget(QLabel("X:"))
        self.analyze_x = QComboBox()
        controls.addWidget(self.analyze_x)

        controls.addWidget(QLabel("Y:"))
        self.analyze_y = QComboBox()
        controls.addWidget(self.analyze_y)

        plot_btn = styled_button("Generate", "trend", "primary")
        plot_btn.clicked.connect(self.update_plot)
        controls.addWidget(plot_btn)
        controls.addStretch()
        layout.addLayout(controls)

        card = Card()
        card_layout = QVBoxLayout(card)
        self.analyze_figure = Figure(figsize=(10, 6))
        self.analyze_canvas = FigureCanvas(self.analyze_figure)
        card_layout.addWidget(self.analyze_canvas)
        layout.addWidget(card, 1)

        return page

    def create_geochemistry_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)

        card = Card()
        card_layout = QVBoxLayout(card)
        card_layout.setContentsMargins(2, 2, 2, 2)
        self.geochem_viewer = QTextEdit()
        self.geochem_viewer.setReadOnly(True)
        self.geochem_viewer.setFont(QFont("Segoe UI", 10))
        self.geochem_viewer.setFrameShape(QFrame.NoFrame)
        card_layout.addWidget(self.geochem_viewer)
        layout.addWidget(card, 1)
        return page

    def create_log_page(self):
        page = QWidget()
        layout = QVBoxLayout(page)
        layout.setContentsMargins(24, 20, 24, 24)
        layout.setSpacing(12)

        controls = QHBoxLayout()
        controls.addStretch()
        clear_btn = styled_button("Clear", "trash", "danger")
        clear_btn.clicked.connect(self.clear_log)
        controls.addWidget(clear_btn)
        layout.addLayout(controls)

        card = Card()
        card_layout = QVBoxLayout(card)
        card_layout.setContentsMargins(2, 2, 2, 2)
        self.log_text = QTextEdit()
        self.log_text.setReadOnly(True)
        self.log_text.setFont(QFont("Consolas", 10))
        self.log_text.setFrameShape(QFrame.NoFrame)
        card_layout.addWidget(self.log_text)
        layout.addWidget(card, 1)

        return page

    # ============================================================
    # LOG FUNCTIONS
    # ============================================================

    def log_message(self, message):
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.append(f"[{timestamp}]  {message}")

    def clear_log(self):
        self.log_text.clear()
        self.log_message("Log cleared")

    # ============================================================
    # DATA FUNCTIONS
    # ============================================================

    def rename_file(self):
        if self.original_filename is None:
            QMessageBox.warning(self, "No File", "No file is currently loaded.")
            return

        new_name, ok = QInputDialog.getText(
            self, "Rename File", "Enter new filename:", QLineEdit.Normal, self.original_filename)

        if ok and new_name:
            self.original_filename = new_name
            self.file_info_label.setText(new_name)
            self.log_message(f"Renamed file to: {new_name}")
            self.set_status('success', f'Renamed to {new_name}')

    def delete_selected(self):
        if self.df is None:
            QMessageBox.warning(self, "No Data", "Please load data first.")
            return

        selected = self.data_table.selectedIndexes()
        if not selected:
            QMessageBox.warning(self, "Nothing Selected", "Select one or more cells first.")
            return

        rows = set(index.row() for index in selected)
        cols = set(index.column() for index in selected)

        reply = QMessageBox.question(
            self, "Delete Selection",
            f"Delete {len(rows)} row(s) and {len(cols)} column(s)?",
            QMessageBox.Yes | QMessageBox.No)

        if reply == QMessageBox.Yes:
            for col in sorted(cols, reverse=True):
                self.df = self.df.drop(self.df.columns[col], axis=1)
            for row in sorted(rows, reverse=True):
                self.df = self.df.drop(self.df.index[row])

            self.df = self.df.reset_index(drop=True)
            self.display_data()
            self.log_message(f"Deleted {len(rows)} row(s) and {len(cols)} column(s)")
            self.set_status('success', f'Deleted {len(rows)} rows, {len(cols)} columns')

    def upload_file(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open Geochemical Data", "",
            "All Files (*);;CSV (*.csv);;Excel (*.xlsx *.xls);;Text (*.txt)"
        )
        if not file_path:
            return

        self.original_filename = os.path.basename(file_path)
        self.set_processing(True, "Loading file…")
        self.log_message(f"Loading: {self.original_filename}")
        self.update_progress(f"Reading {self.original_filename}")

        try:
            self.df = read_any_file(file_path)

            if self.df is not None and not self.df.empty:
                self.df = clean_dataframe(self.df)
                self.cleaned_df = self.df.copy()
                self.derived_df = self.df.copy()
                self.display_data()
                self.file_info_label.setText(self.original_filename)

                numeric_cols = self.df.select_dtypes(include=[np.number]).columns
                self.eda_var.clear()
                self.eda_var.addItems(numeric_cols)
                self.analyze_x.clear()
                self.analyze_x.addItems(numeric_cols)
                self.analyze_y.clear()
                self.analyze_y.addItems(numeric_cols)

                self.log_message(f"Loaded {self.df.shape[0]} rows, {self.df.shape[1]} columns")
                self.set_status('success', f'Loaded {self.df.shape[0]} rows')

                QMessageBox.information(
                    self, "File Loaded",
                    f"{self.df.shape[0]} rows × {self.df.shape[1]} columns loaded successfully.")
            else:
                QMessageBox.warning(self, "Read Error", "Could not read the selected file.")

        except Exception as e:
            self.log_message(f"Error: {str(e)}")
            self.set_status('error', 'Load failed')
            QMessageBox.critical(self, "Error", f"Failed to load file:\n{str(e)}")
        finally:
            self.set_processing(False)

    def clean_data(self):
        if self.df is None or self.df.empty:
            QMessageBox.warning(self, "No Data", "Please load data first.")
            return

        reply = QMessageBox.question(self, "Clean Data", "Clean the current dataset?", QMessageBox.Yes | QMessageBox.No)
        if reply != QMessageBox.Yes:
            return

        self.set_processing(True, "Cleaning data…")
        self.log_message("Cleaning data...")
        self.update_progress("Cleaning data...")

        try:
            rows_before = len(self.df)
            new_cleaned_df = clean_dataframe(self.df.copy())
            rows_after = len(new_cleaned_df)

            self.cleaned_df = new_cleaned_df
            self.derived_df = new_cleaned_df.copy()
            self.df = new_cleaned_df.copy()
            self.display_data()

            self.log_message(f"Cleaned dataset: {rows_after} rows remain")
            self.set_status('success', f'Cleaned: {rows_after} rows')

            QMessageBox.information(
                self, "Cleaning Complete",
                f"Before: {rows_before} rows\nAfter: {rows_after} rows")

        except Exception as e:
            self.log_message(f"Cleaning error: {str(e)}")
            self.set_status('error', 'Cleaning failed')
            QMessageBox.critical(self, "Error", f"Cleaning failed:\n{str(e)}")
        finally:
            self.set_processing(False)

    def run_analysis(self):
        if self.df is None:
            QMessageBox.warning(self, "No Data", "Please load data first.")
            return

        self.set_processing(True, "Running analysis…")
        self.log_message("Starting analysis...")
        self.update_progress("Running analysis...")

        try:
            self.update_progress("Generating statistics...")
            self.generate_statistics()

            self.update_progress("Creating visualizations...")
            self.plot_distribution()
            self.update_plot()

            self.update_progress("Calculating geochemistry...")
            self.calculate_geochemistry()

            self.log_message("Analysis complete")
            self.set_status('success', 'Analysis complete')

            QMessageBox.information(self, "Analysis Complete", "The analysis finished successfully.")

        except Exception as e:
            self.log_message(f"Analysis error: {str(e)}")
            self.set_status('error', 'Analysis failed')
            QMessageBox.critical(self, "Error", f"Analysis failed:\n{str(e)}")
        finally:
            self.set_processing(False)

    def display_data(self):
        if self.df is None:
            return

        display_limit = min(self.row_limit.value(), len(self.df))
        self.data_table.setRowCount(display_limit)
        self.data_table.setColumnCount(len(self.df.columns))
        self.data_table.setHorizontalHeaderLabels(self.df.columns.astype(str))

        for i in range(display_limit):
            for j, col in enumerate(self.df.columns):
                val = self.df.iloc[i, j]
                item = QTableWidgetItem("" if pd.isna(val) else str(val))
                self.data_table.setItem(i, j, item)

        self.data_table.resizeColumnsToContents()
        self.data_info.setText(f"{len(self.df)} rows × {len(self.df.columns)} columns")

    def refresh_data(self):
        self.display_data()

    # ============================================================
    # STATISTICS FUNCTIONS
    # ============================================================

    def generate_statistics(self):
        if self.df is None:
            return

        numeric_df = self.df.select_dtypes(include=[np.number])
        if numeric_df.empty:
            return

        lines = []
        lines.append(f"Dataset: {len(self.df)} rows × {len(self.df.columns)} columns")
        lines.append(f"Numeric columns: {len(numeric_df.columns)}\n")

        lines.append("BASIC STATISTICS")
        lines.append("-" * 60)
        lines.append(numeric_df.describe().to_string())
        lines.append("")

        lines.append("VARIANCE")
        lines.append("-" * 60)
        lines.append(numeric_df.var().to_string())
        lines.append("")

        lines.append("SKEWNESS")
        lines.append("-" * 60)
        lines.append(numeric_df.skew().to_string())
        lines.append("")

        lines.append("KURTOSIS")
        lines.append("-" * 60)
        lines.append(numeric_df.kurtosis().to_string())

        self.log_message("Statistics generated")

    # ============================================================
    # PLOTTING FUNCTIONS
    # ============================================================

    def plot_distribution(self):
        if self.df is None:
            return

        var = self.eda_var.currentText()
        plot_type = self.eda_type.currentText()
        show_combined = self.show_combined.isChecked()

        if not var or var not in self.df.columns:
            return

        data = self.df[var].dropna()
        if len(data) < 2:
            return

        self.eda_figure.clear()

        if show_combined:
            axes = [self.eda_figure.add_subplot(2, 2, i + 1) for i in range(4)]
            plot_types = ["Histogram", "PDF", "CDF", "Box Plot"]
        else:
            axes = [self.eda_figure.add_subplot(111)]
            plot_types = [plot_type]

        accent = Palette.PRIMARY

        for ax, ptype in zip(axes, plot_types):
            if ptype == "Histogram":
                ax.hist(data, bins=20, density=True, alpha=0.75, color=accent, edgecolor='white')
                ax.axvline(data.mean(), color=Palette.DANGER, linestyle='--', label=f'Mean: {data.mean():.2f}')
                ax.axvline(data.median(), color=Palette.SUCCESS, linestyle='--', label=f'Median: {data.median():.2f}')
                ax.set_ylabel('Density')
                ax.legend(frameon=False, fontsize=8)

            elif ptype == "PDF":
                ax.hist(data, bins=20, density=True, alpha=0.35, color=accent)
                x = np.linspace(data.min(), data.max(), 100)
                mu, std = scipy_stats.norm.fit(data)
                pdf = scipy_stats.norm.pdf(x, mu, std)
                ax.plot(x, pdf, color=Palette.DANGER, linewidth=2, label='Normal fit')
                ax.set_ylabel('Density')
                ax.legend(frameon=False, fontsize=8)

            elif ptype == "CDF":
                sorted_data = np.sort(data)
                y = np.arange(1, len(sorted_data) + 1) / len(sorted_data)
                ax.plot(sorted_data, y, color=accent, linewidth=2)
                ax.set_ylabel('Cumulative Probability')

            elif ptype == "Box Plot":
                bp = ax.boxplot(data, vert=True, patch_artist=True)
                for box in bp['boxes']:
                    box.set(facecolor=accent, alpha=0.6)

            ax.set_title(f'{var} — {ptype}', fontweight='bold', fontsize=10)
            ax.set_xlabel(var if ptype != "Box Plot" else '')
            ax.grid(True, alpha=0.25)
            for spine in ('top', 'right'):
                ax.spines[spine].set_visible(False)

        self.eda_figure.tight_layout()
        self.eda_canvas.draw()

    def update_plot(self):
        if self.df is None:
            return

        plot_type = self.plot_type.currentText()
        self.analyze_figure.clear()
        numeric_df = self.df.select_dtypes(include=[np.number])

        def draw_correlation(ax):
            if len(numeric_df.columns) > 1:
                corr = numeric_df.corr()
                sns.heatmap(corr, annot=True, cmap='coolwarm', center=0,
                            fmt='.2f', square=True, linewidths=0.5, ax=ax, cbar=False)
                ax.set_title('Correlation Matrix', fontweight='bold', fontsize=10)

        def draw_pca(ax):
            if len(numeric_df.columns) >= 2:
                clean_data = numeric_df.dropna()
                if len(clean_data) > 1:
                    scaler = StandardScaler()
                    scaled = scaler.fit_transform(clean_data)
                    pca = SklearnPCA()
                    scores = pca.fit_transform(scaled)
                    explained = pca.explained_variance_ratio_
                    ax.scatter(scores[:, 0], scores[:, 1],
                               c=np.linalg.norm(scores[:, :2], axis=1),
                               cmap='viridis', alpha=0.7, edgecolors='white', linewidth=0.5)
                    ax.set_xlabel(f'PC1 ({explained[0] * 100:.1f}%)')
                    ax.set_ylabel(f'PC2 ({explained[1] * 100:.1f}%)')
                    ax.set_title('PCA Scores', fontweight='bold', fontsize=10)
                    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.4)
                    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.4)

        def draw_scatter(ax):
            x_col = self.analyze_x.currentText()
            y_col = self.analyze_y.currentText()
            if x_col and y_col and x_col in self.df.columns and y_col in self.df.columns:
                data = self.df[[x_col, y_col]].dropna()
                ax.scatter(data[x_col], data[y_col], alpha=0.7, color=Palette.PRIMARY, edgecolors='white')
                ax.set_xlabel(x_col)
                ax.set_ylabel(y_col)
                ax.set_title('Scatter Plot', fontweight='bold', fontsize=10)

        def draw_distribution(ax):
            if len(numeric_df.columns) > 0:
                col = numeric_df.columns[0]
                data = numeric_df[col].dropna()
                if len(data) > 1:
                    ax.hist(data, bins=20, density=True, alpha=0.75, color=Palette.PRIMARY, edgecolor='white')
                    ax.set_title(f'Distribution: {col}', fontweight='bold', fontsize=10)
                    ax.set_xlabel(col)
                    ax.set_ylabel('Density')

        if plot_type == "All Plots Combined":
            ax1 = self.analyze_figure.add_subplot(2, 2, 1)
            ax2 = self.analyze_figure.add_subplot(2, 2, 2)
            ax3 = self.analyze_figure.add_subplot(2, 2, 3)
            ax4 = self.analyze_figure.add_subplot(2, 2, 4)
            draw_correlation(ax1)
            draw_pca(ax2)
            draw_scatter(ax3)
            draw_distribution(ax4)
            for ax in (ax2, ax3, ax4):
                ax.grid(True, alpha=0.25)
        elif plot_type == "Correlation Matrix":
            draw_correlation(self.analyze_figure.add_subplot(111))
        elif plot_type == "PCA Scores":
            ax = self.analyze_figure.add_subplot(111)
            draw_pca(ax)
            ax.grid(True, alpha=0.25)
        elif plot_type == "Scatter Plot":
            ax = self.analyze_figure.add_subplot(111)
            draw_scatter(ax)
            ax.grid(True, alpha=0.25)

        self.analyze_figure.tight_layout()
        self.analyze_canvas.draw()

    # ============================================================
    # GEOCHEMISTRY FUNCTIONS
    # ============================================================

    def calculate_geochemistry(self):
        if self.df is None:
            return

        geochem = GeochemistryEngine.calculate_all(self.df)

        html = []
        html.append("<html><head><style>")
        html.append(f"""
            body {{ font-family: 'Segoe UI', Arial, sans-serif; margin: 4px; color: {Palette.TEXT_PRIMARY}; }}
            h1 {{ color: {Palette.PRIMARY}; border-bottom: 2px solid {Palette.PRIMARY}; padding-bottom: 8px; font-size: 20px; }}
            h2 {{ color: {Palette.TEXT_PRIMARY}; margin-top: 22px; font-size: 15px; }}
            .card {{ background-color: {Palette.BG}; border-radius: 8px; padding: 15px; margin: 10px 0; border-left: 4px solid {Palette.PRIMARY}; }}
            .stat-box {{ display: inline-block; margin: 3px; padding: 8px 15px; background: white; border-radius: 6px; border: 1px solid {Palette.BORDER}; }}
            .stat-value {{ font-weight: bold; color: {Palette.PRIMARY}; font-size: 16px; }}
            .stat-label {{ font-size: 10px; color: {Palette.TEXT_SECONDARY}; }}
            .badge {{ display: inline-block; padding: 2px 10px; border-radius: 10px; font-size: 11px; font-weight: 600; }}
            .badge-high {{ background: {Palette.DANGER_LIGHT}; color: {Palette.DANGER}; }}
            .badge-medium {{ background: {Palette.WARNING_LIGHT}; color: {Palette.WARNING}; }}
            .badge-low {{ background: {Palette.SUCCESS_LIGHT}; color: {Palette.SUCCESS}; }}
            .badge-available {{ background: {Palette.SUCCESS_LIGHT}; color: {Palette.SUCCESS}; }}
            .badge-missing {{ background: {Palette.DANGER_LIGHT}; color: {Palette.DANGER}; }}
        """)
        html.append("</style></head><body>")

        html.append("<h1>Exploration Geochemistry Report</h1>")
        html.append(f"<p><b>Generated:</b> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>")
        html.append(f"<p><b>Dataset:</b> {len(self.df)} samples × {len(self.df.columns)} variables</p>")

        avail = geochem.get('available', {})
        html.append("<div class='card'>")
        html.append("<h3>Available Calculations</h3>")
        for key, val in avail.items():
            badge = "badge-available" if val else "badge-missing"
            status = "Available" if val else "Not Available"
            html.append(f"<span class='badge {badge}'>{key.replace('_', ' ').title()}: {status}</span> ")
        html.append("</div>")

        html.append("<h2>Alteration Indices</h2>")
        if geochem['alteration_indices']:
            html.append("<div class='card'>")
            for name, values in geochem['alteration_indices'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{values.min():.4f} - {values.max():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No alteration indices could be calculated.</p></div>")

        html.append("<h2>Element Ratios</h2>")
        if geochem['element_ratios']:
            html.append("<div class='card'>")
            for name, values in geochem['element_ratios'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No element ratios could be calculated.</p></div>")

        html.append("<h2>Mineralization Indices</h2>")
        if geochem['mineralization_indices']:
            html.append("<div class='card'>")
            for name, values in geochem['mineralization_indices'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No mineralization indices could be calculated.</p></div>")

        html.append("<h2>Grade Statistics</h2>")
        if geochem['grade_stats']:
            html.append("<div class='card'>")
            for element, elem_stats in geochem['grade_stats'].items():
                if elem_stats:
                    grade_class = elem_stats.get('Grade Class', 'LOW')
                    badge_class = 'badge-high' if grade_class == 'HIGH' else 'badge-medium' if grade_class == 'MEDIUM' else 'badge-low'
                    html.append(f"""
                    <div style='margin: 10px 0; padding: 10px; background: white; border-radius: 6px; border: 1px solid {Palette.BORDER};'>
                        <b>{element}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{elem_stats.get('Mean', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{elem_stats.get('Median', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Std Dev</div><div class='stat-value'>{elem_stats.get('Std Dev', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{elem_stats.get('Minimum', 0):.4f} - {elem_stats.get('Maximum', 0):.4f}</div></div>
                        <div style='margin-top: 5px;'>
                            <span class='badge {badge_class}'>Grade: {grade_class}</span>
                            <span style='margin-left: 10px;'>Threshold: {elem_stats.get('Threshold', 0):.4f}</span>
                            <span style='margin-left: 10px;'>Above: {elem_stats.get('Above Threshold', 0)} ({elem_stats.get('% Above Threshold', 0):.1f}%)</span>
                        </div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append(f"<div class='card'><p style='color:{Palette.TEXT_SECONDARY};'>No grade statistics could be calculated.</p></div>")

        html.append(f"""
        <div style='margin-top: 20px; padding: 15px; background-color: {Palette.PRIMARY_LIGHT}; border-radius: 8px;'>
            <p><b>Summary</b></p>
            <table style='width: auto;'>
        """)
        html.append(f"<tr><td>Total Samples</td><td><b>{len(self.df)}</b></td></tr>")
        html.append(f"<tr><td>Total Variables</td><td><b>{len(self.df.columns)}</b></td></tr>")
        html.append(f"<tr><td>Alteration Indices</td><td><b>{len(geochem['alteration_indices'])}</b></td></tr>")
        html.append(f"<tr><td>Element Ratios</td><td><b>{len(geochem['element_ratios'])}</b></td></tr>")
        html.append(f"<tr><td>Mineralization Indices</td><td><b>{len(geochem['mineralization_indices'])}</b></td></tr>")
        html.append(f"<tr><td>Grade Statistics</td><td><b>{len(geochem['grade_stats'])}</b></td></tr>")
        html.append("</table></div></body></html>")

        self.geochem_viewer.setHtml("\n".join(html))

    def show_report(self):
        self.nav_list.setCurrentRow(4)  # Geochemistry page

    # ============================================================
    # EXPORT FUNCTIONS
    # ============================================================

    def export_data(self):
        if self.df is None:
            QMessageBox.warning(self, "No Data", "There is no data to export.")
            return

        df = self.df

        formats = {
            "CSV (;)": "csv_semicolon",
            "CSV (,)": "csv_comma",
            "Excel (.xlsx)": "excel",
            "JSON": "json"
        }

        format_choice, ok = QInputDialog.getItem(
            self, "Export Format", "Select format:", list(formats.keys()), 0, False)
        if not ok:
            return

        file_ext_map = {
            "csv_semicolon": "csv",
            "csv_comma": "csv",
            "excel": "xlsx",
            "json": "json"
        }

        file_path, _ = QFileDialog.getSaveFileName(
            self, "Export Data", f"cleaned_data.{file_ext_map[formats[format_choice]]}",
            f"{format_choice} (*.{file_ext_map[formats[format_choice]]})")

        if file_path:
            try:
                self.set_processing(True, "Exporting…")
                self.update_progress(f"Exporting to {format_choice}...")

                fmt = formats[format_choice]
                if fmt == "csv_semicolon":
                    df.to_csv(file_path, sep=';', index=False, encoding='utf-8-sig')
                elif fmt == "csv_comma":
                    df.to_csv(file_path, sep=',', index=False, encoding='utf-8-sig')
                elif fmt == "excel":
                    df.to_excel(file_path, index=False)
                elif fmt == "json":
                    df.to_json(file_path, orient='records', indent=2)

                self.log_message(f"Exported: {file_path}")
                self.set_status('success', 'Export complete')
                QMessageBox.information(self, "Export Complete", f"Data exported to:\n{file_path}")
            except Exception as e:
                self.set_status('error', 'Export failed')
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")
            finally:
                self.set_processing(False)

    # ============================================================
    # HELP AND ABOUT
    # ============================================================

    def show_help(self):
        help_text = f"""{APP_NAME} v{APP_VERSION}

Project Code: {APP_CODE}
Developer: {DEVELOPER}

QUICK START
1. Upload your data
2. Clean the data
3. Run analysis
4. Review visualizations
5. Check geochemistry
6. Generate report

FEATURES

Data Management
• Upload CSV/Excel files
• Clean data (missing values, outliers)
• Rename file
• Delete selected rows/columns

Statistics
• Mean, Median, Std Dev
• Variance, Skewness, Kurtosis
• Correlation Matrix

Visualization
• Correlation Heatmap
• PCA Scores
• Scatter Plot
• Combined view
• EDA: Histogram, PDF, CDF, Box Plot

Geochemistry
• Alteration Indices
• Element Ratios
• Mineralization Indices
• Grade Statistics

Export
• CSV, Excel, JSON

Copyright © 2026 {DEVELOPER}
"""
        QMessageBox.information(self, "Help", help_text)

    def show_about(self):
        about_text = f"""{APP_NAME}

Version: {APP_VERSION}
Project Code: {APP_CODE}

Developer: {DEVELOPER}
Email: {EMAIL}
{BACKGROUND}

Professional geochemical data cleaning and
exploratory data analysis suite for mineral
exploration and geochemistry.

Copyright © 2026 {DEVELOPER}
All rights reserved.
"""
        QMessageBox.about(self, "About", about_text)

    # --------------------------------------------------------
    # STYLESHEET
    # --------------------------------------------------------

    def get_stylesheet(self):
        P = Palette
        return f"""
            QMainWindow {{ background-color: {P.BG}; }}
            QWidget {{ font-family: 'Segoe UI', Arial, sans-serif; font-size: 13px; color: {P.TEXT_PRIMARY}; }}

            /* -- Sidebar -- */
            #Sidebar {{ background-color: {P.SIDEBAR_BG}; }}
            #Brand {{ border-bottom: 1px solid #1F2937; }}
            #BrandTitle {{ color: {P.SIDEBAR_TEXT_ACTIVE}; font-size: 15px; font-weight: 700; }}
            #BrandSubtitle {{ color: {P.SIDEBAR_TEXT}; font-size: 11px; }}
            #NavList {{
                background-color: transparent;
                border: none;
                outline: none;
                padding: 8px;
            }}
            #NavList::item {{
                color: {P.SIDEBAR_TEXT};
                border-radius: 8px;
                padding-left: 10px;
                margin: 2px 4px;
            }}
            #NavList::item:hover {{ background-color: {P.SIDEBAR_BG_HOVER}; color: {P.SIDEBAR_TEXT_ACTIVE}; }}
            #NavList::item:selected {{ background-color: {P.PRIMARY}; color: {P.SIDEBAR_TEXT_ACTIVE}; }}
            #SidebarFooter {{ border-top: 1px solid #1F2937; }}
            #SidebarFooterName {{ color: {P.SIDEBAR_TEXT_ACTIVE}; font-size: 11px; font-weight: 600; }}
            #SidebarFooterRole {{ color: {P.SIDEBAR_TEXT}; font-size: 10px; }}

            /* -- Top bar / status -- */
            #TopBar {{ background-color: {P.SURFACE}; border-bottom: 1px solid {P.BORDER}; }}
            #PageTitle {{ font-size: 17px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}
            #FileInfo {{ color: {P.TEXT_SECONDARY}; font-size: 12px; padding-left: 6px; }}
            #StatusPanel {{ background-color: {P.SURFACE}; border-bottom: 1px solid {P.BORDER}; }}
            #StatusText {{ color: {P.TEXT_SECONDARY}; font-size: 12px; }}
            #ProcessInfo {{ color: {P.TEXT_MUTED}; font-size: 11px; }}
            #StatusProgress {{
                border: none; border-radius: 3px; background-color: {P.BORDER}; height: 6px;
            }}
            #StatusProgress::chunk {{ background-color: {P.PRIMARY}; border-radius: 3px; }}

            /* -- Cards -- */
            #Card {{ background-color: {P.SURFACE}; border-radius: 12px; border: 1px solid {P.BORDER}; }}
            #StatBadge {{ background-color: {P.BG}; border-radius: 6px; }}
            #StatBadgeLabel {{ color: {P.TEXT_MUTED}; font-size: 9px; font-weight: 700; letter-spacing: 0.5px; }}
            #StatBadgeValue {{ color: {P.PRIMARY}; font-size: 15px; font-weight: 700; }}
            #SectionTitle {{ font-size: 14px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}

            /* -- Welcome page -- */
            #WelcomeTitle {{ font-size: 22px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}
            #WelcomeSubtitle {{ font-size: 13px; color: {P.TEXT_SECONDARY}; }}
            #StepIcon {{ background-color: {P.PRIMARY_LIGHT}; border-radius: 8px; }}
            #StepTitle {{ font-size: 13px; font-weight: 700; color: {P.TEXT_PRIMARY}; }}
            #StepDesc {{ font-size: 11px; color: {P.TEXT_SECONDARY}; }}
            #WelcomeTip {{ color: {P.TEXT_SECONDARY}; font-size: 12px; }}
            #MutedLabel {{ color: {P.TEXT_SECONDARY}; }}

            /* -- Buttons -- */
            QPushButton {{
                background-color: {P.SURFACE};
                color: {P.TEXT_PRIMARY};
                padding: 8px 16px;
                border: 1px solid {P.BORDER};
                border-radius: 6px;
                font-weight: 600;
                font-size: 12px;
            }}
            QPushButton:hover {{ background-color: {P.BG}; }}
            QPushButton[kind="primary"] {{ background-color: {P.PRIMARY}; color: white; border: none; }}
            QPushButton[kind="primary"]:hover {{ background-color: {P.PRIMARY_DARK}; }}
            QPushButton[kind="success"] {{ background-color: {P.SUCCESS}; color: white; border: none; }}
            QPushButton[kind="success"]:hover {{ background-color: #047857; }}
            QPushButton[kind="warning"] {{ background-color: {P.WARNING}; color: white; border: none; }}
            QPushButton[kind="warning"]:hover {{ background-color: #B45309; }}
            QPushButton[kind="danger"] {{ background-color: {P.DANGER}; color: white; border: none; }}
            QPushButton[kind="danger"]:hover {{ background-color: #B91C1C; }}
            QPushButton[kind="ghost"] {{ background-color: transparent; border: none; }}
            QPushButton[kind="ghost"]:hover {{ background-color: {P.BG}; }}

            /* -- Inputs -- */
            QComboBox, QSpinBox, QLineEdit {{
                padding: 6px 10px;
                border: 1px solid {P.BORDER};
                border-radius: 6px;
                background-color: {P.SURFACE};
                min-width: 110px;
            }}
            QCheckBox {{ spacing: 6px; }}

            /* -- Tables -- */
            QTableWidget {{
                background-color: {P.SURFACE};
                gridline-color: {P.BORDER};
                border: none;
                border-radius: 10px;
                alternate-background-color: {P.BG};
            }}
            QTableWidget::item {{ padding: 6px; }}
            QTableWidget::item:selected {{ background-color: {P.PRIMARY_LIGHT}; color: {P.PRIMARY}; }}
            QHeaderView::section {{
                background-color: {P.BG};
                padding: 10px;
                border: none;
                border-bottom: 2px solid {P.BORDER};
                font-weight: 700;
                color: {P.TEXT_SECONDARY};
                font-size: 11px;
            }}

            QTextEdit {{
                background-color: {P.SURFACE};
                border: none;
                padding: 12px;
            }}

            QStatusBar {{ background-color: {P.SURFACE}; color: {P.TEXT_SECONDARY}; border-top: 1px solid {P.BORDER}; }}
        """


# ============================================================
# MAIN APPLICATION
# ============================================================

def main():
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    app.setFont(QFont("Segoe UI", 10))
    window = ElementalSocietyAnalysis()
    window.show()
    sys.exit(app.exec_())


if __name__ == '__main__':
    main()

In [ ]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - PROFESSIONAL EDITION v1.0
# Project Code: 1010
# Developer: Mohamed Almoufty Abdoalmoutleb Abaker
# ============================================================

import sys
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar
from matplotlib.figure import Figure
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# APPLICATION METADATA
# ============================================================

APP_NAME = "Elemental Society Analysis"
APP_VERSION = "1.0.0"
APP_CODE = "1010"
DEVELOPER = "Mohamed Almoufty Abdoalmoutleb Abaker"
EMAIL = "almoufty121@gmail.com"
BACKGROUND = "Exploration Geologist | Msc Earth Science Engineering"

# ============================================================
# ELEMENT ALIAS MAPPING
# ============================================================

ELEMENT_ALIAS_MAP = {
    'K': 'K_pct', 'K2O': 'K_pct', 'K_PCT': 'K_pct', 'K%': 'K_pct',
    'MG': 'Mg_pct', 'MGO': 'Mg_pct', 'MG_PCT': 'Mg_pct', 'MG%': 'Mg_pct',
    'CA': 'Ca_pct', 'CAO': 'Ca_pct', 'CA_PCT': 'Ca_pct', 'CA%': 'Ca_pct',
    'NA': 'Na_pct', 'NA2O': 'Na_pct', 'NA_PCT': 'Na_pct', 'NA%': 'Na_pct',
    'FE': 'Fe_pct', 'FE2O3': 'Fe_pct', 'FEO': 'Fe_pct', 'FE_PCT': 'Fe_pct', 'FE%': 'Fe_pct',
    'AU': 'Au_ppm', 'AU_PPM': 'Au_ppm', 'AU_PPB': 'Au_ppm',
    'CU': 'Cu_ppm', 'CU_PPM': 'Cu_ppm', 'CU_PCT': 'Cu_ppm',
    'ZN': 'Zn_ppm', 'ZN_PPM': 'Zn_ppm',
    'PB': 'Pb_ppm', 'PB_PPM': 'Pb_ppm',
    'AG': 'Ag_ppm', 'AG_PPM': 'Ag_ppm',
    'AS': 'As_ppm', 'AS_PPM': 'As_ppm',
    'SB': 'Sb_ppm', 'SB_PPM': 'Sb_ppm',
    'MO': 'Mo_ppm', 'MO_PPM': 'Mo_ppm',
    'SIO2': 'SiO2_pct', 'SIO2_PCT': 'SiO2_pct', 'SIO2%': 'SiO2_pct',
    'AL': 'Al_pct', 'AL2O3': 'Al_pct', 'AL_PCT': 'Al_pct', 'AL%': 'Al_pct',
}

# ============================================================
# DATA CLEANING ENGINE
# ============================================================

def standardize_element_names(df):
    """Normalize chemical headers to canonical standard names."""
    renames = {}
    for col in df.columns:
        clean_col = re.sub(r'[^a-zA-Z0-9%_]', '', str(col)).upper()
        if clean_col in ELEMENT_ALIAS_MAP:
            renames[col] = ELEMENT_ALIAS_MAP[clean_col]
        else:
            for key, std in ELEMENT_ALIAS_MAP.items():
                if key in clean_col:
                    renames[col] = std
                    break
    return df.rename(columns=renames)

def clean_dataframe(df):
    """Universal Cleaning Function for Geochemical Datasets."""
    if df is None or df.empty:
        return df
    
    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]
    df = standardize_element_names(df)

    missing_vals = ['', 'nan', 'NaN', 'null', 'None', 'NA', '-', '--', 'n/a', '?']
    df = df.replace(missing_vals, np.nan)

    text_cols = ['sample', 'id', 'point', 'lithology', 'comments', 'unit', 'rock_type']
    for col in df.columns:
        is_text = any(t in col.lower() for t in text_cols)
        if not is_text:
            cleaned_series = df[col].astype(str).str.replace(r'^[<>]', '', regex=True)
            cleaned_series = pd.to_numeric(cleaned_series.str.replace(',', '.'), errors='coerce')
            if cleaned_series.notna().sum() > 0:
                df[col] = cleaned_series

    df = df.dropna(how='all')
    return df

def read_any_file(path):
    """Read any file format with auto-detection."""
    ext = Path(path).suffix.lower()
    if ext in ['.xlsx', '.xls', '.xlsm']:
        return pd.read_excel(path)
    
    encodings = ["utf-8", "cp1252", "latin-1", "iso-8859-1", "utf-16"]
    seps = [";", ",", "\t", "|"]
    
    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, engine="python", on_bad_lines="skip")
                if df.shape[1] > 1:
                    return df
            except Exception:
                pass
    
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read(3000)
        sep = max([";", ",", "\t", "|"], key=lambda d: txt.count(d))
        return pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")
    except Exception:
        return pd.read_csv(path, engine="python", on_bad_lines="skip")

# ============================================================
# GEOCHEMISTRY ENGINE
# ============================================================

class GeochemistryEngine:
    """Smart geochemistry calculations with auto-detection"""
    
    @staticmethod
    def calculate_all(df):
        results = {
            'alteration_indices': {},
            'element_ratios': {},
            'mineralization_indices': {},
            'grade_stats': {},
            'available': {'alteration': False, 'ratios': False, 'mineralization': False, 'grades': False}
        }
        
        # 1. Alteration Indices
        if 'K_pct' in df.columns and 'Mg_pct' in df.columns and 'Ca_pct' in df.columns and 'Na_pct' in df.columns:
            try:
                results['alteration_indices']['Ishikawa AI'] = 100 * (df['K_pct'] + df['Mg_pct']) / (
                    df['K_pct'] + df['Mg_pct'] + df['Ca_pct'] + df['Na_pct'] + 0.0001)
                results['available']['alteration'] = True
            except: pass
        
        if 'Fe_pct' in df.columns and 'Mg_pct' in df.columns and 'Na_pct' in df.columns and 'K_pct' in df.columns:
            try:
                results['alteration_indices']['CCPI'] = 100 * (df['Fe_pct'] + df['Mg_pct']) / (
                    df['Fe_pct'] + df['Mg_pct'] + df['Na_pct'] + df['K_pct'] + 0.0001)
                results['available']['alteration'] = True
            except: pass
        
        if 'K_pct' in df.columns and 'Na_pct' in df.columns:
            try:
                results['alteration_indices']['Sericite Index'] = df['K_pct'] / (df['K_pct'] + df['Na_pct'] + 0.0001)
                results['available']['alteration'] = True
            except: pass
        
        # 2. Element Ratios
        ratio_pairs = [
            ('Au_ppm', 'Cu_ppm', 'Au/Cu'), ('Zn_ppm', 'Pb_ppm', 'Zn/Pb'),
            ('Cu_ppm', 'Zn_ppm', 'Cu/Zn'), ('Au_ppm', 'Ag_ppm', 'Au/Ag'),
            ('Au_ppm', 'As_ppm', 'Au/As'), ('Cu_ppm', 'Mo_ppm', 'Cu/Mo')
        ]
        for c1, c2, name in ratio_pairs:
            if c1 in df.columns and c2 in df.columns:
                try:
                    results['element_ratios'][name] = df[c1] / (df[c2] + 0.0001)
                    results['available']['ratios'] = True
                except: pass
        
        # 3. Mineralization Indices
        if 'Au_ppm' in df.columns:
            try:
                results['mineralization_indices']['Gold Index'] = df['Au_ppm'] * 100
                results['available']['mineralization'] = True
            except: pass
        
        base_metals = [c for c in ['Cu_ppm', 'Zn_ppm', 'Pb_ppm'] if c in df.columns]
        if base_metals:
            try:
                results['mineralization_indices']['Base Metal Index'] = sum(df[c].fillna(0) for c in base_metals) / len(base_metals)
                results['available']['mineralization'] = True
            except: pass
        
        pathfinders = [c for c in ['As_ppm', 'Sb_ppm', 'Mo_ppm'] if c in df.columns]
        if pathfinders:
            try:
                results['mineralization_indices']['Pathfinder Index'] = sum(df[c].fillna(0) for c in pathfinders) / len(pathfinders)
                results['available']['mineralization'] = True
            except: pass
        
        # 4. Grade Statistics
        key_elements = ['Au_ppm', 'Cu_ppm', 'Zn_ppm', 'Pb_ppm', 'Ag_ppm', 'As_ppm']
        for elem in key_elements:
            if elem in df.columns:
                data = df[elem].dropna()
                if len(data) > 0:
                    threshold = data.quantile(0.75)
                    results['grade_stats'][elem] = {
                        'Mean': data.mean(), 'Median': data.median(), 'Std Dev': data.std(),
                        'Minimum': data.min(), 'Maximum': data.max(),
                        'N': len(data), 'Threshold': threshold,
                        'Above Threshold': (data > threshold).sum(),
                        '% Above Threshold': (data > threshold).sum() / len(data) * 100,
                        'Grade Class': 'HIGH' if data.mean() > threshold * 2 else 'MEDIUM' if data.mean() > threshold else 'LOW'
                    }
                    results['available']['grades'] = True
        
        return results

# ============================================================
# MAIN APPLICATION WINDOW
# ============================================================

class ElementalSocietyAnalysis(QMainWindow):
    def __init__(self):
        super().__init__()
        self.df = None
        self.cleaned_df = None
        self.derived_df = None
        self.is_processing = False
        self.original_filename = None
        self.init_ui()
        
    def init_ui(self):
        self.setWindowTitle(f"🧪 {APP_NAME} v{APP_VERSION} (Code: {APP_CODE})")
        self.setGeometry(50, 50, 1600, 950)
        
        # Central widget
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout()
        main_layout.setContentsMargins(10, 10, 10, 10)
        main_layout.setSpacing(10)
        central.setLayout(main_layout)
        
        # Header
        header = self.create_header()
        main_layout.addWidget(header)
        
        # Main Toolbar
        toolbar = self.create_toolbar()
        main_layout.addWidget(toolbar)
        
        # Status and Processing Combined Panel
        status_panel = self.create_status_panel()
        main_layout.addWidget(status_panel)
        
        # Info Panel
        self.info_label = QLabel("📋 Welcome! Upload your geochemical data to begin analysis.")
        self.info_label.setObjectName("info_label")
        self.info_label.setStyleSheet("""
            font-size: 14px;
            font-weight: 500;
            padding: 15px 20px;
            background: qlineargradient(x1:0, y1:0, x2:1, y2:0,
                stop:0 #1a73e8, stop:1 #1e8e3e);
            color: white;
            border-radius: 8px;
        """)
        main_layout.addWidget(self.info_label)
        
        # Tab Widget
        self.tab_widget = QTabWidget()
        self.tab_widget.setStyleSheet("""
            QTabWidget::pane {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 8px;
                margin-top: -1px;
            }
            QTabBar::tab {
                padding: 12px 24px;
                margin-right: 2px;
                border-top-left-radius: 8px;
                border-top-right-radius: 8px;
                background-color: #f1f3f4;
                color: #5f6368;
                font-weight: 500;
                font-size: 13px;
            }
            QTabBar::tab:selected {
                background-color: white;
                color: #1a73e8;
                border-bottom: 3px solid #1a73e8;
            }
        """)
        main_layout.addWidget(self.tab_widget)
        
        # Create tabs
        self.create_welcome_tab()
        self.create_manage_tab()
        self.create_eda_tab()
        self.create_analyze_tab()
        self.create_geochemistry_tab()
        self.create_log_tab()
        
        # Status bar
        self.statusBar = QStatusBar()
        self.setStatusBar(self.statusBar)
        self.statusBar.showMessage("✅ Ready")
        
        # Apply styles
        self.setStyleSheet(self.get_stylesheet())
        
        # Log
        self.log_message(f"🔬 {APP_NAME} v{APP_VERSION} initialized")
        self.log_message(f"📌 Project Code: {APP_CODE}")
    
    def get_stylesheet(self):
        return """
            QMainWindow { background-color: #f0f2f5; }
            
            QToolBar {
                background-color: white;
                border: none;
                border-bottom: 2px solid #1a73e8;
                spacing: 3px;
                padding: 5px 10px;
            }
            QToolBar QToolButton {
                border: none;
                border-radius: 4px;
                padding: 8px 16px;
                font-weight: 500;
                color: #3c4043;
                font-size: 12px;
            }
            QToolBar QToolButton:hover {
                background-color: #e8f0fe;
                color: #1a73e8;
            }
            QToolBar QToolButton#upload_btn {
                background-color: #1a73e8;
                color: white;
                padding: 8px 20px;
                border-radius: 4px;
            }
            QToolBar QToolButton#upload_btn:hover { background-color: #1557b0; }
            QToolBar QToolButton#analyze_btn {
                background-color: #1e8e3e;
                color: white;
                padding: 8px 20px;
                border-radius: 4px;
            }
            QToolBar QToolButton#analyze_btn:hover { background-color: #166b31; }
            QToolBar QToolButton#clean_btn {
                background-color: #e37400;
                color: white;
                padding: 8px 20px;
                border-radius: 4px;
            }
            QToolBar QToolButton#clean_btn:hover { background-color: #c26200; }
            
            QPushButton {
                background-color: #1a73e8;
                color: white;
                padding: 10px 24px;
                border: none;
                border-radius: 4px;
                font-weight: 500;
                font-size: 13px;
            }
            QPushButton:hover { background-color: #1557b0; }
            QPushButton#danger_btn { background-color: #d93025; }
            QPushButton#danger_btn:hover { background-color: #b3261e; }
            QPushButton#success_btn { background-color: #1e8e3e; }
            QPushButton#success_btn:hover { background-color: #166b31; }
            
            QTableWidget {
                background-color: white;
                gridline-color: #e8eaed;
                border: none;
                border-radius: 8px;
                alternate-background-color: #f8f9fa;
            }
            QTableWidget::item { padding: 8px; }
            QTableWidget::item:selected {
                background-color: #e8f0fe;
                color: #1a73e8;
            }
            QHeaderView::section {
                background-color: #f8f9fa;
                padding: 12px;
                border: none;
                border-bottom: 2px solid #dadce0;
                font-weight: 600;
                color: #3c4043;
            }
            
            QTextEdit {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 8px;
                padding: 10px;
                font-family: 'Segoe UI', sans-serif;
            }
            
            QComboBox {
                padding: 8px 12px;
                border: 1px solid #dadce0;
                border-radius: 4px;
                background-color: white;
                min-width: 120px;
            }
            
            QProgressBar {
                border: none;
                border-radius: 4px;
                background-color: #e8eaed;
                height: 20px;
                text-align: center;
            }
            QProgressBar::chunk {
                background-color: #1a73e8;
                border-radius: 4px;
            }
            
            QGroupBox {
                font-weight: 600;
                border: 1px solid #dadce0;
                border-radius: 8px;
                margin-top: 10px;
                padding-top: 15px;
            }
            QGroupBox::title {
                subcontrol-origin: margin;
                left: 10px;
                padding: 0 10px 0 10px;
                background-color: white;
            }
        """
    
    def create_header(self):
        widget = QWidget()
        widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border-bottom: 2px solid #1a73e8;
                border-radius: 8px 8px 0 0;
                padding: 10px 20px;
            }
        """)
        layout = QHBoxLayout()
        widget.setLayout(layout)
        
        title_label = QLabel("🧪 Elemental Society Analysis")
        title_label.setStyleSheet("font-size: 22px; font-weight: 700; color: #1a73e8;")
        layout.addWidget(title_label)
        
        version_label = QLabel(f"v{APP_VERSION} | Code: {APP_CODE}")
        version_label.setStyleSheet("font-size: 13px; color: #5f6368; padding-left: 10px;")
        layout.addWidget(version_label)
        
        layout.addStretch()
        
        dev_label = QLabel(f"👨‍🔬 {DEVELOPER}")
        dev_label.setStyleSheet("font-size: 13px; color: #5f6368;")
        layout.addWidget(dev_label)
        
        return widget
    
    def create_toolbar(self):
        toolbar = QToolBar()
        toolbar.setMovable(False)
        toolbar.setIconSize(QSize(32, 32))
        
        # Upload
        upload_btn = QToolButton()
        upload_btn.setObjectName("upload_btn")
        upload_btn.setText("📁 Upload")
        upload_btn.setToolTip("Upload geochemical data file")
        upload_btn.clicked.connect(self.upload_file)
        toolbar.addWidget(upload_btn)
        
        toolbar.addSeparator()
        
        # Clean
        clean_btn = QToolButton()
        clean_btn.setObjectName("clean_btn")
        clean_btn.setText("🧹 Clean")
        clean_btn.setToolTip("Clean the loaded data")
        clean_btn.clicked.connect(self.clean_data)
        toolbar.addWidget(clean_btn)
        
        # Analyze
        analyze_btn = QToolButton()
        analyze_btn.setObjectName("analyze_btn")
        analyze_btn.setText("📊 Analyze")
        analyze_btn.setToolTip("Run comprehensive EDA")
        analyze_btn.clicked.connect(self.run_analysis)
        toolbar.addWidget(analyze_btn)
        
        toolbar.addSeparator()
        
        # Export
        export_btn = QToolButton()
        export_btn.setText("💾 Export")
        export_btn.setToolTip("Export cleaned data")
        export_btn.clicked.connect(self.export_data)
        toolbar.addWidget(export_btn)
        
        # Report
        report_btn = QToolButton()
        report_btn.setText("📄 Report")
        report_btn.setToolTip("Generate report")
        report_btn.clicked.connect(self.show_report)
        toolbar.addWidget(report_btn)
        
        toolbar.addSeparator()
        
        # Help
        help_btn = QToolButton()
        help_btn.setText("❓ Help")
        help_btn.setToolTip("Show help")
        help_btn.clicked.connect(self.show_help)
        toolbar.addWidget(help_btn)
        
        about_btn = QToolButton()
        about_btn.setText("ℹ️ About")
        about_btn.setToolTip("About")
        about_btn.clicked.connect(self.show_about)
        toolbar.addWidget(about_btn)
        
        spacer = QWidget()
        spacer.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Preferred)
        toolbar.addWidget(spacer)
        
        self.file_info_label = QLabel("📁 No file loaded")
        self.file_info_label.setStyleSheet("color: #5f6368; padding: 5px 10px; font-size: 12px;")
        toolbar.addWidget(self.file_info_label)
        
        return toolbar
    
    def create_status_panel(self):
        """Create combined status panel with indicators"""
        widget = QWidget()
        widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 8px;
                padding: 5px 15px;
            }
        """)
        layout = QHBoxLayout()
        layout.setContentsMargins(10, 5, 10, 5)
        widget.setLayout(layout)
        
        # Status label
        layout.addWidget(QLabel("Status:"))
        
        # Processing indicator with color
        self.status_indicator = QLabel("●")
        self.status_indicator.setStyleSheet("""
            QLabel {
                font-size: 20px;
                color: #e8eaed;
            }
        """)
        layout.addWidget(self.status_indicator)
        
        # Status text
        self.status_text = QLabel("Ready")
        self.status_text.setStyleSheet("color: #5f6368; font-weight: 500;")
        layout.addWidget(self.status_text)
        
        # Separator
        separator = QFrame()
        separator.setFrameShape(QFrame.VLine)
        separator.setFrameShadow(QFrame.Sunken)
        layout.addWidget(separator)
        
        # Progress bar
        self.progress_bar = QProgressBar()
        self.progress_bar.setVisible(False)
        self.progress_bar.setMaximumWidth(200)
        self.progress_bar.setStyleSheet("""
            QProgressBar {
                border: none;
                border-radius: 4px;
                background-color: #e8eaed;
                height: 16px;
                text-align: center;
            }
            QProgressBar::chunk {
                background-color: #1a73e8;
                border-radius: 4px;
            }
        """)
        layout.addWidget(self.progress_bar)
        
        # Wait indicator
        self.wait_label = QLabel("⏳ Processing...")
        self.wait_label.setStyleSheet("color: #e37400; font-weight: 600;")
        self.wait_label.setVisible(False)
        layout.addWidget(self.wait_label)
        
        layout.addStretch()
        
        # Process info
        self.process_info = QLabel("")
        self.process_info.setStyleSheet("color: #9aa0a6; font-size: 11px;")
        layout.addWidget(self.process_info)
        
        return widget
    
    def set_status(self, status, message="", progress=False):
        """Set the status indicator with color"""
        status_map = {
            'idle': ('#e8eaed', 'Ready', False),
            'processing': ('#e37400', 'Processing...', True),
            'success': ('#1e8e3e', 'Success!', False),
            'error': ('#d93025', 'Error!', False)
        }
        color, text, show_progress = status_map.get(status, status_map['idle'])
        
        self.status_indicator.setStyleSheet(f"""
            QLabel {{
                font-size: 20px;
                color: {color};
            }}
        """)
        self.status_text.setText(message if message else text)
        self.progress_bar.setVisible(show_progress or progress)
        if progress:
            self.progress_bar.setRange(0, 0)
        else:
            self.progress_bar.setVisible(False)
        
        self.wait_label.setVisible(status == 'processing')
        if status == 'processing':
            self.wait_label.setText("⏳ Processing...")
            QApplication.processEvents()
    
    def set_processing(self, status, message="", progress=False):
        """Set processing state"""
        self.is_processing = status
        if status:
            self.set_status('processing', message, progress)
            self.info_label.setText(f"⚙️ {message}")
            self.statusBar.showMessage(f"Processing: {message}")
        else:
            self.set_status('idle', 'Ready', False)
            self.info_label.setText("✅ Ready")
            self.statusBar.showMessage("✅ Ready")
    
    def update_progress(self, message):
        """Update progress message"""
        self.process_info.setText(message)
        self.status_text.setText(message)
        self.log_message(f"⏳ {message}")
        QApplication.processEvents()
    
    # ============================================================
    # TAB CREATION
    # ============================================================
    
    def create_welcome_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        welcome_widget = QWidget()
        welcome_widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 12px;
                padding: 20px;
            }
        """)
        welcome_layout = QVBoxLayout()
        welcome_widget.setLayout(welcome_layout)
        
        welcome_text = QLabel("""
        <h1 style='color: #1a73e8;'>🧪 Welcome to Elemental Society Analysis</h1>
        <h3 style='color: #3c4043;'>Professional Geochemical Data Analysis Suite v1.0</h3>
        <br>
        <p style='font-size: 14px; color: #5f6368;'>
        <b>Complete solution for exploration geochemistry data analysis.</b>
        </p>
        <br>
        <table style='width: 100%;'>
            <tr>
                <td style='padding: 10px; border: none;'><b>📁 1. Upload</b><br>Load CSV or Excel file</td>
                <td style='padding: 10px; border: none;'><b>🧹 2. Clean</b><br>Handle missing values, outliers</td>
            </tr>
            <tr>
                <td style='padding: 10px; border: none;'><b>📊 3. Analyze</b><br>Run comprehensive EDA</td>
                <td style='padding: 10px; border: none;'><b>📉 4. Visualize</b><br>Professional plots</td>
            </tr>
            <tr>
                <td style='padding: 10px; border: none;'><b>🧪 5. Geochemistry</b><br>Alteration indices, ratios</td>
                <td style='padding: 10px; border: none;'><b>📄 6. Report</b><br>Generate HTML report</td>
            </tr>
        </table>
        <br>
        <p style='color: #5f6368;'><b>💡 Tip:</b> Start by clicking <b style='color: #1a73e8;'>Upload</b> above.</p>
        """)
        welcome_text.setWordWrap(True)
        welcome_layout.addWidget(welcome_text)
        
        layout.addWidget(welcome_widget)
        layout.addStretch()
        self.tab_widget.addTab(tab, "🏠 Welcome")
    
    def create_manage_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        # Controls
        controls = QHBoxLayout()
        self.data_info = QLabel("No data loaded")
        controls.addWidget(self.data_info)
        controls.addStretch()
        
        controls.addWidget(QLabel("Show rows:"))
        self.row_limit = QSpinBox()
        self.row_limit.setRange(10, 5000)
        self.row_limit.setValue(500)
        self.row_limit.valueChanged.connect(self.refresh_data)
        controls.addWidget(self.row_limit)
        
        # Modify filename
        rename_btn = QPushButton("✏️ Rename File")
        rename_btn.clicked.connect(self.rename_file)
        controls.addWidget(rename_btn)
        
        # Delete rows/columns
        delete_btn = QPushButton("🗑️ Delete Selected")
        delete_btn.setObjectName("danger_btn")
        delete_btn.clicked.connect(self.delete_selected)
        controls.addWidget(delete_btn)
        
        layout.addLayout(controls)
        
        self.data_table = QTableWidget()
        self.data_table.setAlternatingRowColors(True)
        self.data_table.setSortingEnabled(True)
        self.data_table.setSelectionBehavior(QTableWidget.SelectItems)
        self.data_table.setSelectionMode(QTableWidget.ExtendedSelection)
        layout.addWidget(self.data_table)
        
        self.tab_widget.addTab(tab, "📊 Data")
    
    def create_eda_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        # Controls
        controls = QHBoxLayout()
        controls.addWidget(QLabel("Variable:"))
        self.eda_var = QComboBox()
        self.eda_var.currentTextChanged.connect(self.plot_distribution)
        controls.addWidget(self.eda_var)
        
        controls.addWidget(QLabel("Type:"))
        self.eda_type = QComboBox()
        self.eda_type.addItems(["Histogram", "PDF", "CDF", "Box Plot"])
        self.eda_type.currentTextChanged.connect(self.plot_distribution)
        controls.addWidget(self.eda_type)
        
        # Combined plot option
        self.show_combined = QCheckBox("Show Combined View")
        self.show_combined.toggled.connect(self.plot_distribution)
        controls.addWidget(self.show_combined)
        
        controls.addStretch()
        layout.addLayout(controls)
        
        self.eda_figure = Figure(figsize=(10, 6))
        self.eda_canvas = FigureCanvas(self.eda_figure)
        layout.addWidget(self.eda_canvas)
        
        self.tab_widget.addTab(tab, "📈 EDA")
    
    def create_analyze_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        # Controls
        controls = QHBoxLayout()
        controls.addWidget(QLabel("Plot:"))
        self.plot_type = QComboBox()
        self.plot_type.addItems(["Correlation Matrix", "PCA Scores", "Scatter Plot", "All Plots Combined"])
        self.plot_type.currentTextChanged.connect(self.update_plot)
        controls.addWidget(self.plot_type)
        
        controls.addWidget(QLabel("X:"))
        self.analyze_x = QComboBox()
        controls.addWidget(self.analyze_x)
        
        controls.addWidget(QLabel("Y:"))
        self.analyze_y = QComboBox()
        controls.addWidget(self.analyze_y)
        
        plot_btn = QPushButton("🎨 Generate")
        plot_btn.clicked.connect(self.update_plot)
        controls.addWidget(plot_btn)
        controls.addStretch()
        layout.addLayout(controls)
        
        self.analyze_figure = Figure(figsize=(10, 6))
        self.analyze_canvas = FigureCanvas(self.analyze_figure)
        layout.addWidget(self.analyze_canvas)
        
        self.tab_widget.addTab(tab, "📊 Analysis")
    
    def create_geochemistry_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        self.geochem_viewer = QTextEdit()
        self.geochem_viewer.setReadOnly(True)
        self.geochem_viewer.setFont(QFont("Segoe UI", 10))
        layout.addWidget(self.geochem_viewer)
        
        self.tab_widget.addTab(tab, "🧪 Geochemistry")
    
    def create_log_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        controls = QHBoxLayout()
        clear_btn = QPushButton("🗑️ Clear")
        clear_btn.clicked.connect(self.clear_log)
        controls.addWidget(clear_btn)
        controls.addStretch()
        layout.addLayout(controls)
        
        self.log_text = QTextEdit()
        self.log_text.setReadOnly(True)
        self.log_text.setFont(QFont("Consolas", 10))
        layout.addWidget(self.log_text)
        
        self.tab_widget.addTab(tab, "📝 Log")
    
    # ============================================================
    # LOG FUNCTIONS
    # ============================================================
    
    def log_message(self, message):
        from datetime import datetime
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.append(f"[{timestamp}] {message}")
    
    def clear_log(self):
        self.log_text.clear()
        self.log_message("🗑️ Log cleared")
    
    # ============================================================
    # DATA FUNCTIONS
    # ============================================================
    
    def rename_file(self):
        if self.original_filename is None:
            QMessageBox.warning(self, "Warning", "No file loaded!")
            return
        
        new_name, ok = QInputDialog.getText(self, "Rename File", 
            "Enter new filename:", QLineEdit.Normal, self.original_filename)
        
        if ok and new_name:
            self.original_filename = new_name
            self.file_info_label.setText(f"📁 {new_name}")
            self.log_message(f"✏️ Renamed file to: {new_name}")
            self.set_status('success', f'Renamed: {new_name}')
    
    def delete_selected(self):
        if self.df is None:
            QMessageBox.warning(self, "Warning", "No data loaded!")
            return
        
        selected = self.data_table.selectedIndexes()
        if not selected:
            QMessageBox.warning(self, "Warning", "No cells selected!")
            return
        
        # Get unique rows and columns
        rows = set(index.row() for index in selected)
        cols = set(index.column() for index in selected)
        
        reply = QMessageBox.question(self, "Delete Selection",
            f"Delete {len(rows)} rows and {len(cols)} columns?",
            QMessageBox.Yes | QMessageBox.No)
        
        if reply == QMessageBox.Yes:
            # Delete columns first (from right to left)
            for col in sorted(cols, reverse=True):
                self.df = self.df.drop(self.df.columns[col], axis=1)
            
            # Delete rows (from bottom to top)
            for row in sorted(rows, reverse=True):
                self.df = self.df.drop(self.df.index[row])
            
            self.df = self.df.reset_index(drop=True)
            self.display_data()
            self.log_message(f"🗑️ Deleted {len(rows)} rows and {len(cols)} columns")
            self.set_status('success', f'Deleted {len(rows)} rows, {len(cols)} columns')
    
    def upload_file(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open Geochemical Data", "",
            "All Files (*);;CSV (*.csv);;Excel (*.xlsx *.xls);;Text (*.txt)"
        )
        if not file_path:
            return
        
        self.original_filename = os.path.basename(file_path)
        self.set_processing(True, "Loading file...")
        self.log_message(f"📁 Loading: {self.original_filename}")
        self.update_progress(f"Reading: {self.original_filename}")
        
        try:
            self.df = read_any_file(file_path)
            
            if self.df is not None and not self.df.empty:
                self.df = clean_dataframe(self.df)
                self.cleaned_df = self.df.copy()
                self.derived_df = self.df.copy()
                self.display_data()
                self.file_info_label.setText(f"📁 {self.original_filename}")
                
                # Update selectors
                numeric_cols = self.df.select_dtypes(include=[np.number]).columns
                self.eda_var.clear()
                self.eda_var.addItems(numeric_cols)
                self.analyze_x.clear()
                self.analyze_x.addItems(numeric_cols)
                self.analyze_y.clear()
                self.analyze_y.addItems(numeric_cols)
                
                self.log_message(f"✅ Loaded: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
                self.set_status('success', f'Loaded: {self.df.shape[0]} rows')
                self.info_label.setText(f"📊 Data loaded: {self.df.shape[0]} rows × {self.df.shape[1]} columns")
                
                QMessageBox.information(self, "Success",
                    f"File loaded!\n\n📊 {self.df.shape[0]} rows × {self.df.shape[1]} columns")
            else:
                QMessageBox.warning(self, "Error", "Could not read file.")
                
        except Exception as e:
            self.log_message(f"❌ Error: {str(e)}")
            self.set_status('error', 'Load failed')
            QMessageBox.critical(self, "Error", f"Failed to load:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def clean_data(self):
        if self.df is None or self.df.empty:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
        
        reply = QMessageBox.question(self, "Clean Data", "Clean the dataset?", QMessageBox.Yes | QMessageBox.No)
        if reply != QMessageBox.Yes:
            return
        
        self.set_processing(True, "Cleaning data...")
        self.log_message("🧹 Cleaning data...")
        self.update_progress("Cleaning data...")
        
        try:
            self.cleaned_df = clean_dataframe(self.df.copy())
            self.derived_df = self.cleaned_df.copy()
            self.df = self.cleaned_df.copy()
            self.display_data()
            
            self.log_message(f"✅ Cleaned: {len(self.cleaned_df)} rows")
            self.set_status('success', f'Cleaned: {len(self.cleaned_df)} rows')
            self.info_label.setText(f"🧹 Cleaned: {len(self.cleaned_df)} rows × {len(self.cleaned_df.columns)} columns")
            
            QMessageBox.information(self, "Success",
                f"Cleaning complete!\n\nBefore: {len(self.df)} rows\nAfter: {len(self.cleaned_df)} rows")
            
        except Exception as e:
            self.log_message(f"❌ Cleaning error: {str(e)}")
            self.set_status('error', 'Cleaning failed')
            QMessageBox.critical(self, "Error", f"Cleaning failed:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def run_analysis(self):
        if self.df is None:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
        
        self.set_processing(True, "Running analysis...")
        self.log_message("📊 Starting analysis...")
        self.update_progress("🔬 Running analysis...")
        
        try:
            self.update_progress("📊 Generating statistics...")
            self.generate_statistics()
            
            self.update_progress("📉 Creating visualizations...")
            self.plot_distribution()
            self.update_plot()
            
            self.update_progress("🧪 Calculating geochemistry...")
            self.calculate_geochemistry()
            
            self.log_message("✅ Analysis complete!")
            self.set_status('success', 'Analysis Complete!')
            self.info_label.setText("📊 Analysis Complete!")
            
            QMessageBox.information(self, "Success", "Analysis completed successfully!")
            
        except Exception as e:
            self.log_message(f"❌ Analysis error: {str(e)}")
            self.set_status('error', 'Analysis failed')
            QMessageBox.critical(self, "Error", f"Analysis failed:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def display_data(self):
        if self.df is None:
            return
        
        display_limit = min(self.row_limit.value(), len(self.df))
        self.data_table.setRowCount(display_limit)
        self.data_table.setColumnCount(len(self.df.columns))
        self.data_table.setHorizontalHeaderLabels(self.df.columns.astype(str))
        
        for i in range(display_limit):
            for j, col in enumerate(self.df.columns):
                val = self.df.iloc[i, j]
                if pd.isna(val):
                    self.data_table.setItem(i, j, QTableWidgetItem(""))
                else:
                    self.data_table.setItem(i, j, QTableWidgetItem(str(val)))
        
        self.data_table.resizeColumnsToContents()
        self.data_info.setText(f"📊 {len(self.df)} rows × {len(self.df.columns)} columns")
    
    def refresh_data(self):
        self.display_data()
    
    # ============================================================
    # STATISTICS FUNCTIONS
    # ============================================================
    
    def generate_statistics(self):
        if self.df is None:
            return
        
        numeric_df = self.df.select_dtypes(include=[np.number])
        if numeric_df.empty:
            return
        
        stats_text = []
        stats_text.append("="*80)
        stats_text.append("📊 STATISTICAL SUMMARY")
        stats_text.append("="*80)
        stats_text.append(f"\nDataset: {len(self.df)} rows × {len(self.df.columns)} columns")
        stats_text.append(f"Numeric columns: {len(numeric_df.columns)}\n")
        
        # Basic stats
        stats_text.append("📈 BASIC STATISTICS")
        stats_text.append("-"*60)
        stats_text.append(numeric_df.describe().to_string())
        stats_text.append("\n")
        
        # Variance
        stats_text.append("📊 VARIANCE")
        stats_text.append("-"*60)
        stats_text.append(numeric_df.var().to_string())
        stats_text.append("\n")
        
        # Skewness
        stats_text.append("📊 SKEWNESS")
        stats_text.append("-"*60)
        stats_text.append(numeric_df.skew().to_string())
        stats_text.append("\n")
        
        # Kurtosis
        stats_text.append("📊 KURTOSIS")
        stats_text.append("-"*60)
        stats_text.append(numeric_df.kurtosis().to_string())
        
        self.log_message("📊 Statistics generated")
    
    # ============================================================
    # PLOTTING FUNCTIONS
    # ============================================================
    
    def plot_distribution(self):
        if self.df is None:
            return
        
        var = self.eda_var.currentText()
        plot_type = self.eda_type.currentText()
        show_combined = self.show_combined.isChecked()
        
        if not var or var not in self.df.columns:
            return
        
        data = self.df[var].dropna()
        if len(data) < 2:
            return
        
        self.eda_figure.clear()
        
        if show_combined:
            # Create 2x2 grid for combined view
            ax1 = self.eda_figure.add_subplot(2, 2, 1)
            ax2 = self.eda_figure.add_subplot(2, 2, 2)
            ax3 = self.eda_figure.add_subplot(2, 2, 3)
            ax4 = self.eda_figure.add_subplot(2, 2, 4)
            axes = [ax1, ax2, ax3, ax4]
            plot_types = ["Histogram", "PDF", "CDF", "Box Plot"]
        else:
            axes = [self.eda_figure.add_subplot(111)]
            plot_types = [plot_type]
        
        for ax, ptype in zip(axes, plot_types):
            if ptype == "Histogram":
                ax.hist(data, bins=20, density=True, alpha=0.6, color='steelblue', edgecolor='black')
                ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.2f}')
                ax.axvline(data.median(), color='green', linestyle='--', label=f'Median: {data.median():.2f}')
                ax.set_ylabel('Density')
                ax.legend()
                
            elif ptype == "PDF":
                ax.hist(data, bins=20, density=True, alpha=0.4, color='steelblue')
                x = np.linspace(data.min(), data.max(), 100)
                mu, std = stats.norm.fit(data)
                pdf = stats.norm.pdf(x, mu, std)
                ax.plot(x, pdf, 'r-', linewidth=2, label='Normal PDF')
                ax.set_ylabel('Density')
                ax.legend()
                
            elif ptype == "CDF":
                sorted_data = np.sort(data)
                y = np.arange(1, len(sorted_data)+1) / len(sorted_data)
                ax.plot(sorted_data, y, 'b-', linewidth=2)
                ax.set_ylabel('Cumulative Probability')
                ax.grid(True, alpha=0.3)
                
            elif ptype == "Box Plot":
                ax.boxplot(data, vert=True, patch_artist=True)
            
            ax.set_title(f'{var} - {ptype}', fontweight='bold')
            ax.set_xlabel(var if ptype != "Box Plot" else '')
            ax.grid(True, alpha=0.3)
        
        self.eda_figure.tight_layout()
        self.eda_canvas.draw()
    
    def update_plot(self):
        if self.df is None:
            return
        
        plot_type = self.plot_type.currentText()
        
        self.analyze_figure.clear()
        
        numeric_df = self.df.select_dtypes(include=[np.number])
        
        if plot_type == "All Plots Combined":
            # Create 2x2 grid
            ax1 = self.analyze_figure.add_subplot(2, 2, 1)
            ax2 = self.analyze_figure.add_subplot(2, 2, 2)
            ax3 = self.analyze_figure.add_subplot(2, 2, 3)
            ax4 = self.analyze_figure.add_subplot(2, 2, 4)
            
            # Correlation
            if len(numeric_df.columns) > 1:
                corr = numeric_df.corr()
                sns.heatmap(corr, annot=True, cmap='coolwarm', center=0,
                           fmt='.2f', square=True, linewidths=0.5, ax=ax1)
                ax1.set_title('Correlation Matrix', fontweight='bold')
            
            # PCA
            if len(numeric_df.columns) >= 2:
                clean_data = numeric_df.dropna()
                if len(clean_data) > 1:
                    scaler = StandardScaler()
                    scaled = scaler.fit_transform(clean_data)
                    pca = SklearnPCA()
                    scores = pca.fit_transform(scaled)
                    explained = pca.explained_variance_ratio_
                    
                    scatter = ax2.scatter(scores[:, 0], scores[:, 1],
                                       c=np.linalg.norm(scores[:, :2], axis=1),
                                       cmap='viridis', alpha=0.6,
                                       edgecolors='black', linewidth=0.5)
                    ax2.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
                    ax2.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
                    ax2.set_title('PCA Scores', fontweight='bold')
                    ax2.grid(True, alpha=0.3)
            
            # Scatter
            x_col = self.analyze_x.currentText()
            y_col = self.analyze_y.currentText()
            if x_col and y_col and x_col in self.df.columns and y_col in self.df.columns:
                data = self.df[[x_col, y_col]].dropna()
                ax3.scatter(data[x_col], data[y_col], alpha=0.6, color='steelblue', edgecolors='black')
                ax3.set_xlabel(x_col)
                ax3.set_ylabel(y_col)
                ax3.set_title(f'Scatter Plot', fontweight='bold')
                ax3.grid(True, alpha=0.3)
            
            # Distribution
            if len(numeric_df.columns) > 0:
                col = numeric_df.columns[0]
                data = numeric_df[col].dropna()
                if len(data) > 1:
                    ax4.hist(data, bins=20, density=True, alpha=0.6, color='steelblue', edgecolor='black')
                    ax4.set_title(f'Distribution: {col}', fontweight='bold')
                    ax4.set_xlabel(col)
                    ax4.set_ylabel('Density')
                    ax4.grid(True, alpha=0.3)
            
        elif plot_type == "Correlation Matrix" and len(numeric_df.columns) > 1:
            ax = self.analyze_figure.add_subplot(111)
            corr = numeric_df.corr()
            sns.heatmap(corr, annot=True, cmap='coolwarm', center=0,
                       fmt='.2f', square=True, linewidths=0.5, ax=ax)
            ax.set_title('Correlation Matrix', fontweight='bold')
            
        elif plot_type == "PCA Scores" and len(numeric_df.columns) >= 2:
            ax = self.analyze_figure.add_subplot(111)
            clean_data = numeric_df.dropna()
            if len(clean_data) > 1:
                scaler = StandardScaler()
                scaled = scaler.fit_transform(clean_data)
                pca = SklearnPCA()
                scores = pca.fit_transform(scaled)
                explained = pca.explained_variance_ratio_
                
                scatter = ax.scatter(scores[:, 0], scores[:, 1],
                                   c=np.linalg.norm(scores[:, :2], axis=1),
                                   cmap='viridis', alpha=0.6,
                                   edgecolors='black', linewidth=0.5)
                ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
                ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
                ax.set_title('PCA Scores', fontweight='bold')
                ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
                ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
                ax.grid(True, alpha=0.3)
                self.analyze_figure.colorbar(scatter, ax=ax, label='Distance')
                
        elif plot_type == "Scatter Plot":
            ax = self.analyze_figure.add_subplot(111)
            x_col = self.analyze_x.currentText()
            y_col = self.analyze_y.currentText()
            
            if x_col and y_col and x_col in self.df.columns and y_col in self.df.columns:
                data = self.df[[x_col, y_col]].dropna()
                ax.scatter(data[x_col], data[y_col], alpha=0.6, color='steelblue', edgecolors='black')
                ax.set_xlabel(x_col)
                ax.set_ylabel(y_col)
                ax.set_title(f'Scatter Plot: {y_col} vs {x_col}', fontweight='bold')
                ax.grid(True, alpha=0.3)
        
        self.analyze_figure.tight_layout()
        self.analyze_canvas.draw()
    
    # ============================================================
    # GEOCHEMISTRY FUNCTIONS
    # ============================================================
    
    def calculate_geochemistry(self):
        if self.df is None:
            return
        
        geochem = GeochemistryEngine.calculate_all(self.df)
        
        html = []
        html.append("<html><head><style>")
        html.append("""
            body { font-family: 'Segoe UI', Arial, sans-serif; margin: 15px; }
            h1 { color: #1a73e8; border-bottom: 2px solid #1a73e8; padding-bottom: 8px; }
            h2 { color: #3c4043; margin-top: 20px; }
            .card { background-color: #f8f9fa; border-radius: 8px; padding: 15px; margin: 10px 0; border-left: 4px solid #1a73e8; }
            .stat-box { display: inline-block; margin: 3px; padding: 8px 15px; background: white; border-radius: 6px; border: 1px solid #dadce0; }
            .stat-value { font-weight: bold; color: #1a73e8; font-size: 16px; }
            .stat-label { font-size: 10px; color: #5f6368; }
            .badge { display: inline-block; padding: 2px 10px; border-radius: 10px; font-size: 11px; font-weight: 600; }
            .badge-high { background: #fce8e6; color: #d93025; }
            .badge-medium { background: #fef7e0; color: #e37400; }
            .badge-low { background: #e6f4ea; color: #1e8e3e; }
            .badge-available { background: #e6f4ea; color: #1e8e3e; }
            .badge-missing { background: #fce8e6; color: #d93025; }
        """)
        html.append("</style></head><body>")
        
        html.append(f"<h1>🧪 Exploration Geochemistry Report</h1>")
        html.append(f"<p><b>Generated:</b> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>")
        html.append(f"<p><b>Dataset:</b> {len(self.df)} samples × {len(self.df.columns)} variables</p>")
        
        # Available status
        avail = geochem.get('available', {})
        html.append("<div class='card'>")
        html.append("<h3>📊 Available Calculations</h3>")
        for key, val in avail.items():
            badge = "badge-available" if val else "badge-missing"
            status = "✅ Available" if val else "❌ Not Available"
            html.append(f"<span class='badge {badge}'>{key.replace('_', ' ').title()}: {status}</span> ")
        html.append("</div>")
        
        # Alteration Indices
        html.append("<h2>📊 Alteration Indices</h2>")
        if geochem['alteration_indices']:
            html.append("<div class='card'>")
            for name, values in geochem['alteration_indices'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{values.min():.4f} - {values.max():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("<div class='card'><p style='color:#5f6368;'>No alteration indices could be calculated.</p></div>")
        
        # Element Ratios
        html.append("<h2>📊 Element Ratios</h2>")
        if geochem['element_ratios']:
            html.append("<div class='card'>")
            for name, values in geochem['element_ratios'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("<div class='card'><p style='color:#5f6368;'>No element ratios could be calculated.</p></div>")
        
        # Mineralization Indices
        html.append("<h2>📈 Mineralization Indices</h2>")
        if geochem['mineralization_indices']:
            html.append("<div class='card'>")
            for name, values in geochem['mineralization_indices'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("<div class='card'><p style='color:#5f6368;'>No mineralization indices could be calculated.</p></div>")
        
        # Grade Statistics
        html.append("<h2>📊 Grade Statistics</h2>")
        if geochem['grade_stats']:
            html.append("<div class='card'>")
            for element, stats in geochem['grade_stats'].items():
                if stats:
                    grade_class = stats.get('Grade Class', 'LOW')
                    badge_class = 'badge-high' if grade_class == 'HIGH' else 'badge-medium' if grade_class == 'MEDIUM' else 'badge-low'
                    html.append(f"""
                    <div style='margin: 10px 0; padding: 10px; background: white; border-radius: 6px; border: 1px solid #e8eaed;'>
                        <b>{element}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{stats.get('Mean', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{stats.get('Median', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Std Dev</div><div class='stat-value'>{stats.get('Std Dev', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{stats.get('Minimum', 0):.4f} - {stats.get('Maximum', 0):.4f}</div></div>
                        <div style='margin-top: 5px;'>
                            <span class='badge {badge_class}'>Grade: {grade_class}</span>
                            <span style='margin-left: 10px;'>Threshold: {stats.get('Threshold', 0):.4f}</span>
                            <span style='margin-left: 10px;'>Above: {stats.get('Above Threshold', 0)} ({stats.get('% Above Threshold', 0):.1f}%)</span>
                        </div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("<div class='card'><p style='color:#5f6368;'>No grade statistics could be calculated.</p></div>")
        
        # Summary
        html.append("""
        <div style='margin-top: 20px; padding: 15px; background-color: #e8f0fe; border-radius: 8px;'>
            <p><b>📌 Summary</b></p>
            <table style='width: auto;'>
        """)
        html.append(f"<tr><td>Total Samples</td><td><b>{len(self.df)}</b></td></tr>")
        html.append(f"<tr><td>Total Variables</td><td><b>{len(self.df.columns)}</b></td></tr>")
        html.append(f"<tr><td>Alteration Indices</td><td><b>{len(geochem['alteration_indices'])}</b></td></tr>")
        html.append(f"<tr><td>Element Ratios</td><td><b>{len(geochem['element_ratios'])}</b></td></tr>")
        html.append(f"<tr><td>Mineralization Indices</td><td><b>{len(geochem['mineralization_indices'])}</b></td></tr>")
        html.append(f"<tr><td>Grade Statistics</td><td><b>{len(geochem['grade_stats'])}</b></td></tr>")
        html.append("</table></div></body></html>")
        
        self.geochem_viewer.setHtml("\n".join(html))
    
    def show_report(self):
        for i in range(self.tab_widget.count()):
            if self.tab_widget.tabText(i) == "📄 Report":
                self.tab_widget.setCurrentIndex(i)
                break
    
    # ============================================================
    # EXPORT FUNCTIONS
    # ============================================================
    
    def export_data(self):
        if self.df is None:
            QMessageBox.warning(self, "Warning", "No data to export!")
            return
        
        df = self.df
        
        formats = {
            "CSV (;)": "csv_semicolon",
            "CSV (,)": "csv_comma",
            "Excel (.xlsx)": "excel",
            "JSON": "json"
        }
        
        format_choice, ok = QInputDialog.getItem(
            self, "Export Format", "Select format:", list(formats.keys()), 0, False)
        if not ok:
            return
        
        file_ext_map = {
            "csv_semicolon": "csv",
            "csv_comma": "csv",
            "excel": "xlsx",
            "json": "json"
        }
        
        file_path, _ = QFileDialog.getSaveFileName(
            self, "Export Data", f"cleaned_data.{file_ext_map[formats[format_choice]]}",
            f"{format_choice} (*.{file_ext_map[formats[format_choice]]})")
        
        if file_path:
            try:
                self.set_processing(True, "Exporting...")
                self.update_progress(f"Exporting to {format_choice}...")
                
                if formats[format_choice] == "csv_semicolon":
                    df.to_csv(file_path, sep=';', index=False, encoding='utf-8-sig')
                elif formats[format_choice] == "csv_comma":
                    df.to_csv(file_path, sep=',', index=False, encoding='utf-8-sig')
                elif formats[format_choice] == "excel":
                    df.to_excel(file_path, index=False)
                elif formats[format_choice] == "json":
                    df.to_json(file_path, orient='records', indent=2)
                
                self.log_message(f"✅ Exported: {file_path}")
                self.set_status('success', 'Export complete!')
                QMessageBox.information(self, "Success", f"Data exported to:\n{file_path}")
            except Exception as e:
                self.set_status('error', 'Export failed')
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")
            finally:
                self.set_processing(False)
    
    # ============================================================
    # HELP AND ABOUT
    # ============================================================
    
    def show_help(self):
        help_text = f"""
🔬 {APP_NAME} v{APP_VERSION}

📌 Project Code: {APP_CODE}

👨‍🔬 Developer: {DEVELOPER}

🚀 QUICK START

1. 📁 Upload your data
2. 🧹 Clean the data
3. 📊 Run analysis
4. 📉 View visualizations
5. 🧪 Check geochemistry
6. 📄 Generate report

🔧 FEATURES

Data Management:
• Upload CSV/Excel files
• Clean data (missing values, outliers)
• Rename file
• Delete selected rows/columns

Statistics:
• Mean, Median, Std Dev
• Variance, Skewness, Kurtosis
• Correlation Matrix

Visualization:
• Correlation Heatmap
• PCA Scores
• Scatter Plot
• All Plots Combined View
• EDA: Histogram, PDF, CDF, Box Plot

Geochemistry:
• Alteration Indices
• Element Ratios
• Mineralization Indices
• Grade Statistics

Export:
• CSV, Excel, JSON

Copyright © 2026 {DEVELOPER}
"""
        QMessageBox.information(self, "Help", help_text)
    
    def show_about(self):
        about_text = f"""
🔬 {APP_NAME}

Version: {APP_VERSION}
Project Code: {APP_CODE}

Developer: {DEVELOPER}
Email: {EMAIL}

Description:
Professional geochemical data cleaning and 
exploratory data analysis (EDA) suite for
mineral exploration and geochemistry.

Features:
• Data Cleaning Engine
• Statistical Analysis
• Correlation Matrix
• Principal Component Analysis
• Distribution Analysis (PDF, CDF)
• Exploration Geochemistry
• Alteration Indices
• Grade Statistics
• Combined Visualization
• Data Management

Copyright © 2026 {DEVELOPER}
All rights reserved.
"""
        QMessageBox.about(self, "About", about_text)

# ============================================================
# MAIN APPLICATION
# ============================================================

def main():
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    font = QFont("Segoe UI", 10)
    app.setFont(font)
    window = ElementalSocietyAnalysis()
    window.show()
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()

In [ ]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - PROFESSIONAL EDITION v1.0
# Project Code: 1010
# Developer: Mohamed Almoufty Abdoalmoutleb Abaker
# ============================================================

import sys
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar
from matplotlib.figure import Figure
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# APPLICATION METADATA
# ============================================================

APP_NAME = "Elemental Society Analysis"
APP_VERSION = "1.0.0"
APP_CODE = "1010"
DEVELOPER = "Mohamed Almoufty Abdoalmoutleb Abaker"
EMAIL = "almoufty121@gmail.com"
BACKGROUND = "Exploration Geologist | Msc Earth Science Engineering"

# ============================================================
# ELEMENT ALIAS MAPPING
# ============================================================

ELEMENT_ALIAS_MAP = {
    'K': 'K_pct', 'K2O': 'K_pct', 'K_PCT': 'K_pct', 'K%': 'K_pct',
    'MG': 'Mg_pct', 'MGO': 'Mg_pct', 'MG_PCT': 'Mg_pct', 'MG%': 'Mg_pct',
    'CA': 'Ca_pct', 'CAO': 'Ca_pct', 'CA_PCT': 'Ca_pct', 'CA%': 'Ca_pct',
    'NA': 'Na_pct', 'NA2O': 'Na_pct', 'NA_PCT': 'Na_pct', 'NA%': 'Na_pct',
    'FE': 'Fe_pct', 'FE2O3': 'Fe_pct', 'FEO': 'Fe_pct', 'FE_PCT': 'Fe_pct', 'FE%': 'Fe_pct',
    'AU': 'Au_ppm', 'AU_PPM': 'Au_ppm', 'AU_PPB': 'Au_ppm',
    'CU': 'Cu_ppm', 'CU_PPM': 'Cu_ppm', 'CU_PCT': 'Cu_ppm',
    'ZN': 'Zn_ppm', 'ZN_PPM': 'Zn_ppm',
    'PB': 'Pb_ppm', 'PB_PPM': 'Pb_ppm',
    'AG': 'Ag_ppm', 'AG_PPM': 'Ag_ppm',
    'AS': 'As_ppm', 'AS_PPM': 'As_ppm',
    'SB': 'Sb_ppm', 'SB_PPM': 'Sb_ppm',
    'MO': 'Mo_ppm', 'MO_PPM': 'Mo_ppm',
    'SIO2': 'SiO2_pct', 'SIO2_PCT': 'SiO2_pct', 'SIO2%': 'SiO2_pct',
    'AL': 'Al_pct', 'AL2O3': 'Al_pct', 'AL_PCT': 'Al_pct', 'AL%': 'Al_pct',
}

# ============================================================
# DATA CLEANING ENGINE
# ============================================================

def standardize_element_names(df):
    """Normalize chemical headers to canonical standard names."""
    renames = {}
    for col in df.columns:
        clean_col = re.sub(r'[^a-zA-Z0-9%_]', '', str(col)).upper()
        if clean_col in ELEMENT_ALIAS_MAP:
            renames[col] = ELEMENT_ALIAS_MAP[clean_col]
        else:
            for key, std in ELEMENT_ALIAS_MAP.items():
                if key in clean_col:
                    renames[col] = std
                    break
    return df.rename(columns=renames)

def clean_dataframe(df):
    """Universal Cleaning Function for Geochemical Datasets."""
    if df is None or df.empty:
        return df
    
    df.columns = [str(c).strip() for c in df.columns]
    df = df.loc[:, ~df.columns.duplicated()]
    df = standardize_element_names(df)

    missing_vals = ['', 'nan', 'NaN', 'null', 'None', 'NA', '-', '--', 'n/a', '?']
    df = df.replace(missing_vals, np.nan)

    text_cols = ['sample', 'id', 'point', 'lithology', 'comments', 'unit', 'rock_type']
    for col in df.columns:
        is_text = any(t in col.lower() for t in text_cols)
        if not is_text:
            cleaned_series = df[col].astype(str).str.replace(r'^[<>]', '', regex=True)
            cleaned_series = pd.to_numeric(cleaned_series.str.replace(',', '.'), errors='coerce')
            if cleaned_series.notna().sum() > 0:
                df[col] = cleaned_series

    df = df.dropna(how='all')
    return df

def read_any_file(path):
    """Read any file format with auto-detection."""
    ext = Path(path).suffix.lower()
    if ext in ['.xlsx', '.xls', '.xlsm']:
        return pd.read_excel(path)
    
    encodings = ["utf-8", "cp1252", "latin-1", "iso-8859-1", "utf-16"]
    seps = [";", ",", "\t", "|"]
    
    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, engine="python", on_bad_lines="skip")
                if df.shape[1] > 1:
                    return df
            except Exception:
                pass
    
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            txt = f.read(3000)
        sep = max([";", ",", "\t", "|"], key=lambda d: txt.count(d))
        return pd.read_csv(path, sep=sep, engine="python", on_bad_lines="skip")
    except Exception:
        return pd.read_csv(path, engine="python", on_bad_lines="skip")

# ============================================================
# GEOCHEMISTRY ENGINE
# ============================================================

class GeochemistryEngine:
    """Smart geochemistry calculations with auto-detection"""
    
    @staticmethod
    def calculate_all(df):
        results = {
            'alteration_indices': {},
            'element_ratios': {},
            'mineralization_indices': {},
            'grade_stats': {},
            'available': {'alteration': False, 'ratios': False, 'mineralization': False, 'grades': False}
        }
        
        # Get numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        
        # 1. Alteration Indices
        if 'K_pct' in df.columns and 'Mg_pct' in df.columns and 'Ca_pct' in df.columns and 'Na_pct' in df.columns:
            try:
                results['alteration_indices']['Ishikawa AI'] = 100 * (df['K_pct'] + df['Mg_pct']) / (
                    df['K_pct'] + df['Mg_pct'] + df['Ca_pct'] + df['Na_pct'] + 0.0001)
                results['available']['alteration'] = True
            except: pass
        
        if 'Fe_pct' in df.columns and 'Mg_pct' in df.columns and 'Na_pct' in df.columns and 'K_pct' in df.columns:
            try:
                results['alteration_indices']['CCPI'] = 100 * (df['Fe_pct'] + df['Mg_pct']) / (
                    df['Fe_pct'] + df['Mg_pct'] + df['Na_pct'] + df['K_pct'] + 0.0001)
                results['available']['alteration'] = True
            except: pass
        
        # Sericite Index
        if 'K_pct' in df.columns and 'Na_pct' in df.columns:
            try:
                results['alteration_indices']['Sericite Index'] = df['K_pct'] / (df['K_pct'] + df['Na_pct'] + 0.0001)
                results['available']['alteration'] = True
            except: pass
        
        # 2. Element Ratios
        ratio_pairs = [
            ('Au_ppm', 'Cu_ppm', 'Au/Cu'), ('Zn_ppm', 'Pb_ppm', 'Zn/Pb'),
            ('Cu_ppm', 'Zn_ppm', 'Cu/Zn'), ('Au_ppm', 'Ag_ppm', 'Au/Ag'),
            ('Au_ppm', 'As_ppm', 'Au/As'), ('Cu_ppm', 'Mo_ppm', 'Cu/Mo')
        ]
        for c1, c2, name in ratio_pairs:
            if c1 in df.columns and c2 in df.columns:
                try:
                    results['element_ratios'][name] = df[c1] / (df[c2] + 0.0001)
                    results['available']['ratios'] = True
                except: pass
        
        # 3. Mineralization Indices
        if 'Au_ppm' in df.columns:
            try:
                results['mineralization_indices']['Gold Index'] = df['Au_ppm'] * 100
                results['available']['mineralization'] = True
            except: pass
        
        base_metals = [c for c in ['Cu_ppm', 'Zn_ppm', 'Pb_ppm'] if c in df.columns]
        if base_metals:
            try:
                results['mineralization_indices']['Base Metal Index'] = sum(df[c].fillna(0) for c in base_metals) / len(base_metals)
                results['available']['mineralization'] = True
            except: pass
        
        pathfinders = [c for c in ['As_ppm', 'Sb_ppm', 'Mo_ppm'] if c in df.columns]
        if pathfinders:
            try:
                results['mineralization_indices']['Pathfinder Index'] = sum(df[c].fillna(0) for c in pathfinders) / len(pathfinders)
                results['available']['mineralization'] = True
            except: pass
        
        # 4. Grade Statistics
        key_elements = ['Au_ppm', 'Cu_ppm', 'Zn_ppm', 'Pb_ppm', 'Ag_ppm', 'As_ppm']
        for elem in key_elements:
            if elem in df.columns:
                data = df[elem].dropna()
                if len(data) > 0:
                    threshold = data.quantile(0.75)
                    results['grade_stats'][elem] = {
                        'Mean': data.mean(), 'Median': data.median(), 'Std Dev': data.std(),
                        'Minimum': data.min(), 'Maximum': data.max(),
                        'N': len(data), 'Threshold': threshold,
                        'Above Threshold': (data > threshold).sum(),
                        '% Above Threshold': (data > threshold).sum() / len(data) * 100,
                        'Grade Class': 'HIGH' if data.mean() > threshold * 2 else 'MEDIUM' if data.mean() > threshold else 'LOW'
                    }
                    results['available']['grades'] = True
        
        return results

# ============================================================
# MAIN APPLICATION WINDOW
# ============================================================

class ElementalSocietyAnalysis(QMainWindow):
    def __init__(self):
        super().__init__()
        self.df = None
        self.cleaned_df = None
        self.derived_df = None
        self.is_processing = False
        self.init_ui()
        
    def init_ui(self):
        self.setWindowTitle(f"🧪 {APP_NAME} v{APP_VERSION} (Code: {APP_CODE})")
        self.setGeometry(50, 50, 1600, 950)
        
        # Central widget
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout()
        main_layout.setContentsMargins(10, 10, 10, 10)
        main_layout.setSpacing(10)
        central.setLayout(main_layout)
        
        # Header
        header = self.create_header()
        main_layout.addWidget(header)
        
        # Main Toolbar
        toolbar = self.create_toolbar()
        main_layout.addWidget(toolbar)
        
        # Info Panel
        self.info_label = QLabel("📋 Welcome! Upload your geochemical data to begin analysis.")
        self.info_label.setObjectName("info_label")
        self.info_label.setStyleSheet("""
            font-size: 14px;
            font-weight: 500;
            padding: 15px 20px;
            background: qlineargradient(x1:0, y1:0, x2:1, y2:0,
                stop:0 #1a73e8, stop:1 #1e8e3e);
            color: white;
            border-radius: 8px;
        """)
        main_layout.addWidget(self.info_label)
        
        # Status Widget
        status_widget = self.create_status_widget()
        main_layout.addWidget(status_widget)
        
        # Tab Widget
        self.tab_widget = QTabWidget()
        self.tab_widget.setStyleSheet("""
            QTabWidget::pane {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 8px;
                margin-top: -1px;
            }
            QTabBar::tab {
                padding: 12px 24px;
                margin-right: 2px;
                border-top-left-radius: 8px;
                border-top-right-radius: 8px;
                background-color: #f1f3f4;
                color: #5f6368;
                font-weight: 500;
                font-size: 13px;
            }
            QTabBar::tab:selected {
                background-color: white;
                color: #1a73e8;
                border-bottom: 3px solid #1a73e8;
            }
        """)
        main_layout.addWidget(self.tab_widget)
        
        # Create tabs
        self.create_welcome_tab()
        self.create_manage_tab()
        self.create_eda_tab()
        self.create_analyze_tab()
        self.create_geochemistry_tab()
        self.create_log_tab()
        
        # Status bar
        self.statusBar = QStatusBar()
        self.setStatusBar(self.statusBar)
        
        self.progress_bar = QProgressBar()
        self.progress_bar.setVisible(False)
        self.progress_bar.setMaximumWidth(250)
        self.statusBar.addPermanentWidget(self.progress_bar)
        self.statusBar.showMessage("✅ Ready")
        
        # Apply styles
        self.setStyleSheet(self.get_stylesheet())
        
        # Log
        self.log_message(f"🔬 {APP_NAME} v{APP_VERSION} initialized")
        self.log_message(f"📌 Project Code: {APP_CODE}")
    
    def get_stylesheet(self):
        return """
            QMainWindow { background-color: #f0f2f5; }
            QToolBar {
                background-color: white;
                border: none;
                border-bottom: 2px solid #1a73e8;
                spacing: 3px;
                padding: 5px 10px;
            }
            QToolBar QToolButton {
                border: none;
                border-radius: 4px;
                padding: 8px 16px;
                font-weight: 500;
                color: #3c4043;
                font-size: 12px;
            }
            QToolBar QToolButton:hover {
                background-color: #e8f0fe;
                color: #1a73e8;
            }
            QToolBar QToolButton#upload_btn {
                background-color: #1a73e8;
                color: white;
                padding: 8px 20px;
                border-radius: 4px;
            }
            QToolBar QToolButton#upload_btn:hover { background-color: #1557b0; }
            QToolBar QToolButton#analyze_btn {
                background-color: #1e8e3e;
                color: white;
                padding: 8px 20px;
                border-radius: 4px;
            }
            QToolBar QToolButton#analyze_btn:hover { background-color: #166b31; }
            QPushButton {
                background-color: #1a73e8;
                color: white;
                padding: 10px 24px;
                border: none;
                border-radius: 4px;
                font-weight: 500;
                font-size: 13px;
            }
            QPushButton:hover { background-color: #1557b0; }
            QPushButton#danger_btn { background-color: #d93025; }
            QPushButton#danger_btn:hover { background-color: #b3261e; }
            QPushButton#success_btn { background-color: #1e8e3e; }
            QPushButton#success_btn:hover { background-color: #166b31; }
            QTableWidget {
                background-color: white;
                gridline-color: #e8eaed;
                border: none;
                border-radius: 8px;
                alternate-background-color: #f8f9fa;
            }
            QTableWidget::item { padding: 8px; }
            QTableWidget::item:selected {
                background-color: #e8f0fe;
                color: #1a73e8;
            }
            QHeaderView::section {
                background-color: #f8f9fa;
                padding: 12px;
                border: none;
                border-bottom: 2px solid #dadce0;
                font-weight: 600;
                color: #3c4043;
            }
            QTextEdit {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 8px;
                padding: 10px;
                font-family: 'Segoe UI', sans-serif;
            }
            QComboBox {
                padding: 8px 12px;
                border: 1px solid #dadce0;
                border-radius: 4px;
                background-color: white;
                min-width: 120px;
            }
            QProgressBar {
                border: none;
                border-radius: 4px;
                background-color: #e8eaed;
                height: 20px;
                text-align: center;
            }
            QProgressBar::chunk {
                background-color: #1a73e8;
                border-radius: 4px;
            }
            QGroupBox {
                font-weight: 600;
                border: 1px solid #dadce0;
                border-radius: 8px;
                margin-top: 10px;
                padding-top: 15px;
            }
            QGroupBox::title {
                subcontrol-origin: margin;
                left: 10px;
                padding: 0 10px 0 10px;
                background-color: white;
            }
            QLabel#status_idle {
                padding: 4px 15px;
                border-radius: 12px;
                background-color: #e8eaed;
                color: #3c4043;
                font-weight: 500;
            }
            QLabel#status_processing {
                padding: 4px 15px;
                border-radius: 12px;
                background-color: #fef7e0;
                color: #e37400;
                font-weight: 600;
            }
            QLabel#status_success {
                padding: 4px 15px;
                border-radius: 12px;
                background-color: #e6f4ea;
                color: #1e8e3e;
                font-weight: 600;
            }
            QLabel#status_error {
                padding: 4px 15px;
                border-radius: 12px;
                background-color: #fce8e6;
                color: #d93025;
                font-weight: 600;
            }
        """
    
    def create_header(self):
        widget = QWidget()
        widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border-bottom: 2px solid #1a73e8;
                border-radius: 8px 8px 0 0;
                padding: 10px 20px;
            }
        """)
        layout = QHBoxLayout()
        widget.setLayout(layout)
        
        title_label = QLabel("🧪 Elemental Society Analysis")
        title_label.setStyleSheet("font-size: 22px; font-weight: 700; color: #1a73e8;")
        layout.addWidget(title_label)
        
        version_label = QLabel(f"v{APP_VERSION} | Code: {APP_CODE}")
        version_label.setStyleSheet("font-size: 13px; color: #5f6368; padding-left: 10px;")
        layout.addWidget(version_label)
        
        layout.addStretch()
        
        dev_label = QLabel(f"👨‍🔬 {DEVELOPER}")
        dev_label.setStyleSheet("font-size: 13px; color: #5f6368;")
        layout.addWidget(dev_label)
        
        return widget
    
    def create_toolbar(self):
        toolbar = QToolBar()
        toolbar.setMovable(False)
        toolbar.setIconSize(QSize(32, 32))
        
        # Upload
        upload_btn = QToolButton()
        upload_btn.setObjectName("upload_btn")
        upload_btn.setText("📁 Upload")
        upload_btn.setToolTip("Upload geochemical data file")
        upload_btn.clicked.connect(self.upload_file)
        toolbar.addWidget(upload_btn)
        
        toolbar.addSeparator()
        
        # Clean
        clean_btn = QToolButton()
        clean_btn.setText("🧹 Clean")
        clean_btn.setToolTip("Clean the loaded data")
        clean_btn.clicked.connect(self.clean_data)
        toolbar.addWidget(clean_btn)
        
        # Analyze
        analyze_btn = QToolButton()
        analyze_btn.setObjectName("analyze_btn")
        analyze_btn.setText("📊 Analyze")
        analyze_btn.setToolTip("Run comprehensive EDA")
        analyze_btn.clicked.connect(self.run_analysis)
        toolbar.addWidget(analyze_btn)
        
        toolbar.addSeparator()
        
        # Export
        export_btn = QToolButton()
        export_btn.setText("💾 Export")
        export_btn.setToolTip("Export cleaned data")
        export_btn.clicked.connect(self.export_data)
        toolbar.addWidget(export_btn)
        
        # Report
        report_btn = QToolButton()
        report_btn.setText("📄 Report")
        report_btn.setToolTip("Generate report")
        report_btn.clicked.connect(self.show_report)
        toolbar.addWidget(report_btn)
        
        toolbar.addSeparator()
        
        # Help
        help_btn = QToolButton()
        help_btn.setText("❓ Help")
        help_btn.setToolTip("Show help")
        help_btn.clicked.connect(self.show_help)
        toolbar.addWidget(help_btn)
        
        about_btn = QToolButton()
        about_btn.setText("ℹ️ About")
        about_btn.setToolTip("About")
        about_btn.clicked.connect(self.show_about)
        toolbar.addWidget(about_btn)
        
        spacer = QWidget()
        spacer.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Preferred)
        toolbar.addWidget(spacer)
        
        self.file_info_label = QLabel("📁 No file loaded")
        self.file_info_label.setStyleSheet("color: #5f6368; padding: 5px 10px; font-size: 12px;")
        toolbar.addWidget(self.file_info_label)
        
        return toolbar
    
    def create_status_widget(self):
        widget = QWidget()
        widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 8px;
                padding: 8px 15px;
            }
        """)
        layout = QHBoxLayout()
        layout.setContentsMargins(10, 5, 10, 5)
        widget.setLayout(layout)
        
        layout.addWidget(QLabel("Status:"))
        
        self.status_indicator = QLabel("⚪ Idle")
        self.status_indicator.setObjectName("status_idle")
        layout.addWidget(self.status_indicator)
        
        self.status_message = QLabel("Ready")
        self.status_message.setStyleSheet("color: #5f6368; font-weight: 500;")
        layout.addWidget(self.status_message)
        
        self.wait_label = QLabel("⏳ Processing...")
        self.wait_label.setStyleSheet("color: #e37400; font-weight: 600;")
        self.wait_label.setVisible(False)
        layout.addWidget(self.wait_label)
        
        layout.addStretch()
        
        self.process_info = QLabel("")
        self.process_info.setStyleSheet("color: #9aa0a6; font-size: 11px;")
        layout.addWidget(self.process_info)
        
        return widget
    
    # ============================================================
    # TAB CREATION
    # ============================================================
    
    def create_welcome_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        welcome_widget = QWidget()
        welcome_widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 12px;
                padding: 20px;
            }
        """)
        welcome_layout = QVBoxLayout()
        welcome_widget.setLayout(welcome_layout)
        
        welcome_text = QLabel("""
        <h1 style='color: #1a73e8;'>🧪 Welcome to Elemental Society Analysis</h1>
        <h3 style='color: #3c4043;'>Professional Geochemical Data Analysis Suite v1.0</h3>
        <br>
        <p style='font-size: 14px; color: #5f6368;'>
        <b>Complete solution for exploration geochemistry data analysis.</b>
        </p>
        <br>
        <table style='width: 100%;'>
            <tr>
                <td style='padding: 10px; border: none;'><b>📁 1. Upload</b><br>Load CSV or Excel file</td>
                <td style='padding: 10px; border: none;'><b>🧹 2. Clean</b><br>Handle missing values, outliers</td>
            </tr>
            <tr>
                <td style='padding: 10px; border: none;'><b>📊 3. Analyze</b><br>Run comprehensive EDA</td>
                <td style='padding: 10px; border: none;'><b>📉 4. Visualize</b><br>Professional plots</td>
            </tr>
            <tr>
                <td style='padding: 10px; border: none;'><b>🧪 5. Geochemistry</b><br>Alteration indices, ratios</td>
                <td style='padding: 10px; border: none;'><b>📄 6. Report</b><br>Generate HTML report</td>
            </tr>
        </table>
        <br>
        <p style='color: #5f6368;'><b>💡 Tip:</b> Start by clicking <b style='color: #1a73e8;'>Upload</b> above.</p>
        """)
        welcome_text.setWordWrap(True)
        welcome_layout.addWidget(welcome_text)
        
        layout.addWidget(welcome_widget)
        layout.addStretch()
        self.tab_widget.addTab(tab, "🏠 Welcome")
    
    def create_manage_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        # Controls
        controls = QHBoxLayout()
        self.data_info = QLabel("No data loaded")
        controls.addWidget(self.data_info)
        controls.addStretch()
        controls.addWidget(QLabel("Show rows:"))
        self.row_limit = QSpinBox()
        self.row_limit.setRange(10, 5000)
        self.row_limit.setValue(500)
        self.row_limit.valueChanged.connect(self.refresh_data)
        controls.addWidget(self.row_limit)
        
        layout.addLayout(controls)
        
        self.data_table = QTableWidget()
        self.data_table.setAlternatingRowColors(True)
        self.data_table.setSortingEnabled(True)
        layout.addWidget(self.data_table)
        
        self.tab_widget.addTab(tab, "📊 Data")
    
    def create_eda_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        # Controls
        controls = QHBoxLayout()
        controls.addWidget(QLabel("Variable:"))
        self.eda_var = QComboBox()
        self.eda_var.currentTextChanged.connect(self.plot_distribution)
        controls.addWidget(self.eda_var)
        
        controls.addWidget(QLabel("Type:"))
        self.eda_type = QComboBox()
        self.eda_type.addItems(["Histogram", "PDF", "CDF", "Box Plot"])
        self.eda_type.currentTextChanged.connect(self.plot_distribution)
        controls.addWidget(self.eda_type)
        controls.addStretch()
        layout.addLayout(controls)
        
        # Figure with proper import
        self.eda_figure = Figure(figsize=(10, 6))
        self.eda_canvas = FigureCanvas(self.eda_figure)
        layout.addWidget(self.eda_canvas)
        
        self.tab_widget.addTab(tab, "📈 EDA")
    
    def create_analyze_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        # Controls
        controls = QHBoxLayout()
        controls.addWidget(QLabel("Plot:"))
        self.plot_type = QComboBox()
        self.plot_type.addItems(["Correlation Matrix", "PCA Scores", "Scatter Plot"])
        self.plot_type.currentTextChanged.connect(self.update_plot)
        controls.addWidget(self.plot_type)
        
        controls.addWidget(QLabel("X:"))
        self.analyze_x = QComboBox()
        controls.addWidget(self.analyze_x)
        
        controls.addWidget(QLabel("Y:"))
        self.analyze_y = QComboBox()
        controls.addWidget(self.analyze_y)
        
        plot_btn = QPushButton("🎨 Generate")
        plot_btn.clicked.connect(self.update_plot)
        controls.addWidget(plot_btn)
        controls.addStretch()
        layout.addLayout(controls)
        
        # Figure with proper import
        self.analyze_figure = Figure(figsize=(10, 6))
        self.analyze_canvas = FigureCanvas(self.analyze_figure)
        layout.addWidget(self.analyze_canvas)
        
        self.tab_widget.addTab(tab, "📊 Analysis")
    
    def create_geochemistry_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        self.geochem_viewer = QTextEdit()
        self.geochem_viewer.setReadOnly(True)
        self.geochem_viewer.setFont(QFont("Segoe UI", 10))
        layout.addWidget(self.geochem_viewer)
        
        self.tab_widget.addTab(tab, "🧪 Geochemistry")
    
    def create_log_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        controls = QHBoxLayout()
        clear_btn = QPushButton("🗑️ Clear")
        clear_btn.clicked.connect(self.clear_log)
        controls.addWidget(clear_btn)
        controls.addStretch()
        layout.addLayout(controls)
        
        self.log_text = QTextEdit()
        self.log_text.setReadOnly(True)
        self.log_text.setFont(QFont("Consolas", 10))
        layout.addWidget(self.log_text)
        
        self.tab_widget.addTab(tab, "📝 Log")
    
    # ============================================================
    # STATUS FUNCTIONS
    # ============================================================
    
    def set_status(self, status, message="", wait=False):
        status_map = {
            'idle': ('⚪ Idle', 'status_idle', '#3c4043'),
            'processing': ('⚙️ Processing', 'status_processing', '#e37400'),
            'success': ('✅ Success', 'status_success', '#1e8e3e'),
            'error': ('❌ Error', 'status_error', '#d93025')
        }
        icon, obj_name, color = status_map.get(status, status_map['idle'])
        self.status_indicator.setText(icon)
        self.status_indicator.setObjectName(obj_name)
        self.status_indicator.setStyleSheet(f"""
            QLabel#{obj_name} {{
                padding: 4px 15px;
                border-radius: 12px;
                background-color: {color}20;
                color: {color};
                font-weight: 600;
            }}
        """)
        self.status_message.setText(message)
        self.wait_label.setVisible(wait)
        if wait:
            self.wait_label.setText("⏳ Processing...")
            QApplication.processEvents()
    
    def set_processing(self, status, message="", wait=True):
        self.is_processing = status
        if status:
            self.set_status('processing', message, wait)
            self.progress_bar.setVisible(True)
            self.progress_bar.setRange(0, 0)
            self.info_label.setText(f"⚙️ {message}")
            self.statusBar.showMessage(f"Processing: {message}")
        else:
            self.set_status('idle', 'Ready', False)
            self.progress_bar.setVisible(False)
            self.info_label.setText("✅ Ready")
            self.statusBar.showMessage("✅ Ready")
    
    def update_progress(self, message):
        self.process_info.setText(message)
        self.status_message.setText(message)
        self.log_message(f"⏳ {message}")
        QApplication.processEvents()
    
    # ============================================================
    # LOG FUNCTIONS
    # ============================================================
    
    def log_message(self, message):
        from datetime import datetime
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.append(f"[{timestamp}] {message}")
    
    def clear_log(self):
        self.log_text.clear()
        self.log_message("🗑️ Log cleared")
    
    # ============================================================
    # DATA FUNCTIONS
    # ============================================================
    
    def upload_file(self):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open Geochemical Data", "",
            "All Files (*);;CSV (*.csv);;Excel (*.xlsx *.xls);;Text (*.txt)"
        )
        if not file_path:
            return
        
        self.set_processing(True, "Loading file...")
        self.log_message(f"📁 Loading: {os.path.basename(file_path)}")
        self.update_progress(f"Reading: {os.path.basename(file_path)}")
        
        try:
            self.df = read_any_file(file_path)
            
            if self.df is not None and not self.df.empty:
                self.df = clean_dataframe(self.df)
                self.cleaned_df = self.df.copy()
                self.derived_df = self.df.copy()
                self.display_data()
                self.file_info_label.setText(f"📁 {os.path.basename(file_path)}")
                
                # Update selectors
                numeric_cols = self.df.select_dtypes(include=[np.number]).columns
                self.eda_var.clear()
                self.eda_var.addItems(numeric_cols)
                self.analyze_x.clear()
                self.analyze_x.addItems(numeric_cols)
                self.analyze_y.clear()
                self.analyze_y.addItems(numeric_cols)
                
                self.log_message(f"✅ Loaded: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
                self.set_status('success', f'Loaded: {self.df.shape[0]} rows')
                self.info_label.setText(f"📊 Data loaded: {self.df.shape[0]} rows × {self.df.shape[1]} columns")
                
                QMessageBox.information(self, "Success",
                    f"File loaded!\n\n📊 {self.df.shape[0]} rows × {self.df.shape[1]} columns")
            else:
                QMessageBox.warning(self, "Error", "Could not read file.")
                
        except Exception as e:
            self.log_message(f"❌ Error: {str(e)}")
            self.set_status('error', 'Load failed')
            QMessageBox.critical(self, "Error", f"Failed to load:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def clean_data(self):
        if self.df is None or self.df.empty:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
        
        reply = QMessageBox.question(self, "Clean Data", "Clean the dataset?", QMessageBox.Yes | QMessageBox.No)
        if reply != QMessageBox.Yes:
            return
        
        self.set_processing(True, "Cleaning data...")
        self.log_message("🧹 Cleaning data...")
        self.update_progress("Cleaning data...")
        
        try:
            self.cleaned_df = clean_dataframe(self.df.copy())
            self.derived_df = self.cleaned_df.copy()
            self.display_data()
            
            self.log_message(f"✅ Cleaned: {len(self.cleaned_df)} rows")
            self.set_status('success', f'Cleaned: {len(self.cleaned_df)} rows')
            self.info_label.setText(f"🧹 Cleaned: {len(self.cleaned_df)} rows × {len(self.cleaned_df.columns)} columns")
            
            QMessageBox.information(self, "Success",
                f"Cleaning complete!\n\nBefore: {len(self.df)} rows\nAfter: {len(self.cleaned_df)} rows")
            
        except Exception as e:
            self.log_message(f"❌ Cleaning error: {str(e)}")
            self.set_status('error', 'Cleaning failed')
            QMessageBox.critical(self, "Error", f"Cleaning failed:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def run_analysis(self):
        if self.df is None:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
        
        self.set_processing(True, "Running analysis...")
        self.log_message("📊 Starting analysis...")
        self.update_progress("🔬 Running analysis...")
        
        try:
            self.update_progress("📊 Generating statistics...")
            self.generate_statistics()
            
            self.update_progress("📉 Creating visualizations...")
            self.plot_distribution()
            self.update_plot()
            
            self.update_progress("🧪 Calculating geochemistry...")
            self.calculate_geochemistry()
            
            self.log_message("✅ Analysis complete!")
            self.set_status('success', 'Analysis Complete!')
            self.info_label.setText("📊 Analysis Complete!")
            
            QMessageBox.information(self, "Success", "Analysis completed successfully!")
            
        except Exception as e:
            self.log_message(f"❌ Analysis error: {str(e)}")
            self.set_status('error', 'Analysis failed')
            QMessageBox.critical(self, "Error", f"Analysis failed:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def display_data(self):
        if self.derived_df is not None:
            df = self.derived_df
        elif self.cleaned_df is not None:
            df = self.cleaned_df
        elif self.df is not None:
            df = self.df
        else:
            return
        
        display_limit = min(self.row_limit.value(), len(df))
        self.data_table.setRowCount(display_limit)
        self.data_table.setColumnCount(len(df.columns))
        self.data_table.setHorizontalHeaderLabels(df.columns.astype(str))
        
        for i in range(display_limit):
            for j, col in enumerate(df.columns):
                val = df.iloc[i, j]
                if pd.isna(val):
                    self.data_table.setItem(i, j, QTableWidgetItem(""))
                else:
                    self.data_table.setItem(i, j, QTableWidgetItem(str(val)))
        
        self.data_table.resizeColumnsToContents()
        self.data_info.setText(f"📊 {len(df)} rows × {len(df.columns)} columns")
    
    def refresh_data(self):
        self.display_data()
    
    # ============================================================
    # STATISTICS FUNCTIONS
    # ============================================================
    
    def generate_statistics(self):
        if self.df is None:
            return
        
        numeric_df = self.df.select_dtypes(include=[np.number])
        if numeric_df.empty:
            return
        
        stats_text = []
        stats_text.append("="*80)
        stats_text.append("📊 STATISTICAL SUMMARY")
        stats_text.append("="*80)
        stats_text.append(f"\nDataset: {len(self.df)} rows × {len(self.df.columns)} columns")
        stats_text.append(f"Numeric columns: {len(numeric_df.columns)}\n")
        
        # Basic stats
        stats_text.append("📈 BASIC STATISTICS")
        stats_text.append("-"*60)
        stats_text.append(numeric_df.describe().to_string())
        stats_text.append("\n")
        
        # Variance
        stats_text.append("📊 VARIANCE")
        stats_text.append("-"*60)
        stats_text.append(numeric_df.var().to_string())
        stats_text.append("\n")
        
        # Skewness
        stats_text.append("📊 SKEWNESS")
        stats_text.append("-"*60)
        stats_text.append(numeric_df.skew().to_string())
        stats_text.append("\n")
        
        # Kurtosis
        stats_text.append("📊 KURTOSIS")
        stats_text.append("-"*60)
        stats_text.append(numeric_df.kurtosis().to_string())
        
        self.log_message("📊 Statistics generated")
    
    # ============================================================
    # PLOTTING FUNCTIONS
    # ============================================================
    
    def plot_distribution(self):
        if self.df is None:
            return
        
        var = self.eda_var.currentText()
        plot_type = self.eda_type.currentText()
        
        if not var or var not in self.df.columns:
            return
        
        data = self.df[var].dropna()
        if len(data) < 2:
            return
        
        self.eda_figure.clear()
        ax = self.eda_figure.add_subplot(111)
        
        if plot_type == "Histogram":
            ax.hist(data, bins=20, density=True, alpha=0.6, color='steelblue', edgecolor='black')
            ax.set_ylabel('Density')
            ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.2f}')
            ax.axvline(data.median(), color='green', linestyle='--', label=f'Median: {data.median():.2f}')
            ax.legend()
            
        elif plot_type == "PDF":
            ax.hist(data, bins=20, density=True, alpha=0.4, color='steelblue')
            x = np.linspace(data.min(), data.max(), 100)
            mu, std = stats.norm.fit(data)
            pdf = stats.norm.pdf(x, mu, std)
            ax.plot(x, pdf, 'r-', linewidth=2, label='Normal PDF')
            ax.set_ylabel('Density')
            ax.legend()
            
        elif plot_type == "CDF":
            sorted_data = np.sort(data)
            y = np.arange(1, len(sorted_data)+1) / len(sorted_data)
            ax.plot(sorted_data, y, 'b-', linewidth=2)
            ax.set_ylabel('Cumulative Probability')
            ax.grid(True, alpha=0.3)
            
        elif plot_type == "Box Plot":
            ax.boxplot(data, vert=True, patch_artist=True)
            ax.set_ylabel(var)
        
        ax.set_title(f'{var} - {plot_type}', fontweight='bold')
        ax.set_xlabel(var)
        ax.grid(True, alpha=0.3)
        
        self.eda_figure.tight_layout()
        self.eda_canvas.draw()
    
    def update_plot(self):
        if self.df is None:
            return
        
        plot_type = self.plot_type.currentText()
        
        self.analyze_figure.clear()
        ax = self.analyze_figure.add_subplot(111)
        
        numeric_df = self.df.select_dtypes(include=[np.number])
        
        if plot_type == "Correlation Matrix" and len(numeric_df.columns) > 1:
            corr = numeric_df.corr()
            sns.heatmap(corr, annot=True, cmap='coolwarm', center=0,
                       fmt='.2f', square=True, linewidths=0.5, ax=ax)
            ax.set_title('Correlation Matrix', fontweight='bold')
            
        elif plot_type == "PCA Scores" and len(numeric_df.columns) >= 2:
            clean_data = numeric_df.dropna()
            if len(clean_data) > 1:
                scaler = StandardScaler()
                scaled = scaler.fit_transform(clean_data)
                pca = SklearnPCA()
                scores = pca.fit_transform(scaled)
                explained = pca.explained_variance_ratio_
                
                scatter = ax.scatter(scores[:, 0], scores[:, 1],
                                   c=np.linalg.norm(scores[:, :2], axis=1),
                                   cmap='viridis', alpha=0.6,
                                   edgecolors='black', linewidth=0.5)
                ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
                ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
                ax.set_title('PCA Scores', fontweight='bold')
                ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
                ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
                ax.grid(True, alpha=0.3)
                self.analyze_figure.colorbar(scatter, ax=ax, label='Distance')
                
        elif plot_type == "Scatter Plot":
            x_col = self.analyze_x.currentText()
            y_col = self.analyze_y.currentText()
            
            if x_col and y_col and x_col in self.df.columns and y_col in self.df.columns:
                data = self.df[[x_col, y_col]].dropna()
                ax.scatter(data[x_col], data[y_col], alpha=0.6, color='steelblue', edgecolors='black')
                ax.set_xlabel(x_col)
                ax.set_ylabel(y_col)
                ax.set_title(f'Scatter Plot: {y_col} vs {x_col}', fontweight='bold')
                ax.grid(True, alpha=0.3)
        
        self.analyze_figure.tight_layout()
        self.analyze_canvas.draw()
    
    # ============================================================
    # GEOCHEMISTRY FUNCTIONS
    # ============================================================
    
    def calculate_geochemistry(self):
        if self.df is None:
            return
        
        geochem = GeochemistryEngine.calculate_all(self.df)
        
        html = []
        html.append("<html><head><style>")
        html.append("""
            body { font-family: 'Segoe UI', Arial, sans-serif; margin: 15px; }
            h1 { color: #1a73e8; border-bottom: 2px solid #1a73e8; padding-bottom: 8px; }
            h2 { color: #3c4043; margin-top: 20px; }
            .card { background-color: #f8f9fa; border-radius: 8px; padding: 15px; margin: 10px 0; border-left: 4px solid #1a73e8; }
            .stat-box { display: inline-block; margin: 3px; padding: 8px 15px; background: white; border-radius: 6px; border: 1px solid #dadce0; }
            .stat-value { font-weight: bold; color: #1a73e8; font-size: 16px; }
            .stat-label { font-size: 10px; color: #5f6368; }
            .badge { display: inline-block; padding: 2px 10px; border-radius: 10px; font-size: 11px; font-weight: 600; }
            .badge-high { background: #fce8e6; color: #d93025; }
            .badge-medium { background: #fef7e0; color: #e37400; }
            .badge-low { background: #e6f4ea; color: #1e8e3e; }
            .badge-available { background: #e6f4ea; color: #1e8e3e; }
            .badge-missing { background: #fce8e6; color: #d93025; }
        """)
        html.append("</style></head><body>")
        
        html.append(f"<h1>🧪 Exploration Geochemistry Report</h1>")
        html.append(f"<p><b>Generated:</b> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>")
        html.append(f"<p><b>Dataset:</b> {len(self.df)} samples × {len(self.df.columns)} variables</p>")
        
        # Available status
        avail = geochem.get('available', {})
        html.append("<div class='card'>")
        html.append("<h3>📊 Available Calculations</h3>")
        for key, val in avail.items():
            badge = "badge-available" if val else "badge-missing"
            status = "✅ Available" if val else "❌ Not Available"
            html.append(f"<span class='badge {badge}'>{key.replace('_', ' ').title()}: {status}</span> ")
        html.append("</div>")
        
        # Alteration Indices
        html.append("<h2>📊 Alteration Indices</h2>")
        if geochem['alteration_indices']:
            html.append("<div class='card'>")
            for name, values in geochem['alteration_indices'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{values.min():.4f} - {values.max():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("<div class='card'><p style='color:#5f6368;'>No alteration indices could be calculated.</p></div>")
        
        # Element Ratios
        html.append("<h2>📊 Element Ratios</h2>")
        if geochem['element_ratios']:
            html.append("<div class='card'>")
            for name, values in geochem['element_ratios'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("<div class='card'><p style='color:#5f6368;'>No element ratios could be calculated.</p></div>")
        
        # Mineralization Indices
        html.append("<h2>📈 Mineralization Indices</h2>")
        if geochem['mineralization_indices']:
            html.append("<div class='card'>")
            for name, values in geochem['mineralization_indices'].items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <b>{name}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("<div class='card'><p style='color:#5f6368;'>No mineralization indices could be calculated.</p></div>")
        
        # Grade Statistics
        html.append("<h2>📊 Grade Statistics</h2>")
        if geochem['grade_stats']:
            html.append("<div class='card'>")
            for element, stats in geochem['grade_stats'].items():
                if stats:
                    grade_class = stats.get('Grade Class', 'LOW')
                    badge_class = 'badge-high' if grade_class == 'HIGH' else 'badge-medium' if grade_class == 'MEDIUM' else 'badge-low'
                    html.append(f"""
                    <div style='margin: 10px 0; padding: 10px; background: white; border-radius: 6px; border: 1px solid #e8eaed;'>
                        <b>{element}</b>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{stats.get('Mean', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{stats.get('Median', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Std Dev</div><div class='stat-value'>{stats.get('Std Dev', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{stats.get('Minimum', 0):.4f} - {stats.get('Maximum', 0):.4f}</div></div>
                        <div style='margin-top: 5px;'>
                            <span class='badge {badge_class}'>Grade: {grade_class}</span>
                            <span style='margin-left: 10px;'>Threshold: {stats.get('Threshold', 0):.4f}</span>
                            <span style='margin-left: 10px;'>Above: {stats.get('Above Threshold', 0)} ({stats.get('% Above Threshold', 0):.1f}%)</span>
                        </div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("<div class='card'><p style='color:#5f6368;'>No grade statistics could be calculated.</p></div>")
        
        # Summary
        html.append("""
        <div style='margin-top: 20px; padding: 15px; background-color: #e8f0fe; border-radius: 8px;'>
            <p><b>📌 Summary</b></p>
            <table style='width: auto;'>
        """)
        html.append(f"<tr><td>Total Samples</td><td><b>{len(self.df)}</b></td></tr>")
        html.append(f"<tr><td>Total Variables</td><td><b>{len(self.df.columns)}</b></td></tr>")
        html.append(f"<tr><td>Alteration Indices</td><td><b>{len(geochem['alteration_indices'])}</b></td></tr>")
        html.append(f"<tr><td>Element Ratios</td><td><b>{len(geochem['element_ratios'])}</b></td></tr>")
        html.append(f"<tr><td>Mineralization Indices</td><td><b>{len(geochem['mineralization_indices'])}</b></td></tr>")
        html.append(f"<tr><td>Grade Statistics</td><td><b>{len(geochem['grade_stats'])}</b></td></tr>")
        html.append("</table></div></body></html>")
        
        self.geochem_viewer.setHtml("\n".join(html))
    
    def show_report(self):
        for i in range(self.tab_widget.count()):
            if self.tab_widget.tabText(i) == "📄 Report":
                self.tab_widget.setCurrentIndex(i)
                break
    
    # ============================================================
    # EXPORT FUNCTIONS
    # ============================================================
    
    def export_data(self):
        if self.df is None:
            QMessageBox.warning(self, "Warning", "No data to export!")
            return
        
        df = self.derived_df if self.derived_df is not None else self.cleaned_df if self.cleaned_df is not None else self.df
        
        formats = {
            "CSV (;)": "csv_semicolon",
            "CSV (,)": "csv_comma",
            "Excel (.xlsx)": "excel",
            "JSON": "json"
        }
        
        format_choice, ok = QInputDialog.getItem(
            self, "Export Format", "Select format:", list(formats.keys()), 0, False)
        if not ok:
            return
        
        file_ext_map = {
            "csv_semicolon": "csv",
            "csv_comma": "csv",
            "excel": "xlsx",
            "json": "json"
        }
        
        file_path, _ = QFileDialog.getSaveFileName(
            self, "Export Data", f"cleaned_data.{file_ext_map[formats[format_choice]]}",
            f"{format_choice} (*.{file_ext_map[formats[format_choice]]})")
        
        if file_path:
            try:
                if formats[format_choice] == "csv_semicolon":
                    df.to_csv(file_path, sep=';', index=False, encoding='utf-8-sig')
                elif formats[format_choice] == "csv_comma":
                    df.to_csv(file_path, sep=',', index=False, encoding='utf-8-sig')
                elif formats[format_choice] == "excel":
                    df.to_excel(file_path, index=False)
                elif formats[format_choice] == "json":
                    df.to_json(file_path, orient='records', indent=2)
                
                self.log_message(f"✅ Exported: {file_path}")
                QMessageBox.information(self, "Success", f"Data exported to:\n{file_path}")
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")
    
    # ============================================================
    # HELP AND ABOUT
    # ============================================================
    
    def show_help(self):
        help_text = f"""
🔬 {APP_NAME} v{APP_VERSION}

📌 Project Code: {APP_CODE}

👨‍🔬 Developer: {DEVELOPER}

🚀 QUICK START

1. 📁 Upload your data
2. 🧹 Clean the data
3. 📊 Run analysis
4. 📉 View visualizations
5. 🧪 Check geochemistry
6. 📄 Generate report

🔧 FEATURES

Data Management:
• Upload CSV/Excel files
• Clean data (missing values, outliers)

Statistics:
• Mean, Median, Std Dev
• Variance, Skewness, Kurtosis
• Correlation Matrix

Visualization:
• Correlation Heatmap
• PCA Scores
• Distributions (Histogram, PDF, CDF, Box Plot)

Geochemistry:
• Alteration Indices
• Element Ratios
• Mineralization Indices
• Grade Statistics

Export:
• CSV, Excel, JSON

Copyright © 2026 {DEVELOPER}
"""
        QMessageBox.information(self, "Help", help_text)
    
    def show_about(self):
        about_text = f"""
🔬 {APP_NAME}

Version: {APP_VERSION}
Project Code: {APP_CODE}

Developer: {DEVELOPER}
Email: {EMAIL}

Description:
Professional geochemical data cleaning and 
exploratory data analysis (EDA) suite for
mineral exploration and geochemistry.

Features:
• Data Cleaning Engine
• Statistical Analysis
• Correlation Matrix
• Principal Component Analysis
• Distribution Analysis (PDF, CDF)
• Exploration Geochemistry
• Alteration Indices
• Grade Statistics

Copyright © 2026 {DEVELOPER}
All rights reserved.
"""
        QMessageBox.about(self, "About", about_text)

# ============================================================
# MAIN APPLICATION
# ============================================================

def main():
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    font = QFont("Segoe UI", 10)
    app.setFont(font)
    window = ElementalSocietyAnalysis()
    window.show()
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()

In [ ]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - COMPLETE FIXED VERSION
# Project Code: 1010
# Developer: Mohamed Almoufty Abdoalmoutleb Abaker
# ============================================================

import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar
from matplotlib.figure import Figure
import seaborn as sns
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# APPLICATION METADATA
# ============================================================

APP_NAME = "Elemental Society Analysis"
APP_VERSION = "3.0.0"
APP_CODE = "1010"
DEVELOPER = "Mohamed Almoufty Abdoalmoutleb Abaker"
EMAIL = "almoufty121@gmail.com"
BACKGROUND = "Exploration Geologist | Msc Earth Science Engineering (TIMREX Program)"

# ============================================================
# PROFESSIONAL STYLESHEET
# ============================================================

STYLESHEET = """
    QMainWindow { background-color: #f0f2f5; }
    QLabel#title_label {
        font-size: 24px;
        font-weight: bold;
        color: #1a73e8;
        padding: 10px;
    }
    QToolBar {
        background-color: white;
        border: none;
        spacing: 3px;
        padding: 5px;
        margin: 0px;
    }
    QToolBar#main_toolbar {
        background-color: white;
        border-bottom: 1px solid #dadce0;
        padding: 5px 10px;
    }
    QToolBar QToolButton {
        border: none;
        border-radius: 4px;
        padding: 8px 14px;
        font-weight: 500;
        color: #3c4043;
        font-size: 12px;
    }
    QToolBar QToolButton:hover {
        background-color: #e8f0fe;
        color: #1a73e8;
    }
    QToolBar QToolButton#upload_btn {
        background-color: #1a73e8;
        color: white;
        padding: 8px 20px;
        border-radius: 4px;
    }
    QTabWidget::pane {
        background-color: white;
        border: 1px solid #dadce0;
        border-radius: 8px;
        margin-top: -1px;
    }
    QTabBar::tab {
        padding: 12px 24px;
        margin-right: 2px;
        border-top-left-radius: 8px;
        border-top-right-radius: 8px;
        background-color: #f1f3f4;
        color: #5f6368;
        font-weight: 500;
        font-size: 13px;
    }
    QTabBar::tab:selected {
        background-color: white;
        color: #1a73e8;
        border-bottom: 3px solid #1a73e8;
    }
    QTableWidget {
        background-color: white;
        gridline-color: #e8eaed;
        border: none;
        border-radius: 8px;
        alternate-background-color: #f8f9fa;
    }
    QTableWidget::item {
        padding: 8px;
    }
    QTableWidget::item:selected {
        background-color: #e8f0fe;
        color: #1a73e8;
    }
    QHeaderView::section {
        background-color: #f8f9fa;
        padding: 12px;
        border: none;
        border-bottom: 2px solid #dadce0;
        font-weight: 600;
        color: #3c4043;
    }
    QPushButton {
        background-color: #1a73e8;
        color: white;
        padding: 10px 24px;
        border: none;
        border-radius: 4px;
        font-weight: 500;
        font-size: 13px;
    }
    QPushButton:hover { background-color: #1557b0; }
    QPushButton#danger_btn { background-color: #d93025; }
    QPushButton#danger_btn:hover { background-color: #b3261e; }
    QPushButton#success_btn { background-color: #1e8e3e; }
    QPushButton#success_btn:hover { background-color: #166b31; }
    QLabel#status_idle {
        padding: 4px 15px;
        border-radius: 12px;
        background-color: #e8eaed;
        color: #3c4043;
        font-weight: 500;
    }
    QLabel#status_processing {
        padding: 4px 15px;
        border-radius: 12px;
        background-color: #fef7e0;
        color: #e37400;
        font-weight: 600;
    }
    QLabel#status_success {
        padding: 4px 15px;
        border-radius: 12px;
        background-color: #e6f4ea;
        color: #1e8e3e;
        font-weight: 600;
    }
    QLabel#status_error {
        padding: 4px 15px;
        border-radius: 12px;
        background-color: #fce8e6;
        color: #d93025;
        font-weight: 600;
    }
    QLabel#info_label {
        font-size: 14px;
        font-weight: 500;
        padding: 15px 20px;
        background: qlineargradient(x1:0, y1:0, x2:1, y2:0,
            stop:0 #1a73e8, stop:1 #1e8e3e);
        color: white;
        border-radius: 8px;
    }
    QStatusBar {
        background-color: #f8f9fa;
        border-top: 1px solid #dadce0;
        padding: 5px;
    }
    QTextEdit {
        background-color: white;
        border: 1px solid #dadce0;
        border-radius: 8px;
        padding: 10px;
        font-family: 'Segoe UI', sans-serif;
    }
    QComboBox {
        padding: 8px 12px;
        border: 1px solid #dadce0;
        border-radius: 4px;
        background-color: white;
        min-width: 120px;
    }
    QSpinBox, QDoubleSpinBox {
        padding: 8px 12px;
        border: 1px solid #dadce0;
        border-radius: 4px;
        background-color: white;
    }
    QProgressBar {
        border: none;
        border-radius: 4px;
        background-color: #e8eaed;
        height: 20px;
        text-align: center;
    }
    QProgressBar::chunk {
        background-color: #1a73e8;
        border-radius: 4px;
    }
"""

# ============================================================
# EXPLORATION GEOCHEMISTRY CALCULATIONS (COMPLETELY REWRITTEN)
# ============================================================

class ExplorationGeochemistry:
    """Professional exploration geochemistry calculations - NOW WORKS WITH ANY DATA"""
    
    @staticmethod
    def calculate_alteration_indices(df):
        """
        Calculate alteration indices from available elements.
        Automatically detects which elements are present.
        """
        results = {}
        available_cols = df.columns.tolist()
        
        # 1. Ishikawa Alteration Index (AI)
        # AI = 100 * (K2O + MgO) / (K2O + MgO + CaO + Na2O)
        required = ['K_pct', 'Mg_pct', 'Ca_pct', 'Na_pct']
        if all(col in available_cols for col in required):
            try:
                results['Ishikawa AI'] = 100 * (df['K_pct'] + df['Mg_pct']) / (
                    df['K_pct'] + df['Mg_pct'] + df['Ca_pct'] + df['Na_pct'] + 0.0001)
            except:
                pass
        
        # 2. CCPI (Chlorite-Carbonate-Pyrite Index)
        required = ['Fe_pct', 'Mg_pct', 'Na_pct', 'K_pct']
        if all(col in available_cols for col in required):
            try:
                results['CCPI'] = 100 * (df['Fe_pct'] + df['Mg_pct']) / (
                    df['Fe_pct'] + df['Mg_pct'] + df['Na_pct'] + df['K_pct'] + 0.0001)
            except:
                pass
        
        # 3. Sericite Index
        required = ['K_pct', 'Na_pct']
        if all(col in available_cols for col in required):
            try:
                results['Sericite Index'] = df['K_pct'] / (df['K_pct'] + df['Na_pct'] + 0.0001)
            except:
                pass
        
        # 4. Propylitic Index
        required = ['Ca_pct', 'Mg_pct', 'Fe_pct', 'Na_pct', 'K_pct']
        if all(col in available_cols for col in required):
            try:
                results['Propylitic Index'] = (df['Ca_pct'] + df['Mg_pct'] + df['Fe_pct']) / (
                    df['Na_pct'] + df['K_pct'] + df['Ca_pct'] + df['Mg_pct'] + df['Fe_pct'] + 0.0001)
            except:
                pass
        
        # 5. Potassic Alteration Index
        required = ['K_pct', 'Ca_pct', 'Mg_pct']
        if all(col in available_cols for col in required):
            try:
                results['Potassic Alteration'] = df['K_pct'] / (df['K_pct'] + df['Ca_pct'] + df['Mg_pct'] + 0.0001)
            except:
                pass
        
        return results
    
    @staticmethod
    def calculate_element_ratios(df):
        """Calculate element ratios from available elements"""
        results = {}
        available_cols = df.columns.tolist()
        
        # Define common ratios
        ratios = [
            ('Au_ppm', 'Cu_ppm', 'Au/Cu'),
            ('Au_ppm', 'Ag_ppm', 'Au/Ag'),
            ('Cu_ppm', 'Zn_ppm', 'Cu/Zn'),
            ('Zn_ppm', 'Pb_ppm', 'Zn/Pb'),
            ('Au_ppm', 'As_ppm', 'Au/As'),
            ('Cu_ppm', 'Mo_ppm', 'Cu/Mo'),
            ('Zn_ppm', 'Cd_ppm', 'Zn/Cd'),
            ('Pb_ppm', 'Zn_ppm', 'Pb/Zn'),
            ('Au_ppm', 'Sb_ppm', 'Au/Sb'),
            ('Cu_ppm', 'Pb_ppm', 'Cu/Pb'),
            ('Zn_ppm', 'Cu_ppm', 'Zn/Cu'),
            ('Ag_ppm', 'Au_ppm', 'Ag/Au'),
        ]
        
        for col1, col2, name in ratios:
            if col1 in available_cols and col2 in available_cols:
                try:
                    results[name] = df[col1] / (df[col2] + 0.0001)
                except:
                    pass
        
        return results
    
    @staticmethod
    def calculate_mineralization_indices(df):
        """Calculate mineralization indices from available elements"""
        results = {}
        available_cols = df.columns.tolist()
        
        # Gold Index
        if 'Au_ppm' in available_cols:
            try:
                results['Gold Index'] = df['Au_ppm'] * 100
            except:
                pass
        
        # Base Metal Index
        base_metals = ['Cu_ppm', 'Zn_ppm', 'Pb_ppm']
        available = [col for col in base_metals if col in available_cols]
        if available:
            try:
                results['Base Metal Index'] = sum(df[col].fillna(0) for col in available) / len(available)
            except:
                pass
        
        # Pathfinder Index
        pathfinders = ['As_ppm', 'Sb_ppm', 'Bi_ppm', 'Mo_ppm']
        available = [col for col in pathfinders if col in available_cols]
        if available:
            try:
                results['Pathfinder Index'] = sum(df[col].fillna(0) for col in available) / len(available)
            except:
                pass
        
        return results
    
    @staticmethod
    def calculate_grade_statistics(df, element):
        """Calculate grade statistics for a specific element"""
        if element not in df.columns:
            return None
        
        data = df[element].dropna()
        if len(data) == 0:
            return None
        
        # Basic stats
        stats = {
            'Mean': data.mean(),
            'Median': data.median(),
            'Std Dev': data.std(),
            'Minimum': data.min(),
            'Maximum': data.max(),
            'Q1': data.quantile(0.25),
            'Q3': data.quantile(0.75),
            'IQR': data.quantile(0.75) - data.quantile(0.25),
            'N': len(data)
        }
        
        # Threshold (use 75th percentile as default)
        threshold = data.quantile(0.75)
        stats['Threshold'] = threshold
        stats['Above Threshold'] = (data > threshold).sum()
        stats['% Above Threshold'] = (data > threshold).sum() / len(data) * 100
        
        # Grade classification
        if data.mean() > threshold * 2:
            stats['Grade Class'] = 'HIGH'
            stats['Grade Color'] = '#d93025'
        elif data.mean() > threshold:
            stats['Grade Class'] = 'MEDIUM'
            stats['Grade Color'] = '#e37400'
        else:
            stats['Grade Class'] = 'LOW'
            stats['Grade Color'] = '#1e8e3e'
        
        return stats

# ============================================================
# DATA CLEANER
# ============================================================

class DataCleaner:
    def __init__(self, df):
        self.df = df.copy()
        self.cleaned_df = None
        self.log = []
        self.original_shape = df.shape
    
    def clean_all(self):
        """Run complete cleaning pipeline"""
        self.log = []
        
        # Clean column names
        self.df.columns = self.df.columns.str.strip().str.replace(' ', '_')
        self.log.append("✅ Column names cleaned")
        
        # Handle detection limits
        for col in self.df.columns:
            if col not in ['Sample_ID', 'Lithology', 'Comments']:
                mask = self.df[col].astype(str).str.startswith('<', na=False)
                if mask.sum() > 0:
                    for idx in self.df[mask].index:
                        val = str(self.df.loc[idx, col])
                        try:
                            limit = float(val.replace('<', '').strip())
                            self.df.loc[idx, col] = limit / 2
                        except:
                            self.df.loc[idx, col] = np.nan
                    self.log.append(f"✅ Handled detection limits in {col}")
        
        # Convert to numeric
        for col in self.df.columns:
            if col not in ['Sample_ID', 'Lithology', 'Comments']:
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
                self.log.append(f"✅ Converted {col} to numeric")
        
        # Handle missing values
        for col in self.df.columns:
            if col not in ['Sample_ID', 'Lithology', 'Comments']:
                missing = self.df[col].isnull().sum()
                if missing > 0:
                    self.df[col] = self.df[col].fillna(self.df[col].median())
                    self.log.append(f"✅ Filled {missing} missing values in {col}")
        
        # Handle outliers
        for col in self.df.columns:
            if col not in ['Sample_ID', 'Lithology', 'Comments']:
                if self.df[col].dtype in ['float64', 'int64']:
                    Q1 = self.df[col].quantile(0.25)
                    Q3 = self.df[col].quantile(0.75)
                    IQR = Q3 - Q1
                    lower = Q1 - 1.5 * IQR
                    upper = Q3 + 1.5 * IQR
                    outliers = ((self.df[col] < lower) | (self.df[col] > upper)).sum()
                    if outliers > 0:
                        self.df[col] = self.df[col].clip(lower=lower, upper=upper)
                        self.log.append(f"✅ Capped {outliers} outliers in {col}")
        
        self.cleaned_df = self.df.copy()
        return self.cleaned_df
    
    def get_summary(self):
        return {
            'original_shape': self.original_shape,
            'final_shape': self.cleaned_df.shape if self.cleaned_df is not None else None,
            'rows_removed': self.original_shape[0] - (self.cleaned_df.shape[0] if self.cleaned_df is not None else 0),
            'log': self.log
        }

# ============================================================
# PROFESSIONAL EDA ENGINE
# ============================================================

class ProfessionalEDA:
    def __init__(self, df, progress_callback=None):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        self.progress_callback = progress_callback
        self.alteration_indices = {}
        self.element_ratios = {}
        self.mineralization_indices = {}
        self.grade_stats = {}
        self.run_all()
    
    def _update_progress(self, message):
        if self.progress_callback:
            self.progress_callback(message)
    
    def run_all(self):
        """Run all analyses"""
        self._update_progress("📊 Calculating basic statistics...")
        self.basic_statistics()
        
        self._update_progress("📈 Computing variance...")
        self.variance_analysis()
        
        self._update_progress("📉 Computing correlation matrix...")
        self.correlation_analysis()
        
        self._update_progress("🎯 Detecting outliers...")
        self.outlier_detection()
        
        self._update_progress("📊 Running PCA analysis...")
        self.perform_pca()
        
        self._update_progress("🔬 Analyzing normality...")
        self.normality_analysis()
        
        self._update_progress("🧪 Calculating geochemistry indices...")
        self.calculate_geochemistry()
        
        self._update_progress("📊 Calculating grade statistics...")
        self.calculate_grade_statistics()
    
    def basic_statistics(self):
        self.results['basic_stats'] = {
            'mean': self.numeric_df.mean(),
            'median': self.numeric_df.median(),
            'std': self.numeric_df.std(),
            'min': self.numeric_df.min(),
            'max': self.numeric_df.max(),
            'range': self.numeric_df.max() - self.numeric_df.min(),
            'q25': self.numeric_df.quantile(0.25),
            'q75': self.numeric_df.quantile(0.75),
            'iqr': self.numeric_df.quantile(0.75) - self.numeric_df.quantile(0.25),
            'skew': self.numeric_df.skew(),
            'kurtosis': self.numeric_df.kurtosis(),
            'count': self.numeric_df.count()
        }
    
    def variance_analysis(self):
        self.results['variance'] = {
            'variance': self.numeric_df.var(),
            'std_dev': self.numeric_df.std(),
            'cv': (self.numeric_df.std() / self.numeric_df.mean()) * 100
        }
    
    def correlation_analysis(self):
        if len(self.numeric_df.columns) > 1:
            self.results['correlation'] = self.numeric_df.corr()
            strong = []
            corr = self.results['correlation']
            for i, col1 in enumerate(corr.columns):
                for col2 in corr.columns[i+1:]:
                    val = corr.loc[col1, col2]
                    if abs(val) > 0.5:
                        strong.append((col1, col2, val))
            self.results['strong_correlations'] = sorted(strong, key=lambda x: abs(x[2]), reverse=True)
    
    def outlier_detection(self):
        self.results['outliers'] = {}
        for col in self.numeric_df.columns:
            Q1 = self.numeric_df[col].quantile(0.25)
            Q3 = self.numeric_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            outliers_count = ((self.numeric_df[col] < lower) | (self.numeric_df[col] > upper)).sum()
            self.results['outliers'][col] = {
                'count': outliers_count,
                'percentage': outliers_count / len(self.numeric_df) * 100 if len(self.numeric_df) > 0 else 0,
                'lower_bound': lower,
                'upper_bound': upper
            }
    
    def perform_pca(self):
        if len(self.numeric_df.columns) >= 2:
            clean_data = self.numeric_df.dropna()
            if len(clean_data) > 1:
                scaler = StandardScaler()
                scaled = scaler.fit_transform(clean_data)
                pca = SklearnPCA()
                scores = pca.fit_transform(scaled)
                self.results['pca'] = {
                    'explained_variance': pca.explained_variance_ratio_,
                    'scores': scores,
                    'loadings': pd.DataFrame(
                        pca.components_.T,
                        columns=[f'PC{i+1}' for i in range(pca.n_components_)],
                        index=clean_data.columns
                    ),
                    'n_components': pca.n_components_
                }
    
    def normality_analysis(self):
        self.results['normality'] = {}
        for col in self.numeric_df.columns:
            skew_val = self.numeric_df[col].skew()
            kurt_val = self.numeric_df[col].kurtosis()
            is_normal = (-1 < skew_val < 1) and (-2 < kurt_val < 2)
            self.results['normality'][col] = {
                'skewness': skew_val,
                'kurtosis': kurt_val,
                'is_normal': is_normal,
                'interpretation': 'Normal' if is_normal else 'Not normal'
            }
    
    def calculate_geochemistry(self):
        """Calculate all geochemistry indices"""
        self.alteration_indices = ExplorationGeochemistry.calculate_alteration_indices(self.df)
        self.element_ratios = ExplorationGeochemistry.calculate_element_ratios(self.df)
        self.mineralization_indices = ExplorationGeochemistry.calculate_mineralization_indices(self.df)
    
    def calculate_grade_statistics(self):
        """Calculate grade statistics for key elements"""
        key_elements = ['Au_ppm', 'Cu_ppm', 'Zn_ppm', 'Pb_ppm', 'Ag_ppm', 'As_ppm', 'Mo_ppm']
        for element in key_elements:
            if element in self.df.columns:
                stats = ExplorationGeochemistry.calculate_grade_statistics(self.df, element)
                if stats:
                    self.grade_stats[element] = stats
    
    def get_geochemistry_html(self):
        """Generate beautiful HTML for geochemistry"""
        html = []
        html.append("""
        <html>
        <head>
        <style>
            body { font-family: 'Segoe UI', Arial, sans-serif; margin: 20px; color: #202124; }
            h1 { color: #1a73e8; border-bottom: 3px solid #1a73e8; padding-bottom: 10px; }
            h2 { color: #3c4043; margin-top: 30px; border-bottom: 2px solid #e8eaed; padding-bottom: 8px; }
            h3 { color: #5f6368; margin-top: 20px; }
            table { border-collapse: collapse; width: 100%; margin: 15px 0; }
            th { background-color: #1a73e8; color: white; padding: 12px; text-align: left; }
            td { padding: 10px; border-bottom: 1px solid #e8eaed; }
            tr:hover { background-color: #f8f9fa; }
            .card { 
                background-color: #f8f9fa; 
                border-radius: 8px; 
                padding: 15px; 
                margin: 10px 0; 
                border-left: 4px solid #1a73e8;
            }
            .stat-box { 
                display: inline-block; 
                margin: 5px; 
                padding: 10px 20px; 
                background-color: white; 
                border-radius: 8px; 
                border: 1px solid #dadce0;
                min-width: 80px;
            }
            .stat-value { 
                font-weight: bold; 
                color: #1a73e8; 
                font-size: 18px; 
            }
            .stat-label { 
                font-size: 11px; 
                color: #5f6368; 
            }
            .badge { 
                display: inline-block; 
                padding: 3px 12px; 
                border-radius: 12px; 
                font-size: 12px; 
                font-weight: 600; 
            }
            .badge-high { background-color: #fce8e6; color: #d93025; }
            .badge-medium { background-color: #fef7e0; color: #e37400; }
            .badge-low { background-color: #e6f4ea; color: #1e8e3e; }
            .badge-normal { background-color: #e6f4ea; color: #1e8e3e; }
            .badge-warning { background-color: #fef7e0; color: #e37400; }
        </style>
        </head>
        <body>
        """)
        
        # Title
        html.append(f"""
        <h1>🧪 Exploration Geochemistry Report</h1>
        <p><strong>Generated:</strong> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        <p><strong>Dataset:</strong> {len(self.df)} samples × {len(self.df.columns)} variables</p>
        """)
        
        # 1. Alteration Indices
        if self.alteration_indices:
            html.append("<h2>📊 Alteration Indices</h2>")
            html.append("<div class='card'>")
            for name, values in self.alteration_indices.items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <h3>{name}</h3>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Std Dev</div><div class='stat-value'>{values.std():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{values.min():.4f} - {values.max():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("""
            <div class='card'>
                <p style='color: #5f6368;'>No alteration indices could be calculated. 
                Requires elements like: K_pct, Mg_pct, Ca_pct, Na_pct, Fe_pct</p>
            </div>
            """)
        
        # 2. Element Ratios
        if self.element_ratios:
            html.append("<h2>📊 Element Ratios</h2>")
            html.append("<div class='card'>")
            for name, values in self.element_ratios.items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <h3>{name}</h3>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{values.min():.4f} - {values.max():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("""
            <div class='card'>
                <p style='color: #5f6368;'>No element ratios could be calculated. 
                Requires pairs like: Au_ppm/Cu_ppm, Zn_ppm/Pb_ppm, etc.</p>
            </div>
            """)
        
        # 3. Mineralization Indices
        if self.mineralization_indices:
            html.append("<h2>📈 Mineralization Indices</h2>")
            html.append("<div class='card'>")
            for name, values in self.mineralization_indices.items():
                if isinstance(values, pd.Series) and len(values) > 0:
                    html.append(f"""
                    <div style='margin: 10px 0;'>
                        <h3>{name}</h3>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{values.mean():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{values.median():.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{values.min():.4f} - {values.max():.4f}</div></div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("""
            <div class='card'>
                <p style='color: #5f6368;'>No mineralization indices could be calculated. 
                Requires elements like: Au_ppm, Cu_ppm, Zn_ppm, Pb_ppm, As_ppm</p>
            </div>
            """)
        
        # 4. Grade Statistics
        if self.grade_stats:
            html.append("<h2>📊 Grade Statistics</h2>")
            html.append("<div class='card'>")
            for element, stats in self.grade_stats.items():
                if stats:
                    grade_class = stats.get('Grade Class', 'UNKNOWN')
                    grade_color = stats.get('Grade Color', '#1e8e3e')
                    badge_class = 'badge-high' if grade_class == 'HIGH' else 'badge-medium' if grade_class == 'MEDIUM' else 'badge-low'
                    
                    html.append(f"""
                    <div style='margin: 15px 0; padding: 15px; background-color: white; border-radius: 8px; border: 1px solid #e8eaed;'>
                        <h3 style='margin-top: 0;'>{element}</h3>
                        <div class='stat-box'><div class='stat-label'>Mean</div><div class='stat-value'>{stats.get('Mean', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Median</div><div class='stat-value'>{stats.get('Median', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Std Dev</div><div class='stat-value'>{stats.get('Std Dev', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>Range</div><div class='stat-value'>{stats.get('Minimum', 0):.4f} - {stats.get('Maximum', 0):.4f}</div></div>
                        <div class='stat-box'><div class='stat-label'>N</div><div class='stat-value'>{stats.get('N', 0)}</div></div>
                        <div style='margin-top: 10px;'>
                            <span class='badge {badge_class}'>Grade: {grade_class}</span>
                            <span style='margin-left: 10px;'>Threshold: {stats.get('Threshold', 0):.4f}</span>
                            <span style='margin-left: 10px;'>Above Threshold: {stats.get('Above Threshold', 0)} ({stats.get('% Above Threshold', 0):.1f}%)</span>
                        </div>
                    </div>
                    """)
            html.append("</div>")
        else:
            html.append("""
            <div class='card'>
                <p style='color: #5f6368;'>No grade statistics could be calculated. 
                Requires elements like: Au_ppm, Cu_ppm, Zn_ppm, Pb_ppm, Ag_ppm</p>
            </div>
            """)
        
        # Summary
        html.append("""
        <div style='margin-top: 30px; padding: 15px; background-color: #e8f0fe; border-radius: 8px;'>
            <p><b>📌 Summary</b></p>
            <table style='width: auto;'>
        """)
        
        html.append(f"<tr><td>Total Samples</td><td><b>{len(self.df)}</b></td></tr>")
        html.append(f"<tr><td>Total Variables</td><td><b>{len(self.df.columns)}</b></td></tr>")
        html.append(f"<tr><td>Alteration Indices</td><td><b>{len(self.alteration_indices)}</b></td></tr>")
        html.append(f"<tr><td>Element Ratios</td><td><b>{len(self.element_ratios)}</b></td></tr>")
        html.append(f"<tr><td>Mineralization Indices</td><td><b>{len(self.mineralization_indices)}</b></td></tr>")
        html.append(f"<tr><td>Grade Statistics</td><td><b>{len(self.grade_stats)}</b></td></tr>")
        
        html.append("""
            </table>
        </div>
        """)
        
        html.append("""
        </body>
        </html>
        """)
        
        return "\n".join(html)
    
    def get_eda_html(self):
        """Generate HTML for EDA"""
        html = []
        html.append("""
        <html>
        <head>
        <style>
            body { font-family: 'Segoe UI', Arial, sans-serif; margin: 20px; color: #202124; }
            h1 { color: #1a73e8; border-bottom: 3px solid #1a73e8; padding-bottom: 10px; }
            h2 { color: #3c4043; margin-top: 30px; border-bottom: 2px solid #e8eaed; padding-bottom: 8px; }
            table { border-collapse: collapse; width: 100%; margin: 15px 0; }
            th { background-color: #1a73e8; color: white; padding: 12px; text-align: left; }
            td { padding: 10px; border-bottom: 1px solid #e8eaed; }
            tr:hover { background-color: #f8f9fa; }
            .card { background-color: #f8f9fa; border-radius: 8px; padding: 15px; margin: 10px 0; border-left: 4px solid #1a73e8; }
            .badge { display: inline-block; padding: 3px 12px; border-radius: 12px; font-size: 12px; font-weight: 600; }
            .badge-normal { background-color: #e6f4ea; color: #1e8e3e; }
            .badge-warning { background-color: #fef7e0; color: #e37400; }
            .badge-danger { background-color: #fce8e6; color: #d93025; }
        </style>
        </head>
        <body>
        """)
        
        html.append(f"""
        <h1>📊 Exploratory Data Analysis Report</h1>
        <p><strong>Generated:</strong> {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        <p><strong>Dataset:</strong> {len(self.df)} samples × {len(self.df.columns)} variables</p>
        <p><strong>Numeric Variables:</strong> {len(self.numeric_df.columns)}</p>
        """)
        
        # Basic Statistics
        if 'basic_stats' in self.results:
            html.append("<h2>📊 Basic Statistics</h2>")
            stats_df = pd.DataFrame(self.results['basic_stats'])
            html.append(stats_df.round(4).to_html(classes='table'))
        
        # Strong Correlations
        if 'strong_correlations' in self.results:
            html.append("<h2>📉 Strong Correlations</h2>")
            html.append("<div class='card'>")
            for col1, col2, val in self.results['strong_correlations'][:10]:
                direction = "positive" if val > 0 else "negative"
                badge = "badge-normal" if val > 0 else "badge-danger"
                html.append(f"""
                <p><b>{col1}</b> ↔ <b>{col2}</b>: 
                <span class='badge {badge}'>{val:.4f}</span> ({direction} correlation)</p>
                """)
            html.append("</div>")
        
        # Outliers
        html.append("<h2>🎯 Outlier Detection</h2>")
        html.append("<div class='card'>")
        for col, data in self.results.get('outliers', {}).items():
            if data['count'] > 0:
                badge = "badge-danger" if data['percentage'] > 5 else "badge-warning"
                html.append(f"""
                <p><b>{col}</b>: <span class='badge {badge}'>{data['count']} outliers ({data['percentage']:.1f}%)</span>
                <span style='color:#5f6368;font-size:12px;'> (Range: {data['lower_bound']:.4f} - {data['upper_bound']:.4f})</span></p>
                """)
        html.append("</div>")
        
        # Normality
        html.append("<h2>🔬 Normality Test</h2>")
        html.append("<div class='card'>")
        for col, data in self.results.get('normality', {}).items():
            status = "✅ Normal" if data['is_normal'] else "❌ Not normal"
            badge = "badge-normal" if data['is_normal'] else "badge-danger"
            html.append(f"<p><b>{col}</b>: <span class='badge {badge}'>{status}</span> (skew={data['skewness']:.4f})</p>")
        html.append("</div>")
        
        # PCA
        if 'pca' in self.results:
            html.append("<h2>📉 Principal Component Analysis</h2>")
            pca = self.results['pca']
            html.append("<div class='card'>")
            html.append(f"<p><b>Number of Components:</b> {pca['n_components']}</p>")
            html.append(f"<p><b>Total Explained Variance:</b> {pca['explained_variance'].sum()*100:.1f}%</p>")
            html.append("<p><b>Top 5 Loadings (PC1):</b></p>")
            loadings = pca['loadings']['PC1'].abs().sort_values(ascending=False)[:5]
            for var, loading in loadings.items():
                html.append(f"<p style='margin-left:20px;'>{var}: {loading:.4f}</p>")
            html.append("</div>")
        
        html.append("</body></html>")
        return "\n".join(html)

# ============================================================
# MAIN APPLICATION
# ============================================================

class EDAWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.df = None
        self.cleaned_df = None
        self.eda = None
        self.cleaner = None
        self.original_df = None
        self.current_file_path = None
        self.is_processing = False
        self.init_ui()
        
    def init_ui(self):
        self.setWindowTitle(f"🧪 {APP_NAME} v{APP_VERSION} (Code: {APP_CODE})")
        self.setGeometry(50, 50, 1600, 950)
        self.setWindowIcon(self.create_icon())
        
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout()
        main_layout.setContentsMargins(10, 10, 10, 10)
        main_layout.setSpacing(10)
        central.setLayout(main_layout)
        
        # Header
        header = self.create_header()
        main_layout.addWidget(header)
        
        # Toolbar
        toolbar = self.create_unified_toolbar()
        main_layout.addWidget(toolbar)
        
        # Info Panel
        self.info_label = QLabel("📋 Welcome! Upload your geochemical data to begin analysis.")
        self.info_label.setObjectName("info_label")
        main_layout.addWidget(self.info_label)
        
        # Status Widget
        status_widget = self.create_status_widget()
        main_layout.addWidget(status_widget)
        
        # Tab Widget
        self.tab_widget = QTabWidget()
        self.tab_widget.setStyleSheet("""
            QTabWidget::pane {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 8px;
                margin-top: -1px;
            }
            QTabBar::tab {
                padding: 12px 24px;
                margin-right: 2px;
                border-top-left-radius: 8px;
                border-top-right-radius: 8px;
                background-color: #f1f3f4;
                color: #5f6368;
                font-weight: 500;
                font-size: 13px;
            }
            QTabBar::tab:selected {
                background-color: white;
                color: #1a73e8;
                border-bottom: 3px solid #1a73e8;
            }
        """)
        main_layout.addWidget(self.tab_widget)
        
        # Create tabs
        self.create_welcome_tab()
        self.create_data_tab()
        self.create_eda_tab()
        self.create_visualization_tab()
        self.create_report_tab()
        self.create_geochemistry_tab()
        self.create_log_tab()
        
        # Status bar
        self.statusBar = QStatusBar()
        self.setStatusBar(self.statusBar)
        
        self.progress_bar = QProgressBar()
        self.progress_bar.setVisible(False)
        self.progress_bar.setMaximumWidth(250)
        self.statusBar.addPermanentWidget(self.progress_bar)
        
        self.statusBar.showMessage("✅ Ready")
        self.setStyleSheet(STYLESHEET)
        
        self.log_message(f"🔬 {APP_NAME} v{APP_VERSION} initialized")
        self.log_message(f"📌 Project Code: {APP_CODE}")
        self.log_message("📌 Ready for data analysis")
    
    def create_icon(self):
        pixmap = QPixmap(64, 64)
        pixmap.fill(Qt.transparent)
        painter = QPainter(pixmap)
        painter.setRenderHint(QPainter.Antialiasing)
        painter.setBrush(QBrush(QColor(26, 115, 232)))
        painter.setPen(QPen(QColor(33, 33, 33), 2))
        painter.drawEllipse(10, 10, 44, 44)
        painter.setBrush(QBrush(QColor(255, 193, 7)))
        painter.drawEllipse(25, 20, 6, 6)
        painter.drawEllipse(35, 30, 6, 6)
        painter.drawEllipse(20, 40, 6, 6)
        painter.end()
        return QIcon(pixmap)
    
    def create_header(self):
        widget = QWidget()
        widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border-bottom: 1px solid #dadce0;
                border-radius: 8px 8px 0 0;
                padding: 10px 20px;
            }
        """)
        layout = QHBoxLayout()
        widget.setLayout(layout)
        title_label = QLabel("🧪 Elemental Society Analysis")
        title_label.setStyleSheet("font-size: 22px; font-weight: 700; color: #1a73e8;")
        layout.addWidget(title_label)
        version_label = QLabel(f"v{APP_VERSION} | Code: {APP_CODE}")
        version_label.setStyleSheet("font-size: 13px; color: #5f6368; padding-left: 10px;")
        layout.addWidget(version_label)
        layout.addStretch()
        dev_label = QLabel(f"👨‍🔬 {DEVELOPER}")
        dev_label.setStyleSheet("font-size: 13px; color: #5f6368;")
        layout.addWidget(dev_label)
        return widget
    
    def create_unified_toolbar(self):
        toolbar = QToolBar()
        toolbar.setObjectName("main_toolbar")
        toolbar.setMovable(False)
        toolbar.setIconSize(QSize(28, 28))
        toolbar.setStyleSheet("""
            QToolBar {
                background-color: white;
                border: none;
                border-bottom: 1px solid #dadce0;
                spacing: 2px;
                padding: 5px 10px;
            }
            QToolBar QToolButton {
                border: none;
                border-radius: 4px;
                padding: 6px 12px;
                font-weight: 500;
                color: #3c4043;
                font-size: 12px;
            }
            QToolBar QToolButton:hover {
                background-color: #e8f0fe;
                color: #1a73e8;
            }
            QToolBar QToolButton#upload_btn {
                background-color: #1a73e8;
                color: white;
                padding: 6px 16px;
                border-radius: 4px;
            }
        """)
        
        upload_menu = QMenu()
        upload_new = QAction("📄 New Data", self)
        upload_new.triggered.connect(lambda: self.upload_file('new'))
        upload_menu.addAction(upload_new)
        upload_append = QAction("➕ Append", self)
        upload_append.triggered.connect(lambda: self.upload_file('append'))
        upload_menu.addAction(upload_append)
        upload_replace = QAction("🔄 Replace", self)
        upload_replace.triggered.connect(lambda: self.upload_file('replace'))
        upload_menu.addAction(upload_replace)
        
        upload_btn = QToolButton()
        upload_btn.setObjectName("upload_btn")
        upload_btn.setText("📁 Upload")
        upload_btn.setMenu(upload_menu)
        upload_btn.setPopupMode(QToolButton.InstantPopup)
        toolbar.addWidget(upload_btn)
        toolbar.addSeparator()
        
        clean_btn = QToolButton()
        clean_btn.setText("🧹 Clean")
        clean_btn.clicked.connect(self.clean_data)
        toolbar.addWidget(clean_btn)
        
        analyze_btn = QToolButton()
        analyze_btn.setText("📊 Analyze")
        analyze_btn.clicked.connect(self.run_eda)
        toolbar.addWidget(analyze_btn)
        toolbar.addSeparator()
        
        save_btn = QToolButton()
        save_btn.setText("💾 Save")
        save_btn.clicked.connect(self.save_state)
        toolbar.addWidget(save_btn)
        
        restore_btn = QToolButton()
        restore_btn.setText("↩️ Restore")
        restore_btn.clicked.connect(self.restore_state)
        toolbar.addWidget(restore_btn)
        
        undo_btn = QToolButton()
        undo_btn.setText("◀️ Undo")
        undo_btn.clicked.connect(self.undo_last_action)
        toolbar.addWidget(undo_btn)
        toolbar.addSeparator()
        
        delete_btn = QToolButton()
        delete_btn.setText("🗑️ Delete")
        delete_btn.setObjectName("danger_btn")
        delete_btn.clicked.connect(self.delete_data)
        toolbar.addWidget(delete_btn)
        
        export_btn = QToolButton()
        export_btn.setText("💾 Export")
        export_btn.clicked.connect(self.export_data)
        toolbar.addWidget(export_btn)
        
        report_btn = QToolButton()
        report_btn.setText("📄 Report")
        report_btn.clicked.connect(self.show_report)
        toolbar.addWidget(report_btn)
        toolbar.addSeparator()
        
        help_btn = QToolButton()
        help_btn.setText("❓ Help")
        help_btn.clicked.connect(self.show_help)
        toolbar.addWidget(help_btn)
        
        about_btn = QToolButton()
        about_btn.setText("ℹ️ About")
        about_btn.clicked.connect(self.show_about)
        toolbar.addWidget(about_btn)
        
        spacer = QWidget()
        spacer.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Preferred)
        toolbar.addWidget(spacer)
        
        self.file_info_label = QLabel("📁 No file loaded")
        self.file_info_label.setStyleSheet("color: #5f6368; padding: 5px 10px; font-size: 12px;")
        toolbar.addWidget(self.file_info_label)
        
        return toolbar
    
    def create_status_widget(self):
        widget = QWidget()
        widget.setStyleSheet("""
            QWidget {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 8px;
                padding: 8px 15px;
            }
        """)
        layout = QHBoxLayout()
        layout.setContentsMargins(10, 5, 10, 5)
        widget.setLayout(layout)
        
        layout.addWidget(QLabel("Status:"))
        
        self.status_indicator = QLabel("⚪ Idle")
        self.status_indicator.setObjectName("status_idle")
        layout.addWidget(self.status_indicator)
        
        self.status_message = QLabel("Ready")
        self.status_message.setStyleSheet("color: #5f6368; font-weight: 500;")
        layout.addWidget(self.status_message)
        
        self.wait_label = QLabel("⏳ Please wait...")
        self.wait_label.setStyleSheet("color: #e37400; font-weight: 600;")
        self.wait_label.setVisible(False)
        layout.addWidget(self.wait_label)
        
        layout.addStretch()
        
        self.process_info = QLabel("")
        self.process_info.setStyleSheet("color: #9aa0a6; font-size: 11px;")
        layout.addWidget(self.process_info)
        
        return widget
    
    def create_welcome_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        welcome_widget = QWidget()
        welcome_widget.setObjectName("card")
        welcome_widget.setStyleSheet("""
            QWidget#card {
                background-color: white;
                border: 1px solid #dadce0;
                border-radius: 12px;
                padding: 20px;
            }
        """)
        welcome_layout = QVBoxLayout()
        welcome_widget.setLayout(welcome_layout)
        
        welcome_text = QLabel("""
        <h1 style='color: #1a73e8;'>🧪 Welcome to Elemental Society Analysis</h1>
        <h3 style='color: #3c4043;'>Professional Geochemical Data Analysis Suite</h3>
        <br>
        <p style='font-size: 14px; color: #5f6368;'>
        <b>Your complete solution for exploration geochemistry data analysis.</b>
        </p>
        <br>
        <table style='width: 100%;'>
            <tr><td style='padding: 10px; border: none;'><b>📁 1. Upload Data</b><br>Load your CSV or Excel file</td>
            <td style='padding: 10px; border: none;'><b>🧹 2. Clean Data</b><br>Handle missing values, outliers</td></tr>
            <tr><td style='padding: 10px; border: none;'><b>📊 3. Analyze</b><br>Run comprehensive statistical analysis</td>
            <td style='padding: 10px; border: none;'><b>📉 4. Visualize</b><br>Generate professional plots</td></tr>
            <tr><td style='padding: 10px; border: none;'><b>🧪 5. Geochemistry</b><br>Alteration indices, grade estimates</td>
            <td style='padding: 10px; border: none;'><b>📄 6. Report</b><br>Generate comprehensive HTML report</td></tr>
        </table>
        <br>
        <p style='color: #5f6368;'><b>💡 Tip:</b> Start by clicking the <b style='color: #1a73e8;'>Upload</b> button above.</p>
        """)
        welcome_text.setWordWrap(True)
        welcome_layout.addWidget(welcome_text)
        
        layout.addWidget(welcome_widget)
        layout.addStretch()
        self.tab_widget.addTab(tab, "🏠 Welcome")
    
    def create_data_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        controls = QHBoxLayout()
        self.data_info = QLabel("No data loaded")
        controls.addWidget(self.data_info)
        controls.addStretch()
        controls.addWidget(QLabel("Show rows:"))
        self.row_limit = QSpinBox()
        self.row_limit.setRange(10, 5000)
        self.row_limit.setValue(1000)
        self.row_limit.valueChanged.connect(self.refresh_data_display)
        controls.addWidget(self.row_limit)
        
        layout.addLayout(controls)
        
        self.data_table = QTableWidget()
        self.data_table.setAlternatingRowColors(True)
        self.data_table.setSortingEnabled(True)
        layout.addWidget(self.data_table)
        
        self.tab_widget.addTab(tab, "📊 Data")
    
    def create_eda_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        self.eda_text = QTextEdit()
        self.eda_text.setReadOnly(True)
        self.eda_text.setFont(QFont("Segoe UI", 10))
        layout.addWidget(self.eda_text)
        self.tab_widget.addTab(tab, "📈 EDA Results")
    
    def create_visualization_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        controls = QHBoxLayout()
        controls.addWidget(QLabel("Plot:"))
        self.plot_type = QComboBox()
        self.plot_type.addItems(["Correlation Matrix", "PCA Scores", "Distributions", "Box Plots"])
        controls.addWidget(self.plot_type)
        controls.addWidget(QLabel("Theme:"))
        self.color_theme = QComboBox()
        self.color_theme.addItems(["coolwarm", "viridis", "plasma", "inferno"])
        controls.addWidget(self.color_theme)
        plot_btn = QPushButton("🎨 Generate")
        plot_btn.clicked.connect(self.generate_plot)
        controls.addWidget(plot_btn)
        controls.addStretch()
        layout.addLayout(controls)
        
        self.figure = Figure(figsize=(12, 7))
        self.figure.patch.set_facecolor('white')
        self.canvas = FigureCanvas(self.figure)
        layout.addWidget(self.canvas)
        self.toolbar = NavigationToolbar(self.canvas, self)
        layout.addWidget(self.toolbar)
        
        self.tab_widget.addTab(tab, "📉 Visualizations")
    
    def create_report_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        controls = QHBoxLayout()
        export_btn = QPushButton("💾 Export HTML")
        export_btn.clicked.connect(self.export_html_report)
        controls.addWidget(export_btn)
        copy_btn = QPushButton("📋 Copy")
        copy_btn.clicked.connect(self.copy_report)
        controls.addWidget(copy_btn)
        controls.addStretch()
        layout.addLayout(controls)
        
        self.report_viewer = QTextEdit()
        self.report_viewer.setReadOnly(True)
        self.report_viewer.setFont(QFont("Segoe UI", 10))
        layout.addWidget(self.report_viewer)
        
        self.tab_widget.addTab(tab, "📄 Reports")
    
    def create_geochemistry_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        self.geochem_viewer = QTextEdit()
        self.geochem_viewer.setReadOnly(True)
        self.geochem_viewer.setFont(QFont("Segoe UI", 10))
        layout.addWidget(self.geochem_viewer)
        
        self.tab_widget.addTab(tab, "🧪 Geochemistry")
    
    def create_log_tab(self):
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        controls = QHBoxLayout()
        clear_btn = QPushButton("🗑️ Clear")
        clear_btn.clicked.connect(self.clear_log)
        controls.addWidget(clear_btn)
        copy_btn = QPushButton("📋 Copy")
        copy_btn.clicked.connect(self.copy_log)
        controls.addWidget(copy_btn)
        controls.addStretch()
        layout.addLayout(controls)
        
        self.log_text = QTextEdit()
        self.log_text.setReadOnly(True)
        self.log_text.setFont(QFont("Consolas", 10))
        layout.addWidget(self.log_text)
        
        self.tab_widget.addTab(tab, "📝 Log")
    
    # ============================================================
    # STATUS FUNCTIONS
    # ============================================================
    
    def set_status(self, status, message="", wait=False):
        status_map = {
            'idle': ('⚪ Idle', 'status_idle', '#3c4043'),
            'processing': ('⚙️ Processing', 'status_processing', '#e37400'),
            'success': ('✅ Success', 'status_success', '#1e8e3e'),
            'error': ('❌ Error', 'status_error', '#d93025')
        }
        icon, obj_name, color = status_map.get(status, status_map['idle'])
        self.status_indicator.setText(icon)
        self.status_indicator.setObjectName(obj_name)
        self.status_indicator.setStyleSheet(f"""
            QLabel#{obj_name} {{
                padding: 4px 15px;
                border-radius: 12px;
                background-color: {color}20;
                color: {color};
                font-weight: 600;
            }}
        """)
        self.status_message.setText(message)
        self.wait_label.setVisible(wait)
        if wait:
            self.wait_label.setText("⏳ Processing...")
            QApplication.processEvents()
    
    def set_processing(self, status, message="", wait=True):
        self.is_processing = status
        if status:
            self.set_status('processing', message, wait)
            self.progress_bar.setVisible(True)
            self.progress_bar.setRange(0, 0)
            self.info_label.setText(f"⚙️ {message}")
            self.statusBar.showMessage(f"Processing: {message}")
        else:
            self.set_status('idle', 'Ready', False)
            self.progress_bar.setVisible(False)
            self.info_label.setText("✅ Ready")
            self.statusBar.showMessage("✅ Ready")
    
    def update_progress(self, message):
        self.process_info.setText(message)
        self.status_message.setText(message)
        self.log_message(f"⏳ {message}")
        QApplication.processEvents()
    
    # ============================================================
    # LOG FUNCTIONS
    # ============================================================
    
    def log_message(self, message):
        from datetime import datetime
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.append(f"[{timestamp}] {message}")
    
    def clear_log(self):
        self.log_text.clear()
        self.log_message("🗑️ Log cleared")
    
    def copy_log(self):
        clipboard = QApplication.clipboard()
        clipboard.setText(self.log_text.toPlainText())
        self.log_message("📋 Log copied")
        QMessageBox.information(self, "Success", "Log copied to clipboard!")
    
    def copy_report(self):
        clipboard = QApplication.clipboard()
        clipboard.setText(self.report_viewer.toPlainText())
        self.log_message("📋 Report copied")
        QMessageBox.information(self, "Success", "Report copied to clipboard!")
    
    # ============================================================
    # DATA LOADING
    # ============================================================
    
    def upload_file(self, mode='new'):
        file_path, _ = QFileDialog.getOpenFileName(
            self, "Open Geochemical Data", "",
            "All Files (*);;CSV (*.csv);;Excel (*.xlsx *.xls);;Text (*.txt)"
        )
        if not file_path:
            return
        
        self.current_file_path = file_path
        self.set_processing(True, "Loading file...")
        self.log_message(f"📁 Loading: {os.path.basename(file_path)}")
        self.update_progress(f"Reading file: {os.path.basename(file_path)}")
        
        try:
            delimiters = [';', ',', '\t', '|']
            encodings = ['utf-8', 'cp1252', 'latin-1']
            df = None
            
            for encoding in encodings:
                for delim in delimiters:
                    try:
                        df = pd.read_csv(file_path, sep=delim, encoding=encoding, on_bad_lines='skip')
                        if df is not None and not df.empty and df.shape[1] > 1:
                            break
                    except:
                        continue
                if df is not None and not df.empty:
                    break
            
            if df is None or df.empty:
                try:
                    df = pd.read_excel(file_path)
                except:
                    pass
            
            if df is not None and not df.empty:
                if mode == 'new' or self.df is None:
                    self.df = df
                elif mode == 'append':
                    if set(self.df.columns) == set(df.columns):
                        self.df = pd.concat([self.df, df], ignore_index=True)
                    else:
                        self.df = df
                elif mode == 'replace':
                    self.df = df
                
                self.original_df = self.df.copy()
                self.cleaned_df = None
                self.eda = None
                self.display_data()
                self.file_info_label.setText(f"📁 {os.path.basename(file_path)}")
                
                self.log_message(f"✅ Loaded: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
                self.set_status('success', f'Loaded: {self.df.shape[0]} rows')
                self.statusBar.showMessage(f"✅ Loaded: {os.path.basename(file_path)}")
                self.info_label.setText(f"📊 Data loaded: {self.df.shape[0]} rows × {self.df.shape[1]} columns")
                
                QMessageBox.information(self, "Success",
                    f"File loaded!\n\n📊 {self.df.shape[0]} rows × {self.df.shape[1]} columns")
            else:
                QMessageBox.warning(self, "Error", "Could not read file.")
                
        except Exception as e:
            self.log_message(f"❌ Error: {str(e)}")
            self.set_status('error', 'Load failed')
            QMessageBox.critical(self, "Error", f"Failed to load:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def display_data(self, df=None):
        if df is None:
            if self.cleaned_df is not None:
                df = self.cleaned_df
            elif self.df is not None:
                df = self.df
            else:
                return
        
        display_limit = min(self.row_limit.value(), len(df))
        self.data_table.setRowCount(display_limit)
        self.data_table.setColumnCount(len(df.columns))
        self.data_table.setHorizontalHeaderLabels(df.columns.astype(str))
        
        for i in range(display_limit):
            for j, col in enumerate(df.columns):
                val = df.iloc[i, j]
                if pd.isna(val):
                    self.data_table.setItem(i, j, QTableWidgetItem(""))
                else:
                    self.data_table.setItem(i, j, QTableWidgetItem(str(val)))
        
        self.data_table.resizeColumnsToContents()
        self.data_info.setText(f"📊 {len(df)} rows × {len(df.columns)} columns (showing {display_limit})")
    
    def refresh_data_display(self):
        self.display_data()
    
    # ============================================================
    # CLEANING FUNCTIONS
    # ============================================================
    
    def clean_data(self):
        if self.df is None or self.df.empty:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
        
        reply = QMessageBox.question(self, "Clean Data", 
            "Clean the current dataset?", QMessageBox.Yes | QMessageBox.No)
        if reply != QMessageBox.Yes:
            return
        
        self.set_processing(True, "Cleaning data...")
        self.log_message("🧹 Starting data cleaning...")
        self.update_progress("Cleaning data...")
        
        try:
            self.cleaner = DataCleaner(self.df)
            self.cleaned_df = self.cleaner.clean_all()
            if self.original_df is None:
                self.original_df = self.df.copy()
            self.display_data(self.cleaned_df)
            
            self.log_message(f"✅ Cleaning complete: {len(self.cleaned_df)} rows")
            self.set_status('success', f'Cleaned: {len(self.cleaned_df)} rows')
            self.info_label.setText(f"🧹 Cleaned: {len(self.cleaned_df)} rows × {len(self.cleaned_df.columns)} columns")
            
            summary = self.cleaner.get_summary()
            QMessageBox.information(self, "Cleaning Complete",
                f"Before: {summary['original_shape'][0]} rows\n"
                f"After: {summary['final_shape'][0]} rows\n"
                f"Removed: {summary['rows_removed']} rows")
            
        except Exception as e:
            self.log_message(f"❌ Cleaning error: {str(e)}")
            self.set_status('error', 'Cleaning failed')
            QMessageBox.critical(self, "Error", f"Cleaning failed:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    # ============================================================
    # EDA FUNCTIONS
    # ============================================================
    
    def run_eda(self):
        df_to_analyze = self.cleaned_df if self.cleaned_df is not None else self.df
        
        if df_to_analyze is None or df_to_analyze.empty:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
        
        self.set_processing(True, "Running EDA...")
        self.log_message("📊 Starting EDA analysis...")
        self.update_progress("🔬 Preparing EDA analysis...")
        
        try:
            self.eda = ProfessionalEDA(df_to_analyze, self.update_progress)
            
            self.update_progress("📊 Generating results...")
            self.display_eda_results()
            self.display_reports()
            self.display_geochemistry()
            
            self.log_message("✅ EDA complete!")
            self.set_status('success', 'EDA Complete!')
            self.info_label.setText(f"📊 EDA Complete: {len(self.eda.numeric_df.columns)} variables analyzed")
            
            QMessageBox.information(self, "Success",
                "EDA completed!\n\n"
                f"📊 Variables: {len(self.eda.numeric_df.columns)}\n"
                f"📈 Statistics computed\n"
                f"📉 Correlations analyzed\n"
                f"📊 PCA performed\n"
                f"🧪 Geochemistry calculated\n"
                f"📋 Reports generated")
            
        except Exception as e:
            self.log_message(f"❌ EDA error: {str(e)}")
            self.set_status('error', 'EDA failed')
            QMessageBox.critical(self, "Error", f"EDA failed:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def display_eda_results(self):
        if self.eda is None:
            return
        
        results = []
        results.append("="*100)
        results.append("📊 ELEMENTAL SOCIETY ANALYSIS - EDA RESULTS")
        results.append("="*100)
        results.append(f"\n📋 Dataset: {len(self.eda.df)} rows × {len(self.eda.df.columns)} columns")
        results.append(f"📊 Numeric columns: {len(self.eda.numeric_df.columns)}")
        
        results.append("\n📊 BASIC STATISTICS")
        results.append("-"*80)
        stats_df = pd.DataFrame(self.eda.results.get('basic_stats', {}))
        if not stats_df.empty:
            results.append(stats_df.round(4).to_string())
        
        if 'strong_correlations' in self.eda.results:
            results.append("\n📉 STRONG CORRELATIONS (>0.5)")
            results.append("-"*80)
            for col1, col2, val in self.eda.results['strong_correlations'][:5]:
                direction = "positive" if val > 0 else "negative"
                results.append(f"  {col1} ↔ {col2}: {val:.4f} ({direction})")
        
        results.append("\n🎯 OUTLIER DETECTION")
        results.append("-"*80)
        for col, data in self.eda.results.get('outliers', {}).items():
            if data['count'] > 0:
                results.append(f"  {col}: {data['count']} outliers ({data['percentage']:.1f}%)")
        
        results.append("\n📊 NORMALITY TEST")
        results.append("-"*80)
        for col, data in self.eda.results.get('normality', {}).items():
            status = "✅ Normal" if data['is_normal'] else "❌ Not normal"
            results.append(f"  {col}: {status} (skew={data['skewness']:.4f})")
        
        if 'pca' in self.eda.results:
            results.append("\n📉 PRINCIPAL COMPONENT ANALYSIS")
            results.append("-"*80)
            pca = self.eda.results['pca']
            results.append(f"Total explained variance: {pca['explained_variance'].sum()*100:.1f}%")
            results.append("\nTop 5 Loadings (PC1):")
            loadings = pca['loadings']['PC1'].abs().sort_values(ascending=False)[:5]
            for var, loading in loadings.items():
                results.append(f"  {var}: {loading:.4f}")
        
        self.eda_text.setText("\n".join(results))
    
    def display_reports(self):
        if self.eda is None:
            return
        eda_html = self.eda.get_eda_html()
        self.report_viewer.setHtml(eda_html)
    
    def display_geochemistry(self):
        if self.eda is None:
            return
        geochem_html = self.eda.get_geochemistry_html()
        self.geochem_viewer.setHtml(geochem_html)
    
    def export_html_report(self):
        if self.eda is None:
            QMessageBox.warning(self, "Warning", "Please run EDA first!")
            return
        
        file_path, _ = QFileDialog.getSaveFileName(
            self, "Export HTML Report", "eda_report.html", "HTML Files (*.html)")
        if file_path:
            try:
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(self.eda.get_eda_html())
                self.log_message(f"✅ HTML Report exported: {file_path}")
                QMessageBox.information(self, "Success", f"Report saved to:\n{file_path}")
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")
    
    # ============================================================
    # VISUALIZATION
    # ============================================================
    
    def generate_plot(self):
        if self.eda is None:
            QMessageBox.warning(self, "Warning", "Please run EDA first!")
            return
        
        plot_type = self.plot_type.currentText()
        theme = self.color_theme.currentText()
        self.figure.clear()
        
        try:
            self.set_processing(True, f"Generating {plot_type}...")
            self.update_progress(f"Creating {plot_type} plot...")
            
            if plot_type == "Correlation Matrix":
                self.plot_correlation(theme)
            elif plot_type == "PCA Scores":
                self.plot_pca(theme)
            elif plot_type == "Distributions":
                self.plot_distributions()
            elif plot_type == "Box Plots":
                self.plot_boxplots()
            
            self.canvas.draw()
            self.log_message(f"📉 Generated {plot_type} plot")
            self.set_status('success', f'{plot_type} generated')
            
        except Exception as e:
            self.log_message(f"❌ Plot error: {str(e)}")
            self.set_status('error', 'Plot failed')
            QMessageBox.critical(self, "Error", f"Failed to generate plot:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def plot_correlation(self, theme='coolwarm'):
        if 'correlation' not in self.eda.results:
            return
        ax = self.figure.add_subplot(111)
        mask = np.triu(np.ones_like(self.eda.results['correlation'], dtype=bool))
        sns.heatmap(self.eda.results['correlation'], mask=mask, annot=True,
                   cmap=theme, center=0, fmt='.2f', square=True, linewidths=0.5, ax=ax)
        ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
    
    def plot_pca(self, theme='viridis'):
        if 'pca' not in self.eda.results:
            return
        pca = self.eda.results['pca']
        scores = pca['scores']
        explained = pca['explained_variance']
        ax = self.figure.add_subplot(111)
        scatter = ax.scatter(scores[:, 0], scores[:, 1],
                            c=np.linalg.norm(scores[:, :2], axis=1),
                            cmap=theme, alpha=0.6, edgecolors='black', linewidth=0.5)
        ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
        ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
        ax.set_title('PCA Scores', fontweight='bold')
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax.grid(True, alpha=0.3)
        self.figure.colorbar(scatter, ax=ax, label='Distance')
    
    def plot_distributions(self):
        cols = self.eda.numeric_df.columns[:4]
        n_cols = min(2, len(cols))
        n_rows = (len(cols) + n_cols - 1) // n_cols
        for i, col in enumerate(cols):
            ax = self.figure.add_subplot(n_rows, n_cols, i+1)
            data = self.eda.numeric_df[col].dropna()
            if len(data) > 0:
                ax.hist(data, bins=20, density=True, alpha=0.6,
                       color='steelblue', edgecolor='black')
                ax.set_xlabel(col, fontsize=10)
                ax.set_ylabel('Density', fontsize=10)
                ax.set_title(f'{col} Distribution', fontweight='bold')
                ax.axvline(data.mean(), color='red', linestyle='--',
                          label=f'Mean: {data.mean():.2f}')
                ax.axvline(data.median(), color='green', linestyle='--',
                          label=f'Median: {data.median():.2f}')
                ax.legend(fontsize=8)
                ax.grid(True, alpha=0.3)
        self.figure.tight_layout()
    
    def plot_boxplots(self):
        ax = self.figure.add_subplot(111)
        self.eda.numeric_df.boxplot(ax=ax, rot=45)
        ax.set_title('Box Plots', fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    # ============================================================
    # EXPORT FUNCTIONS
    # ============================================================
    
    def export_data(self):
        if self.cleaned_df is None and self.df is None:
            QMessageBox.warning(self, "Warning", "No data to export!")
            return
        
        df = self.cleaned_df if self.cleaned_df is not None else self.df
        
        formats = {"CSV (;)": "csv_semicolon", "CSV (,)": "csv_comma", "Excel (.xlsx)": "excel", "JSON": "json"}
        format_choice, ok = QInputDialog.getItem(self, "Export Format", "Select format:", list(formats.keys()), 0, False)
        if not ok:
            return
        
        file_ext_map = {"csv_semicolon": "csv", "csv_comma": "csv", "excel": "xlsx", "json": "json"}
        file_path, _ = QFileDialog.getSaveFileName(
            self, "Export Data", f"cleaned_data.{file_ext_map[formats[format_choice]]}",
            f"{format_choice} (*.{file_ext_map[formats[format_choice]]})")
        
        if file_path:
            try:
                self.set_processing(True, "Exporting data...")
                self.update_progress(f"Exporting to {format_choice}...")
                
                if formats[format_choice] == "csv_semicolon":
                    df.to_csv(file_path, sep=';', index=False, encoding='utf-8-sig')
                elif formats[format_choice] == "csv_comma":
                    df.to_csv(file_path, sep=',', index=False, encoding='utf-8-sig')
                elif formats[format_choice] == "excel":
                    df.to_excel(file_path, index=False)
                elif formats[format_choice] == "json":
                    df.to_json(file_path, orient='records', indent=2)
                
                self.log_message(f"✅ Exported: {file_path}")
                self.set_status('success', 'Export complete!')
                QMessageBox.information(self, "Success", f"Data exported to:\n{file_path}")
            except Exception as e:
                self.set_status('error', 'Export failed')
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")
            finally:
                self.set_processing(False)
    
    def show_report(self):
        if self.eda is None:
            QMessageBox.warning(self, "Warning", "Please run EDA first!")
            return
        for i in range(self.tab_widget.count()):
            if self.tab_widget.tabText(i) == "📄 Reports":
                self.tab_widget.setCurrentIndex(i)
                break
    
    # ============================================================
    # PROCESS CONTROL
    # ============================================================
    
    def save_state(self):
        if self.df is not None:
            self.original_df = self.df.copy()
            self.log_message("💾 State saved")
            self.set_status('success', 'State saved!')
            QMessageBox.information(self, "Success", "State saved successfully!")
        else:
            QMessageBox.warning(self, "Warning", "No data to save!")
    
    def restore_state(self):
        if self.original_df is not None:
            reply = QMessageBox.question(self, "Restore State", "Restore last saved state?", QMessageBox.Yes | QMessageBox.No)
            if reply == QMessageBox.Yes:
                self.df = self.original_df.copy()
                self.cleaned_df = None
                self.eda = None
                self.display_data()
                self.log_message("↩️ State restored")
                self.set_status('success', 'State restored!')
        else:
            QMessageBox.warning(self, "Warning", "No saved state found!")
    
    def undo_last_action(self):
        if self.original_df is not None:
            reply = QMessageBox.question(self, "Undo", "Undo last action?", QMessageBox.Yes | QMessageBox.No)
            if reply == QMessageBox.Yes:
                self.df = self.original_df.copy()
                self.cleaned_df = None
                self.eda = None
                self.display_data()
                self.log_message("◀️ Action undone")
                self.set_status('success', 'Undone!')
        else:
            QMessageBox.warning(self, "Warning", "Nothing to undo!")
    
    def delete_data(self):
        if self.df is None:
            QMessageBox.warning(self, "Warning", "No data to delete!")
            return
        
        reply = QMessageBox.question(self, "Delete Data", "Delete all loaded data?", QMessageBox.Yes | QMessageBox.No)
        if reply == QMessageBox.Yes:
            self.df = None
            self.cleaned_df = None
            self.eda = None
            self.original_df = None
            self.data_table.clear()
            self.data_table.setRowCount(0)
            self.data_table.setColumnCount(0)
            self.eda_text.clear()
            self.report_viewer.clear()
            self.geochem_viewer.clear()
            self.figure.clear()
            self.canvas.draw()
            self.data_info.setText("No data loaded")
            self.file_info_label.setText("📁 No file loaded")
            self.info_label.setText("🗑️ Data deleted. Upload new data to continue.")
            self.log_message("🗑️ All data deleted")
            self.set_status('idle', 'Data deleted')
    
    # ============================================================
    # HELP AND ABOUT
    # ============================================================
    
    def show_help(self):
        help_text = f"""
🔬 {APP_NAME} v{APP_VERSION}

📌 Project Code: {APP_CODE}

👨‍🔬 Developer: {DEVELOPER}
📧 {EMAIL}

🚀 QUICK START

1. 📁 Upload your data (CSV or Excel)
2. 🧹 Clean the data
3. 📊 Run EDA analysis
4. 📉 Visualize results
5. 🧪 View geochemistry calculations
6. 📄 Generate HTML reports

🔧 FEATURES

Data Management:
• Upload with New/Append/Replace modes
• Save/Restore state • Undo actions

Data Cleaning:
• Detection limit handling (< values)
• Missing value imputation
• Outlier detection (IQR method)

Statistical Analysis:
• Basic statistics (mean, median, std, etc.)
• Variance analysis • Correlation matrix
• Normality tests • PCA

Exploration Geochemistry:
• Alteration Indices (Ishikawa AI, CCPI, Sericite, Propylitic)
• Element Ratios (Au/Cu, Zn/Pb, etc.)
• Mineralization Indices
• Grade Statistics and Classification

Visualization:
• Correlation matrix • PCA scores
• Distributions • Box plots

Export:
• CSV • Excel • JSON • HTML Reports

⌨️ SHORTCUTS: Ctrl+O Upload | Ctrl+C Clean | Ctrl+A Analyze | Ctrl+E Export

Copyright © 2026 {DEVELOPER}
"""
        QMessageBox.information(self, "Help", help_text)
    
    def show_about(self):
        about_text = f"""
🔬 {APP_NAME}

Version: {APP_VERSION}
Project Code: {APP_CODE}

Developer: {DEVELOPER}
Email: {EMAIL}
Background: {BACKGROUND}

📋 Description:
Professional geochemical data cleaning and 
exploratory data analysis (EDA) suite for
mineral exploration and geochemistry.

✨ Features:
• Data Cleaning Engine
• Statistical Analysis
• PCA
• Exploration Geochemistry
• Alteration Indices
• Grade Estimates
• Visualization
• HTML Reports

📌 Project Code 1010 - Data Cleaning

Copyright © 2026 {DEVELOPER}
All rights reserved.
"""
        QMessageBox.about(self, "About", about_text)


# ============================================================
# MAIN APPLICATION
# ============================================================

def main():
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    font = QFont("Segoe UI", 10)
    app.setFont(font)
    window = EDAWindow()
    window.show()
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()

In [ ]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - ENHANCED PROFESSIONAL VERSION (FIXED)
# Project Code: 1010
# Developer: Mohamed Almoufty Abdoalmoutleb Abaker
# ============================================================

import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.backends.backend_qt5agg import NavigationToolbar2QT as NavigationToolbar
import seaborn as sns
from PyQt5.QtWidgets import *
from PyQt5.QtCore import *
from PyQt5.QtGui import *
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# APPLICATION METADATA
# ============================================================

APP_NAME = "Elemental Society Analysis"
APP_VERSION = "2.0.0"
APP_CODE = "1010"
DEVELOPER = "Mohamed Almoufty Abdoalmoutleb Abaker"
EMAIL = "almoufty121@gmail.com"
BACKGROUND = "Exploration Geologist_Msc of Earth Science Engineering(TIMREX Program)"

# ============================================================
# CUSTOM ICONS (Using Unicode/Emoji)
# ============================================================

class Icons:
    UPLOAD = "📁"
    CLEAN = "🧹"
    ANALYZE = "📊"
    EXPORT = "💾"
    REPORT = "📄"
    HELP = "❓"
    ABOUT = "ℹ️"
    SAVE = "💾"
    CANCEL = "✖️"
    RESTORE = "↩️"
    BACK = "◀️"
    NEW = "📄"
    APPEND = "➕"
    REPLACE = "🔄"
    PROCESS = "⚙️"
    DONE = "✅"
    ERROR = "❌"
    WARNING = "⚠️"
    INFO = "ℹ️"

# ============================================================
# DATA CLEANING ENGINE
# ============================================================

class DataCleaner:
    def __init__(self, df):
        self.df = df.copy()
        self.cleaned_df = None
        self.log = []
        self.original_shape = df.shape
    
    def clean_all(self):
        """Run complete cleaning pipeline"""
        self.log = []
        
        # Clean column names
        self.df.columns = self.df.columns.str.strip().str.replace(' ', '_')
        self.log.append("✅ Column names cleaned")
        
        # Handle detection limits
        for col in self.df.columns:
            if col not in ['Sample_ID', 'Lithology']:
                mask = self.df[col].astype(str).str.startswith('<', na=False)
                if mask.sum() > 0:
                    for idx in self.df[mask].index:
                        val = str(self.df.loc[idx, col])
                        try:
                            limit = float(val.replace('<', '').strip())
                            self.df.loc[idx, col] = limit / 2
                        except:
                            self.df.loc[idx, col] = np.nan
                    self.log.append(f"✅ Handled detection limits in {col}")
        
        # Convert to numeric
        for col in self.df.columns:
            if col not in ['Sample_ID', 'Lithology']:
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
                self.log.append(f"✅ Converted {col} to numeric")
        
        # Handle missing values
        for col in self.df.columns:
            if col not in ['Sample_ID', 'Lithology']:
                missing = self.df[col].isnull().sum()
                if missing > 0:
                    self.df[col] = self.df[col].fillna(self.df[col].median())
                    self.log.append(f"✅ Filled {missing} missing values in {col}")
        
        # Handle outliers
        for col in self.df.columns:
            if col not in ['Sample_ID', 'Lithology']:
                Q1 = self.df[col].quantile(0.25)
                Q3 = self.df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower = Q1 - 1.5 * IQR
                upper = Q3 + 1.5 * IQR
                outliers = ((self.df[col] < lower) | (self.df[col] > upper)).sum()
                if outliers > 0:
                    self.df[col] = self.df[col].clip(lower=lower, upper=upper)
                    self.log.append(f"✅ Capped {outliers} outliers in {col}")
        
        # Remove duplicates
        initial_len = len(self.df)
        self.df = self.df.drop_duplicates()
        duplicates = initial_len - len(self.df)
        if duplicates > 0:
            self.log.append(f"✅ Removed {duplicates} duplicates")
        
        self.cleaned_df = self.df.copy()
        return self.cleaned_df
    
    def get_summary(self):
        """Get cleaning summary"""
        return {
            'original_shape': self.original_shape,
            'final_shape': self.cleaned_df.shape if self.cleaned_df is not None else None,
            'rows_removed': self.original_shape[0] - (self.cleaned_df.shape[0] if self.cleaned_df is not None else 0),
            'columns_removed': self.original_shape[1] - (self.cleaned_df.shape[1] if self.cleaned_df is not None else 0),
            'log': self.log
        }

# ============================================================
# EDA ENGINE
# ============================================================

class ProfessionalEDA:
    def __init__(self, df):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        self.run_all()
    
    def run_all(self):
        """Run all analyses"""
        # Basic statistics
        self.results['basic_stats'] = {
            'mean': self.numeric_df.mean(),
            'median': self.numeric_df.median(),
            'std': self.numeric_df.std(),
            'min': self.numeric_df.min(),
            'max': self.numeric_df.max(),
            'range': self.numeric_df.max() - self.numeric_df.min(),
            'q25': self.numeric_df.quantile(0.25),
            'q75': self.numeric_df.quantile(0.75),
            'iqr': self.numeric_df.quantile(0.75) - self.numeric_df.quantile(0.25),
            'skew': self.numeric_df.skew(),
            'kurtosis': self.numeric_df.kurtosis(),
            'count': self.numeric_df.count()
        }
        
        # Variance
        self.results['variance'] = {
            'variance': self.numeric_df.var(),
            'std_dev': self.numeric_df.std(),
            'cv': (self.numeric_df.std() / self.numeric_df.mean()) * 100
        }
        
        # Correlation
        if len(self.numeric_df.columns) > 1:
            self.results['correlation'] = self.numeric_df.corr()
            strong = []
            corr = self.results['correlation']
            for i, col1 in enumerate(corr.columns):
                for col2 in corr.columns[i+1:]:
                    val = corr.loc[col1, col2]
                    if abs(val) > 0.5:
                        strong.append((col1, col2, val))
            self.results['strong_correlations'] = sorted(strong, key=lambda x: abs(x[2]), reverse=True)
        
        # Outliers
        self.results['outliers'] = {}
        for col in self.numeric_df.columns:
            Q1 = self.numeric_df[col].quantile(0.25)
            Q3 = self.numeric_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            self.results['outliers'][col] = ((self.numeric_df[col] < lower) | (self.numeric_df[col] > upper)).sum()
        
        # PCA
        if len(self.numeric_df.columns) >= 2:
            clean_data = self.numeric_df.dropna()
            if len(clean_data) > 1:
                scaler = StandardScaler()
                scaled = scaler.fit_transform(clean_data)
                pca = SklearnPCA()
                scores = pca.fit_transform(scaled)
                self.results['pca'] = {
                    'explained_variance': pca.explained_variance_ratio_,
                    'scores': scores,
                    'loadings': pd.DataFrame(
                        pca.components_.T,
                        columns=[f'PC{i+1}' for i in range(pca.n_components_)],
                        index=clean_data.columns
                    )
                }
        
        # Normality
        self.results['normality'] = {}
        for col in self.numeric_df.columns:
            skew_val = self.numeric_df[col].skew()
            kurt_val = self.numeric_df[col].kurtosis()
            is_normal = (-1 < skew_val < 1) and (-2 < kurt_val < 2)
            self.results['normality'][col] = {
                'skewness': skew_val,
                'kurtosis': kurt_val,
                'is_normal': is_normal,
                'interpretation': 'Normal' if is_normal else 'Not normal'
            }
    
    def get_report(self):
        """Generate comprehensive report"""
        report = []
        report.append("="*80)
        report.append(f"📊 {APP_NAME} - EDA REPORT")
        report.append("="*80)
        
        report.append(f"\n📋 Dataset: {len(self.df)} rows × {len(self.df.columns)} columns")
        report.append(f"📊 Numeric columns: {len(self.numeric_df.columns)}")
        
        report.append("\n📊 BASIC STATISTICS")
        report.append("-"*40)
        stats_df = pd.DataFrame(self.results.get('basic_stats', {}))
        if not stats_df.empty:
            report.append(stats_df.round(4).to_string())
        
        report.append("\n📈 VARIANCE")
        report.append("-"*40)
        var_df = pd.DataFrame(self.results.get('variance', {}))
        if not var_df.empty:
            report.append(var_df.round(4).to_string())
        
        if 'strong_correlations' in self.results:
            report.append("\n📉 STRONG CORRELATIONS (>0.5)")
            report.append("-"*40)
            for col1, col2, val in self.results['strong_correlations'][:5]:
                direction = "positive" if val > 0 else "negative"
                report.append(f"  {col1} ↔ {col2}: {val:.4f} ({direction})")
        
        report.append("\n🎯 OUTLIERS")
        report.append("-"*40)
        for col, count in self.results.get('outliers', {}).items():
            report.append(f"  {col}: {count} outliers")
        
        report.append("\n📊 NORMALITY")
        report.append("-"*40)
        for col, data in self.results.get('normality', {}).items():
            status = "✅ Normal" if data['is_normal'] else "❌ Not normal"
            report.append(f"  {col}: {status} (skew={data['skewness']:.4f})")
        
        if 'pca' in self.results:
            report.append("\n📉 PCA")
            report.append("-"*40)
            explained = self.results['pca']['explained_variance']
            cum_var = np.cumsum(explained)
            report.append(f"Total explained variance: {explained.sum()*100:.1f}%")
            for i, (ev, cum) in enumerate(zip(explained[:5], cum_var[:5])):
                report.append(f"  PC{i+1}: {ev*100:.1f}% (cumulative: {cum*100:.1f}%)")
            report.append("\nLoadings (PC1):")
            loadings = self.results['pca']['loadings']['PC1'].abs().sort_values(ascending=False)
            for var, loading in list(loadings.items())[:5]:
                report.append(f"  {var}: {loading:.4f}")
        
        report.append("\n"+"="*80)
        report.append(f"📌 Generated by {APP_NAME} v{APP_VERSION}")
        report.append(f"👨‍🔬 Developer: {DEVELOPER}")
        report.append(f"📧 {EMAIL}")
        report.append("="*80)
        
        return "\n".join(report)

# ============================================================
# MAIN APPLICATION WINDOW
# ============================================================

class EDAWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        self.df = None
        self.cleaned_df = None
        self.eda = None
        self.cleaner = None
        self.original_df = None  # For restore functionality
        self.current_file_path = None
        self.is_processing = False
        self.init_ui()
        
    def init_ui(self):
        """Initialize the user interface"""
        self.setWindowTitle(f"🧪 {APP_NAME} v{APP_VERSION} (Code: {APP_CODE})")
        self.setGeometry(100, 100, 1500, 950)
        
        # Set application icon
        self.setWindowIcon(self.create_icon())
        
        # Central widget
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout()
        central.setLayout(main_layout)
        
        # Title bar with logo
        title_widget = self.create_title_bar()
        main_layout.addWidget(title_widget)
        
        # Main toolbar
        toolbar = self.create_main_toolbar()
        main_layout.addWidget(toolbar)
        
        # Process control toolbar
        process_toolbar = self.create_process_toolbar()
        main_layout.addWidget(process_toolbar)
        
        # Info label
        self.info_label = QLabel("📋 Ready. Load data to begin analysis.")
        self.info_label.setStyleSheet("""
            font-size: 14px; 
            font-weight: bold; 
            padding: 15px; 
            background: qlineargradient(x1:0, y1:0, x2:1, y2:0,
                stop:0 #2196F3, stop:1 #4CAF50);
            color: white; 
            border-radius: 8px;
        """)
        main_layout.addWidget(self.info_label)
        
        # Tab widget
        self.tab_widget = QTabWidget()
        main_layout.addWidget(self.tab_widget)
        
        # Create tabs
        self.create_data_tab()
        self.create_eda_tab()
        self.create_visualization_tab()
        self.create_report_tab()
        self.create_log_tab()
        
        # Status bar
        self.statusBar = QStatusBar()
        self.setStatusBar(self.statusBar)
        self.statusBar.showMessage("✅ Ready")
        
        # Progress bar in status bar
        self.progress_bar = QProgressBar()
        self.progress_bar.setVisible(False)
        self.progress_bar.setMaximumWidth(200)
        self.statusBar.addPermanentWidget(self.progress_bar)
        
        # Apply styles
        self.apply_styles()
        
        # Log initial message
        self.log_message(f"🔬 {APP_NAME} v{APP_VERSION} initialized")
        self.log_message(f"📌 Project Code: {APP_CODE}")
        self.log_message(f"👨‍🔬 Developer: {DEVELOPER}")
        self.log_message("📌 Ready for data analysis")
    
    def create_icon(self):
        """Create application icon"""
        pixmap = QPixmap(64, 64)
        pixmap.fill(Qt.transparent)
        painter = QPainter(pixmap)
        painter.setRenderHint(QPainter.Antialiasing)
        
        # Draw flask
        painter.setBrush(QBrush(QColor(33, 150, 243)))
        painter.setPen(QPen(Qt.black, 2))
        painter.drawEllipse(10, 10, 44, 44)
        
        # Draw data points
        painter.setBrush(QBrush(QColor(255, 200, 0)))
        painter.drawEllipse(25, 20, 6, 6)
        painter.drawEllipse(35, 30, 6, 6)
        painter.drawEllipse(20, 40, 6, 6)
        
        painter.end()
        return QIcon(pixmap)
    
    def create_title_bar(self):
        """Create the title bar with developer info"""
        widget = QWidget()
        layout = QHBoxLayout()
        widget.setLayout(layout)
        
        # App name and version
        title_label = QLabel(f"🔬 {APP_NAME}")
        title_label.setStyleSheet("font-size: 22px; font-weight: bold; color: #2196F3;")
        layout.addWidget(title_label)
        
        version_label = QLabel(f"v{APP_VERSION} | Code: {APP_CODE}")
        version_label.setStyleSheet("font-size: 12px; color: #666; padding-left: 10px;")
        layout.addWidget(version_label)
        
        layout.addStretch()
        
        # Developer info
        dev_label = QLabel(f"👨‍🔬 {DEVELOPER}")
        dev_label.setStyleSheet("font-size: 12px; color: #666;")
        layout.addWidget(dev_label)
        
        # Background info
        bg_label = QLabel(f"📋 {BACKGROUND}")
        bg_label.setStyleSheet("font-size: 10px; color: #888; padding-left: 10px;")
        layout.addWidget(bg_label)
        
        return widget
    
    def create_main_toolbar(self):
        """Create the main toolbar with icons"""
        toolbar = QToolBar()
        toolbar.setMovable(False)
        toolbar.setIconSize(QSize(40, 40))
        toolbar.setStyleSheet("QToolBar { spacing: 3px; padding: 5px; }")
        
        # Upload with sub-menu for append/replace
        upload_menu = QMenu()
        
        upload_new_action = QAction(f"{Icons.NEW} New Data", self)
        upload_new_action.triggered.connect(lambda: self.upload_file(mode='new'))
        upload_menu.addAction(upload_new_action)
        
        upload_append_action = QAction(f"{Icons.APPEND} Append to Existing", self)
        upload_append_action.triggered.connect(lambda: self.upload_file(mode='append'))
        upload_menu.addAction(upload_append_action)
        
        upload_replace_action = QAction(f"{Icons.REPLACE} Replace Current", self)
        upload_replace_action.triggered.connect(lambda: self.upload_file(mode='replace'))
        upload_menu.addAction(upload_replace_action)
        
        upload_btn = QAction(f"{Icons.UPLOAD} Upload", self)
        upload_btn.setMenu(upload_menu)
        upload_btn.triggered.connect(lambda: self.upload_file(mode='new'))
        toolbar.addAction(upload_btn)
        
        toolbar.addSeparator()
        
        # Clean
        clean_btn = QAction(f"{Icons.CLEAN} Clean", self)
        clean_btn.triggered.connect(self.clean_data)
        toolbar.addAction(clean_btn)
        
        # Analyze
        analyze_btn = QAction(f"{Icons.ANALYZE} Analyze", self)
        analyze_btn.triggered.connect(self.run_eda)
        toolbar.addAction(analyze_btn)
        
        toolbar.addSeparator()
        
        # Export
        export_btn = QAction(f"{Icons.EXPORT} Export", self)
        export_btn.triggered.connect(self.export_data)
        toolbar.addAction(export_btn)
        
        # Report
        report_btn = QAction(f"{Icons.REPORT} Report", self)
        report_btn.triggered.connect(self.show_report)
        toolbar.addAction(report_btn)
        
        toolbar.addSeparator()
        
        # Help
        help_btn = QAction(f"{Icons.HELP} Help", self)
        help_btn.triggered.connect(self.show_help)
        toolbar.addAction(help_btn)
        
        # About
        about_btn = QAction(f"{Icons.ABOUT} About", self)
        about_btn.triggered.connect(self.show_about)
        toolbar.addAction(about_btn)
        
        return toolbar
    
    def create_process_toolbar(self):
        """Create process control toolbar (Save, Restore, Back, Cancel)"""
        toolbar = QToolBar()
        toolbar.setMovable(False)
        toolbar.setIconSize(QSize(32, 32))
        toolbar.setStyleSheet("""
            QToolBar { 
                spacing: 5px; 
                padding: 5px; 
                background-color: #f8f9fa;
                border: 1px solid #dee2e6;
                border-radius: 4px;
            }
        """)
        
        # Save State
        save_btn = QAction(f"{Icons.SAVE} Save State", self)
        save_btn.triggered.connect(self.save_state)
        save_btn.setToolTip("Save current data and analysis state")
        toolbar.addAction(save_btn)
        
        # Restore State
        restore_btn = QAction(f"{Icons.RESTORE} Restore", self)
        restore_btn.triggered.connect(self.restore_state)
        restore_btn.setToolTip("Restore last saved state")
        toolbar.addAction(restore_btn)
        
        toolbar.addSeparator()
        
        # Back (Undo Last Action)
        back_btn = QAction(f"{Icons.BACK} Back", self)
        back_btn.triggered.connect(self.undo_last_action)
        back_btn.setToolTip("Undo last action")
        toolbar.addAction(back_btn)
        
        # Cancel Current Process
        cancel_btn = QAction(f"{Icons.CANCEL} Cancel", self)
        cancel_btn.triggered.connect(self.cancel_process)
        cancel_btn.setToolTip("Cancel current operation")
        toolbar.addAction(cancel_btn)
        
        toolbar.addSeparator()
        
        # Reset All
        reset_btn = QAction(f"{Icons.WARNING} Reset All", self)
        reset_btn.triggered.connect(self.reset_all)
        reset_btn.setToolTip("Reset everything to initial state")
        toolbar.addAction(reset_btn)
        
        # Add stretch using a spacer widget instead of addStretch()
        spacer = QWidget()
        spacer.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Preferred)
        toolbar.addWidget(spacer)
        
        # Process status indicator
        self.process_status = QLabel("⚪ Idle")
        self.process_status.setStyleSheet("""
            QLabel {
                padding: 5px 15px;
                border-radius: 4px;
                font-weight: bold;
                background-color: #e9ecef;
            }
        """)
        toolbar.addWidget(self.process_status)
        
        # Current file info
        self.file_info_label = QLabel("No file loaded")
        self.file_info_label.setStyleSheet("color: #666; padding: 5px;")
        toolbar.addWidget(self.file_info_label)
        
        return toolbar
    
    def create_data_tab(self):
        """Create data viewing tab"""
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        # Data info and controls
        info_widget = QWidget()
        info_layout = QHBoxLayout()
        info_widget.setLayout(info_layout)
        
        self.data_info = QLabel("No data loaded")
        info_layout.addWidget(self.data_info)
        info_layout.addStretch()
        
        # Row count control
        info_layout.addWidget(QLabel("Show rows:"))
        self.row_limit = QSpinBox()
        self.row_limit.setRange(10, 5000)
        self.row_limit.setValue(1000)
        self.row_limit.valueChanged.connect(self.refresh_data_display)
        info_layout.addWidget(self.row_limit)
        
        layout.addWidget(info_widget)
        
        # Data table
        self.data_table = QTableWidget()
        self.data_table.setAlternatingRowColors(True)
        self.data_table.setSortingEnabled(True)
        layout.addWidget(self.data_table)
        
        self.tab_widget.addTab(tab, "📊 Data")
    
    def create_eda_tab(self):
        """Create EDA results tab"""
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        self.eda_text = QTextEdit()
        self.eda_text.setReadOnly(True)
        self.eda_text.setFont(QFont("Consolas", 10))
        layout.addWidget(self.eda_text)
        
        self.tab_widget.addTab(tab, "📈 EDA Results")
    
    def create_visualization_tab(self):
        """Create visualization tab"""
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        # Controls
        controls = QHBoxLayout()
        
        controls.addWidget(QLabel("Plot Type:"))
        self.plot_type = QComboBox()
        self.plot_type.addItems(["Correlation Matrix", "PCA Scores", "Distributions", "Box Plots"])
        controls.addWidget(self.plot_type)
        
        controls.addWidget(QLabel("Color Theme:"))
        self.color_theme = QComboBox()
        self.color_theme.addItems(["coolwarm", "viridis", "plasma", "inferno", "magma"])
        controls.addWidget(self.color_theme)
        
        plot_btn = QPushButton("Generate Plot")
        plot_btn.clicked.connect(self.generate_plot)
        controls.addWidget(plot_btn)
        
        layout.addLayout(controls)
        
        # Matplotlib figure
        self.figure = plt.figure(figsize=(12, 7))
        self.canvas = FigureCanvas(self.figure)
        layout.addWidget(self.canvas)
        
        # Navigation toolbar
        self.toolbar = NavigationToolbar(self.canvas, self)
        layout.addWidget(self.toolbar)
        
        self.tab_widget.addTab(tab, "📉 Visualizations")
    
    def create_report_tab(self):
        """Create report tab"""
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        # Report controls
        report_controls = QHBoxLayout()
        
        export_report_btn = QPushButton("💾 Export Report")
        export_report_btn.clicked.connect(self.export_report)
        report_controls.addWidget(export_report_btn)
        
        copy_report_btn = QPushButton("📋 Copy Report")
        copy_report_btn.clicked.connect(self.copy_report)
        report_controls.addWidget(copy_report_btn)
        
        report_controls.addStretch()
        layout.addLayout(report_controls)
        
        self.report_text = QTextEdit()
        self.report_text.setReadOnly(True)
        self.report_text.setFont(QFont("Consolas", 10))
        layout.addWidget(self.report_text)
        
        self.tab_widget.addTab(tab, "📄 Report")
    
    def create_log_tab(self):
        """Create log tab"""
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        # Log controls
        log_controls = QHBoxLayout()
        
        clear_log_btn = QPushButton("🗑️ Clear Log")
        clear_log_btn.clicked.connect(self.clear_log)
        log_controls.addWidget(clear_log_btn)
        
        copy_log_btn = QPushButton("📋 Copy Log")
        copy_log_btn.clicked.connect(self.copy_log)
        log_controls.addWidget(copy_log_btn)
        
        log_controls.addStretch()
        layout.addLayout(log_controls)
        
        self.log_text = QTextEdit()
        self.log_text.setReadOnly(True)
        self.log_text.setFont(QFont("Consolas", 10))
        layout.addWidget(self.log_text)
        
        self.tab_widget.addTab(tab, "📝 Log")
    
    def apply_styles(self):
        """Apply stylesheet"""
        self.setStyleSheet("""
            QMainWindow { background-color: #f5f5f5; }
            QTableWidget { 
                background-color: white; 
                gridline-color: #d0d0d0;
                alternate-background-color: #fafafa;
            }
            QTableWidget::item:selected { 
                background-color: #2196F3; 
                color: white; 
            }
            QHeaderView::section { 
                background-color: #e0e0e0; 
                padding: 5px; 
                font-weight: bold;
            }
            QPushButton { 
                background-color: #2196F3; 
                color: white; 
                padding: 8px 15px; 
                border-radius: 4px; 
                border: none; 
                font-weight: bold;
            }
            QPushButton:hover { background-color: #1976D2; }
            QPushButton:pressed { background-color: #0D47A1; }
            QTabWidget::pane { border: 1px solid #d0d0d0; border-radius: 4px; }
            QTabBar::tab { 
                padding: 10px 20px; 
                margin-right: 2px;
                border-top-left-radius: 4px;
                border-top-right-radius: 4px;
                background-color: #e0e0e0;
                font-weight: bold;
            }
            QTabBar::tab:selected { 
                background-color: #2196F3; 
                color: white; 
            }
            QTabBar::tab:hover { background-color: #d0d0d0; }
            QComboBox { 
                padding: 5px; 
                border: 1px solid #d0d0d0; 
                border-radius: 4px;
            }
            QTextEdit { 
                background-color: white; 
                border: 1px solid #d0d0d0; 
                border-radius: 4px; 
            }
            QMenuBar { background-color: #f0f0f0; }
            QMenuBar::item:selected { background-color: #2196F3; color: white; }
            QToolBar { spacing: 5px; }
        """)
    
    # ============================================================
    # LOG FUNCTIONS
    # ============================================================
    
    def log_message(self, message):
        """Add message to log"""
        from datetime import datetime
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.append(f"[{timestamp}] {message}")
    
    def clear_log(self):
        """Clear the log"""
        self.log_text.clear()
        self.log_message("🗑️ Log cleared")
    
    def copy_log(self):
        """Copy log to clipboard"""
        clipboard = QApplication.clipboard()
        clipboard.setText(self.log_text.toPlainText())
        self.log_message("📋 Log copied to clipboard")
    
    def copy_report(self):
        """Copy report to clipboard"""
        clipboard = QApplication.clipboard()
        clipboard.setText(self.report_text.toPlainText())
        self.log_message("📋 Report copied to clipboard")
    
    # ============================================================
    # PROCESS CONTROL FUNCTIONS
    # ============================================================
    
    def set_processing(self, status, message=""):
        """Set processing state"""
        self.is_processing = status
        if status:
            self.process_status.setText(f"⚙️ Processing... {message}")
            self.process_status.setStyleSheet("""
                QLabel {
                    padding: 5px 15px;
                    border-radius: 4px;
                    font-weight: bold;
                    background-color: #fff3cd;
                    color: #856404;
                }
            """)
            self.progress_bar.setVisible(True)
            self.progress_bar.setRange(0, 0)
        else:
            self.process_status.setText("⚪ Idle")
            self.process_status.setStyleSheet("""
                QLabel {
                    padding: 5px 15px;
                    border-radius: 4px;
                    font-weight: bold;
                    background-color: #e9ecef;
                }
            """)
            self.progress_bar.setVisible(False)
    
    def save_state(self):
        """Save current state"""
        if self.df is not None:
            self.original_df = self.df.copy()
            self.log_message("💾 State saved successfully")
            self.statusBar.showMessage("✅ State saved")
            QMessageBox.information(self, "Success", "Current state saved successfully!")
        else:
            QMessageBox.warning(self, "Warning", "No data to save!")
    
    def restore_state(self):
        """Restore last saved state"""
        if self.original_df is not None:
            reply = QMessageBox.question(self, "Restore State", 
                "This will restore the last saved state. Continue?",
                QMessageBox.Yes | QMessageBox.No)
            
            if reply == QMessageBox.Yes:
                self.df = self.original_df.copy()
                self.cleaned_df = None
                self.eda = None
                self.display_data()
                self.log_message("↩️ State restored")
                self.statusBar.showMessage("✅ State restored")
                QMessageBox.information(self, "Success", "State restored successfully!")
        else:
            QMessageBox.warning(self, "Warning", "No saved state found!")
    
    def undo_last_action(self):
        """Undo last action"""
        if self.original_df is not None:
            reply = QMessageBox.question(self, "Undo", 
                "This will undo the last action. Continue?",
                QMessageBox.Yes | QMessageBox.No)
            
            if reply == QMessageBox.Yes:
                self.df = self.original_df.copy()
                self.cleaned_df = None
                self.eda = None
                self.display_data()
                self.log_message("◀️ Last action undone")
                self.statusBar.showMessage("✅ Action undone")
        else:
            QMessageBox.warning(self, "Warning", "Nothing to undo!")
    
    def cancel_process(self):
        """Cancel current process"""
        if self.is_processing:
            reply = QMessageBox.question(self, "Cancel Process", 
                "Cancel current operation?",
                QMessageBox.Yes | QMessageBox.No)
            
            if reply == QMessageBox.Yes:
                self.set_processing(False)
                self.log_message("✖️ Process cancelled by user")
                self.statusBar.showMessage("❌ Process cancelled")
        else:
            QMessageBox.information(self, "Info", "No process is currently running")
    
    def reset_all(self):
        """Reset everything to initial state"""
        reply = QMessageBox.question(self, "Reset All", 
            "This will remove all data and reset everything. Continue?",
            QMessageBox.Yes | QMessageBox.No)
        
        if reply == QMessageBox.Yes:
            self.df = None
            self.cleaned_df = None
            self.eda = None
            self.original_df = None
            self.current_file_path = None
            
            self.data_table.clear()
            self.data_table.setRowCount(0)
            self.data_table.setColumnCount(0)
            self.eda_text.clear()
            self.report_text.clear()
            self.figure.clear()
            self.canvas.draw()
            
            self.data_info.setText("No data loaded")
            self.file_info_label.setText("No file loaded")
            self.info_label.setText("📋 Reset complete. Ready for new data.")
            
            self.log_message("🔄 All data reset")
            self.statusBar.showMessage("✅ Reset complete")
            QMessageBox.information(self, "Success", "All data has been reset!")
    
    # ============================================================
    # DATA LOADING FUNCTIONS
    # ============================================================
    
    def upload_file(self, mode='new'):
        """Upload data file with mode selection"""
        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Open Geochemical Data",
            "",
            "All Files (*);;CSV (*.csv);;Excel (*.xlsx *.xls);;Text (*.txt);;TSV (*.tsv)"
        )
        
        if not file_path:
            return
        
        self.current_file_path = file_path
        self.set_processing(True, "Loading file...")
        self.log_message(f"📁 Loading: {os.path.basename(file_path)}")
        self.statusBar.showMessage(f"Loading: {os.path.basename(file_path)}...")
        
        try:
            # Try different delimiters and encodings
            delimiters = [';', ',', '\t', '|']
            encodings = ['utf-8', 'cp1252', 'latin-1']
            
            df = None
            for encoding in encodings:
                for delim in delimiters:
                    try:
                        df = pd.read_csv(file_path, sep=delim, encoding=encoding, on_bad_lines='skip')
                        if df is not None and not df.empty and df.shape[1] > 1:
                            break
                    except:
                        continue
                if df is not None and not df.empty:
                    break
            
            # If still None, try Excel
            if df is None or df.empty:
                try:
                    df = pd.read_excel(file_path)
                except:
                    pass
            
            if df is not None and not df.empty:
                # Handle mode
                if mode == 'new' or self.df is None:
                    self.df = df
                elif mode == 'append':
                    # Check if columns match
                    if set(self.df.columns) == set(df.columns):
                        self.df = pd.concat([self.df, df], ignore_index=True)
                    else:
                        QMessageBox.warning(self, "Warning", 
                            "Column mismatch! Data will be loaded as new.")
                        self.df = df
                elif mode == 'replace':
                    self.df = df
                
                # Save original for restore
                self.original_df = self.df.copy()
                
                self.cleaned_df = None
                self.eda = None
                self.display_data()
                self.file_info_label.setText(f"📁 {os.path.basename(file_path)}")
                
                self.log_message(f"✅ Loaded: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
                self.statusBar.showMessage(f"✅ Loaded: {os.path.basename(file_path)}")
                self.info_label.setText(f"📊 Data loaded: {self.df.shape[0]} rows × {self.df.shape[1]} columns")
                
                QMessageBox.information(self, "Success",
                    f"File loaded successfully!\n\n"
                    f"📊 Shape: {self.df.shape[0]} rows × {self.df.shape[1]} columns\n"
                    f"📋 Mode: {mode.upper()}")
            else:
                QMessageBox.warning(self, "Error", "Could not read file.")
                
        except Exception as e:
            self.log_message(f"❌ Error: {str(e)}")
            QMessageBox.critical(self, "Error", f"Failed to load:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def refresh_data_display(self):
        """Refresh data display with current row limit"""
        self.display_data()
    
    def display_data(self, df=None):
        """Display data in table"""
        if df is None:
            if self.cleaned_df is not None:
                df = self.cleaned_df
            elif self.df is not None:
                df = self.df
            else:
                return
        
        display_limit = min(self.row_limit.value(), len(df))
        self.data_table.setRowCount(display_limit)
        self.data_table.setColumnCount(len(df.columns))
        self.data_table.setHorizontalHeaderLabels(df.columns.astype(str))
        
        for i in range(display_limit):
            for j, col in enumerate(df.columns):
                val = df.iloc[i, j]
                if pd.isna(val):
                    self.data_table.setItem(i, j, QTableWidgetItem(""))
                else:
                    self.data_table.setItem(i, j, QTableWidgetItem(str(val)))
        
        self.data_table.resizeColumnsToContents()
        self.data_info.setText(f"📊 {len(df)} rows × {len(df.columns)} columns (showing {display_limit})")
    
    # ============================================================
    # DATA CLEANING FUNCTIONS
    # ============================================================
    
    def clean_data(self):
        """Clean the loaded data"""
        if self.df is None or self.df.empty:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
        
        reply = QMessageBox.question(self, "Clean Data", 
            "This will clean the current dataset. Continue?",
            QMessageBox.Yes | QMessageBox.No)
        
        if reply != QMessageBox.Yes:
            return
        
        self.set_processing(True, "Cleaning data...")
        self.log_message("🧹 Starting data cleaning...")
        self.statusBar.showMessage("Cleaning data...")
        
        try:
            self.cleaner = DataCleaner(self.df)
            self.cleaned_df = self.cleaner.clean_all()
            
            # Save original for undo
            if self.original_df is None:
                self.original_df = self.df.copy()
            
            self.display_data(self.cleaned_df)
            
            self.log_message(f"✅ Cleaning complete: {len(self.cleaned_df)} rows")
            self.statusBar.showMessage(f"✅ Cleaned: {len(self.cleaned_df)} rows")
            self.info_label.setText(f"🧹 Cleaned: {len(self.cleaned_df)} rows × {len(self.cleaned_df.columns)} columns")
            
            # Show cleaning summary
            summary = self.cleaner.get_summary()
            QMessageBox.information(self, "Cleaning Complete",
                f"Cleaning complete!\n\n"
                f"📊 Before: {summary['original_shape'][0]} rows\n"
                f"📊 After: {summary['final_shape'][0]} rows\n"
                f"🗑️ Removed: {summary['rows_removed']} rows")
            
        except Exception as e:
            self.log_message(f"❌ Cleaning error: {str(e)}")
            QMessageBox.critical(self, "Error", f"Cleaning failed:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    # ============================================================
    # EDA FUNCTIONS
    # ============================================================
    
    def run_eda(self):
        """Run EDA analysis"""
        df_to_analyze = self.cleaned_df if self.cleaned_df is not None else self.df
        
        if df_to_analyze is None or df_to_analyze.empty:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
        
        self.set_processing(True, "Running EDA...")
        self.log_message("📊 Running EDA...")
        self.statusBar.showMessage("Analyzing data...")
        
        try:
            self.eda = ProfessionalEDA(df_to_analyze)
            
            # Display results
            self.display_eda_results()
            self.display_report()
            
            self.log_message("✅ EDA complete!")
            self.statusBar.showMessage("✅ EDA complete")
            
            QMessageBox.information(self, "Success",
                "EDA completed successfully!\n\n"
                f"📊 Variables analyzed: {len(self.eda.numeric_df.columns)}\n"
                f"📈 Statistics computed\n"
                f"📉 Correlations analyzed\n"
                f"📊 PCA performed\n"
                f"📋 Full report generated")
            
        except Exception as e:
            self.log_message(f"❌ EDA error: {str(e)}")
            QMessageBox.critical(self, "Error", f"EDA failed:\n{str(e)}")
        finally:
            self.set_processing(False)
    
    def display_eda_results(self):
        """Display EDA results"""
        if self.eda is None:
            return
        
        results = []
        results.append("="*80)
        results.append("📊 EDA RESULTS")
        results.append("="*80)
        
        results.append(f"\n📋 Dataset: {len(self.eda.df)} rows × {len(self.eda.df.columns)} columns")
        results.append(f"📊 Numeric columns: {len(self.eda.numeric_df.columns)}")
        
        results.append("\n📊 BASIC STATISTICS")
        results.append("-"*40)
        stats_df = pd.DataFrame(self.eda.results.get('basic_stats', {}))
        if not stats_df.empty:
            results.append(stats_df.round(4).to_string())
        
        if 'strong_correlations' in self.eda.results:
            results.append("\n📉 STRONG CORRELATIONS")
            results.append("-"*40)
            for col1, col2, val in self.eda.results['strong_correlations'][:5]:
                direction = "positive" if val > 0 else "negative"
                results.append(f"  {col1} ↔ {col2}: {val:.4f} ({direction})")
        
        results.append("\n🎯 OUTLIERS")
        results.append("-"*40)
        for col, count in self.eda.results.get('outliers', {}).items():
            results.append(f"  {col}: {count} outliers")
        
        results.append("\n📊 NORMALITY")
        results.append("-"*40)
        for col, data in self.eda.results.get('normality', {}).items():
            status = "✅ Normal" if data['is_normal'] else "❌ Not normal"
            results.append(f"  {col}: {status} (skew={data['skewness']:.4f})")
        
        self.eda_text.setText("\n".join(results))
    
    def display_report(self):
        """Display full report"""
        if self.eda is None:
            return
        
        report = self.eda.get_report()
        self.report_text.setText(report)
    
    # ============================================================
    # VISUALIZATION FUNCTIONS
    # ============================================================
    
    def generate_plot(self):
        """Generate selected plot"""
        if self.eda is None:
            QMessageBox.warning(self, "Warning", "Please run EDA first!")
            return
        
        plot_type = self.plot_type.currentText()
        theme = self.color_theme.currentText()
        
        self.figure.clear()
        
        try:
            if plot_type == "Correlation Matrix":
                self.plot_correlation(theme)
            elif plot_type == "PCA Scores":
                self.plot_pca(theme)
            elif plot_type == "Distributions":
                self.plot_distributions()
            elif plot_type == "Box Plots":
                self.plot_boxplots()
            
            self.canvas.draw()
            self.log_message(f"📉 Generated {plot_type} plot")
        except Exception as e:
            self.log_message(f"❌ Plot error: {str(e)}")
            QMessageBox.critical(self, "Error", f"Failed to generate plot:\n{str(e)}")
    
    def plot_correlation(self, theme='coolwarm'):
        """Plot correlation matrix"""
        if 'correlation' not in self.eda.results:
            return
        
        ax = self.figure.add_subplot(111)
        mask = np.triu(np.ones_like(self.eda.results['correlation'], dtype=bool))
        sns.heatmap(self.eda.results['correlation'], mask=mask, annot=True,
                   cmap=theme, center=0, fmt='.2f',
                   square=True, linewidths=0.5, ax=ax)
        ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
    
    def plot_pca(self, theme='viridis'):
        """Plot PCA scores"""
        if 'pca' not in self.eda.results:
            return
        
        pca = self.eda.results['pca']
        scores = pca['scores']
        explained = pca['explained_variance']
        
        ax = self.figure.add_subplot(111)
        scatter = ax.scatter(scores[:, 0], scores[:, 1],
                            c=np.linalg.norm(scores[:, :2], axis=1),
                            cmap=theme, alpha=0.6,
                            edgecolors='black', linewidth=0.5)
        ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
        ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
        ax.set_title('PCA Scores', fontweight='bold')
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
        ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
        ax.grid(True, alpha=0.3)
        self.figure.colorbar(scatter, ax=ax, label='Distance')
    
    def plot_distributions(self):
        """Plot distributions"""
        cols = self.eda.numeric_df.columns[:4]
        n_cols = min(2, len(cols))
        n_rows = (len(cols) + n_cols - 1) // n_cols
        
        for i, col in enumerate(cols):
            ax = self.figure.add_subplot(n_rows, n_cols, i+1)
            data = self.eda.numeric_df[col].dropna()
            if len(data) > 0:
                ax.hist(data, bins=20, density=True, alpha=0.6,
                       color='steelblue', edgecolor='black')
                ax.set_xlabel(col, fontsize=10)
                ax.set_ylabel('Density', fontsize=10)
                ax.set_title(f'{col} Distribution', fontweight='bold')
                ax.axvline(data.mean(), color='red', linestyle='--',
                          label=f'Mean: {data.mean():.2f}')
                ax.axvline(data.median(), color='green', linestyle='--',
                          label=f'Median: {data.median():.2f}')
                ax.legend(fontsize=8)
                ax.grid(True, alpha=0.3)
        
        self.figure.tight_layout()
    
    def plot_boxplots(self):
        """Plot box plots"""
        ax = self.figure.add_subplot(111)
        self.eda.numeric_df.boxplot(ax=ax, rot=45)
        ax.set_title('Box Plots', fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    # ============================================================
    # EXPORT FUNCTIONS
    # ============================================================
    
    def export_data(self):
        """Export cleaned data"""
        if self.cleaned_df is None and self.df is None:
            QMessageBox.warning(self, "Warning", "No data to export!")
            return
        
        df = self.cleaned_df if self.cleaned_df is not None else self.df
        
        formats = {
            "CSV (;)": "csv_semicolon",
            "CSV (,)": "csv_comma",
            "Excel (.xlsx)": "excel",
            "JSON": "json"
        }
        
        format_choice, ok = QInputDialog.getItem(
            self, "Export Format", "Select format:", list(formats.keys()), 0, False
        )
        
        if not ok:
            return
        
        file_ext_map = {
            "csv_semicolon": "csv",
            "csv_comma": "csv",
            "excel": "xlsx",
            "json": "json"
        }
        
        file_path, _ = QFileDialog.getSaveFileName(
            self, "Export Data", f"cleaned_data.{file_ext_map[formats[format_choice]]}",
            f"{format_choice} (*.{file_ext_map[formats[format_choice]]})"
        )
        
        if file_path:
            try:
                if formats[format_choice] == "csv_semicolon":
                    df.to_csv(file_path, sep=';', index=False, encoding='utf-8-sig')
                elif formats[format_choice] == "csv_comma":
                    df.to_csv(file_path, sep=',', index=False, encoding='utf-8-sig')
                elif formats[format_choice] == "excel":
                    df.to_excel(file_path, index=False)
                elif formats[format_choice] == "json":
                    df.to_json(file_path, orient='records', indent=2)
                
                self.log_message(f"✅ Exported: {file_path}")
                QMessageBox.information(self, "Success", f"Data exported to:\n{file_path}")
                
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")
    
    def export_report(self):
        """Export report"""
        if self.eda is None:
            QMessageBox.warning(self, "Warning", "Please run EDA first!")
            return
        
        file_path, _ = QFileDialog.getSaveFileName(
            self, "Export Report", "eda_report.txt", "Text Files (*.txt)"
        )
        
        if file_path:
            try:
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(self.eda.get_report())
                self.log_message(f"✅ Report exported: {file_path}")
                QMessageBox.information(self, "Success", f"Report saved to:\n{file_path}")
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")
    
    def show_report(self):
        """Show full report"""
        if self.eda is None:
            QMessageBox.warning(self, "Warning", "Please run EDA first!")
            return
        
        report = self.eda.get_report()
        dialog = QDialog(self)
        dialog.setWindowTitle("📄 Full EDA Report")
        dialog.setGeometry(200, 200, 1000, 700)
        
        layout = QVBoxLayout()
        dialog.setLayout(layout)
        
        text_edit = QTextEdit()
        text_edit.setFont(QFont("Consolas", 10))
        text_edit.setText(report)
        layout.addWidget(text_edit)
        
        # Buttons
        btn_layout = QHBoxLayout()
        
        copy_btn = QPushButton("📋 Copy")
        copy_btn.clicked.connect(lambda: self.copy_text(text_edit.toPlainText()))
        btn_layout.addWidget(copy_btn)
        
        save_btn = QPushButton("💾 Save")
        save_btn.clicked.connect(lambda: self.save_report_text(text_edit.toPlainText()))
        btn_layout.addWidget(save_btn)
        
        btn_layout.addStretch()
        
        close_btn = QPushButton("Close")
        close_btn.clicked.connect(dialog.accept)
        btn_layout.addWidget(close_btn)
        
        layout.addLayout(btn_layout)
        
        dialog.exec_()
    
    def copy_text(self, text):
        """Copy text to clipboard"""
        clipboard = QApplication.clipboard()
        clipboard.setText(text)
        self.log_message("📋 Text copied to clipboard")
        QMessageBox.information(self, "Success", "Copied to clipboard!")
    
    def save_report_text(self, text):
        """Save report text"""
        file_path, _ = QFileDialog.getSaveFileName(
            self, "Save Report", "eda_report.txt", "Text Files (*.txt)"
        )
        
        if file_path:
            try:
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(text)
                self.log_message(f"✅ Report saved: {file_path}")
                QMessageBox.information(self, "Success", f"Report saved to:\n{file_path}")
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Save failed:\n{str(e)}")
    
    # ============================================================
    # HELP AND ABOUT
    # ============================================================
    
    def show_help(self):
        """Show help dialog"""
        help_text = f"""
🔬 {APP_NAME} v{APP_VERSION}

📌 Project Code: {APP_CODE}

👨‍🔬 Developer: {DEVELOPER}
📧 {EMAIL}
📋 Background: {BACKGROUND}

FEATURES:

1. Data Management
   - Upload with multiple modes (New/Append/Replace)
   - Save/Restore state
   - Undo last action
   - Cancel processing

2. Data Cleaning
   - Universal file support (CSV, Excel, TSV, Text)
   - Detection limit handling (< values)
   - Missing value imputation
   - Outlier detection and capping
   - Duplicate removal

3. Statistical Analysis
   - Basic statistics (mean, median, std, etc.)
   - Variance analysis
   - Correlation (Pearson)
   - Outlier detection (IQR method)
   - Normality tests (Skewness/Kurtosis)

4. Multivariate Analysis
   - Principal Component Analysis (PCA)
   - Eigen decomposition

5. Visualization
   - Correlation heatmaps
   - PCA score plots
   - Distribution histograms
   - Box plots

6. Export
   - CSV (semicolon or comma)
   - Excel (.xlsx)
   - JSON
   - Report

SHORTCUTS:
- Ctrl+O: Upload Data
- Ctrl+C: Clean Data
- Ctrl+A: Run Analysis
- Ctrl+E: Export Data
- Ctrl+R: Generate Report
- Ctrl+S: Save State
- Ctrl+Z: Undo

COPYRIGHT © 2026 {DEVELOPER}
All rights reserved. Proprietary software.
"""
        QMessageBox.information(self, "Help", help_text)
    
    def show_about(self):
        """Show about dialog"""
        about_text = f"""
🔬 {APP_NAME}

Version: {APP_VERSION}
Project Code: {APP_CODE}

Developer: {DEVELOPER}
Email: {EMAIL}
Background: {BACKGROUND}

Description:
Professional geochemical data cleaning and 
exploratory data analysis (EDA) suite for
mineral exploration and geochemistry.

Features:
- Data Cleaning Engine
- Statistical Analysis
- Principal Component Analysis (PCA)
- Comprehensive Visualization
- Professional Reporting
- Multi-format Export

Copyright © 2026 {DEVELOPER}
All rights reserved.

This software is protected under international
copyright laws. Unauthorized use prohibited.

Project Code 1010 - Data Cleaning
"""
        QMessageBox.about(self, "About", about_text)

# ============================================================
# MAIN APPLICATION
# ============================================================

def main():
    app = QApplication(sys.argv)
    app.setStyle('Fusion')
    
    # Set application font
    font = QFont("Segoe UI", 10)
    app.setFont(font)
    
    window = EDAWindow()
    window.show()
    
    sys.exit(app.exec_())

if __name__ == '__main__':
    main()

In [1]:
# 1. CODE NUMBER: 1010 
# ============================================================
# GEOCHEMICAL DATA ANALYSIS - JUPYTER VERSION
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# PART 1: DATA CLEANING ENGINE
# ============================================================

def clean_any_data(path, sheet_name=None, **kwargs):
    """Universal data cleaner for Jupyter notebooks"""
    print(f"\n📁 READING FILE: {os.path.basename(path)}")
    print("=" * 60)
    
    file_ext = Path(path).suffix.lower()
    df = None
    encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1', 'utf-16', 'cp1250']
    
    if file_ext in ['.xlsx', '.xls', '.xlsm']:
        df = read_excel_file(path, sheet_name)
    elif file_ext in ['.csv', '.txt', '.dat', '.tsv']:
        df = read_text_file(path, encodings, **kwargs)
        if df is None or df.empty:
            df = manual_parse_text_file(path)
    else:
        df = manual_parse_text_file(path)
    
    if df is None or df.empty:
        print("❌ Could not read file")
        return None]
    
    print(f"📊 Raw shape: {df.shape}")
    df = clean_dataframe(df)
    print(f"✅ Final shape: {df.shape}")
    return df

def read_excel_file(path, sheet_name):
    try:
        df = pd.read_excel(path, sheet_name=sheet_name, engine='openpyxl')
        print("✅ Excel file loaded")
        return df
    except:
        try:
            df = pd.read_excel(path, sheet_name=sheet_name, engine='xlrd')
            print("✅ Excel file loaded (xlrd)")
            return df
        except Exception as e:
            print(f"⚠️ Excel read failed: {e}")
            return None

def read_text_file(path, encodings, **kwargs):
    delimiters = [';', ',', '\t', '|', ' ']
    for encoding in encodings:
        for delim in delimiters:
            try:
                df = pd.read_csv(path, sep=delim, encoding=encoding, engine='python', on_bad_lines='skip', **kwargs)
                if df is not None and not df.empty and df.shape[1] > 1:
                    print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                    return df
            except:
                continue
    return None

def manual_parse_text_file(path):
    try:
        with open(path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        df = pd.DataFrame(data)
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    return None

def clean_dataframe(df):
    if df is None or df.empty:
        return df
    df = split_single_column(df)
    df = clean_column_names(df)
    df = remove_header_rows(df)
    df = standardize_missing_values(df)
    df = clean_numeric_columns(df)
    df = clean_text_columns(df)
    df = remove_empty(df)
    return df

def split_single_column(df):
    if df is None or df.empty or df.shape[1] != 1:
        return df
    col_name = df.columns[0]
    col_data = df[col_name].astype(str)
    sample = col_data.iloc[0] if len(col_data) > 0 else ''
    delimiters = [';', ',', '\t', '|']
    for delim in delimiters:
        if delim in sample:
            split_data = col_data.str.split(delim, expand=True)
            try:
                pd.to_numeric(split_data.iloc[0, 0])
                split_data.columns = [f'col_{i+1}' for i in range(split_data.shape[1])]
            except:
                headers = split_data.iloc[0].tolist()
                split_data = split_data.iloc[1:].copy()
                headers = [str(h).strip() for h in headers]
                split_data.columns = headers
            return split_data.reset_index(drop=True)
    return df

def clean_column_names(df):
    if df is None or df.empty:
        return df
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.loc[:, ~df.columns.astype(str).str.contains(r',+$', na=False)]
    df.columns = [f'col_{i}' if str(col).strip() == '' or pd.isna(col) else col for i, col in enumerate(df.columns)]
    df.columns = [re.sub(r'[^a-zA-Z0-9_ ]', '_', str(col)).strip('_') for col in df.columns]
    seen = {}
    new_cols = []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

def remove_header_rows(df):
    if df is None or df.empty or len(df) == 0:
        return df
    first_row = df.iloc[0].astype(str)
    header_keywords = ['sample', 'point', 'id', 'name', 'date', 'time', 'value', 'measurement', 'parameter', 'unit', 'ppm']
    header_score = sum(1 for val in first_row if any(keyword in str(val).lower() for keyword in header_keywords))
    if len(first_row) > 0 and header_score / len(first_row) > 0.3:
        df = df.iloc[1:].reset_index(drop=True)
    return df

def standardize_missing_values(df):
    if df is None or df.empty:
        return df
    missing_values = ['', 'nan', 'NaN', 'NAN', 'null', 'NULL', 'None', 'na', 'NA', '-', '--', '---', 'n/a', 'N/A', '?']
    df = df.replace(missing_values, np.nan)
    for col in df.columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'^<.*$', 'nan', regex=True)
            df[col] = df[col].astype(str).str.replace(r'^>.*$', 'nan', regex=True)
    return df

def clean_numeric_columns(df):
    if df is None or df.empty:
        return df
    text_cols = ['sample', 'point_id', 'point', 'id', 'name', 'lithology', 'description', 'comment', 'remarks', 'unit', 'method', 'date', 'time']
    def clean_numeric(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip().replace(',', '.')
        if x.startswith('<') or x.startswith('>'):
            return np.nan
        if x in ['', 'nan', 'NaN', 'NAN', 'null', 'NULL']:
            return np.nan
        x = re.sub(r'[^\d.\-eE+]', '', x)
        try:
            return float(x)
        except:
            return np.nan
    for col in df.columns:
        col_lower = col.lower()
        is_text = any(text in col_lower for text in text_cols)
        if not is_text:
            try:
                df[col] = df[col].map(clean_numeric)
            except:
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    pass
    return df

def clean_text_columns(df):
    if df is None or df.empty:
        return df
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace('', np.nan)
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
    return df

def remove_empty(df):
    if df is None or df.empty:
        return df
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    return df

# ============================================================
# PART 2: STATISTICAL ANALYSIS ENGINE
# ============================================================

class StatisticalAnalyzer:
    """Comprehensive statistical analysis for geochemical data"""
    
    def __init__(self, df):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        self.run_all_analyses()
    
    def run_all_analyses(self):
        """Run all statistical analyses"""
        self.calculate_variance()
        self.calculate_covariance()
        self.calculate_correlation()
        self.calculate_eigen()
        self.perform_pca()
        self.standardize_data()
        self.exploratory_analysis()
    
    def calculate_variance(self):
        """Calculate variance for all numeric columns"""
        self.results['variance'] = self.numeric_df.var()
        self.results['variance_dict'] = self.numeric_df.var().to_dict()
        self.results['std'] = self.numeric_df.std()
        self.results['mean'] = self.numeric_df.mean()
        self.results['median'] = self.numeric_df.median()
    
    def calculate_covariance(self):
        """Calculate covariance matrix"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 1:
            self.results['covariance_matrix'] = clean_data.cov()
        else:
            self.results['covariance_matrix'] = None
    
    def calculate_correlation(self):
        """Calculate correlation matrix"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 1:
            self.results['correlation_matrix'] = clean_data.corr()
        else:
            self.results['correlation_matrix'] = None
    
    def calculate_eigen(self):
        """Calculate eigenvalues and eigenvectors"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2:
            self.results['eigenvalues'] = None
            self.results['eigenvectors'] = None
            return
        
        try:
            cov = clean_data.cov().values
            eigenvals, eigenvecs = np.linalg.eig(cov)
            
            # Sort by eigenvalues (largest first)
            idx = np.argsort(eigenvals)[::-1]
            eigenvals = eigenvals[idx]
            eigenvecs = eigenvecs[:, idx]
            
            self.results['eigenvalues'] = eigenvals
            self.results['eigenvectors'] = eigenvecs
            self.results['explained_variance'] = eigenvals / eigenvals.sum() * 100
        except Exception as e:
            self.results['eigenvalues'] = None
            self.results['eigenvectors'] = None
            print(f"⚠️ Eigen decomposition failed: {e}")
    
    def perform_pca(self):
        """Perform Principal Component Analysis"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2 or self.results['eigenvectors'] is None:
            self.results['pca_scores'] = None
            self.results['pca_loadings'] = None
            return
        
        try:
            centered = clean_data - clean_data.mean()
            pca_scores = centered @ self.results['eigenvectors']
            self.results['pca_scores'] = pca_scores
            self.results['pca_loadings'] = self.results['eigenvectors']
            self.results['pca_columns'] = [f'PC{i+1}' for i in range(pca_scores.shape[1])]
            self.results['pca_df'] = pd.DataFrame(pca_scores, columns=self.results['pca_columns'])
        except Exception as e:
            print(f"⚠️ PCA failed: {e}")
    
    def standardize_data(self):
        """Standardize data (Z-score normalization)"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 0:
            self.results['standardized'] = (clean_data - clean_data.mean()) / clean_data.std()
            self.results['min_max_scaled'] = (clean_data - clean_data.min()) / (clean_data.max() - clean_data.min())
    
    def exploratory_analysis(self):
        """Full exploratory data analysis"""
        self.results['eda'] = {}
        self.results['eda']['shape'] = self.df.shape
        self.results['eda']['numeric_shape'] = self.numeric_df.shape
        self.results['eda']['missing'] = self.df.isnull().sum()
        self.results['eda']['missing_pct'] = self.df.isnull().sum() / len(self.df) * 100
        self.results['eda']['skewness'] = self.numeric_df.skew()
        self.results['eda']['kurtosis'] = self.numeric_df.kurtosis()
        self.results['eda']['quantiles'] = self.numeric_df.quantile([0.25, 0.5, 0.75])
        
        if len(self.numeric_df.columns) > 1:
            corr = self.numeric_df.corr()
            self.results['eda']['correlation_summary'] = {
                'max_corr': corr.max().max(),
                'min_corr': corr.min().min(),
                'high_corr_pairs': [(col1, col2, corr.loc[col1, col2]) 
                                   for i, col1 in enumerate(corr.columns)
                                   for col2 in corr.columns[i+1:]
                                   if abs(corr.loc[col1, col2]) > 0.7]
            }
    
    def get_summary(self):
        """Get summary statistics"""
        return {
            'variance': self.results.get('variance_dict', {}),
            'mean': self.results.get('mean', {}),
            'std': self.results.get('std', {}),
            'correlation_matrix': self.results.get('correlation_matrix', None),
            'eigenvalues': self.results.get('eigenvalues', None),
            'eigenvectors': self.results.get('eigenvectors', None),
            'explained_variance': self.results.get('explained_variance', None),
            'pca_scores': self.results.get('pca_df', None),
            'missing_values': self.results.get('eda', {}).get('missing', None),
            'skewness': self.results.get('eda', {}).get('skewness', None)
        }
    
    def get_pca_report(self, n_components=2):
        """Get a detailed PCA report"""
        if self.results['pca_scores'] is None:
            return "PCA could not be performed."
        
        report = []
        report.append("="*70)
        report.append("PRINCIPAL COMPONENT ANALYSIS REPORT")
        report.append("="*70)
        report.append(f"\nSamples: {self.results['pca_scores'].shape[0]}")
        report.append(f"Variables: {self.results['pca_scores'].shape[1]}")
        report.append("\nEigenvalues and Explained Variance:")
        report.append("-"*40)
        
        for i in range(len(self.results['eigenvalues'])):
            ev = self.results['eigenvalues'][i]
            ev_pct = self.results['explained_variance'][i]
            cum_pct = self.results['explained_variance'][:i+1].sum()
            report.append(f"  PC{i+1}: λ = {ev:.4f}  ({ev_pct:.1f}%  cumulative: {cum_pct:.1f}%)")
        
        report.append("\nPrincipal Component Loadings (first 3):")
        report.append("-"*40)
        variables = self.numeric_df.dropna().columns.tolist()
        for i in range(min(3, self.results['eigenvectors'].shape[1])):
            report.append(f"  PC{i+1}:")
            for j, var in enumerate(variables):
                loading = self.results['eigenvectors'][j, i]
                report.append(f"    {var}: {loading:.4f}")
        
        return "\n".join(report)
    
    def get_eda_report(self):
        """Get a full EDA report"""
        if self.results['eda'] is None:
            return "EDA could not be performed."
        
        report = []
        report.append("="*70)
        report.append("EXPLORATORY DATA ANALYSIS REPORT")
        report.append("="*70)
        report.append(f"\nShape: {self.results['eda']['shape'][0]} rows × {self.results['eda']['shape'][1]} columns")
        report.append(f"Numeric Columns: {self.results['eda']['numeric_shape'][1]}")
        
        report.append("\nMissing Values:")
        report.append("-"*40)
        missing = self.results['eda']['missing']
        missing_pct = self.results['eda']['missing_pct']
        for col in missing.index:
            if missing[col] > 0:
                report.append(f"  {col}: {missing[col]} ({missing_pct[col]:.1f}%)")
        
        report.append("\nSkewness:")
        report.append("-"*40)
        skewness = self.results['eda']['skewness']
        for col, val in skewness.items():
            if not pd.isna(val):
                shape = "highly skewed" if abs(val) > 1 else "moderately skewed" if abs(val) > 0.5 else "symmetric"
                report.append(f"  {col}: {val:.4f} ({shape})")
        
        report.append("\nKurtosis:")
        report.append("-"*40)
        kurtosis = self.results['eda']['kurtosis']
        for col, val in kurtosis.items():
            if not pd.isna(val):
                shape = "heavy tails" if val > 3 else "light tails"
                report.append(f"  {col}: {val:.4f} ({shape})")
        
        return "\n".join(report)

# ============================================================
# PART 3: VISUALIZATION FUNCTIONS
# ============================================================

def plot_correlation_matrix(df, title="Correlation Matrix", figsize=(10, 8)):
    """Plot correlation matrix heatmap"""
    numeric_df = df.select_dtypes(include=[np.number])
    if len(numeric_df.columns) < 2:
        print("Need at least 2 numeric columns for correlation plot")
        return
    
    plt.figure(figsize=figsize)
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=0.5, fmt='.2f')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_pca_scores(analyzer, figsize=(10, 6)):
    """Plot PCA scores"""
    if analyzer.results['pca_scores'] is None:
        print("No PCA results available")
        return
    
    pca_df = analyzer.results['pca_df']
    explained_var = analyzer.results['explained_variance']
    
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Scatter plot of PC1 vs PC2
    ax1 = axes[0]
    ax1.scatter(pca_df['PC1'], pca_df['PC2'], alpha=0.7, c='steelblue', edgecolors='white', linewidth=0.5)
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlabel(f'PC1 ({explained_var[0]:.1f}%)')
    ax1.set_ylabel(f'PC2 ({explained_var[1]:.1f}%)')
    ax1.set_title('PCA Scores')
    ax1.grid(True, alpha=0.3)
    
    # Explained variance bar plot
    ax2 = axes[1]
    components = [f'PC{i+1}' for i in range(len(explained_var))]
    ax2.bar(components, explained_var, color='steelblue', edgecolor='white')
    ax2.set_ylabel('Explained Variance (%)')
    ax2.set_title('Explained Variance by Component')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_feature_distributions(df, cols=None, figsize=(12, 8)):
    """Plot histograms for numeric columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:6]  # Limit to first 6 columns
    
    n_cols = min(len(cols), 6)
    n_rows = (len(cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten() if n_rows > 1 else [axes]
    
    for i, col in enumerate(cols):
        if i < len(axes) and col in numeric_df.columns:
            axes[i].hist(numeric_df[col].dropna(), bins=20, edgecolor='black', alpha=0.7, color='steelblue')
            axes[i].set_title(col, fontsize=10)
            axes[i].set_xlabel('Value')
            axes[i].set_ylabel('Frequency')
            axes[i].grid(True, alpha=0.3)
    
    # Remove empty subplots
    for i in range(len(cols), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

def plot_scatter_matrix(df, cols=None, figsize=(12, 10)):
    """Plot scatter matrix for selected columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:4]
    
    selected_data = numeric_df[cols].dropna()
    if len(selected_data) < 2:
        print("Not enough data for scatter matrix")
        return
    
    pd.plotting.scatter_matrix(selected_data, figsize=figsize, alpha=0.5, 
                                diagonal='hist', color='steelblue')
    plt.suptitle('Scatter Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ============================================================
# PART 4: COMPLETE WORKFLOW IN JUPYTER
# ============================================================

def load_and_analyze(file_path, sheet_name=None):
    """
    Complete workflow: load, clean, and analyze geochemical data
    """
    # Load and clean
    df = clean_any_data(file_path, sheet_name)
    if df is None:
        print("❌ Failed to load data")
        return None, None
    
    print("\n" + "="*60)
    print("DATASET SUMMARY")
    print("="*60)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Numeric columns: {len(df.select_dtypes(include=[np.number]).columns)}")
    print("\nFirst 5 rows:")
    print(df.head())
    
    # Analyze
    print("\n" + "="*60)
    print("STATISTICAL ANALYSIS")
    print("="*60)
    
    analyzer = StatisticalAnalyzer(df)
    
    # Print reports
    print("\n" + analyzer.get_eda_report())
    print("\n" + analyzer.get_pca_report())
    
    # Display visualizations
    print("\n" + "="*60)
    print("VISUALIZATIONS")
    print("="*60)
    
    # Plot correlation matrix
    plot_correlation_matrix(df, title="Geochemical Correlation Matrix")
    
    # Plot PCA scores
    plot_pca_scores(analyzer)
    
    # Plot feature distributions
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        plot_feature_distributions(df, cols=numeric_cols[:6])
    
    return df, analyzer

# ============================================================
# EXAMPLE USAGE IN JUPYTER
# ============================================================

# Example 1: Load and analyze a file
# df, analyzer = load_and_analyze('your_data.csv')

# Example 2: Load data only
# df = clean_any_data('your_data.csv')

# Example 3: Analyze existing dataframe
# analyzer = StatisticalAnalyzer(df)

# Example 4: Get specific results
# summary = analyzer.get_summary()
# print("Variance:", summary['variance'])
# print("Correlation Matrix:", summary['correlation_matrix'])
# print("PCA Scores:", summary['pca_scores'])

# Example 5: Export cleaned data
# df.to_csv('cleaned_data.csv', sep=';', index=False, encoding='utf-8-sig')

# Example 6: Save PCA results
# if analyzer.results['pca_df'] is not None:
#     analyzer.results['pca_df'].to_csv('pca_scores.csv', sep=';', index=False)

print("\n" + "="*60)
print("✅ GEOCHEMICAL DATA ANALYSIS TOOLKIT LOADED")
print("="*60)
print("\nAvailable functions:")
print("  • clean_any_data(path) - Load and clean data")
print("  • StatisticalAnalyzer(df) - Full statistical analysis")
print("  • plot_correlation_matrix(df) - Correlation heatmap")
print("  • plot_pca_scores(analyzer) - PCA visualization")
print("  • plot_feature_distributions(df) - Histograms")
print("  • plot_scatter_matrix(df) - Scatter matrix")
print("\nExample workflow:")
print("  df = clean_any_data('your_file.csv')")
print("  analyzer = StatisticalAnalyzer(df)")
print("  plot_correlation_matrix(df)")
print("  plot_pca_scores(analyzer)")

SyntaxError: unmatched ']' (4195570447.py, line 38)

In [3]:
# ============================================================
# GEOCHEMICAL DATA ANALYSIS - JUPYTER VERSION
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# PART 1: DATA CLEANING ENGINE
# ============================================================

def clean_any_data(path, sheet_name=None, **kwargs):
    """Universal data cleaner for Jupyter notebooks"""
    print(f"\n📁 READING FILE: {os.path.basename(path)}")
    print("=" * 60)
    
    file_ext = Path(path).suffix.lower()
    df = None
    encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1', 'utf-16', 'cp1250']
    
    if file_ext in ['.xlsx', '.xls', '.xlsm']:
        df = read_excel_file(path, sheet_name)
    elif file_ext in ['.csv', '.txt', '.dat', '.tsv']:
        df = read_text_file(path, encodings, **kwargs)
        if df is None or df.empty:
            df = manual_parse_text_file(path)
    else:
        df = manual_parse_text_file(path)
    
    if df is None or df.empty:
        print("❌ Could not read file")
        return None]
    
    print(f"📊 Raw shape: {df.shape}")
    df = clean_dataframe(df)
    print(f"✅ Final shape: {df.shape}")
    return df

def read_excel_file(path, sheet_name):
    try:
        df = pd.read_excel(path, sheet_name=sheet_name, engine='openpyxl')
        print("✅ Excel file loaded")
        return df
    except:
        try:
            df = pd.read_excel(path, sheet_name=sheet_name, engine='xlrd')
            print("✅ Excel file loaded (xlrd)")
            return df
        except Exception as e:
            print(f"⚠️ Excel read failed: {e}")
            return None

def read_text_file(path, encodings, **kwargs):
    delimiters = [';', ',', '\t', '|', ' ']
    for encoding in encodings:
        for delim in delimiters:
            try:
                df = pd.read_csv(path, sep=delim, encoding=encoding, engine='python', on_bad_lines='skip', **kwargs)
                if df is not None and not df.empty and df.shape[1] > 1:
                    print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                    return df
            except:
                continue
    return None

def manual_parse_text_file(path):
    try:
        with open(path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        df = pd.DataFrame(data)
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    return None

def clean_dataframe(df):
    if df is None or df.empty:
        return df
    df = split_single_column(df)
    df = clean_column_names(df)
    df = remove_header_rows(df)
    df = standardize_missing_values(df)
    df = clean_numeric_columns(df)
    df = clean_text_columns(df)
    df = remove_empty(df)
    return df

def split_single_column(df):
    if df is None or df.empty or df.shape[1] != 1:
        return df
    col_name = df.columns[0]
    col_data = df[col_name].astype(str)
    sample = col_data.iloc[0] if len(col_data) > 0 else ''
    delimiters = [';', ',', '\t', '|']
    for delim in delimiters:
        if delim in sample:
            split_data = col_data.str.split(delim, expand=True)
            try:
                pd.to_numeric(split_data.iloc[0, 0])
                split_data.columns = [f'col_{i+1}' for i in range(split_data.shape[1])]
            except:
                headers = split_data.iloc[0].tolist()
                split_data = split_data.iloc[1:].copy()
                headers = [str(h).strip() for h in headers]
                split_data.columns = headers
            return split_data.reset_index(drop=True)
    return df

def clean_column_names(df):
    if df is None or df.empty:
        return df
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.loc[:, ~df.columns.astype(str).str.contains(r',+$', na=False)]
    df.columns = [f'col_{i}' if str(col).strip() == '' or pd.isna(col) else col for i, col in enumerate(df.columns)]
    df.columns = [re.sub(r'[^a-zA-Z0-9_ ]', '_', str(col)).strip('_') for col in df.columns]
    seen = {}
    new_cols = []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

def remove_header_rows(df):
    if df is None or df.empty or len(df) == 0:
        return df
    first_row = df.iloc[0].astype(str)
    header_keywords = ['sample', 'point', 'id', 'name', 'date', 'time', 'value', 'measurement', 'parameter', 'unit', 'ppm']
    header_score = sum(1 for val in first_row if any(keyword in str(val).lower() for keyword in header_keywords))
    if len(first_row) > 0 and header_score / len(first_row) > 0.3:
        df = df.iloc[1:].reset_index(drop=True)
    return df

def standardize_missing_values(df):
    if df is None or df.empty:
        return df
    missing_values = ['', 'nan', 'NaN', 'NAN', 'null', 'NULL', 'None', 'na', 'NA', '-', '--', '---', 'n/a', 'N/A', '?']
    df = df.replace(missing_values, np.nan)
    for col in df.columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'^<.*$', 'nan', regex=True)
            df[col] = df[col].astype(str).str.replace(r'^>.*$', 'nan', regex=True)
    return df

def clean_numeric_columns(df):
    if df is None or df.empty:
        return df
    text_cols = ['sample', 'point_id', 'point', 'id', 'name', 'lithology', 'description', 'comment', 'remarks', 'unit', 'method', 'date', 'time']
    def clean_numeric(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip().replace(',', '.')
        if x.startswith('<') or x.startswith('>'):
            return np.nan
        if x in ['', 'nan', 'NaN', 'NAN', 'null', 'NULL']:
            return np.nan
        x = re.sub(r'[^\d.\-eE+]', '', x)
        try:
            return float(x)
        except:
            return np.nan
    for col in df.columns:
        col_lower = col.lower()
        is_text = any(text in col_lower for text in text_cols)
        if not is_text:
            try:
                df[col] = df[col].map(clean_numeric)
            except:
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    pass
    return df

def clean_text_columns(df):
    if df is None or df.empty:
        return df
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace('', np.nan)
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
    return df

def remove_empty(df):
    if df is None or df.empty:
        return df
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    return df

# ============================================================
# PART 2: STATISTICAL ANALYSIS ENGINE
# ============================================================

class StatisticalAnalyzer:
    """Comprehensive statistical analysis for geochemical data"""
    
    def __init__(self, df):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        self.run_all_analyses()
    
    def run_all_analyses(self):
        """Run all statistical analyses"""
        self.calculate_variance()
        self.calculate_covariance()
        self.calculate_correlation()
        self.calculate_eigen()
        self.perform_pca()
        self.standardize_data()
        self.exploratory_analysis()
    
    def calculate_variance(self):
        """Calculate variance for all numeric columns"""
        self.results['variance'] = self.numeric_df.var()
        self.results['variance_dict'] = self.numeric_df.var().to_dict()
        self.results['std'] = self.numeric_df.std()
        self.results['mean'] = self.numeric_df.mean()
        self.results['median'] = self.numeric_df.median()
    
    def calculate_covariance(self):
        """Calculate covariance matrix"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 1:
            self.results['covariance_matrix'] = clean_data.cov()
        else:
            self.results['covariance_matrix'] = None
    
    def calculate_correlation(self):
        """Calculate correlation matrix"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 1:
            self.results['correlation_matrix'] = clean_data.corr()
        else:
            self.results['correlation_matrix'] = None
    
    def calculate_eigen(self):
        """Calculate eigenvalues and eigenvectors"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2:
            self.results['eigenvalues'] = None
            self.results['eigenvectors'] = None
            return
        
        try:
            cov = clean_data.cov().values
            eigenvals, eigenvecs = np.linalg.eig(cov)
            
            # Sort by eigenvalues (largest first)
            idx = np.argsort(eigenvals)[::-1]
            eigenvals = eigenvals[idx]
            eigenvecs = eigenvecs[:, idx]
            
            self.results['eigenvalues'] = eigenvals
            self.results['eigenvectors'] = eigenvecs
            self.results['explained_variance'] = eigenvals / eigenvals.sum() * 100
        except Exception as e:
            self.results['eigenvalues'] = None
            self.results['eigenvectors'] = None
            print(f"⚠️ Eigen decomposition failed: {e}")
    
    def perform_pca(self):
        """Perform Principal Component Analysis"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2 or self.results['eigenvectors'] is None:
            self.results['pca_scores'] = None
            self.results['pca_loadings'] = None
            return
        
        try:
            centered = clean_data - clean_data.mean()
            pca_scores = centered @ self.results['eigenvectors']
            self.results['pca_scores'] = pca_scores
            self.results['pca_loadings'] = self.results['eigenvectors']
            self.results['pca_columns'] = [f'PC{i+1}' for i in range(pca_scores.shape[1])]
            self.results['pca_df'] = pd.DataFrame(pca_scores, columns=self.results['pca_columns'])
        except Exception as e:
            print(f"⚠️ PCA failed: {e}")
    
    def standardize_data(self):
        """Standardize data (Z-score normalization)"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 0:
            self.results['standardized'] = (clean_data - clean_data.mean()) / clean_data.std()
            self.results['min_max_scaled'] = (clean_data - clean_data.min()) / (clean_data.max() - clean_data.min())
    
    def exploratory_analysis(self):
        """Full exploratory data analysis"""
        self.results['eda'] = {}
        self.results['eda']['shape'] = self.df.shape
        self.results['eda']['numeric_shape'] = self.numeric_df.shape
        self.results['eda']['missing'] = self.df.isnull().sum()
        self.results['eda']['missing_pct'] = self.df.isnull().sum() / len(self.df) * 100
        self.results['eda']['skewness'] = self.numeric_df.skew()
        self.results['eda']['kurtosis'] = self.numeric_df.kurtosis()
        self.results['eda']['quantiles'] = self.numeric_df.quantile([0.25, 0.5, 0.75])
        
        if len(self.numeric_df.columns) > 1:
            corr = self.numeric_df.corr()
            self.results['eda']['correlation_summary'] = {
                'max_corr': corr.max().max(),
                'min_corr': corr.min().min(),
                'high_corr_pairs': [(col1, col2, corr.loc[col1, col2]) 
                                   for i, col1 in enumerate(corr.columns)
                                   for col2 in corr.columns[i+1:]
                                   if abs(corr.loc[col1, col2]) > 0.7]
            }
    
    def get_summary(self):
        """Get summary statistics"""
        return {
            'variance': self.results.get('variance_dict', {}),
            'mean': self.results.get('mean', {}),
            'std': self.results.get('std', {}),
            'correlation_matrix': self.results.get('correlation_matrix', None),
            'eigenvalues': self.results.get('eigenvalues', None),
            'eigenvectors': self.results.get('eigenvectors', None),
            'explained_variance': self.results.get('explained_variance', None),
            'pca_scores': self.results.get('pca_df', None),
            'missing_values': self.results.get('eda', {}).get('missing', None),
            'skewness': self.results.get('eda', {}).get('skewness', None)
        }
    
    def get_pca_report(self, n_components=2):
        """Get a detailed PCA report"""
        if self.results['pca_scores'] is None:
            return "PCA could not be performed."
        
        report = []
        report.append("="*70)
        report.append("PRINCIPAL COMPONENT ANALYSIS REPORT")
        report.append("="*70)
        report.append(f"\nSamples: {self.results['pca_scores'].shape[0]}")
        report.append(f"Variables: {self.results['pca_scores'].shape[1]}")
        report.append("\nEigenvalues and Explained Variance:")
        report.append("-"*40)
        
        for i in range(len(self.results['eigenvalues'])):
            ev = self.results['eigenvalues'][i]
            ev_pct = self.results['explained_variance'][i]
            cum_pct = self.results['explained_variance'][:i+1].sum()
            report.append(f"  PC{i+1}: λ = {ev:.4f}  ({ev_pct:.1f}%  cumulative: {cum_pct:.1f}%)")
        
        report.append("\nPrincipal Component Loadings (first 3):")
        report.append("-"*40)
        variables = self.numeric_df.dropna().columns.tolist()
        for i in range(min(3, self.results['eigenvectors'].shape[1])):
            report.append(f"  PC{i+1}:")
            for j, var in enumerate(variables):
                loading = self.results['eigenvectors'][j, i]
                report.append(f"    {var}: {loading:.4f}")
        
        return "\n".join(report)
    
    def get_eda_report(self):
        """Get a full EDA report"""
        if self.results['eda'] is None:
            return "EDA could not be performed."
        
        report = []
        report.append("="*70)
        report.append("EXPLORATORY DATA ANALYSIS REPORT")
        report.append("="*70)
        report.append(f"\nShape: {self.results['eda']['shape'][0]} rows × {self.results['eda']['shape'][1]} columns")
        report.append(f"Numeric Columns: {self.results['eda']['numeric_shape'][1]}")
        
        report.append("\nMissing Values:")
        report.append("-"*40)
        missing = self.results['eda']['missing']
        missing_pct = self.results['eda']['missing_pct']
        for col in missing.index:
            if missing[col] > 0:
                report.append(f"  {col}: {missing[col]} ({missing_pct[col]:.1f}%)")
        
        report.append("\nSkewness:")
        report.append("-"*40)
        skewness = self.results['eda']['skewness']
        for col, val in skewness.items():
            if not pd.isna(val):
                shape = "highly skewed" if abs(val) > 1 else "moderately skewed" if abs(val) > 0.5 else "symmetric"
                report.append(f"  {col}: {val:.4f} ({shape})")
        
        report.append("\nKurtosis:")
        report.append("-"*40)
        kurtosis = self.results['eda']['kurtosis']
        for col, val in kurtosis.items():
            if not pd.isna(val):
                shape = "heavy tails" if val > 3 else "light tails"
                report.append(f"  {col}: {val:.4f} ({shape})")
        
        return "\n".join(report)

# ============================================================
# PART 3: VISUALIZATION FUNCTIONS
# ============================================================

def plot_correlation_matrix(df, title="Correlation Matrix", figsize=(10, 8)):
    """Plot correlation matrix heatmap"""
    numeric_df = df.select_dtypes(include=[np.number])
    if len(numeric_df.columns) < 2:
        print("Need at least 2 numeric columns for correlation plot")
        return
    
    plt.figure(figsize=figsize)
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=0.5, fmt='.2f')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_pca_scores(analyzer, figsize=(10, 6)):
    """Plot PCA scores"""
    if analyzer.results['pca_scores'] is None:
        print("No PCA results available")
        return
    
    pca_df = analyzer.results['pca_df']
    explained_var = analyzer.results['explained_variance']
    
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Scatter plot of PC1 vs PC2
    ax1 = axes[0]
    ax1.scatter(pca_df['PC1'], pca_df['PC2'], alpha=0.7, c='steelblue', edgecolors='white', linewidth=0.5)
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlabel(f'PC1 ({explained_var[0]:.1f}%)')
    ax1.set_ylabel(f'PC2 ({explained_var[1]:.1f}%)')
    ax1.set_title('PCA Scores')
    ax1.grid(True, alpha=0.3)
    
    # Explained variance bar plot
    ax2 = axes[1]
    components = [f'PC{i+1}' for i in range(len(explained_var))]
    ax2.bar(components, explained_var, color='steelblue', edgecolor='white')
    ax2.set_ylabel('Explained Variance (%)')
    ax2.set_title('Explained Variance by Component')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_feature_distributions(df, cols=None, figsize=(12, 8)):
    """Plot histograms for numeric columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:6]  # Limit to first 6 columns
    
    n_cols = min(len(cols), 6)
    n_rows = (len(cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten() if n_rows > 1 else [axes]
    
    for i, col in enumerate(cols):
        if i < len(axes) and col in numeric_df.columns:
            axes[i].hist(numeric_df[col].dropna(), bins=20, edgecolor='black', alpha=0.7, color='steelblue')
            axes[i].set_title(col, fontsize=10)
            axes[i].set_xlabel('Value')
            axes[i].set_ylabel('Frequency')
            axes[i].grid(True, alpha=0.3)
    
    # Remove empty subplots
    for i in range(len(cols), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

def plot_scatter_matrix(df, cols=None, figsize=(12, 10)):
    """Plot scatter matrix for selected columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:4]
    
    selected_data = numeric_df[cols].dropna()
    if len(selected_data) < 2:
        print("Not enough data for scatter matrix")
        return
    
    pd.plotting.scatter_matrix(selected_data, figsize=figsize, alpha=0.5, 
                                diagonal='hist', color='steelblue')
    plt.suptitle('Scatter Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ============================================================
# PART 4: COMPLETE WORKFLOW IN JUPYTER
# ============================================================

def load_and_analyze(file_path, sheet_name=None):
    """
    Complete workflow: load, clean, and analyze geochemical data
    """
    # Load and clean
    df = clean_any_data(file_path, sheet_name)
    if df is None:
        print("❌ Failed to load data")
        return None, None
    
    print("\n" + "="*60)
    print("DATASET SUMMARY")
    print("="*60)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Numeric columns: {len(df.select_dtypes(include=[np.number]).columns)}")
    print("\nFirst 5 rows:")
    print(df.head())
    
    # Analyze
    print("\n" + "="*60)
    print("STATISTICAL ANALYSIS")
    print("="*60)
    
    analyzer = StatisticalAnalyzer(df)
    
    # Print reports
    print("\n" + analyzer.get_eda_report())
    print("\n" + analyzer.get_pca_report())
    
    # Display visualizations
    print("\n" + "="*60)
    print("VISUALIZATIONS")
    print("="*60)
    
    # Plot correlation matrix
    plot_correlation_matrix(df, title="Geochemical Correlation Matrix")
    
    # Plot PCA scores
    plot_pca_scores(analyzer)
    
    # Plot feature distributions
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        plot_feature_distributions(df, cols=numeric_cols[:6])
    
    return df, analyzer

# ============================================================
# EXAMPLE USAGE IN JUPYTER
# ============================================================

# Example 1: Load and analyze a file
# df, analyzer = load_and_analyze('your_data.csv')

# Example 2: Load data only
# df = clean_any_data('your_data.csv')

# Example 3: Analyze existing dataframe
# analyzer = StatisticalAnalyzer(df)

# Example 4: Get specific results
# summary = analyzer.get_summary()
# print("Variance:", summary['variance'])
# print("Correlation Matrix:", summary['correlation_matrix'])
# print("PCA Scores:", summary['pca_scores'])

# Example 5: Export cleaned data
# df.to_csv('cleaned_data.csv', sep=';', index=False, encoding='utf-8-sig')

# Example 6: Save PCA results
# if analyzer.results['pca_df'] is not None:
#     analyzer.results['pca_df'].to_csv('pca_scores.csv', sep=';', index=False)

print("\n" + "="*60)
print("✅ GEOCHEMICAL DATA ANALYSIS TOOLKIT LOADED")
print("="*60)
print("\nAvailable functions:")
print("  • clean_any_data(path) - Load and clean data")
print("  • StatisticalAnalyzer(df) - Full statistical analysis")
print("  • plot_correlation_matrix(df) - Correlation heatmap")
print("  • plot_pca_scores(analyzer) - PCA visualization")
print("  • plot_feature_distributions(df) - Histograms")
print("  • plot_scatter_matrix(df) - Scatter matrix")
print("\nExample workflow:")
print("  df = clean_any_data('your_file.csv')")
print("  analyzer = StatisticalAnalyzer(df)")
print("  plot_correlation_matrix(df)")
print("  plot_pca_scores(analyzer)")

SyntaxError: unmatched ']' (2181669008.py, line 37)

In [19]:
# ============================================================
# GEOCHEMICAL DATA ANALYSIS - JUPYTER VERSION
# ============================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# PART 1: DATA CLEANING ENGINE
# ============================================================

def clean_any_data(path, sheet_name=None, **kwargs):
    """Universal data cleaner for Jupyter notebooks"""
    print(f"\n📁 READING FILE: {os.path.basename(path)}")
    print("=" * 60)
    
    file_ext = Path(path).suffix.lower()
    df = None
    encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1', 'utf-16', 'cp1250']
    
    if file_ext in ['.xlsx', '.xls', '.xlsm']:
        df = read_excel_file(path, sheet_name)
    elif file_ext in ['.csv', '.txt', '.dat', '.tsv']:
        df = read_text_file(path, encodings, **kwargs)
        if df is None or df.empty:
            df = manual_parse_text_file(path)
    else:
        df = manual_parse_text_file(path)
    
    if df is None or df.empty:
        print("❌ Could not read file")
        return None
    
    print(f"📊 Raw shape: {df.shape}")
    df = clean_dataframe(df)
    print(f"✅ Final shape: {df.shape}")
    return df

def read_excel_file(path, sheet_name):
    try:
        df = pd.read_excel(path, sheet_name=sheet_name, engine='openpyxl')
        print("✅ Excel file loaded")
        return df
    except:
        try:
            df = pd.read_excel(path, sheet_name=sheet_name, engine='xlrd')
            print("✅ Excel file loaded (xlrd)")
            return df
        except Exception as e:
            print(f"⚠️ Excel read failed: {e}")
            return None

def read_text_file(path, encodings, **kwargs):
    delimiters = [';', ',', '\t', '|', ' ']
    for encoding in encodings:
        for delim in delimiters:
            try:
                df = pd.read_csv(path, sep=delim, encoding=encoding, engine='python', on_bad_lines='skip', **kwargs)
                if df is not None and not df.empty and df.shape[1] > 1:
                    print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                    return df
            except:
                continue
    return None

def manual_parse_text_file(path):
    try:
        with open(path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        df = pd.DataFrame(data)
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    return None

def clean_dataframe(df):
    if df is None or df.empty:
        return df
    df = split_single_column(df)
    df = clean_column_names(df)
    df = remove_header_rows(df)
    df = standardize_missing_values(df)
    df = clean_numeric_columns(df)
    df = clean_text_columns(df)
    df = remove_empty(df)
    return df

def split_single_column(df):
    if df is None or df.empty or df.shape[1] != 1:
        return df
    col_name = df.columns[0]
    col_data = df[col_name].astype(str)
    sample = col_data.iloc[0] if len(col_data) > 0 else ''
    delimiters = [';', ',', '\t', '|']
    for delim in delimiters:
        if delim in sample:
            split_data = col_data.str.split(delim, expand=True)
            try:
                pd.to_numeric(split_data.iloc[0, 0])
                split_data.columns = [f'col_{i+1}' for i in range(split_data.shape[1])]
            except:
                headers = split_data.iloc[0].tolist()
                split_data = split_data.iloc[1:].copy()
                headers = [str(h).strip() for h in headers]
                split_data.columns = headers
            return split_data.reset_index(drop=True)
    return df

def clean_column_names(df):
    if df is None or df.empty:
        return df
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.loc[:, ~df.columns.astype(str).str.contains(r',+$', na=False)]
    df.columns = [f'col_{i}' if str(col).strip() == '' or pd.isna(col) else col for i, col in enumerate(df.columns)]
    df.columns = [re.sub(r'[^a-zA-Z0-9_ ]', '_', str(col)).strip('_') for col in df.columns]
    seen = {}
    new_cols = []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

def remove_header_rows(df):
    if df is None or df.empty or len(df) == 0:
        return df
    first_row = df.iloc[0].astype(str)
    header_keywords = ['sample', 'point', 'id', 'name', 'date', 'time', 'value', 'measurement', 'parameter', 'unit', 'ppm']
    header_score = sum(1 for val in first_row if any(keyword in str(val).lower() for keyword in header_keywords))
    if len(first_row) > 0 and header_score / len(first_row) > 0.3:
        df = df.iloc[1:].reset_index(drop=True)
    return df

def standardize_missing_values(df):
    if df is None or df.empty:
        return df
    missing_values = ['', 'nan', 'NaN', 'NAN', 'null', 'NULL', 'None', 'na', 'NA', '-', '--', '---', 'n/a', 'N/A', '?']
    df = df.replace(missing_values, np.nan)
    for col in df.columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'^<.*$', 'nan', regex=True)
            df[col] = df[col].astype(str).str.replace(r'^>.*$', 'nan', regex=True)
    return df

def clean_numeric_columns(df):
    if df is None or df.empty:
        return df
    text_cols = ['sample', 'point_id', 'point', 'id', 'name', 'lithology', 'description', 'comment', 'remarks', 'unit', 'method', 'date', 'time']
    def clean_numeric(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip().replace(',', '.')
        if x.startswith('<') or x.startswith('>'):
            return np.nan
        if x in ['', 'nan', 'NaN', 'NAN', 'null', 'NULL']:
            return np.nan
        x = re.sub(r'[^\d.\-eE+]', '', x)
        try:
            return float(x)
        except:
            return np.nan
    for col in df.columns:
        col_lower = col.lower()
        is_text = any(text in col_lower for text in text_cols)
        if not is_text:
            try:
                df[col] = df[col].map(clean_numeric)
            except:
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    pass
    return df

def clean_text_columns(df):
    if df is None or df.empty:
        return df
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace('', np.nan)
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
    return df

def remove_empty(df):
    if df is None or df.empty:
        return df
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    return df

# ============================================================
# PART 2: STATISTICAL ANALYSIS ENGINE
# ============================================================

class StatisticalAnalyzer:
    """Comprehensive statistical analysis for geochemical data"""
    
    def __init__(self, df):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        self.run_all_analyses()
    
    def run_all_analyses(self):
        """Run all statistical analyses"""
        self.calculate_variance()
        self.calculate_covariance()
        self.calculate_correlation()
        self.calculate_eigen()
        self.perform_pca()
        self.standardize_data()
        self.exploratory_analysis()
    
    def calculate_variance(self):
        """Calculate variance for all numeric columns"""
        self.results['variance'] = self.numeric_df.var()
        self.results['variance_dict'] = self.numeric_df.var().to_dict()
        self.results['std'] = self.numeric_df.std()
        self.results['mean'] = self.numeric_df.mean()
        self.results['median'] = self.numeric_df.median()
    
    def calculate_covariance(self):
        """Calculate covariance matrix"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 1:
            self.results['covariance_matrix'] = clean_data.cov()
        else:
            self.results['covariance_matrix'] = None
    
    def calculate_correlation(self):
        """Calculate correlation matrix"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 1:
            self.results['correlation_matrix'] = clean_data.corr()
        else:
            self.results['correlation_matrix'] = None
    
    def calculate_eigen(self):
        """Calculate eigenvalues and eigenvectors"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2:
            self.results['eigenvalues'] = None
            self.results['eigenvectors'] = None
            return
        
        try:
            cov = clean_data.cov().values
            eigenvals, eigenvecs = np.linalg.eig(cov)
            
            # Sort by eigenvalues (largest first)
            idx = np.argsort(eigenvals)[::-1]
            eigenvals = eigenvals[idx]
            eigenvecs = eigenvecs[:, idx]
            
            self.results['eigenvalues'] = eigenvals
            self.results['eigenvectors'] = eigenvecs
            self.results['explained_variance'] = eigenvals / eigenvals.sum() * 100
        except Exception as e:
            self.results['eigenvalues'] = None
            self.results['eigenvectors'] = None
            print(f"⚠️ Eigen decomposition failed: {e}")
    
    def perform_pca(self):
        """Perform Principal Component Analysis"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2 or self.results['eigenvectors'] is None:
            self.results['pca_scores'] = None
            self.results['pca_loadings'] = None
            return
        
        try:
            centered = clean_data - clean_data.mean()
            pca_scores = centered @ self.results['eigenvectors']
            self.results['pca_scores'] = pca_scores
            self.results['pca_loadings'] = self.results['eigenvectors']
            self.results['pca_columns'] = [f'PC{i+1}' for i in range(pca_scores.shape[1])]
            self.results['pca_df'] = pd.DataFrame(pca_scores, columns=self.results['pca_columns'])
        except Exception as e:
            print(f"⚠️ PCA failed: {e}")
    
    def standardize_data(self):
        """Standardize data (Z-score normalization)"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 0:
            self.results['standardized'] = (clean_data - clean_data.mean()) / clean_data.std()
            self.results['min_max_scaled'] = (clean_data - clean_data.min()) / (clean_data.max() - clean_data.min())
    
    def exploratory_analysis(self):
        """Full exploratory data analysis"""
        self.results['eda'] = {}
        self.results['eda']['shape'] = self.df.shape
        self.results['eda']['numeric_shape'] = self.numeric_df.shape
        self.results['eda']['missing'] = self.df.isnull().sum()
        self.results['eda']['missing_pct'] = self.df.isnull().sum() / len(self.df) * 100
        self.results['eda']['skewness'] = self.numeric_df.skew()
        self.results['eda']['kurtosis'] = self.numeric_df.kurtosis()
        self.results['eda']['quantiles'] = self.numeric_df.quantile([0.25, 0.5, 0.75])
        
        if len(self.numeric_df.columns) > 1:
            corr = self.numeric_df.corr()
            self.results['eda']['correlation_summary'] = {
                'max_corr': corr.max().max(),
                'min_corr': corr.min().min(),
                'high_corr_pairs': [(col1, col2, corr.loc[col1, col2]) 
                                   for i, col1 in enumerate(corr.columns)
                                   for col2 in corr.columns[i+1:]
                                   if abs(corr.loc[col1, col2]) > 0.7]
            }
    
    def get_summary(self):
        """Get summary statistics"""
        return {
            'variance': self.results.get('variance_dict', {}),
            'mean': self.results.get('mean', {}),
            'std': self.results.get('std', {}),
            'correlation_matrix': self.results.get('correlation_matrix', None),
            'eigenvalues': self.results.get('eigenvalues', None),
            'eigenvectors': self.results.get('eigenvectors', None),
            'explained_variance': self.results.get('explained_variance', None),
            'pca_scores': self.results.get('pca_df', None),
            'missing_values': self.results.get('eda', {}).get('missing', None),
            'skewness': self.results.get('eda', {}).get('skewness', None)
        }
    
    def get_pca_report(self, n_components=2):
        """Get a detailed PCA report"""
        if self.results['pca_scores'] is None:
            return "PCA could not be performed."
        
        report = []
        report.append("="*70)
        report.append("PRINCIPAL COMPONENT ANALYSIS REPORT")
        report.append("="*70)
        report.append(f"\nSamples: {self.results['pca_scores'].shape[0]}")
        report.append(f"Variables: {self.results['pca_scores'].shape[1]}")
        report.append("\nEigenvalues and Explained Variance:")
        report.append("-"*40)
        
        for i in range(len(self.results['eigenvalues'])):
            ev = self.results['eigenvalues'][i]
            ev_pct = self.results['explained_variance'][i]
            cum_pct = self.results['explained_variance'][:i+1].sum()
            report.append(f"  PC{i+1}: λ = {ev:.4f}  ({ev_pct:.1f}%  cumulative: {cum_pct:.1f}%)")
        
        report.append("\nPrincipal Component Loadings (first 3):")
        report.append("-"*40)
        variables = self.numeric_df.dropna().columns.tolist()
        for i in range(min(3, self.results['eigenvectors'].shape[1])):
            report.append(f"  PC{i+1}:")
            for j, var in enumerate(variables):
                loading = self.results['eigenvectors'][j, i]
                report.append(f"    {var}: {loading:.4f}")
        
        return "\n".join(report)
    
    def get_eda_report(self):
        """Get a full EDA report"""
        if self.results['eda'] is None:
            return "EDA could not be performed."
        
        report = []
        report.append("="*70)
        report.append("EXPLORATORY DATA ANALYSIS REPORT")
        report.append("="*70)
        report.append(f"\nShape: {self.results['eda']['shape'][0]} rows × {self.results['eda']['shape'][1]} columns")
        report.append(f"Numeric Columns: {self.results['eda']['numeric_shape'][1]}")
        
        report.append("\nMissing Values:")
        report.append("-"*40)
        missing = self.results['eda']['missing']
        missing_pct = self.results['eda']['missing_pct']
        for col in missing.index:
            if missing[col] > 0:
                report.append(f"  {col}: {missing[col]} ({missing_pct[col]:.1f}%)")
        
        report.append("\nSkewness:")
        report.append("-"*40)
        skewness = self.results['eda']['skewness']
        for col, val in skewness.items():
            if not pd.isna(val):
                shape = "highly skewed" if abs(val) > 1 else "moderately skewed" if abs(val) > 0.5 else "symmetric"
                report.append(f"  {col}: {val:.4f} ({shape})")
        
        report.append("\nKurtosis:")
        report.append("-"*40)
        kurtosis = self.results['eda']['kurtosis']
        for col, val in kurtosis.items():
            if not pd.isna(val):
                shape = "heavy tails" if val > 3 else "light tails"
                report.append(f"  {col}: {val:.4f} ({shape})")
        
        return "\n".join(report)

# ============================================================
# PART 3: VISUALIZATION FUNCTIONS
# ============================================================

def plot_correlation_matrix(df, title="Correlation Matrix", figsize=(10, 8)):
    """Plot correlation matrix heatmap"""
    numeric_df = df.select_dtypes(include=[np.number])
    if len(numeric_df.columns) < 2:
        print("Need at least 2 numeric columns for correlation plot")
        return
    
    plt.figure(figsize=figsize)
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, 
                square=True, linewidths=0.5, fmt='.2f')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_pca_scores(analyzer, figsize=(10, 6)):
    """Plot PCA scores"""
    if analyzer.results['pca_scores'] is None:
        print("No PCA results available")
        return
    
    pca_df = analyzer.results['pca_df']
    explained_var = analyzer.results['explained_variance']
    
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # Scatter plot of PC1 vs PC2
    ax1 = axes[0]
    ax1.scatter(pca_df['PC1'], pca_df['PC2'], alpha=0.7, c='steelblue', edgecolors='white', linewidth=0.5)
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlabel(f'PC1 ({explained_var[0]:.1f}%)')
    ax1.set_ylabel(f'PC2 ({explained_var[1]:.1f}%)')
    ax1.set_title('PCA Scores')
    ax1.grid(True, alpha=0.3)
    
    # Explained variance bar plot
    ax2 = axes[1]
    components = [f'PC{i+1}' for i in range(len(explained_var))]
    ax2.bar(components, explained_var, color='steelblue', edgecolor='white')
    ax2.set_ylabel('Explained Variance (%)')
    ax2.set_title('Explained Variance by Component')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_feature_distributions(df, cols=None, figsize=(12, 8)):
    """Plot histograms for numeric columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:6]  # Limit to first 6 columns
    
    n_cols = min(len(cols), 6)
    n_rows = (len(cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten() if n_rows > 1 else [axes]
    
    for i, col in enumerate(cols):
        if i < len(axes) and col in numeric_df.columns:
            axes[i].hist(numeric_df[col].dropna(), bins=20, edgecolor='black', alpha=0.7, color='steelblue')
            axes[i].set_title(col, fontsize=10)
            axes[i].set_xlabel('Value')
            axes[i].set_ylabel('Frequency')
            axes[i].grid(True, alpha=0.3)
    
    # Remove empty subplots
    for i in range(len(cols), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

def plot_scatter_matrix(df, cols=None, figsize=(12, 10)):
    """Plot scatter matrix for selected columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:4]
    
    selected_data = numeric_df[cols].dropna()
    if len(selected_data) < 2:
        print("Not enough data for scatter matrix")
        return
    
    pd.plotting.scatter_matrix(selected_data, figsize=figsize, alpha=0.5, 
                                diagonal='hist', color='steelblue')
    plt.suptitle('Scatter Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ============================================================
# PART 4: COMPLETE WORKFLOW IN JUPYTER
# ============================================================

def load_and_analyze(file_path, sheet_name=None):
    """
    Complete workflow: load, clean, and analyze geochemical data
    """
    # Load and clean
    df = clean_any_data(file_path = 'study area_BRAZIL.csv', sheet_name = 'study area_BRAZIL.csv')
    if df is None:
        print("❌ Failed to load data")
        return None, None
    
    print("\n" + "="*60)
    print("DATASET SUMMARY")
    print("="*60)
    print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Numeric columns: {len(df.select_dtypes(include=[np.number]).columns)}")
    print("\nFirst 5 rows:")
    print(df.head())
    
    # Analyze
    print("\n" + "="*60)
    print("STATISTICAL ANALYSIS")
    print("="*60)
    
    analyzer = StatisticalAnalyzer(df)
    
    # Print reports
    print("\n" + analyzer.get_eda_report())
    print("\n" + analyzer.get_pca_report())
    
    # Display visualizations
    print("\n" + "="*60)
    print("VISUALIZATIONS")
    print("="*60)
    
    # Plot correlation matrix
    plot_correlation_matrix(df, title="Geochemical Correlation Matrix")
    
    # Plot PCA scores
    plot_pca_scores(analyzer)
    
    # Plot feature distributions
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols) > 0:
        plot_feature_distributions(df, cols=numeric_cols[:6])
    
    return df, analyzer

# ============================================================
# EXAMPLE USAGE IN JUPYTER
# ============================================================

# Example 1: Load and analyze a file
# df, analyzer = load_and_analyze('your_data.csv')

# Example 2: Load data only
# df = clean_any_data('your_data.csv')

# Example 3: Analyze existing dataframe
# analyzer = StatisticalAnalyzer(df)

# Example 4: Get specific results
# summary = analyzer.get_summary()
# print("Variance:", summary['variance'])
# print("Correlation Matrix:", summary['correlation_matrix'])
# print("PCA Scores:", summary['pca_scores'])

# Example 5: Export cleaned data
# df.to_csv('cleaned_data.csv', sep=';', index=False, encoding='utf-8-sig')

# Example 6: Save PCA results
# if analyzer.results['pca_df'] is not None:
#     analyzer.results['pca_df'].to_csv('pca_scores.csv', sep=';', index=False)

print("\n" + "="*60)
print("✅ GEOCHEMICAL DATA ANALYSIS TOOLKIT LOADED")
print("="*60)
print("\nAvailable functions:")
print("  • clean_any_data(path) - Load and clean data")
print("  • StatisticalAnalyzer(df) - Full statistical analysis")
print("  • plot_correlation_matrix(df) - Correlation heatmap")
print("  • plot_pca_scores(analyzer) - PCA visualization")
print("  • plot_feature_distributions(df) - Histograms")
print("  • plot_scatter_matrix(df) - Scatter matrix")
print("\nExample workflow:")
print("  df = clean_any_data('your_file.csv')")
print("  analyzer = StatisticalAnalyzer(df)")
print("  plot_correlation_matrix(df)")
print("  plot_pca_scores(analyzer)")


✅ GEOCHEMICAL DATA ANALYSIS TOOLKIT LOADED

Available functions:
  • clean_any_data(path) - Load and clean data
  • StatisticalAnalyzer(df) - Full statistical analysis
  • plot_correlation_matrix(df) - Correlation heatmap
  • plot_pca_scores(analyzer) - PCA visualization
  • plot_feature_distributions(df) - Histograms
  • plot_scatter_matrix(df) - Scatter matrix

Example workflow:
  df = clean_any_data('your_file.csv')
  analyzer = StatisticalAnalyzer(df)
  plot_correlation_matrix(df)
  plot_pca_scores(analyzer)


In [13]:
df = clean_any_data('study area_BRAZIL.csv')
print(df)
analyzer = StatisticalAnalyzer(df)
plot_correlation_matrix(df)
plot_pca_scores(analyzer)


📁 READING FILE: study area_BRAZIL.csv
❌ Could not read file
None


AttributeError: 'NoneType' object has no attribute 'select_dtypes'

In [21]:
import sys
import os
import pandas as pd
import numpy as np
import re
from pathlib import Path
from scipy import stats
from scipy.optimize import approx_fprime
from scipy.stats import pearsonr, spearmanr, kendalltau, norm, lognorm, skew, kurtosis
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Check if running in Jupyter
try:
    from IPython import get_ipython
    if 'IPKernelApp' in get_ipython().config:
        IN_JUPYTER = True
    else:
        IN_JUPYTER = False
except:
    IN_JUPYTER = False

# Import Qt
try:
    from PyQt5.QtWidgets import *
    from PyQt5.QtCore import *
    from PyQt5.QtGui import *
    QT_VERSION = 5
except ImportError:
    try:
        from PyQt6.QtWidgets import *
        from PyQt6.QtCore import *
        from PyQt6.QtGui import *
        QT_VERSION = 6
    except ImportError:
        print("⚠️ PyQt not installed. Install with: pip install PyQt5")
        QT_VERSION = None

# ============================================================
# PART 1: DATA CLEANING ENGINE
# ============================================================

def clean_any_data(path, sheet_name=None, **kwargs):
    """Universal data cleaner that handles any file format and structure."""
    print(f"\n📁 READING FILE: {os.path.basename(path)}")
    print("=" * 70)
    
    file_ext = Path(path).suffix.lower()
    df = None
    encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1', 'utf-16', 'cp1250']
    
    if file_ext in ['.xlsx', '.xls', '.xlsm']:
        df = read_excel_file(path, sheet_name)
        if df is None:
            return None
    elif file_ext in ['.csv', '.txt', '.dat', '.tsv']:
        df = read_text_file(path, encodings, **kwargs)
        if df is None or df.empty:
            df = manual_parse_text_file(path)
    else:
        print(f"⚠️ Unknown file type: {file_ext}")
        df = manual_parse_text_file(path)
    
    if df is None or df.empty:
        print("❌ Could not read file")
        return None
    
    print(f"📊 Raw data shape: {df.shape}")
    df = clean_dataframe(df)
    print(f"\n✅ Final shape: {df.shape}")
    return df

def read_excel_file(path, sheet_name):
    try:
        df = pd.read_excel(path, sheet_name=sheet_name, engine='openpyxl')
        print("✅ Read as Excel file (openpyxl)")
        return df
    except:
        try:
            df = pd.read_excel(path, sheet_name=sheet_name, engine='xlrd')
            print("✅ Read as Excel file (xlrd)")
            return df
        except Exception as e:
            print(f"⚠️ Excel read failed: {e}")
            return None

def read_text_file(path, encodings, **kwargs):
    delimiters = [';', ',', '\t', '|', ' ']
    for encoding in encodings:
        for delim in delimiters:
            try:
                df = pd.read_csv(path, sep=delim, encoding=encoding, engine='python', on_bad_lines='skip', **kwargs)
                if df is not None and not df.empty and df.shape[1] > 1:
                    print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                    return df
            except:
                continue
    return None

def manual_parse_text_file(path):
    try:
        with open(path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        df = pd.DataFrame(data)
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    return None

def clean_dataframe(df):
    if df is None or df.empty:
        return df
    df = split_single_column(df)
    df = clean_column_names(df)
    df = remove_header_rows(df)
    df = standardize_missing_values(df)
    df = clean_numeric_columns(df)
    df = clean_text_columns(df)
    df = remove_empty(df)
    return df

def split_single_column(df):
    if df is None or df.empty or df.shape[1] != 1:
        return df
    col_name = df.columns[0]
    col_data = df[col_name].astype(str)
    sample = col_data.iloc[0] if len(col_data) > 0 else ''
    delimiters = [';', ',', '\t', '|']
    for delim in delimiters:
        if delim in sample:
            split_data = col_data.str.split(delim, expand=True)
            try:
                pd.to_numeric(split_data.iloc[0, 0])
                split_data.columns = [f'col_{i+1}' for i in range(split_data.shape[1])]
            except:
                headers = split_data.iloc[0].tolist()
                split_data = split_data.iloc[1:].copy()
                headers = [str(h).strip() for h in headers]
                split_data.columns = headers
            return split_data.reset_index(drop=True)
    return df

def clean_column_names(df):
    if df is None or df.empty:
        return df
    df.columns = df.columns.str.strip()
    df = df.loc[:, ~df.columns.duplicated()]
    df = df.loc[:, ~df.columns.astype(str).str.contains(r',+$', na=False)]
    df.columns = [f'col_{i}' if str(col).strip() == '' or pd.isna(col) else col for i, col in enumerate(df.columns)]
    df.columns = [re.sub(r'[^a-zA-Z0-9_ ]', '_', str(col)).strip('_') for col in df.columns]
    seen = {}
    new_cols = []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
    return df

def remove_header_rows(df):
    if df is None or df.empty or len(df) == 0:
        return df
    first_row = df.iloc[0].astype(str)
    header_keywords = ['sample', 'point', 'id', 'name', 'date', 'time', 'value', 'measurement', 'parameter', 'unit', 'ppm']
    header_score = sum(1 for val in first_row if any(keyword in str(val).lower() for keyword in header_keywords))
    if len(first_row) > 0 and header_score / len(first_row) > 0.3:
        df = df.iloc[1:].reset_index(drop=True)
    return df

def standardize_missing_values(df):
    if df is None or df.empty:
        return df
    missing_values = ['', 'nan', 'NaN', 'NAN', 'null', 'NULL', 'None', 'na', 'NA', '-', '--', '---', 'n/a', 'N/A', '?']
    df = df.replace(missing_values, np.nan)
    for col in df.columns:
        if col in df.columns:
            df[col] = df[col].astype(str).str.replace(r'^<.*$', 'nan', regex=True)
            df[col] = df[col].astype(str).str.replace(r'^>.*$', 'nan', regex=True)
    return df

def clean_numeric_columns(df):
    if df is None or df.empty:
        return df
    text_cols = ['sample', 'point_id', 'point', 'id', 'name', 'lithology', 'description', 'comment', 'remarks', 'unit', 'method', 'date', 'time']
    def clean_numeric(x):
        if pd.isna(x):
            return np.nan
        x = str(x).strip().replace(',', '.')
        if x.startswith('<') or x.startswith('>'):
            return np.nan
        if x in ['', 'nan', 'NaN', 'NAN', 'null', 'NULL']:
            return np.nan
        x = re.sub(r'[^\d.\-eE+]', '', x)
        try:
            return float(x)
        except:
            return np.nan
    for col in df.columns:
        col_lower = col.lower()
        is_text = any(text in col_lower for text in text_cols)
        if not is_text:
            try:
                df[col] = df[col].map(clean_numeric)
            except:
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    pass
    return df

def clean_text_columns(df):
    if df is None or df.empty:
        return df
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace('', np.nan)
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)
    return df

def remove_empty(df):
    if df is None or df.empty:
        return df
    df = df.dropna(how='all')
    df = df.dropna(axis=1, how='all')
    return df

# ============================================================
# PART 2: PROFESSIONAL EDA ENGINE
# ============================================================

class ProfessionalEDA:
    """
    Professional Exploratory Data Analysis Engine for Geochemical Data
    Features: Variance, Covariance, Correlation (Pearson, Spearman, Kendall),
    PCA, Outlier Detection, PDF Fitting, Jacobian Matrix, and more.
    """
    
    def __init__(self, df, sample_col=None):
        """
        Initialize the EDA engine
        
        Parameters:
        -----------
        df : pandas.DataFrame
            The cleaned dataframe
        sample_col : str, optional
            Column name containing sample IDs
        """
        self.df = df
        self.sample_col = sample_col if sample_col in df.columns else None
        
        # Separate numeric and categorical columns
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.categorical_df = df.select_dtypes(include=['object', 'category'])
        
        # Remove columns with all NaN
        self.numeric_df = self.numeric_df.dropna(axis=1, how='all')
        
        # Results storage
        self.results = {}
        self.cleaned_data = None
        
        print(f"\n{'='*70}")
        print("🔬 PROFESSIONAL EDA ENGINE INITIALIZED")
        print(f"{'='*70}")
        print(f"Total samples: {len(df)}")
        print(f"Numeric variables: {len(self.numeric_df.columns)}")
        print(f"Categorical variables: {len(self.categorical_df.columns)}")
        
        # Run complete analysis
        self.run_complete_analysis()
    
    def run_complete_analysis(self):
        """Execute all statistical analyses"""
        print("\n📊 Running comprehensive EDA...")
        
        # 1. Basic Statistics
        self.calculate_basic_stats()
        
        # 2. Variance Analysis
        self.calculate_variance()
        
        # 3. Covariance Matrix
        self.calculate_covariance()
        
        # 4. Correlation Matrices (Pearson, Spearman, Kendall)
        self.calculate_correlations()
        
        # 5. Outlier Detection
        self.detect_outliers()
        
        # 6. Probability Density Functions
        self.fit_pdfs()
        
        # 7. Jacobian Matrix (sensitivity analysis)
        self.compute_jacobian()
        
        # 8. Principal Component Analysis
        self.perform_pca()
        
        # 9. Eigen decomposition
        self.calculate_eigen()
        
        # 10. Advanced EDA
        self.advanced_eda()
        
        print("✅ Analysis complete!")
    
    def calculate_basic_stats(self):
        """Calculate basic descriptive statistics"""
        self.results['basic_stats'] = {
            'mean': self.numeric_df.mean(),
            'median': self.numeric_df.median(),
            'std': self.numeric_df.std(),
            'min': self.numeric_df.min(),
            'max': self.numeric_df.max(),
            'range': self.numeric_df.max() - self.numeric_df.min(),
            'q25': self.numeric_df.quantile(0.25),
            'q75': self.numeric_df.quantile(0.75),
            'iqr': self.numeric_df.quantile(0.75) - self.numeric_df.quantile(0.25),
            'skewness': self.numeric_df.skew(),
            'kurtosis': self.numeric_df.kurtosis()
        }
        print("✅ Basic statistics calculated")
    
    def calculate_variance(self):
        """Calculate variance and related metrics"""
        self.results['variance'] = {
            'variance': self.numeric_df.var(),
            'std_dev': self.numeric_df.std(),
            'coef_variation': (self.numeric_df.std() / self.numeric_df.mean()) * 100,
            'mean_absolute_dev': self.numeric_df.mad()
        }
        print("✅ Variance analysis complete")
    
    def calculate_covariance(self):
        """Calculate covariance matrix"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 1:
            self.results['covariance'] = {
                'matrix': clean_data.cov(),
                'trace': np.trace(clean_data.cov().values),
                'determinant': np.linalg.det(clean_data.cov().values) if clean_data.shape[1] > 1 else None
            }
        print("✅ Covariance matrix computed")
    
    def calculate_correlations(self):
        """Calculate multiple correlation matrices"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) > 1:
            self.results['correlations'] = {
                'pearson': clean_data.corr(method='pearson'),
                'spearman': clean_data.corr(method='spearman'),
                'kendall': clean_data.corr(method='kendall')
            }
            
            # Find strongest correlations for each method
            for method in ['pearson', 'spearman', 'kendall']:
                corr_matrix = self.results['correlations'][method]
                if corr_matrix is not None:
                    # Get upper triangle
                    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
                    # Find strongest pairs
                    max_corr = upper.max().max()
                    min_corr = upper.min().min()
                    self.results['correlations'][f'{method}_summary'] = {
                        'max': max_corr,
                        'min': min_corr,
                        'strong_pairs': self._get_strong_correlations(corr_matrix)
                    }
        print("✅ Correlation matrices computed (Pearson, Spearman, Kendall)")
    
    def _get_strong_correlations(self, corr_matrix, threshold=0.7):
        """Extract strong correlation pairs"""
        strong_pairs = []
        for i, col1 in enumerate(corr_matrix.columns):
            for col2 in corr_matrix.columns[i+1:]:
                val = corr_matrix.loc[col1, col2]
                if abs(val) > threshold:
                    strong_pairs.append({
                        'var1': col1,
                        'var2': col2,
                        'correlation': val,
                        'strength': 'strong positive' if val > 0 else 'strong negative'
                    })
        return sorted(strong_pairs, key=lambda x: abs(x['correlation']), reverse=True)
    
    def detect_outliers(self, method='iqr', threshold=1.5):
        """
        Detect outliers using multiple methods
        
        Methods:
        - 'iqr': Interquartile Range method
        - 'zscore': Z-score method
        - 'mad': Median Absolute Deviation method
        """
        self.results['outliers'] = {}
        clean_data = self.numeric_df.dropna()
        
        # IQR Method
        if method == 'iqr' or method == 'all':
            Q1 = clean_data.quantile(0.25)
            Q3 = clean_data.quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR
            
            outliers_iqr = ((clean_data < lower_bound) | (clean_data > upper_bound))
            self.results['outliers']['iqr'] = {
                'count': outliers_iqr.sum(),
                'percentage': (outliers_iqr.sum() / len(clean_data)) * 100,
                'mask': outliers_iqr,
                'lower_bound': lower_bound,
                'upper_bound': upper_bound,
                'variables': outliers_iqr.sum().to_dict()
            }
        
        # Z-Score Method
        if method == 'zscore' or method == 'all':
            z_scores = np.abs(stats.zscore(clean_data))
            outliers_z = z_scores > 3
            self.results['outliers']['zscore'] = {
                'count': outliers_z.sum(),
                'percentage': (outliers_z.sum() / len(clean_data)) * 100,
                'mask': outliers_z,
                'variables': outliers_z.sum().to_dict()
            }
        
        # MAD Method (Median Absolute Deviation)
        if method == 'mad' or method == 'all':
            median = clean_data.median()
            mad = np.abs(clean_data - median).median()
            mad_score = 0.6745 * (clean_data - median) / mad
            outliers_mad = np.abs(mad_score) > 3.5
            self.results['outliers']['mad'] = {
                'count': outliers_mad.sum(),
                'percentage': (outliers_mad.sum() / len(clean_data)) * 100,
                'mask': outliers_mad,
                'variables': outliers_mad.sum().to_dict()
            }
        
        print("✅ Outlier detection complete")
    
    def fit_pdfs(self):
        """
        Fit Probability Density Functions to each numeric variable
        Supports: Normal, Log-normal, Exponential, Weibull, Gamma
        """
        self.results['pdf'] = {}
        
        for col in self.numeric_df.columns:
            data = self.numeric_df[col].dropna()
            if len(data) < 10:
                continue
            
            pdf_results = {'variable': col, 'fits': {}}
            
            # Try different distributions
            distributions = {
                'normal': stats.norm,
                'lognormal': stats.lognorm,
                'exponential': stats.expon,
                'weibull': stats.weibull_min,
                'gamma': stats.gamma,
                'logistic': stats.logistic
            }
            
            # Use Kolmogorov-Smirnov test
            for name, dist in distributions.items():
                try:
                    params = dist.fit(data)
                    ks_stat, p_value = stats.kstest(data, dist.cdf, args=params)
                    pdf_results['fits'][name] = {
                        'params': params,
                        'ks_statistic': ks_stat,
                        'p_value': p_value,
                        'params_dict': self._get_param_names(dist, params)
                    }
                except:
                    continue
            
            # Find best fitting distribution (lowest KS statistic)
            if pdf_results['fits']:
                best_fit = min(pdf_results['fits'].items(), key=lambda x: x[1]['ks_statistic'])
                pdf_results['best_fit'] = best_fit[0]
                pdf_results['best_fit_info'] = best_fit[1]
            
            self.results['pdf'][col] = pdf_results
        
        print("✅ PDF fitting complete")
    
    def _get_param_names(self, dist, params):
        """Get parameter names for distribution"""
        param_names = {
            stats.norm: ['loc', 'scale'],
            stats.lognorm: ['shape', 'loc', 'scale'],
            stats.expon: ['loc', 'scale'],
            stats.weibull_min: ['shape', 'loc', 'scale'],
            stats.gamma: ['shape', 'loc', 'scale'],
            stats.logistic: ['loc', 'scale']
        }
        names = param_names.get(type(dist), [f'param_{i}' for i in range(len(params))])
        return dict(zip(names, params))
    
    def compute_jacobian(self, func=None, epsilon=1e-6):
        """
        Compute the Jacobian matrix for sensitivity analysis
        
        Parameters:
        -----------
        func : callable, optional
            Function that takes a 1D array and returns outputs.
            If None, uses a default function (mean of variables)
        epsilon : float
            Step size for finite differences
        """
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2:
            return
        
        # Default function: calculate mean of each variable
        if func is None:
            def func(x):
                return np.array([np.mean(x) for _ in range(len(x))])
        
        # Compute Jacobian at the mean of each variable
        x0 = clean_data.mean().values
        
        try:
            J = approx_fprime(x0, func, epsilon)
            
            self.results['jacobian'] = {
                'matrix': J,
                'variables': clean_data.columns.tolist(),
                'sensitivity': {
                    col: J[i] for i, col in enumerate(clean_data.columns)
                },
                'magnitudes': np.abs(J).sum(axis=1) if J.ndim > 1 else np.abs(J)
            }
            
            # Identify most sensitive variables
            if J.ndim > 1:
                sensitivities = np.abs(J).sum(axis=0)
            else:
                sensitivities = np.abs(J)
            
            self.results['jacobian']['most_sensitive'] = sorted(
                zip(clean_data.columns, sensitivities),
                key=lambda x: x[1],
                reverse=True
            )[:5]
            
            print("✅ Jacobian matrix computed")
        except Exception as e:
            print(f"⚠️ Jacobian computation failed: {e}")
    
    def perform_pca(self, n_components=None):
        """
        Perform Principal Component Analysis
        
        Parameters:
        -----------
        n_components : int, optional
            Number of components to keep. If None, use all.
        """
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2:
            print("⚠️ Not enough data for PCA")
            return
        
        # Standardize data
        scaler = StandardScaler()
        scaled_data = scaler.fit_transform(clean_data)
        
        # Perform PCA
        if n_components is None:
            n_components = min(clean_data.shape[1], len(clean_data))
        
        pca = SklearnPCA(n_components=n_components)
        pca_scores = pca.fit_transform(scaled_data)
        
        self.results['pca'] = {
            'explained_variance': pca.explained_variance_,
            'explained_variance_ratio': pca.explained_variance_ratio_,
            'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
            'components': pca.components_,
            'scores': pca_scores,
            'loadings': pd.DataFrame(
                pca.components_.T,
                columns=[f'PC{i+1}' for i in range(pca.n_components_)],
                index=clean_data.columns
            ),
            'n_components': pca.n_components_,
            'n_samples': len(clean_data),
            'scaler': scaler
        }
        
        print(f"✅ PCA complete: {pca.n_components_} components, "
              f"{sum(pca.explained_variance_ratio_)*100:.1f}% variance explained")
    
    def calculate_eigen(self):
        """Calculate eigenvalues and eigenvectors from covariance matrix"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2:
            return
        
        try:
            cov = clean_data.cov().values
            eigenvals, eigenvecs = np.linalg.eig(cov)
            
            # Sort by eigenvalues (largest first)
            idx = np.argsort(eigenvals)[::-1]
            eigenvals = eigenvals[idx]
            eigenvecs = eigenvecs[:, idx]
            
            self.results['eigen'] = {
                'values': eigenvals,
                'vectors': eigenvecs,
                'explained_variance': eigenvals / eigenvals.sum() * 100,
                'variables': clean_data.columns.tolist()
            }
            print("✅ Eigen decomposition complete")
        except Exception as e:
            print(f"⚠️ Eigen decomposition failed: {e}")
    
    def advanced_eda(self):
        """Perform advanced EDA including tests and diagnostics"""
        clean_data = self.numeric_df.dropna()
        if len(clean_data) < 2:
            return
        
        # 1. Normality tests (Shapiro-Wilk)
        self.results['normality_tests'] = {}
        for col in clean_data.columns:
            data = clean_data[col].dropna()
            if len(data) > 3 and len(data) < 5000:  # Shapiro-Wilk limit
                stat, p = stats.shapiro(data)
                self.results['normality_tests'][col] = {
                    'statistic': stat,
                    'p_value': p,
                    'is_normal': p > 0.05,
                    'interpretation': 'Normal' if p > 0.05 else 'Not normal'
                }
        
        # 2. Homogeneity tests (Levene's test)
        if len(clean_data.columns) >= 2:
            groups = [clean_data[col].dropna().values for col in clean_data.columns[:5]]
            if all(len(g) > 1 for g in groups):
                stat, p = stats.levene(*groups)
                self.results['homogeneity'] = {
                    'statistic': stat,
                    'p_value': p,
                    'equal_variances': p > 0.05
                }
        
        # 3. Multivariate normality (Mardia's test - simplified)
        if len(clean_data) > 3:
            try:
                from scipy.spatial.distance import mahalanobis
                cov = clean_data.cov().values
                inv_cov = np.linalg.inv(cov)
                center = clean_data.mean().values
                
                # Calculate Mahalanobis distances
                distances = np.array([mahalanobis(x, center, inv_cov) for x in clean_data.values])
                
                # These should follow chi-square distribution
                chi2 = stats.chi2(df=clean_data.shape[1])
                p_values = 1 - chi2.cdf(distances)
                
                self.results['multivariate'] = {
                    'mahalanobis_distances': distances,
                    'p_values': p_values,
                    'mean_mahalanobis': np.mean(distances),
                    'outliers_mv': np.sum(p_values < 0.001)
                }
            except:
                pass
        
        # 4. Correlation significance tests
        self.results['correlation_tests'] = {}
        corr_matrix = clean_data.corr()
        for i, col1 in enumerate(corr_matrix.columns):
            for col2 in corr_matrix.columns[i+1:]:
                r, p = pearsonr(clean_data[col1], clean_data[col2])
                self.results['correlation_tests'][f'{col1}_vs_{col2}'] = {
                    'correlation': r,
                    'p_value': p,
                    'significant': p < 0.05,
                    'method': 'Pearson'
                }
        
        print("✅ Advanced EDA complete")
    
    def get_cleaned_data(self, outlier_method='iqr'):
        """
        Get cleaned data with outliers removed
        
        Parameters:
        -----------
        outlier_method : str
            Method to use for outlier removal ('iqr', 'zscore', 'mad')
        """
        if outlier_method not in self.results['outliers']:
            print(f"⚠️ {outlier_method} not found, using IQR")
            outlier_method = 'iqr'
        
        outlier_mask = self.results['outliers'][outlier_method]['mask']
        self.cleaned_data = self.df.loc[~outlier_mask.any(axis=1)]
        return self.cleaned_data
    
    def get_report(self, include_all=True):
        """Generate comprehensive EDA report"""
        report = []
        report.append("="*80)
        report.append("🔬 ELEMENTAL SOCIETY ANALYSIS - EDA REPORT")
        report.append("="*80)
        report.append(f"Generated by: Elemental Society Analysis v1.0 (Code: 1010)")
        report.append(f"Dataset: {len(self.df)} samples × {len(self.df.columns)} variables")
        report.append(f"Numeric variables: {len(self.numeric_df.columns)}")
        report.append("")
        
        # 1. Basic Statistics
        report.append("📊 1. BASIC STATISTICS")
        report.append("-"*80)
        stats_df = pd.DataFrame(self.results['basic_stats'])
        report.append(stats_df.round(4).to_string())
        report.append("")
        
        # 2. Variance
        report.append("📈 2. VARIANCE ANALYSIS")
        report.append("-"*80)
        variance_df = pd.DataFrame(self.results['variance'])
        report.append(variance_df.round(4).to_string())
        report.append("")
        
        # 3. Correlations
        report.append("📉 3. STRONG CORRELATIONS")
        report.append("-"*80)
        if 'correlations' in self.results:
            for method in ['pearson', 'spearman', 'kendall']:
                if f'{method}_summary' in self.results['correlations']:
                    report.append(f"\n{method.upper()} Method:")
                    for pair in self.results['correlations'][f'{method}_summary']['strong_pairs'][:5]:
                        report.append(f"  {pair['var1']} ↔ {pair['var2']}: {pair['correlation']:.4f} ({pair['strength']})")
        report.append("")
        
        # 4. Outliers
        report.append("🎯 4. OUTLIER DETECTION")
        report.append("-"*80)
        if 'outliers' in self.results:
            for method, data in self.results['outliers'].items():
                report.append(f"\n{method.upper()} Method:")
                report.append(f"  Total outliers: {data['count']}")
                report.append(f"  Percentage: {data['percentage']:.2f}%")
                report.append("  Variables with outliers:")
                for var, count in sorted(data['variables'].items(), key=lambda x: x[1], reverse=True)[:5]:
                    if count > 0:
                        report.append(f"    {var}: {count}")
        report.append("")
        
        # 5. PDF Fitting
        report.append("📊 5. BEST FIT DISTRIBUTIONS")
        report.append("-"*80)
        if 'pdf' in self.results:
            for col, data in self.results['pdf'].items():
                if 'best_fit' in data:
                    report.append(f"{col}: {data['best_fit']} (KS={data['best_fit_info']['ks_statistic']:.4f}, p={data['best_fit_info']['p_value']:.4f})")
        report.append("")
        
        # 6. PCA
        report.append("📉 6. PRINCIPAL COMPONENT ANALYSIS")
        report.append("-"*80)
        if 'pca' in self.results:
            pca_data = self.results['pca']
            report.append(f"Number of components: {pca_data['n_components']}")
            report.append("Explained variance:")
            for i, (ev_ratio, cum_var) in enumerate(zip(pca_data['explained_variance_ratio'], 
                                                       pca_data['cumulative_variance'])):
                report.append(f"  PC{i+1}: {ev_ratio*100:.1f}% (cumulative: {cum_var*100:.1f}%)")
            report.append("\nTop variable loadings (PC1):")
            loadings = pca_data['loadings']['PC1'].abs().sort_values(ascending=False)[:5]
            for var, loading in loadings.items():
                report.append(f"  {var}: {loading:.4f}")
        report.append("")
        
        # 7. Jacobian (Sensitivity)
        report.append("🔧 7. SENSITIVITY ANALYSIS (JACOBIAN)")
        report.append("-"*80)
        if 'jacobian' in self.results:
            report.append("Most sensitive variables:")
            for var, sens in self.results['jacobian']['most_sensitive']:
                report.append(f"  {var}: {sens:.4f}")
        report.append("")
        
        # 8. Normality Tests
        report.append("📊 8. NORMALITY TESTS")
        report.append("-"*80)
        if 'normality_tests' in self.results:
            for col, data in self.results['normality_tests'].items():
                status = "✅ Normal" if data['is_normal'] else "❌ Not normal"
                report.append(f"  {col}: {status} (p={data['p_value']:.4f})")
        report.append("")
        
        report.append("="*80)
        report.append("📌 EDA Complete - Elemental Society Analysis v1.0")
        report.append("="*80)
        
        return "\n".join(report)

# ============================================================
# PART 3: GUI APPLICATION
# ============================================================

class ElementalSocietyEDA(QMainWindow):
    """Main application window for Elemental Society Analysis"""
    
    def __init__(self):
        super().__init__()
        self.df = None
        self.cleaned_df = None
        self.eda = None
        self.current_file = None
        self.log_messages = []
        self.init_ui()
        
    def init_ui(self):
        """Initialize the user interface"""
        self.setWindowTitle("🧪 Elemental Society Analysis - Professional EDA Suite v2.0")
        self.setGeometry(100, 100, 1400, 900)
        
        # Central widget
        central = QWidget()
        self.setCentralWidget(central)
        main_layout = QVBoxLayout()
        central.setLayout(main_layout)
        
        # Toolbar
        self.create_toolbar()
        
        # Info label
        self.info_label = QLabel("📋 Elemental Society Analysis - Ready for data")
        self.info_label.setStyleSheet("font-size: 14px; font-weight: bold; padding: 15px; background: linear-gradient(to right, #2196F3, #4CAF50); color: white; border-radius: 8px;")
        main_layout.addWidget(self.info_label)
        
        # Tab widget
        self.tab_widget = QTabWidget()
        main_layout.addWidget(self.tab_widget)
        
        # Data tab
        self.create_data_tab()
        
        # Statistics tab
        self.create_stats_tab()
        
        # PCA tab
        self.create_pca_tab()
        
        # EDA Report tab
        self.create_report_tab()
        
        # Log tab
        self.create_log_tab()
        
        # Status bar
        self.statusBar = QStatusBar()
        self.setStatusBar(self.statusBar)
        self.statusBar.showMessage("✅ Ready")
        
        # Styles
        self.apply_styles()
        
        self.add_log("🔬 Elemental Society Analysis v2.0 initialized")
        self.add_log("📌 Professional EDA Suite ready")
    
    def create_toolbar(self):
        """Create the toolbar"""
        toolbar = self.addToolBar("Main")
        toolbar.setIconSize(QSize(32, 32))
        
        # Upload
        upload_btn = QAction("📁 Upload", self)
        upload_btn.triggered.connect(self.upload_file)
        toolbar.addAction(upload_btn)
        
        # Clean
        clean_btn = QAction("🧹 Clean", self)
        clean_btn.triggered.connect(self.clean_data)
        toolbar.addAction(clean_btn)
        
        # Analyze
        analyze_btn = QAction("📊 Analyze", self)
        analyze_btn.triggered.connect(self.run_eda)
        toolbar.addAction(analyze_btn)
        
        toolbar.addSeparator()
        
        # Report
        report_btn = QAction("📄 Full Report", self)
        report_btn.triggered.connect(self.show_full_report)
        toolbar.addAction(report_btn)
        
        # Export
        export_btn = QAction("💾 Export", self)
        export_btn.triggered.connect(self.export_data)
        toolbar.addAction(export_btn)
        
        toolbar.addSeparator()
        
        # Help
        help_btn = QAction("❓ Help", self)
        help_btn.triggered.connect(self.show_help)
        toolbar.addAction(help_btn)
    
    def create_data_tab(self):
        """Create data viewing tab"""
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        self.table = QTableWidget()
        self.table.setAlternatingRowColors(True)
        self.table.setSortingEnabled(True)
        layout.addWidget(self.table)
        
        self.tab_widget.addTab(tab, "📊 Data")
    
    def create_stats_tab(self):
        """Create statistics tab"""
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        self.stats_text = QTextEdit()
        self.stats_text.setReadOnly(True)
        self.stats_text.setFont(QFont("Consolas", 10))
        layout.addWidget(self.stats_text)
        
        self.tab_widget.addTab(tab, "📈 Statistics")
    
    def create_pca_tab(self):
        """Create PCA tab"""
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        self.pca_text = QTextEdit()
        self.pca_text.setReadOnly(True)
        self.pca_text.setFont(QFont("Consolas", 10))
        layout.addWidget(self.pca_text)
        
        self.tab_widget.addTab(tab, "📉 PCA")
    
    def create_report_tab(self):
        """Create EDA report tab"""
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        self.report_text = QTextEdit()
        self.report_text.setReadOnly(True)
        self.report_text.setFont(QFont("Consolas", 10))
        layout.addWidget(self.report_text)
        
        self.tab_widget.addTab(tab, "📄 EDA Report")
    
    def create_log_tab(self):
        """Create log tab"""
        tab = QWidget()
        layout = QVBoxLayout()
        tab.setLayout(layout)
        
        self.log_text = QTextEdit()
        self.log_text.setReadOnly(True)
        self.log_text.setFont(QFont("Consolas", 10))
        layout.addWidget(self.log_text)
        
        self.tab_widget.addTab(tab, "📝 Log")
    
    def apply_styles(self):
        """Apply application styles"""
        self.setStyleSheet("""
            QMainWindow { background-color: #f5f5f5; }
            QTableWidget { 
                background-color: white; 
                gridline-color: #d0d0d0;
                alternate-background-color: #fafafa;
            }
            QTableWidget::item:selected { 
                background-color: #2196F3; 
                color: white; 
            }
            QHeaderView::section { 
                background-color: #e0e0e0; 
                padding: 5px; 
                font-weight: bold;
            }
            QPushButton { 
                background-color: #2196F3; 
                color: white; 
                padding: 8px 15px; 
                border-radius: 4px; 
                border: none; 
            }
            QPushButton:hover { background-color: #1976D2; }
            QTabWidget::pane { border: 1px solid #d0d0d0; border-radius: 4px; }
            QTabBar::tab { 
                padding: 8px 15px; 
                margin-right: 2px;
                border-top-left-radius: 4px;
                border-top-right-radius: 4px;
            }
            QTabBar::tab:selected { 
                background-color: #2196F3; 
                color: white; 
            }
            QTabBar::tab:hover { background-color: #e0e0e0; }
        """)
    
    def add_log(self, message):
        """Add message to log"""
        from datetime import datetime
        timestamp = datetime.now().strftime("%H:%M:%S")
        self.log_text.append(f"[{timestamp}] {message}")
        self.log_messages.append(message)
    
    def upload_file(self):
        """Upload data file"""
        file_path, _ = QFileDialog.getOpenFileName(
            self,
            "Open Geochemical Data",
            "",
            "All Files (*);;CSV (*.csv);;Excel (*.xlsx *.xls);;Text (*.txt);;TSV (*.tsv)"
        )
        
        if file_path:
            self.current_file = file_path
            self.add_log(f"📁 Loading: {os.path.basename(file_path)}")
            self.statusBar.showMessage(f"Loading: {os.path.basename(file_path)}...")
            
            try:
                self.df = clean_any_data(file_path)
                
                if self.df is not None and not self.df.empty:
                    self.display_data()
                    self.add_log(f"✅ Loaded: {self.df.shape[0]} rows, {self.df.shape[1]} columns")
                    self.statusBar.showMessage(f"✅ Loaded: {os.path.basename(file_path)}")
                    self.info_label.setText(f"📊 Data loaded: {self.df.shape[0]} rows × {self.df.shape[1]} columns")
                    
                    QMessageBox.information(self, "Success",
                        f"File loaded successfully!\n\n"
                        f"📊 Shape: {self.df.shape[0]} rows × {self.df.shape[1]} columns\n"
                        f"📋 Numeric columns: {len(self.df.select_dtypes(include=[np.number]).columns)}")
                else:
                    QMessageBox.warning(self, "Error", "No data could be read.")
                    
            except Exception as e:
                self.add_log(f"❌ Error: {str(e)}")
                QMessageBox.critical(self, "Error", f"Failed to load:\n{str(e)}")
    
    def clean_data(self):
        """Clean the loaded data"""
        if self.df is None or self.df.empty:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
        
        self.add_log("🧹 Starting data cleaning...")
        self.statusBar.showMessage("Cleaning data...")
        
        try:
            self.cleaned_df = clean_dataframe(self.df.copy())
            self.display_data(self.cleaned_df)
            
            rows_removed = len(self.df) - len(self.cleaned_df)
            self.add_log(f"✅ Cleaning complete: {len(self.cleaned_df)} rows, {len(self.cleaned_df.columns)} columns")
            self.add_log(f"🗑️ Removed {rows_removed} rows")
            self.statusBar.showMessage(f"✅ Cleaned: {len(self.cleaned_df)} rows")
            self.info_label.setText(f"🧹 Cleaned: {len(self.cleaned_df)} rows × {len(self.cleaned_df.columns)} columns")
            
            QMessageBox.information(self, "Success",
                f"Cleaning complete!\n\n"
                f"📊 Before: {len(self.df)} rows\n"
                f"📊 After: {len(self.cleaned_df)} rows\n"
                f"🗑️ Removed: {rows_removed} rows")
            
        except Exception as e:
            self.add_log(f"❌ Cleaning error: {str(e)}")
            QMessageBox.critical(self, "Error", f"Cleaning failed:\n{str(e)}")
    
    def run_eda(self):
        """Run comprehensive EDA"""
        df_to_analyze = self.cleaned_df if self.cleaned_df is not None else self.df
        
        if df_to_analyze is None or df_to_analyze.empty:
            QMessageBox.warning(self, "Warning", "Please load data first!")
            return
        
        self.add_log("📊 Starting comprehensive EDA...")
        self.statusBar.showMessage("Running EDA...")
        
        try:
            self.eda = ProfessionalEDA(df_to_analyze)
            
            # Display results
            self.display_statistics()
            self.display_pca()
            self.display_report()
            
            self.add_log("✅ EDA complete!")
            self.statusBar.showMessage("✅ EDA complete")
            
            QMessageBox.information(self, "Success",
                "EDA completed successfully!\n\n"
                f"📊 Variables analyzed: {len(self.eda.numeric_df.columns)}\n"
                f"📈 Variance computed\n"
                f"📉 Correlations (Pearson, Spearman, Kendall)\n"
                f"📊 PDF fitting complete\n"
                f"📉 PCA performed\n"
                f"🔧 Jacobian computed\n"
                f"📋 Full report generated")
            
        except Exception as e:
            self.add_log(f"❌ EDA error: {str(e)}")
            QMessageBox.critical(self, "Error", f"EDA failed:\n{str(e)}")
    
    def display_data(self, df=None):
        """Display data in table"""
        if df is None:
            if self.cleaned_df is not None:
                df = self.cleaned_df
            elif self.df is not None:
                df = self.df
            else:
                return
        
        display_limit = min(1000, len(df))
        self.table.setRowCount(display_limit)
        self.table.setColumnCount(len(df.columns))
        self.table.setHorizontalHeaderLabels(df.columns.astype(str))
        
        for i in range(display_limit):
            for j, col in enumerate(df.columns):
                val = df.iloc[i, j]
                if pd.isna(val):
                    self.table.setItem(i, j, QTableWidgetItem(""))
                else:
                    self.table.setItem(i, j, QTableWidgetItem(str(val)))
        
        self.table.resizeColumnsToContents()
    
    def display_statistics(self):
        """Display statistics in stats tab"""
        if self.eda is None:
            return
        
        stats_text = []
        stats_text.append("="*80)
        stats_text.append("📊 COMPREHENSIVE STATISTICAL SUMMARY")
        stats_text.append("="*80)
        stats_text.append("")
        
        # Basic statistics
        stats_text.append("1. BASIC STATISTICS")
        stats_text.append("-"*80)
        basic_df = pd.DataFrame(self.eda.results['basic_stats'])
        stats_text.append(basic_df.round(4).to_string())
        stats_text.append("")
        
        # Variance
        stats_text.append("2. VARIANCE ANALYSIS")
        stats_text.append("-"*80)
        variance_df = pd.DataFrame(self.eda.results['variance'])
        stats_text.append(variance_df.round(4).to_string())
        stats_text.append("")
        
        # Outliers
        stats_text.append("3. OUTLIER DETECTION")
        stats_text.append("-"*80)
        if 'outliers' in self.eda.results:
            for method, data in self.eda.results['outliers'].items():
                stats_text.append(f"\n{method.upper()}:")
                stats_text.append(f"  Total: {data['count']} ({data['percentage']:.2f}%)")
                stats_text.append("  Variables with outliers:")
                for var, count in sorted(data['variables'].items(), key=lambda x: x[1], reverse=True)[:5]:
                    if count > 0:
                        stats_text.append(f"    {var}: {count}")
        
        self.stats_text.setText("\n".join(stats_text))
    
    def display_pca(self):
        """Display PCA results"""
        if self.eda is None or 'pca' not in self.eda.results:
            return
        
        pca_data = self.eda.results['pca']
        pca_text = []
        pca_text.append("="*80)
        pca_text.append("📉 PRINCIPAL COMPONENT ANALYSIS")
        pca_text.append("="*80)
        pca_text.append(f"\nComponents: {pca_data['n_components']}")
        pca_text.append(f"Samples: {pca_data['n_samples']}")
        pca_text.append("\nExplained Variance:")
        pca_text.append("-"*40)
        
        for i, (ev_ratio, cum_var) in enumerate(zip(pca_data['explained_variance_ratio'], 
                                                   pca_data['cumulative_variance'])):
            pca_text.append(f"PC{i+1}: {ev_ratio*100:.1f}% (cumulative: {cum_var*100:.1f}%)")
        
        pca_text.append("\nLoadings (PC1-PC3):")
        pca_text.append("-"*40)
        for col in pca_data['loadings'].index[:5]:
            loadings = pca_data['loadings'].loc[col]
            pca_text.append(f"{col}: PC1={loadings[0]:.4f}, PC2={loadings[1]:.4f}, PC3={loadings[2]:.4f}")
        
        self.pca_text.setText("\n".join(pca_text))
    
    def display_report(self):
        """Display full EDA report"""
        if self.eda is None:
            return
        
        report = self.eda.get_report()
        self.report_text.setText(report)
    
    def show_full_report(self):
        """Show full report in a dialog"""
        if self.eda is None:
            QMessageBox.warning(self, "Warning", "Please run EDA first!")
            return
        
        report = self.eda.get_report()
        dialog = QDialog(self)
        dialog.setWindowTitle("📄 Full EDA Report")
        dialog.setGeometry(200, 200, 1000, 700)
        
        layout = QVBoxLayout()
        dialog.setLayout(layout)
        
        text_edit = QTextEdit()
        text_edit.setFont(QFont("Consolas", 10))
        text_edit.setText(report)
        layout.addWidget(text_edit)
        
        # Save button
        save_btn = QPushButton("💾 Save Report")
        save_btn.clicked.connect(lambda: self.save_report(text_edit.toPlainText()))
        layout.addWidget(save_btn)
        
        close_btn = QPushButton("Close")
        close_btn.clicked.connect(dialog.accept)
        layout.addWidget(close_btn)
        
        dialog.exec_()
    
    def save_report(self, report_text):
        """Save report to file"""
        file_path, _ = QFileDialog.getSaveFileName(
            self,
            "Save EDA Report",
            "eda_report.txt",
            "Text Files (*.txt);;All Files (*)"
        )
        
        if file_path:
            try:
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(report_text)
                QMessageBox.information(self, "Success", f"Report saved to:\n{file_path}")
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Failed to save:\n{str(e)}")
    
    def export_data(self):
        """Export cleaned data"""
        if self.cleaned_df is None and self.df is None:
            QMessageBox.warning(self, "Warning", "No data to export!")
            return
        
        df = self.cleaned_df if self.cleaned_df is not None else self.df
        
        # Ask for format
        formats = {
            "CSV (;)": "csv_semicolon",
            "CSV (,)": "csv_comma",
            "Excel (.xlsx)": "excel",
            "JSON": "json"
        }
        
        format_choice, ok = QInputDialog.getItem(
            self,
            "Export Format",
            "Select export format:",
            list(formats.keys()),
            0,
            False
        )
        
        if not ok:
            return
        
        file_ext_map = {
            "csv_semicolon": "csv",
            "csv_comma": "csv",
            "excel": "xlsx",
            "json": "json"
        }
        
        file_path, _ = QFileDialog.getSaveFileName(
            self,
            "Export Data",
            f"cleaned_data.{file_ext_map[formats[format_choice]]}",
            f"{format_choice} (*.{file_ext_map[formats[format_choice]]})"
        )
        
        if file_path:
            try:
                if formats[format_choice] == "csv_semicolon":
                    df.to_csv(file_path, sep=';', index=False, encoding='utf-8-sig')
                elif formats[format_choice] == "csv_comma":
                    df.to_csv(file_path, sep=',', index=False, encoding='utf-8-sig')
                elif formats[format_choice] == "excel":
                    df.to_excel(file_path, index=False)
                elif formats[format_choice] == "json":
                    df.to_json(file_path, orient='records', indent=2)
                
                self.add_log(f"✅ Exported: {file_path}")
                QMessageBox.information(self, "Success", f"Data exported to:\n{file_path}")
                
            except Exception as e:
                QMessageBox.critical(self, "Error", f"Export failed:\n{str(e)}")
    
    def show_help(self):
        """Show help dialog"""
        help_text = """
🔬 **Elemental Society Analysis - Professional EDA Suite v2.0**

**Project Code:** 1010 (Data Cleaning)

**Features:**

1. **Data Cleaning**
   - Universal file support (CSV, Excel, TSV, Text)
   - Auto-detection of delimiters and encoding
   - Detection limit handling (< values)
   - Unit unification (ppm, %, ppb)
   - Column name normalization
   - Duplicate removal

2. **Statistical Analysis**
   - Basic statistics (mean, median, std, etc.)
   - Variance analysis
   - Covariance matrix
   - Correlation (Pearson, Spearman, Kendall)
   - Outlier detection (IQR, Z-score, MAD)
   - Probability density function fitting

3. **Advanced Analytics**
   - Principal Component Analysis (PCA)
   - Eigen decomposition
   - Jacobian matrix (sensitivity analysis)
   - Normality tests
   - Multivariate analysis

4. **Visualization & Reporting**
   - Full EDA report generation
   - Export to multiple formats (CSV, Excel, JSON)
   - Interactive data preview

**Developed by:** Mohamed Almoufty Abaker
**Email:** mohamed.almoufty@example.com
**License:** Proprietary - All rights reserved
"""
        
        QMessageBox.information(self, "Help", help_text)


# ============================================================
# PART 4: LAUNCH THE APPLICATION
# ============================================================

def launch_app():
    """Launch the application"""
    if QT_VERSION is None:
        print("❌ PyQt not installed. Install with: pip install PyQt5")
        return None, None
    
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    
    app.setStyle('Fusion')
    font = QFont("Segoe UI", 10)
    app.setFont(font)
    
    window = ElementalSocietyEDA()
    window.show()
    
    return app, window

def main():
    """Main entry point"""
    app, window = launch_app()
    if not IN_JUPYTER:
        sys.exit(app.exec_())
    else:
        return app, window

if __name__ == '__main__':
    try:
        from IPython import get_ipython
        if 'IPKernelApp' in get_ipython().config:
            print("📓 Running in Jupyter Notebook")
            print("🚀 Launching Elemental Society Analysis...")
            app, window = main()
            print("✅ Application launched! Close the window to stop.")
        else:
            main()
    except:
        main()

📓 Running in Jupyter Notebook
🚀 Launching Elemental Society Analysis...
✅ Application launched! Close the window to stop.


In [ ]:
# Basic statistics
eda.results['basic_stats']

# Correlation matrix
eda.results['correlations']['pearson']

# PCA results
eda.results['pca']['loadings']

# Outlier detection
eda.results['outliers']['iqr']

# PDF fits
eda.results['pdf']

# Jacobian (sensitivity)
eda.results['jacobian']['most_sensitive']

In [5]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - JUPYTER COMPLETE VERSION (FULLY FIXED)
# Project Code: 1010 (Data Cleaning)
# Developed by: Mohamed Almoufty Abaker - Geologist & Writer
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("🔬 ELEMENTAL SOCIETY ANALYSIS - JUPYTER EDA SUITE")
print("="*80)
print("📌 Project Code: 1010 (Data Cleaning)")
print("👨‍🔬 Developed by: Mohamed Almoufty Abaker")
print("="*80)

# ============================================================
# PART 1: FILE LOADING ENGINE
# ============================================================

def load_file(file_path, **kwargs):
    """
    Universal file loader for geochemical data
    
    Parameters:
    -----------
    file_path : str
        Path to the file
    **kwargs : additional arguments for pd.read_csv or pd.read_excel
    
    Returns:
    --------
    pandas.DataFrame : Loaded data
    """
    print(f"\n📁 LOADING FILE: {file_path}")
    print("-" * 50)
    
    # Check file extension
    file_ext = file_path.split('.')[-1].lower()
    
    # Try different delimiters and encodings for CSV/text files
    if file_ext in ['csv', 'txt', 'dat', 'tsv']:
        delimiters = [';', ',', '\t', '|', ' ']
        encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1']
        
        df = None
        for encoding in encodings:
            for delim in delimiters:
                try:
                    df = pd.read_csv(file_path, sep=delim, encoding=encoding, 
                                   engine='python', on_bad_lines='skip', **kwargs)
                    if df is not None and not df.empty and df.shape[1] > 1:
                        print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                        print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                        return df
                except:
                    continue
        
        # If all fail, try manual parsing
        if df is None or df.empty:
            print("⚠️ Auto-detection failed, trying manual parsing...")
            df = manual_parse_file(file_path)
            if df is not None:
                print(f"✅ Manual parsing successful! Shape: {df.shape}")
                return df
    
    # Excel files
    elif file_ext in ['xlsx', 'xls', 'xlsm']:
        try:
            df = pd.read_excel(file_path, engine='openpyxl', **kwargs)
            print(f"✅ Read as Excel file (openpyxl)")
            print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
            return df
        except:
            try:
                df = pd.read_excel(file_path, engine='xlrd', **kwargs)
                print(f"✅ Read as Excel file (xlrd)")
                print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                return df
            except Exception as e:
                print(f"❌ Excel read failed: {e}")
                return None
    
    else:
        print(f"⚠️ Unknown file type: {file_ext}")
        df = manual_parse_file(file_path)
        if df is not None:
            print(f"✅ Manual parsing successful! Shape: {df.shape}")
            return df
    
    print("❌ Could not read file")
    return None

def manual_parse_file(file_path):
    """Manually parse text files when auto-detection fails"""
    try:
        with open(file_path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    # Detect delimiter
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        
        df = pd.DataFrame(data)
        
        # Check if first row is header
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    
    return None

# ============================================================
# PART 2: DATA CLEANING ENGINE
# ============================================================

class DataCleaner:
    """Professional Data Cleaning Engine for Geochemical Data"""
    
    def __init__(self, df):
        self.df = df.copy()
        self.cleaned_df = None
        self.cleaning_log = []
        print(f"\n🧹 DATA CLEANER INITIALIZED")
        print(f"Original shape: {self.df.shape}")
        print(f"Columns: {len(self.df.columns)}")
        print(f"Rows: {len(self.df)}")
    
    def clean_all(self, handle_missing=True, handle_outliers=True, 
                  handle_detection_limits=True, remove_duplicates=True):
        """
        Run complete cleaning pipeline
        """
        self.cleaning_log = []
        print("\n🧹 RUNNING CLEANING PIPELINE")
        print("-" * 50)
        
        # 1. Clean column names
        self._clean_column_names()
        
        # 2. Handle detection limits BEFORE converting to numeric
        if handle_detection_limits:
            self._handle_detection_limits()
        
        # 3. Convert to numeric
        self._convert_to_numeric()
        
        # 4. Remove duplicates
        if remove_duplicates:
            self._remove_duplicates()
        
        # 5. Handle missing values
        if handle_missing:
            self._handle_missing_values()
        
        # 6. Handle outliers
        if handle_outliers:
            self._handle_outliers()
        
        self.cleaned_df = self.df.copy()
        
        print(f"\n✅ Cleaning complete!")
        print(f"Final shape: {self.cleaned_df.shape}")
        print(f"Rows: {len(self.cleaned_df)}")
        print(f"Columns: {len(self.cleaned_df.columns)}")
        
        return self.cleaned_df
    
    def _clean_column_names(self):
        """Clean column names"""
        self.df.columns = self.df.columns.str.strip()
        self.df.columns = self.df.columns.str.replace(' ', '_')
        self.df.columns = self.df.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
        self.df.columns = self.df.columns.str.replace('__', '_', regex=False)
        
        # Make unique
        seen = {}
        new_cols = []
        for col in self.df.columns:
            if col in seen:
                seen[col] += 1
                new_cols.append(f"{col}_{seen[col]}")
            else:
                seen[col] = 0
                new_cols.append(col)
        self.df.columns = new_cols
        
        self.cleaning_log.append("✅ Column names cleaned")
    
    def _handle_detection_limits(self):
        """Handle detection limits (< values) - BEFORE converting to numeric"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            
            # Check if column contains < values
            mask = self.df[col].astype(str).str.startswith('<', na=False)
            if mask.sum() > 0:
                for idx in self.df[mask].index:
                    val = str(self.df.loc[idx, col])
                    try:
                        # Extract number after <
                        limit = float(val.replace('<', '').strip())
                        # Replace with half detection limit
                        self.df.loc[idx, col] = limit / 2
                    except:
                        self.df.loc[idx, col] = np.nan
                self.cleaning_log.append(f"✅ Handled {mask.sum()} detection limits in {col}")
    
    def _convert_to_numeric(self):
        """Convert columns to numeric"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
        self.cleaning_log.append("✅ Converted columns to numeric")
    
    def _remove_duplicates(self):
        """Remove duplicate rows"""
        initial_len = len(self.df)
        self.df = self.df.drop_duplicates()
        removed = initial_len - len(self.df)
        if removed > 0:
            self.cleaning_log.append(f"✅ Removed {removed} duplicate rows")
    
    def _handle_missing_values(self):
        """Handle missing values"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            missing = self.df[col].isnull().sum()
            if missing > 0:
                if self.df[col].dtype in ['float64', 'int64']:
                    self.df[col] = self.df[col].fillna(self.df[col].median())
                    self.cleaning_log.append(f"✅ Filled {missing} missing values in {col}")
    
    def _handle_outliers(self, method='iqr'):
        """Handle outliers using IQR method"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            
            if self.df[col].dtype not in ['float64', 'int64']:
                continue
            
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            
            outliers = ((self.df[col] < lower) | (self.df[col] > upper)).sum()
            if outliers > 0:
                self.df[col] = self.df[col].clip(lower=lower, upper=upper)
                self.cleaning_log.append(f"✅ Capped {outliers} outliers in {col}")
    
    def get_log(self):
        """Get cleaning log"""
        return "\n".join(self.cleaning_log)

# ============================================================
# PART 3: PROFESSIONAL EDA ENGINE
# ============================================================

class ProfessionalEDA:
    """Comprehensive Exploratory Data Analysis Engine"""
    
    def __init__(self, df):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        print(f"\n📊 EDA ENGINE INITIALIZED")
        print(f"Numeric columns: {len(self.numeric_df.columns)}")
        print(f"Samples: {len(self.df)}")
    
    def run_all(self):
        """Run all analyses"""
        print("\n📊 RUNNING COMPREHENSIVE EDA")
        print("-" * 50)
        
        self.basic_statistics()
        self.variance_analysis()
        self.correlation_analysis()
        self.outlier_detection()
        self.normality_analysis()
        self.perform_pca()
        
        print("\n✅ EDA complete!")
        return self.results
    
    def basic_statistics(self):
        """Calculate basic statistics"""
        self.results['basic_stats'] = {
            'mean': self.numeric_df.mean(),
            'median': self.numeric_df.median(),
            'std': self.numeric_df.std(),
            'min': self.numeric_df.min(),
            'max': self.numeric_df.max(),
            'range': self.numeric_df.max() - self.numeric_df.min(),
            'q25': self.numeric_df.quantile(0.25),
            'q75': self.numeric_df.quantile(0.75),
            'iqr': self.numeric_df.quantile(0.75) - self.numeric_df.quantile(0.25),
            'skew': self.numeric_df.skew(),
            'kurtosis': self.numeric_df.kurtosis(),
            'count': self.numeric_df.count()
        }
        print("  ✅ Basic statistics")
        return self.results['basic_stats']
    
    def variance_analysis(self):
        """Calculate variance metrics"""
        self.results['variance'] = {
            'variance': self.numeric_df.var(),
            'std_dev': self.numeric_df.std(),
            'cv': (self.numeric_df.std() / self.numeric_df.mean()) * 100
        }
        print("  ✅ Variance analysis")
        return self.results['variance']
    
    def correlation_analysis(self):
        """Calculate correlation matrix"""
        if len(self.numeric_df.columns) > 1:
            self.results['correlation'] = self.numeric_df.corr()
            
            # Find strong correlations
            strong = []
            corr = self.results['correlation']
            for i, col1 in enumerate(corr.columns):
                for col2 in corr.columns[i+1:]:
                    val = corr.loc[col1, col2]
                    if abs(val) > 0.5:
                        strong.append((col1, col2, val))
            self.results['strong_correlations'] = sorted(strong, key=lambda x: abs(x[2]), reverse=True)
            print(f"  ✅ Correlation matrix ({len(strong)} strong correlations)")
        return self.results.get('correlation')
    
    def outlier_detection(self):
        """Detect outliers using IQR method"""
        self.results['outliers'] = {}
        for col in self.numeric_df.columns:
            Q1 = self.numeric_df[col].quantile(0.25)
            Q3 = self.numeric_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            self.results['outliers'][col] = ((self.numeric_df[col] < lower) | 
                                            (self.numeric_df[col] > upper)).sum()
        print("  ✅ Outlier detection")
        return self.results['outliers']
    
    def normality_analysis(self):
        """Analyze normality using skewness and kurtosis"""
        self.results['normality'] = {}
        for col in self.numeric_df.columns:
            skew_val = self.numeric_df[col].skew()
            kurt_val = self.numeric_df[col].kurtosis()
            is_normal = (-1 < skew_val < 1) and (-2 < kurt_val < 2)
            self.results['normality'][col] = {
                'skewness': skew_val,
                'kurtosis': kurt_val,
                'is_normal': is_normal
            }
        print("  ✅ Normality analysis")
        return self.results['normality']
    
    def perform_pca(self):
        """Perform Principal Component Analysis"""
        if len(self.numeric_df.columns) >= 2:
            clean_data = self.numeric_df.dropna()
            if len(clean_data) > 1:
                scaler = StandardScaler()
                scaled = scaler.fit_transform(clean_data)
                pca = SklearnPCA()
                scores = pca.fit_transform(scaled)
                self.results['pca'] = {
                    'explained_variance': pca.explained_variance_ratio_,
                    'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
                    'scores': scores,
                    'loadings': pd.DataFrame(
                        pca.components_.T,
                        columns=[f'PC{i+1}' for i in range(pca.n_components_)],
                        index=clean_data.columns
                    ),
                    'n_components': pca.n_components_,
                    'n_samples': len(clean_data)
                }
                print(f"  ✅ PCA ({pca.n_components_} components, "
                      f"{pca.explained_variance_ratio_.sum()*100:.1f}% variance)")
        return self.results.get('pca')
    
    def get_summary(self):
        """Get summary statistics"""
        return {
            'shape': self.df.shape,
            'numeric_cols': len(self.numeric_df.columns),
            'total_cols': len(self.df.columns),
            'basic_stats': self.results.get('basic_stats', {}),
            'variance': self.results.get('variance', {}),
            'strong_correlations': self.results.get('strong_correlations', []),
            'outliers': self.results.get('outliers', {}),
            'normality': self.results.get('normality', {})
        }
    
    def get_report(self):
        """Generate comprehensive report"""
        report = []
        report.append("="*80)
        report.append("📊 ELEMENTAL SOCIETY ANALYSIS - EDA REPORT")
        report.append("="*80)
        
        # Dataset info
        report.append(f"\n📋 Dataset: {len(self.df)} rows × {len(self.df.columns)} columns")
        report.append(f"📊 Numeric columns: {len(self.numeric_df.columns)}")
        
        # Basic stats
        report.append("\n📊 BASIC STATISTICS")
        report.append("-"*40)
        stats_df = pd.DataFrame(self.results.get('basic_stats', {}))
        if not stats_df.empty:
            report.append(stats_df.round(4).to_string())
        
        # Variance
        report.append("\n📈 VARIANCE")
        report.append("-"*40)
        var_df = pd.DataFrame(self.results.get('variance', {}))
        if not var_df.empty:
            report.append(var_df.round(4).to_string())
        
        # Correlations
        if 'strong_correlations' in self.results:
            report.append("\n📉 STRONG CORRELATIONS (>0.5)")
            report.append("-"*40)
            for col1, col2, val in self.results['strong_correlations'][:5]:
                direction = "positive" if val > 0 else "negative"
                report.append(f"  {col1} ↔ {col2}: {val:.4f} ({direction})")
        
        # Outliers
        report.append("\n🎯 OUTLIERS")
        report.append("-"*40)
        for col, count in self.results.get('outliers', {}).items():
            report.append(f"  {col}: {count} outliers")
        
        # Normality
        report.append("\n📊 NORMALITY")
        report.append("-"*40)
        for col, data in self.results.get('normality', {}).items():
            status = "✅ Normal" if data['is_normal'] else "❌ Not normal"
            report.append(f"  {col}: {status} (skew={data['skewness']:.4f})")
        
        # PCA
        if 'pca' in self.results:
            report.append("\n📉 PCA")
            report.append("-"*40)
            explained = self.results['pca']['explained_variance']
            cum_var = np.cumsum(explained)
            report.append(f"Total explained variance: {explained.sum()*100:.1f}%")
            for i, (ev, cum) in enumerate(zip(explained[:5], cum_var[:5])):
                report.append(f"  PC{i+1}: {ev*100:.1f}% (cumulative: {cum*100:.1f}%)")
            report.append("\nLoadings (PC1):")
            loadings = self.results['pca']['loadings']['PC1'].abs().sort_values(ascending=False)
            for var, loading in list(loadings.items())[:5]:
                report.append(f"  {var}: {loading:.4f}")
        
        report.append("\n"+"="*80)
        report.append(f"📌 Report generated by Elemental Society Analysis v2.0")
        report.append("="*80)
        
        return "\n".join(report)

# ============================================================
# PART 4: VISUALIZATION FUNCTIONS
# ============================================================

def plot_correlation(df, title="Correlation Matrix", figsize=(10, 8)):
    """Plot correlation matrix heatmap"""
    numeric_df = df.select_dtypes(include=[np.number])
    if len(numeric_df.columns) < 2:
        print("⚠️ Need at least 2 numeric columns")
        return
    
    plt.figure(figsize=figsize)
    corr = numeric_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', center=0,
                fmt='.2f', square=True, linewidths=0.5)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_pca_scores(pca_results, figsize=(10, 6)):
    """Plot PCA scores"""
    if pca_results is None:
        print("⚠️ No PCA results available")
        return
    
    scores = pca_results['scores']
    explained = pca_results['explained_variance']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # PCA scores
    scatter = ax1.scatter(scores[:, 0], scores[:, 1],
                         c=np.linalg.norm(scores[:, :2], axis=1),
                         cmap='viridis', alpha=0.6,
                         edgecolors='black', linewidth=0.5)
    ax1.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
    ax1.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
    ax1.set_title('PCA Scores', fontweight='bold')
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax1.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax1, label='Distance')
    
    # Explained variance
    n_components = min(5, len(explained))
    ax2.bar(range(1, n_components+1), explained[:n_components]*100,
            color='steelblue', edgecolor='black')
    ax2.set_xlabel('Principal Component')
    ax2.set_ylabel('Explained Variance (%)')
    ax2.set_title('Explained Variance by Component', fontweight='bold')
    ax2.set_xticks(range(1, n_components+1))
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_distributions(df, cols=None, figsize=(12, 8)):
    """Plot distributions for selected columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:4]
    
    cols = [col for col in cols if col in numeric_df.columns]
    if not cols:
        print("⚠️ No valid columns")
        return
    
    n_cols = min(2, len(cols))
    n_rows = (len(cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1:
        if n_cols == 1:
            axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(cols):
        if i >= len(axes):
            break
        data = numeric_df[col].dropna()
        if len(data) > 0:
            axes[i].hist(data, bins=20, density=True, alpha=0.6,
                        color='steelblue', edgecolor='black')
            axes[i].set_xlabel(col, fontsize=10)
            axes[i].set_ylabel('Density', fontsize=10)
            axes[i].set_title(f'{col} Distribution', fontweight='bold')
            axes[i].axvline(data.mean(), color='red', linestyle='--',
                          label=f'Mean: {data.mean():.2f}')
            axes[i].axvline(data.median(), color='green', linestyle='--',
                          label=f'Median: {data.median():.2f}')
            axes[i].legend(fontsize=8)
            axes[i].grid(True, alpha=0.3)
    
    for i in range(len(cols), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

# ============================================================
# PART 5: COMPLETE WORKFLOW
# ============================================================

def analyze_geochemical_data(file_path, **kwargs):
    """
    Complete workflow: Load → Clean → Analyze → Report
    
    Parameters:
    -----------
    file_path : str
        Path to the geochemical data file
    **kwargs : additional arguments for cleaning
    
    Returns:
    --------
    tuple : (cleaned_df, eda_results)
    """
    print("\n" + "="*80)
    print("🚀 ELEMENTAL SOCIETY ANALYSIS - COMPLETE WORKFLOW")
    print("="*80)
    
    # Step 1: Load data
    df = load_file(file_path)
    if df is None:
        print("❌ Failed to load data")
        return None, None
    
    # Step 2: Clean data
    cleaner = DataCleaner(df)
    cleaned_df = cleaner.clean_all(**kwargs)
    
    # Show cleaning log
    print("\n📝 Cleaning Log:")
    print(cleaner.get_log())
    
    # Step 3: Analyze
    eda = ProfessionalEDA(cleaned_df)
    eda.run_all()
    
    # Step 4: Report
    print("\n" + eda.get_report())
    
    # Step 5: Visualizations
    print("\n📊 GENERATING VISUALIZATIONS...")
    
    # Correlation matrix
    plot_correlation(cleaned_df, title="Correlation Matrix")
    
    # PCA scores
    if 'pca' in eda.results:
        plot_pca_scores(eda.results['pca'])
    
    # Distributions
    plot_distributions(cleaned_df)
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    
    return cleaned_df, eda

# ============================================================
# PART 6: QUICK ANALYSIS FUNCTIONS
# ============================================================

def quick_analysis(df):
    """
    Quick analysis on existing DataFrame
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data to analyze
    
    Returns:
    --------
    tuple : (cleaned_df, eda_results)
    """
    print("\n" + "="*80)
    print("🚀 QUICK ANALYSIS")
    print("="*80)
    
    # Clean
    cleaner = DataCleaner(df)
    cleaned_df = cleaner.clean_all()
    
    # Analyze
    eda = ProfessionalEDA(cleaned_df)
    eda.run_all()
    
    # Report
    print("\n" + eda.get_report())
    
    return cleaned_df, eda

# ============================================================
# PART 7: DEMO WITH SAMPLE DATA (FULLY FIXED)
# ============================================================

def create_sample_data(n_samples=50):
    """Create sample geochemical data for testing"""
    np.random.seed(42)
    
    # Base patterns
    base = np.random.randn(n_samples) * 2
    
    # Create data as lists first (to handle mixed types)
    sample_ids = [f'Sample_{i+1:03d}' for i in range(n_samples)]
    au_data = []
    cu_data = []
    zn_data = []
    pb_data = []
    as_data = []
    lithologies = []
    
    for i in range(n_samples):
        # Au with some < values
        if i in np.random.choice(n_samples, size=5, replace=False):
            au_data.append(f"<{np.random.uniform(0.005, 0.02):.6f}")
        else:
            au_data.append(1.5 + 0.8 * base[i] + 0.2 * np.random.randn())
        
        # Cu with some outliers
        if i in np.random.choice(n_samples, size=3, replace=False):
            cu_data.append(np.random.uniform(200, 500))
        else:
            cu_data.append(30 + 15 * base[i] + 5 * np.random.randn())
        
        # Zn with some missing values
        if i in np.random.choice(n_samples, size=3, replace=False):
            zn_data.append(np.nan)
        else:
            zn_data.append(45 + 5 * base[i] + 10 * np.random.randn())
        
        # Pb
        pb_data.append(15 + 3 * np.random.randn())
        
        # As
        as_data.append(2 + 0.5 * base[i] + 0.3 * np.random.randn())
        
        # Lithology
        lithologies.append(np.random.choice(['Granite', 'Gabbro', 'Sandstone', 'Rhyolite']))
    
    # Create DataFrame
    df = pd.DataFrame({
        'Sample_ID': sample_ids,
        'Au_ppm': au_data,
        'Cu_ppm': cu_data,
        'Zn_ppm': zn_data,
        'Pb_ppm': pb_data,
        'As_ppm': as_data,
        'Lithology': lithologies
    })
    
    return df

def demo_analysis():
    """Run demo analysis on sample data"""
    print("\n" + "="*80)
    print("📊 DEMO ANALYSIS ON SAMPLE DATA")
    print("="*80)
    
    # Create sample data
    df = create_sample_data()
    print(f"\nSample data shape: {df.shape}")
    print("\nFirst 5 rows:")
    print(df.head())
    print("\nData types:")
    print(df.dtypes)
    
    # Run analysis
    cleaned_df, eda = quick_analysis(df)
    
    # Show some results
    print("\n📊 KEY INSIGHTS:")
    print("-" * 40)
    
    if 'strong_correlations' in eda.results:
        print("\nStrong Correlations:")
        for col1, col2, val in eda.results['strong_correlations'][:3]:
            print(f"  {col1} ↔ {col2}: {val:.4f}")
    
    if 'outliers' in eda.results:
        print("\nOutliers per column:")
        for col, count in eda.results['outliers'].items():
            if count > 0:
                print(f"  {col}: {count}")
    
    if 'normality' in eda.results:
        print("\nNormality:")
        for col, data in eda.results['normality'].items():
            status = "Normal" if data['is_normal'] else "Not normal"
            print(f"  {col}: {status}")
    
    return cleaned_df, eda

# ============================================================
# MAIN EXECUTION
# ============================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🧪 ELEMENTAL SOCIETY ANALYSIS - JUPYTER SUITE v2.0")
    print("="*80)
    print("""
📌 AVAILABLE FUNCTIONS:

    1. LOAD DATA:
       df = load_file('your_file.csv')
    
    2. CLEAN DATA:
       cleaner = DataCleaner(df)
       cleaned_df = cleaner.clean_all()
    
    3. ANALYZE DATA:
       eda = ProfessionalEDA(cleaned_df)
       eda.run_all()
    
    4. COMPLETE WORKFLOW:
       cleaned_df, eda = analyze_geochemical_data('your_file.csv')
    
    5. QUICK ANALYSIS:
       cleaned_df, eda = quick_analysis(df)
    
    6. DEMO:
       cleaned_df, eda = demo_analysis()
    
    7. VISUALIZATIONS:
       plot_correlation(df)
       plot_pca_scores(eda.results['pca'])
       plot_distributions(df)
    
    8. GET REPORT:
       report = eda.get_report()
       print(report)
    
    9. ACCESS RESULTS:
       eda.results['basic_stats']    # Statistics
       eda.results['correlation']    # Correlation matrix
       eda.results['pca']            # PCA results
       eda.results['outliers']       # Outliers
       eda.results['normality']      # Normality tests
    """)
    
    # Run demo
    print("\n" + "="*80)
    print("📊 RUNNING DEMO ANALYSIS")
    print("="*80)
    cleaned_df, eda = demo_analysis()

🔬 ELEMENTAL SOCIETY ANALYSIS - JUPYTER EDA SUITE
📌 Project Code: 1010 (Data Cleaning)
👨‍🔬 Developed by: Mohamed Almoufty Abaker

🧪 ELEMENTAL SOCIETY ANALYSIS - JUPYTER SUITE v2.0

📌 AVAILABLE FUNCTIONS:

    1. LOAD DATA:
       df = load_file('your_file.csv')
    
    2. CLEAN DATA:
       cleaner = DataCleaner(df)
       cleaned_df = cleaner.clean_all()
    
    3. ANALYZE DATA:
       eda = ProfessionalEDA(cleaned_df)
       eda.run_all()
    
    4. COMPLETE WORKFLOW:
       cleaned_df, eda = analyze_geochemical_data('your_file.csv')
    
    5. QUICK ANALYSIS:
       cleaned_df, eda = quick_analysis(df)
    
    6. DEMO:
       cleaned_df, eda = demo_analysis()
    
    7. VISUALIZATIONS:
       plot_correlation(df)
       plot_pca_scores(eda.results['pca'])
       plot_distributions(df)
    
    8. GET REPORT:
       report = eda.get_report()
       print(report)
    
    9. ACCESS RESULTS:
       eda.results['basic_stats']    # Statistics
       eda.results['correlation']    # C

In [9]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - JUPYTER COMPLETE VERSION
# Project Code: 1010 (Data Cleaning)
# Developed by: Mohamed Almoufty Abaker - Geologist & Writer
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("🔬 ELEMENTAL SOCIETY ANALYSIS - JUPYTER EDA SUITE")
print("="*80)
print("📌 Project Code: 1010 (Data Cleaning)")
print("👨‍🔬 Developed by: Mohamed Almoufty Abaker")
print("="*80)

# ============================================================
# PART 1: FILE LOADING ENGINE
# ============================================================

def load_file(file_path, **kwargs):
    """
    Universal file loader for geochemical data
    
    Parameters:
    -----------
    file_path : str
        Path to the file
    **kwargs : additional arguments for pd.read_csv or pd.read_excel
    
    Returns:
    --------
    pandas.DataFrame : Loaded data
    """
    print(f"\n📁 LOADING FILE: {file_path}")
    print("-" * 50)
    
    # Check file extension
    file_ext = file_path.split('.')[-1].lower()
    
    # Try different delimiters and encodings for CSV/text files
    if file_ext in ['csv', 'txt', 'dat', 'tsv']:
        delimiters = [';', ',', '\t', '|', ' ']
        encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1']
        
        df = None
        for encoding in encodings:
            for delim in delimiters:
                try:
                    df = pd.read_csv(file_path, sep=delim, encoding=encoding, 
                                   engine='python', on_bad_lines='skip', **kwargs)
                    if df is not None and not df.empty and df.shape[1] > 1:
                        print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                        print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                        return df
                except:
                    continue
        
        # If all fail, try manual parsing
        if df is None or df.empty:
            print("⚠️ Auto-detection failed, trying manual parsing...")
            df = manual_parse_file(file_path)
            if df is not None:
                print(f"✅ Manual parsing successful! Shape: {df.shape}")
                return df
    
    # Excel files
    elif file_ext in ['xlsx', 'xls', 'xlsm']:
        try:
            df = pd.read_excel(file_path, engine='openpyxl', **kwargs)
            print(f"✅ Read as Excel file (openpyxl)")
            print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
            return df
        except:
            try:
                df = pd.read_excel(file_path, engine='xlrd', **kwargs)
                print(f"✅ Read as Excel file (xlrd)")
                print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                return df
            except Exception as e:
                print(f"❌ Excel read failed: {e}")
                return None
    
    else:
        print(f"⚠️ Unknown file type: {file_ext}")
        df = manual_parse_file(file_path)
        if df is not None:
            print(f"✅ Manual parsing successful! Shape: {df.shape}")
            return df
    
    print("❌ Could not read file")
    return None

def manual_parse_file(file_path):
    """Manually parse text files when auto-detection fails"""
    try:
        with open(file_path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    # Detect delimiter
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        
        df = pd.DataFrame(data)
        
        # Check if first row is header
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    
    return None

# ============================================================
# PART 2: DATA CLEANING ENGINE
# ============================================================

class DataCleaner:
    """Professional Data Cleaning Engine for Geochemical Data"""
    
    def __init__(self, df):
        self.df = df.copy()
        self.cleaned_df = None
        self.cleaning_log = []
        print(f"\n🧹 DATA CLEANER INITIALIZED")
        print(f"Original shape: {self.df.shape}")
        print(f"Columns: {len(self.df.columns)}")
        print(f"Rows: {len(self.df)}")
    
    def clean_all(self, handle_missing=True, handle_outliers=True, 
                  handle_detection_limits=True, remove_duplicates=True):
        """
        Run complete cleaning pipeline
        
        Parameters:
        -----------
        handle_missing : bool
            Whether to handle missing values
        handle_outliers : bool
            Whether to handle outliers
        handle_detection_limits : bool
            Whether to handle detection limits (< values)
        remove_duplicates : bool
            Whether to remove duplicate rows
        """
        self.cleaning_log = []
        print("\n🧹 RUNNING CLEANING PIPELINE")
        print("-" * 50)
        
        # 1. Clean column names
        self._clean_column_names()
        
        # 2. Remove duplicates
        if remove_duplicates:
            self._remove_duplicates()
        
        # 3. Handle detection limits
        if handle_detection_limits:
            self._handle_detection_limits()
        
        # 4. Convert to numeric
        self._convert_to_numeric()
        
        # 5. Handle missing values
        if handle_missing:
            self._handle_missing_values()
        
        # 6. Handle outliers
        if handle_outliers:
            self._handle_outliers()
        
        self.cleaned_df = self.df.copy()
        
        print(f"\n✅ Cleaning complete!")
        print(f"Final shape: {self.cleaned_df.shape}")
        print(f"Rows: {len(self.cleaned_df)}")
        print(f"Columns: {len(self.cleaned_df.columns)}")
        
        return self.cleaned_df
    
    def _clean_column_names(self):
        """Clean column names"""
        self.df.columns = self.df.columns.str.strip()
        self.df.columns = self.df.columns.str.replace(' ', '_')
        self.df.columns = self.df.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
        self.df.columns = self.df.columns.str.replace('__', '_', regex=False)
        
        # Make unique
        seen = {}
        new_cols = []
        for col in self.df.columns:
            if col in seen:
                seen[col] += 1
                new_cols.append(f"{col}_{seen[col]}")
            else:
                seen[col] = 0
                new_cols.append(col)
        self.df.columns = new_cols
        
        self.cleaning_log.append("✅ Column names cleaned")
    
    def _remove_duplicates(self):
        """Remove duplicate rows"""
        initial_len = len(self.df)
        self.df = self.df.drop_duplicates()
        removed = initial_len - len(self.df)
        if removed > 0:
            self.cleaning_log.append(f"✅ Removed {removed} duplicate rows")
    
    def _handle_detection_limits(self):
        """Handle detection limits (< values)"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            
            mask = self.df[col].astype(str).str.startswith('<', na=False)
            if mask.sum() > 0:
                for idx in self.df[mask].index:
                    val = str(self.df.loc[idx, col])
                    try:
                        limit = float(val.replace('<', '').strip())
                        self.df.loc[idx, col] = limit / 2  # Half detection limit
                    except:
                        self.df.loc[idx, col] = np.nan
                self.cleaning_log.append(f"✅ Handled {mask.sum()} detection limits in {col}")
    
    def _convert_to_numeric(self):
        """Convert columns to numeric"""
        for col in self.df.columns:
            if col not in ['Sample_ID', 'Lithology', 'Comments']:
                self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
        self.cleaning_log.append("✅ Converted columns to numeric")
    
    def _handle_missing_values(self):
        """Handle missing values"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            missing = self.df[col].isnull().sum()
            if missing > 0:
                self.df[col] = self.df[col].fillna(self.df[col].median())
                self.cleaning_log.append(f"✅ Filled {missing} missing values in {col}")
    
    def _handle_outliers(self, method='iqr'):
        """Handle outliers using IQR method"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            
            outliers = ((self.df[col] < lower) | (self.df[col] > upper)).sum()
            if outliers > 0:
                self.df[col] = self.df[col].clip(lower=lower, upper=upper)
                self.cleaning_log.append(f"✅ Capped {outliers} outliers in {col}")
    
    def get_log(self):
        """Get cleaning log"""
        return "\n".join(self.cleaning_log)

# ============================================================
# PART 3: PROFESSIONAL EDA ENGINE
# ============================================================

class ProfessionalEDA:
    """Comprehensive Exploratory Data Analysis Engine"""
    
    def __init__(self, df):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        print(f"\n📊 EDA ENGINE INITIALIZED")
        print(f"Numeric columns: {len(self.numeric_df.columns)}")
        print(f"Samples: {len(self.df)}")
    
    def run_all(self):
        """Run all analyses"""
        print("\n📊 RUNNING COMPREHENSIVE EDA")
        print("-" * 50)
        
        self.basic_statistics()
        self.variance_analysis()
        self.correlation_analysis()
        self.outlier_detection()
        self.normality_analysis()
        self.perform_pca()
        
        print("\n✅ EDA complete!")
        return self.results
    
    def basic_statistics(self):
        """Calculate basic statistics"""
        self.results['basic_stats'] = {
            'mean': self.numeric_df.mean(),
            'median': self.numeric_df.median(),
            'std': self.numeric_df.std(),
            'min': self.numeric_df.min(),
            'max': self.numeric_df.max(),
            'range': self.numeric_df.max() - self.numeric_df.min(),
            'q25': self.numeric_df.quantile(0.25),
            'q75': self.numeric_df.quantile(0.75),
            'iqr': self.numeric_df.quantile(0.75) - self.numeric_df.quantile(0.25),
            'skew': self.numeric_df.skew(),
            'kurtosis': self.numeric_df.kurtosis(),
            'count': self.numeric_df.count()
        }
        print("  ✅ Basic statistics")
        return self.results['basic_stats']
    
    def variance_analysis(self):
        """Calculate variance metrics"""
        self.results['variance'] = {
            'variance': self.numeric_df.var(),
            'std_dev': self.numeric_df.std(),
            'cv': (self.numeric_df.std() / self.numeric_df.mean()) * 100,
            'mad': self.numeric_df.mad()
        }
        print("  ✅ Variance analysis")
        return self.results['variance']
    
    def correlation_analysis(self):
        """Calculate correlation matrix"""
        if len(self.numeric_df.columns) > 1:
            self.results['correlation'] = self.numeric_df.corr()
            
            # Find strong correlations
            strong = []
            corr = self.results['correlation']
            for i, col1 in enumerate(corr.columns):
                for col2 in corr.columns[i+1:]:
                    val = corr.loc[col1, col2]
                    if abs(val) > 0.5:
                        strong.append((col1, col2, val))
            self.results['strong_correlations'] = sorted(strong, key=lambda x: abs(x[2]), reverse=True)
            print(f"  ✅ Correlation matrix ({len(strong)} strong correlations)")
        return self.results.get('correlation')
    
    def outlier_detection(self):
        """Detect outliers using IQR method"""
        self.results['outliers'] = {}
        for col in self.numeric_df.columns:
            Q1 = self.numeric_df[col].quantile(0.25)
            Q3 = self.numeric_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            self.results['outliers'][col] = ((self.numeric_df[col] < lower) | 
                                            (self.numeric_df[col] > upper)).sum()
        print("  ✅ Outlier detection")
        return self.results['outliers']
    
    def normality_analysis(self):
        """Analyze normality using skewness and kurtosis"""
        self.results['normality'] = {}
        for col in self.numeric_df.columns:
            skew_val = self.numeric_df[col].skew()
            kurt_val = self.numeric_df[col].kurtosis()
            is_normal = (-1 < skew_val < 1) and (-2 < kurt_val < 2)
            self.results['normality'][col] = {
                'skewness': skew_val,
                'kurtosis': kurt_val,
                'is_normal': is_normal
            }
        print("  ✅ Normality analysis")
        return self.results['normality']
    
    def perform_pca(self):
        """Perform Principal Component Analysis"""
        if len(self.numeric_df.columns) >= 2:
            clean_data = self.numeric_df.dropna()
            if len(clean_data) > 1:
                scaler = StandardScaler()
                scaled = scaler.fit_transform(clean_data)
                pca = SklearnPCA()
                scores = pca.fit_transform(scaled)
                self.results['pca'] = {
                    'explained_variance': pca.explained_variance_ratio_,
                    'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
                    'scores': scores,
                    'loadings': pd.DataFrame(
                        pca.components_.T,
                        columns=[f'PC{i+1}' for i in range(pca.n_components_)],
                        index=clean_data.columns
                    ),
                    'n_components': pca.n_components_,
                    'n_samples': len(clean_data)
                }
                print(f"  ✅ PCA ({pca.n_components_} components, "
                      f"{pca.explained_variance_ratio_.sum()*100:.1f}% variance)")
        return self.results.get('pca')
    
    def get_summary(self):
        """Get summary statistics"""
        return {
            'shape': self.df.shape,
            'numeric_cols': len(self.numeric_df.columns),
            'total_cols': len(self.df.columns),
            'basic_stats': self.results.get('basic_stats', {}),
            'variance': self.results.get('variance', {}),
            'strong_correlations': self.results.get('strong_correlations', []),
            'outliers': self.results.get('outliers', {}),
            'normality': self.results.get('normality', {})
        }
    
    def get_report(self):
        """Generate comprehensive report"""
        report = []
        report.append("="*80)
        report.append("📊 ELEMENTAL SOCIETY ANALYSIS - EDA REPORT")
        report.append("="*80)
        
        # Dataset info
        report.append(f"\n📋 Dataset: {len(self.df)} rows × {len(self.df.columns)} columns")
        report.append(f"📊 Numeric columns: {len(self.numeric_df.columns)}")
        
        # Basic stats
        report.append("\n📊 BASIC STATISTICS")
        report.append("-"*40)
        stats_df = pd.DataFrame(self.results.get('basic_stats', {}))
        if not stats_df.empty:
            report.append(stats_df.round(4).to_string())
        
        # Variance
        report.append("\n📈 VARIANCE")
        report.append("-"*40)
        var_df = pd.DataFrame(self.results.get('variance', {}))
        if not var_df.empty:
            report.append(var_df.round(4).to_string())
        
        # Correlations
        if 'strong_correlations' in self.results:
            report.append("\n📉 STRONG CORRELATIONS (>0.5)")
            report.append("-"*40)
            for col1, col2, val in self.results['strong_correlations'][:5]:
                direction = "positive" if val > 0 else "negative"
                report.append(f"  {col1} ↔ {col2}: {val:.4f} ({direction})")
        
        # Outliers
        report.append("\n🎯 OUTLIERS")
        report.append("-"*40)
        for col, count in self.results.get('outliers', {}).items():
            report.append(f"  {col}: {count} outliers")
        
        # Normality
        report.append("\n📊 NORMALITY")
        report.append("-"*40)
        for col, data in self.results.get('normality', {}).items():
            status = "✅ Normal" if data['is_normal'] else "❌ Not normal"
            report.append(f"  {col}: {status} (skew={data['skewness']:.4f})")
        
        # PCA
        if 'pca' in self.results:
            report.append("\n📉 PCA")
            report.append("-"*40)
            explained = self.results['pca']['explained_variance']
            cum_var = np.cumsum(explained)
            report.append(f"Total explained variance: {explained.sum()*100:.1f}%")
            for i, (ev, cum) in enumerate(zip(explained[:5], cum_var[:5])):
                report.append(f"  PC{i+1}: {ev*100:.1f}% (cumulative: {cum*100:.1f}%)")
            report.append("\nLoadings (PC1):")
            loadings = self.results['pca']['loadings']['PC1'].abs().sort_values(ascending=False)
            for var, loading in list(loadings.items())[:5]:
                report.append(f"  {var}: {loading:.4f}")
        
        report.append("\n"+"="*80)
        report.append(f"📌 Report generated by Elemental Society Analysis v2.0")
        report.append("="*80)
        
        return "\n".join(report)

# ============================================================
# PART 4: VISUALIZATION FUNCTIONS
# ============================================================

def plot_correlation(df, title="Correlation Matrix", figsize=(10, 8)):
    """Plot correlation matrix heatmap"""
    numeric_df = df.select_dtypes(include=[np.number])
    if len(numeric_df.columns) < 2:
        print("⚠️ Need at least 2 numeric columns")
        return
    
    plt.figure(figsize=figsize)
    corr = numeric_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', center=0,
                fmt='.2f', square=True, linewidths=0.5)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_pca_scores(pca_results, figsize=(10, 6)):
    """Plot PCA scores"""
    if pca_results is None:
        print("⚠️ No PCA results available")
        return
    
    scores = pca_results['scores']
    explained = pca_results['explained_variance']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # PCA scores
    scatter = ax1.scatter(scores[:, 0], scores[:, 1],
                         c=np.linalg.norm(scores[:, :2], axis=1),
                         cmap='viridis', alpha=0.6,
                         edgecolors='black', linewidth=0.5)
    ax1.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
    ax1.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
    ax1.set_title('PCA Scores', fontweight='bold')
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax1.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax1, label='Distance')
    
    # Explained variance
    n_components = min(5, len(explained))
    ax2.bar(range(1, n_components+1), explained[:n_components]*100,
            color='steelblue', edgecolor='black')
    ax2.set_xlabel('Principal Component')
    ax2.set_ylabel('Explained Variance (%)')
    ax2.set_title('Explained Variance by Component', fontweight='bold')
    ax2.set_xticks(range(1, n_components+1))
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_distributions(df, cols=None, figsize=(12, 8)):
    """Plot distributions for selected columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:4]
    
    cols = [col for col in cols if col in numeric_df.columns]
    if not cols:
        print("⚠️ No valid columns")
        return
    
    n_cols = min(2, len(cols))
    n_rows = (len(cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1:
        if n_cols == 1:
            axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(cols):
        if i >= len(axes):
            break
        data = numeric_df[col].dropna()
        if len(data) > 0:
            axes[i].hist(data, bins=20, density=True, alpha=0.6,
                        color='steelblue', edgecolor='black')
            axes[i].set_xlabel(col, fontsize=10)
            axes[i].set_ylabel('Density', fontsize=10)
            axes[i].set_title(f'{col} Distribution', fontweight='bold')
            axes[i].axvline(data.mean(), color='red', linestyle='--',
                          label=f'Mean: {data.mean():.2f}')
            axes[i].axvline(data.median(), color='green', linestyle='--',
                          label=f'Median: {data.median():.2f}')
            axes[i].legend(fontsize=8)
            axes[i].grid(True, alpha=0.3)
    
    for i in range(len(cols), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

# ============================================================
# PART 5: COMPLETE WORKFLOW
# ============================================================

def analyze_geochemical_data(file_path, **kwargs):
    """
    Complete workflow: Load → Clean → Analyze → Report
    
    Parameters:
    -----------
    file_path : str
        Path to the geochemical data file
    **kwargs : additional arguments for cleaning
    
    Returns:
    --------
    tuple : (cleaned_df, eda_results)
    """
    print("\n" + "="*80)
    print("🚀 ELEMENTAL SOCIETY ANALYSIS - COMPLETE WORKFLOW")
    print("="*80)
    
    # Step 1: Load data
    df = load_file(file_path)
    if df is None:
        print("❌ Failed to load data")
        return None, None
    
    # Step 2: Clean data
    cleaner = DataCleaner(df)
    cleaned_df = cleaner.clean_all(**kwargs)
    
    # Show cleaning log
    print("\n📝 Cleaning Log:")
    print(cleaner.get_log())
    
    # Step 3: Analyze
    eda = ProfessionalEDA(cleaned_df)
    eda.run_all()
    
    # Step 4: Report
    print("\n" + eda.get_report())
    
    # Step 5: Visualizations
    print("\n📊 GENERATING VISUALIZATIONS...")
    
    # Correlation matrix
    plot_correlation(cleaned_df, title="Correlation Matrix")
    
    # PCA scores
    if 'pca' in eda.results:
        plot_pca_scores(eda.results['pca'])
    
    # Distributions
    plot_distributions(cleaned_df)
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    
    return cleaned_df, eda

# ============================================================
# PART 6: QUICK ANALYSIS FUNCTIONS
# ============================================================

def quick_analysis(df):
    """
    Quick analysis on existing DataFrame
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data to analyze
    
    Returns:
    --------
    tuple : (cleaned_df, eda_results)
    """
    print("\n" + "="*80)
    print("🚀 QUICK ANALYSIS")
    print("="*80)
    
    # Clean
    cleaner = DataCleaner(df)
    cleaned_df = cleaner.clean_all()
    
    # Analyze
    eda = ProfessionalEDA(cleaned_df)
    eda.run_all()
    
    # Report
    print("\n" + eda.get_report())
    
    return cleaned_df, eda

# ============================================================
# PART 7: DEMO WITH SAMPLE DATA
# ============================================================

def create_sample_data(n_samples=50):
    """Create sample geochemical data for testing"""
    np.random.seed(42)
    
    # Base patterns
    base = np.random.randn(n_samples) * 2
    
    data = {
        'Sample_ID': [f'Sample_{i+1:03d}' for i in range(n_samples)],
        'Au_ppm': 1.5 + 0.8 * base + 0.2 * np.random.randn(n_samples),
        'Cu_ppm': 30 + 15 * base + 5 * np.random.randn(n_samples),
        'Zn_ppm': 45 + 5 * base + 10 * np.random.randn(n_samples),
        'Pb_ppm': 15 + 3 * np.random.randn(n_samples),
        'As_ppm': 2 + 0.5 * base + 0.3 * np.random.randn(n_samples)
    }
    
    # Add some < values (detection limits)
    for i in np.random.choice(n_samples, size=5, replace=False):
        data['Au_ppm'][i] = f"<{np.random.uniform(0.005, 0.02):.6f}"
    
    # Add some outliers
    for i in np.random.choice(n_samples, size=3, replace=False):
        data['Cu_ppm'][i] = np.random.uniform(200, 500)
    
    # Add some missing values
    for i in np.random.choice(n_samples, size=3, replace=False):
        data['Zn_ppm'][i] = np.nan
    
    # Add categorical column
    lithologies = ['Granite', 'Gabbro', 'Sandstone', 'Rhyolite']
    data['Lithology'] = np.random.choice(lithologies, n_samples)
    
    return pd.DataFrame(data)

def demo_analysis():
    """Run demo analysis on sample data"""
    print("\n" + "="*80)
    print("📊 DEMO ANALYSIS ON SAMPLE DATA")
    print("="*80)
    
    # Create sample data
    df = create_sample_data()
    print(f"\nSample data shape: {df.shape}")
    print("\nFirst 5 rows:")
    print(df.head())
    
    # Run analysis
    cleaned_df, eda = quick_analysis(df)
    
    return cleaned_df, eda

# ============================================================
# MAIN EXECUTION
# ============================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🧪 ELEMENTAL SOCIETY ANALYSIS - JUPYTER SUITE v2.0")
    print("="*80)
    print("""
📌 AVAILABLE FUNCTIONS:

    1. LOAD DATA:
       df = load_file('your_file.csv')
    
    2. CLEAN DATA:
       cleaner = DataCleaner(df)
       cleaned_df = cleaner.clean_all()
    
    3. ANALYZE DATA:
       eda = ProfessionalEDA(cleaned_df)
       eda.run_all()
    
    4. COMPLETE WORKFLOW:
       cleaned_df, eda = analyze_geochemical_data('your_file.csv')
    
    5. QUICK ANALYSIS:
       cleaned_df, eda = quick_analysis(df)
    
    6. DEMO:
       cleaned_df, eda = demo_analysis()
    
    7. VISUALIZATIONS:
       plot_correlation(df)
       plot_pca_scores(eda.results['pca'])
       plot_distributions(df)
    
    8. GET REPORT:
       report = eda.get_report()
       print(report)
    
    9. ACCESS RESULTS:
       eda.results['basic_stats']    # Statistics
       eda.results['correlation']    # Correlation matrix
       eda.results['pca']            # PCA results
       eda.results['outliers']       # Outliers
       eda.results['normality']      # Normality tests
    """)
    
    # Run demo
    print("\n" + "="*80)
    print("📊 RUNNING DEMO ANALYSIS")
    print("="*80)
    demo_analysis()

🔬 ELEMENTAL SOCIETY ANALYSIS - JUPYTER EDA SUITE
📌 Project Code: 1010 (Data Cleaning)
👨‍🔬 Developed by: Mohamed Almoufty Abaker

🧪 ELEMENTAL SOCIETY ANALYSIS - JUPYTER SUITE v2.0

📌 AVAILABLE FUNCTIONS:

    1. LOAD DATA:
       df = load_file('your_file.csv')
    
    2. CLEAN DATA:
       cleaner = DataCleaner(df)
       cleaned_df = cleaner.clean_all()
    
    3. ANALYZE DATA:
       eda = ProfessionalEDA(cleaned_df)
       eda.run_all()
    
    4. COMPLETE WORKFLOW:
       cleaned_df, eda = analyze_geochemical_data('your_file.csv')
    
    5. QUICK ANALYSIS:
       cleaned_df, eda = quick_analysis(df)
    
    6. DEMO:
       cleaned_df, eda = demo_analysis()
    
    7. VISUALIZATIONS:
       plot_correlation(df)
       plot_pca_scores(eda.results['pca'])
       plot_distributions(df)
    
    8. GET REPORT:
       report = eda.get_report()
       print(report)
    
    9. ACCESS RESULTS:
       eda.results['basic_stats']    # Statistics
       eda.results['correlation']    # C

ValueError: could not convert string to float: '<0.019193'

In [15]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - JUPYTER COMPLETE VERSION (FULLY FIXED)
# Project Code: 1010 (Data Cleaning)
# Developed by: Mohamed Almoufty Abaker - Geologist & Writer
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("🔬 ELEMENTAL SOCIETY ANALYSIS - JUPYTER EDA SUITE")
print("="*80)
print("📌 Project Code: 1010 (Data Cleaning)")
print("👨‍🔬 Developed by: Mohamed Almoufty Abaker")
print("="*80)

# ============================================================
# PART 1: FILE LOADING ENGINE
# ============================================================

def load_file(file_path, **kwargs):
    """
    Universal file loader for geochemical data
    
    Parameters:
    -----------
    file_path : str
        Path to the file
    **kwargs : additional arguments for pd.read_csv or pd.read_excel
    
    Returns:
    --------
    pandas.DataFrame : Loaded data
    """
    print(f"\n📁 LOADING FILE: {file_path}")
    print("-" * 50)
    
    # Check file extension
    file_ext = file_path.split('.')[-1].lower()
    
    # Try different delimiters and encodings for CSV/text files
    if file_ext in ['csv', 'txt', 'dat', 'tsv']:
        delimiters = [';', ',', '\t', '|', ' ']
        encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1']
        
        df = None
        for encoding in encodings:
            for delim in delimiters:
                try:
                    df = pd.read_csv(file_path, sep=delim, encoding=encoding, 
                                   engine='python', on_bad_lines='skip', **kwargs)
                    if df is not None and not df.empty and df.shape[1] > 1:
                        print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                        print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                        return df
                except:
                    continue
        
        # If all fail, try manual parsing
        if df is None or df.empty:
            print("⚠️ Auto-detection failed, trying manual parsing...")
            df = manual_parse_file(file_path)
            if df is not None:
                print(f"✅ Manual parsing successful! Shape: {df.shape}")
                return df
    
    # Excel files
    elif file_ext in ['xlsx', 'xls', 'xlsm']:
        try:
            df = pd.read_excel(file_path, engine='openpyxl', **kwargs)
            print(f"✅ Read as Excel file (openpyxl)")
            print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
            return df
        except:
            try:
                df = pd.read_excel(file_path, engine='xlrd', **kwargs)
                print(f"✅ Read as Excel file (xlrd)")
                print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                return df
            except Exception as e:
                print(f"❌ Excel read failed: {e}")
                return None
    
    else:
        print(f"⚠️ Unknown file type: {file_ext}")
        df = manual_parse_file(file_path)
        if df is not None:
            print(f"✅ Manual parsing successful! Shape: {df.shape}")
            return df
    
    print("❌ Could not read file")
    return None

def manual_parse_file(file_path):
    """Manually parse text files when auto-detection fails"""
    try:
        with open(file_path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    # Detect delimiter
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        
        df = pd.DataFrame(data)
        
        # Check if first row is header
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    
    return None

# ============================================================
# PART 2: DATA CLEANING ENGINE
# ============================================================

class DataCleaner:
    """Professional Data Cleaning Engine for Geochemical Data"""
    
    def __init__(self, df):
        self.df = df.copy()
        self.cleaned_df = None
        self.cleaning_log = []
        print(f"\n🧹 DATA CLEANER INITIALIZED")
        print(f"Original shape: {self.df.shape}")
        print(f"Columns: {len(self.df.columns)}")
        print(f"Rows: {len(self.df)}")
    
    def clean_all(self, handle_missing=True, handle_outliers=True, 
                  handle_detection_limits=True, remove_duplicates=True):
        """
        Run complete cleaning pipeline
        """
        self.cleaning_log = []
        print("\n🧹 RUNNING CLEANING PIPELINE")
        print("-" * 50)
        
        # 1. Clean column names
        self._clean_column_names()
        
        # 2. Handle detection limits BEFORE converting to numeric
        if handle_detection_limits:
            self._handle_detection_limits()
        
        # 3. Convert to numeric
        self._convert_to_numeric()
        
        # 4. Remove duplicates
        if remove_duplicates:
            self._remove_duplicates()
        
        # 5. Handle missing values
        if handle_missing:
            self._handle_missing_values()
        
        # 6. Handle outliers
        if handle_outliers:
            self._handle_outliers()
        
        self.cleaned_df = self.df.copy()
        
        print(f"\n✅ Cleaning complete!")
        print(f"Final shape: {self.cleaned_df.shape}")
        print(f"Rows: {len(self.cleaned_df)}")
        print(f"Columns: {len(self.cleaned_df.columns)}")
        
        return self.cleaned_df
    
    def _clean_column_names(self):
        """Clean column names"""
        self.df.columns = self.df.columns.str.strip()
        self.df.columns = self.df.columns.str.replace(' ', '_')
        self.df.columns = self.df.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
        self.df.columns = self.df.columns.str.replace('__', '_', regex=False)
        
        # Make unique
        seen = {}
        new_cols = []
        for col in self.df.columns:
            if col in seen:
                seen[col] += 1
                new_cols.append(f"{col}_{seen[col]}")
            else:
                seen[col] = 0
                new_cols.append(col)
        self.df.columns = new_cols
        
        self.cleaning_log.append("✅ Column names cleaned")
    
    def _handle_detection_limits(self):
        """Handle detection limits (< values) - BEFORE converting to numeric"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            
            # Check if column contains < values
            mask = self.df[col].astype(str).str.startswith('<', na=False)
            if mask.sum() > 0:
                for idx in self.df[mask].index:
                    val = str(self.df.loc[idx, col])
                    try:
                        # Extract number after <
                        limit = float(val.replace('<', '').strip())
                        # Replace with half detection limit
                        self.df.loc[idx, col] = limit / 2
                    except:
                        self.df.loc[idx, col] = np.nan
                self.cleaning_log.append(f"✅ Handled {mask.sum()} detection limits in {col}")
    
    def _convert_to_numeric(self):
        """Convert columns to numeric"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
        self.cleaning_log.append("✅ Converted columns to numeric")
    
    def _remove_duplicates(self):
        """Remove duplicate rows"""
        initial_len = len(self.df)
        self.df = self.df.drop_duplicates()
        removed = initial_len - len(self.df)
        if removed > 0:
            self.cleaning_log.append(f"✅ Removed {removed} duplicate rows")
    
    def _handle_missing_values(self):
        """Handle missing values"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            missing = self.df[col].isnull().sum()
            if missing > 0:
                if self.df[col].dtype in ['float64', 'int64']:
                    self.df[col] = self.df[col].fillna(self.df[col].median())
                    self.cleaning_log.append(f"✅ Filled {missing} missing values in {col}")
    
    def _handle_outliers(self, method='iqr'):
        """Handle outliers using IQR method"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            
            if self.df[col].dtype not in ['float64', 'int64']:
                continue
            
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            
            outliers = ((self.df[col] < lower) | (self.df[col] > upper)).sum()
            if outliers > 0:
                self.df[col] = self.df[col].clip(lower=lower, upper=upper)
                self.cleaning_log.append(f"✅ Capped {outliers} outliers in {col}")
    
    def get_log(self):
        """Get cleaning log"""
        return "\n".join(self.cleaning_log)

# ============================================================
# PART 3: PROFESSIONAL EDA ENGINE
# ============================================================

class ProfessionalEDA:
    """Comprehensive Exploratory Data Analysis Engine"""
    
    def __init__(self, df):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        print(f"\n📊 EDA ENGINE INITIALIZED")
        print(f"Numeric columns: {len(self.numeric_df.columns)}")
        print(f"Samples: {len(self.df)}")
    
    def run_all(self):
        """Run all analyses"""
        print("\n📊 RUNNING COMPREHENSIVE EDA")
        print("-" * 50)
        
        self.basic_statistics()
        self.variance_analysis()
        self.correlation_analysis()
        self.outlier_detection()
        self.normality_analysis()
        self.perform_pca()
        
        print("\n✅ EDA complete!")
        return self.results
    
    def basic_statistics(self):
        """Calculate basic statistics"""
        self.results['basic_stats'] = {
            'mean': self.numeric_df.mean(),
            'median': self.numeric_df.median(),
            'std': self.numeric_df.std(),
            'min': self.numeric_df.min(),
            'max': self.numeric_df.max(),
            'range': self.numeric_df.max() - self.numeric_df.min(),
            'q25': self.numeric_df.quantile(0.25),
            'q75': self.numeric_df.quantile(0.75),
            'iqr': self.numeric_df.quantile(0.75) - self.numeric_df.quantile(0.25),
            'skew': self.numeric_df.skew(),
            'kurtosis': self.numeric_df.kurtosis(),
            'count': self.numeric_df.count()
        }
        print("  ✅ Basic statistics")
        return self.results['basic_stats']
    
    def variance_analysis(self):
        """Calculate variance metrics"""
        self.results['variance'] = {
            'variance': self.numeric_df.var(),
            'std_dev': self.numeric_df.std(),
            'cv': (self.numeric_df.std() / self.numeric_df.mean()) * 100
        }
        print("  ✅ Variance analysis")
        return self.results['variance']
    
    def correlation_analysis(self):
        """Calculate correlation matrix"""
        if len(self.numeric_df.columns) > 1:
            self.results['correlation'] = self.numeric_df.corr()
            
            # Find strong correlations
            strong = []
            corr = self.results['correlation']
            for i, col1 in enumerate(corr.columns):
                for col2 in corr.columns[i+1:]:
                    val = corr.loc[col1, col2]
                    if abs(val) > 0.5:
                        strong.append((col1, col2, val))
            self.results['strong_correlations'] = sorted(strong, key=lambda x: abs(x[2]), reverse=True)
            print(f"  ✅ Correlation matrix ({len(strong)} strong correlations)")
        return self.results.get('correlation')
    
    def outlier_detection(self):
        """Detect outliers using IQR method"""
        self.results['outliers'] = {}
        for col in self.numeric_df.columns:
            Q1 = self.numeric_df[col].quantile(0.25)
            Q3 = self.numeric_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            self.results['outliers'][col] = ((self.numeric_df[col] < lower) | 
                                            (self.numeric_df[col] > upper)).sum()
        print("  ✅ Outlier detection")
        return self.results['outliers']
    
    def normality_analysis(self):
        """Analyze normality using skewness and kurtosis"""
        self.results['normality'] = {}
        for col in self.numeric_df.columns:
            skew_val = self.numeric_df[col].skew()
            kurt_val = self.numeric_df[col].kurtosis()
            is_normal = (-1 < skew_val < 1) and (-2 < kurt_val < 2)
            self.results['normality'][col] = {
                'skewness': skew_val,
                'kurtosis': kurt_val,
                'is_normal': is_normal
            }
        print("  ✅ Normality analysis")
        return self.results['normality']
    
    def perform_pca(self):
        """Perform Principal Component Analysis"""
        if len(self.numeric_df.columns) >= 2:
            clean_data = self.numeric_df.dropna()
            if len(clean_data) > 1:
                scaler = StandardScaler()
                scaled = scaler.fit_transform(clean_data)
                pca = SklearnPCA()
                scores = pca.fit_transform(scaled)
                self.results['pca'] = {
                    'explained_variance': pca.explained_variance_ratio_,
                    'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
                    'scores': scores,
                    'loadings': pd.DataFrame(
                        pca.components_.T,
                        columns=[f'PC{i+1}' for i in range(pca.n_components_)],
                        index=clean_data.columns
                    ),
                    'n_components': pca.n_components_,
                    'n_samples': len(clean_data)
                }
                print(f"  ✅ PCA ({pca.n_components_} components, "
                      f"{pca.explained_variance_ratio_.sum()*100:.1f}% variance)")
        return self.results.get('pca')
    
    def get_summary(self):
        """Get summary statistics"""
        return {
            'shape': self.df.shape,
            'numeric_cols': len(self.numeric_df.columns),
            'total_cols': len(self.df.columns),
            'basic_stats': self.results.get('basic_stats', {}),
            'variance': self.results.get('variance', {}),
            'strong_correlations': self.results.get('strong_correlations', []),
            'outliers': self.results.get('outliers', {}),
            'normality': self.results.get('normality', {})
        }
    
    def get_report(self):
        """Generate comprehensive report"""
        report = []
        report.append("="*80)
        report.append("📊 ELEMENTAL SOCIETY ANALYSIS - EDA REPORT")
        report.append("="*80)
        
        # Dataset info
        report.append(f"\n📋 Dataset: {len(self.df)} rows × {len(self.df.columns)} columns")
        report.append(f"📊 Numeric columns: {len(self.numeric_df.columns)}")
        
        # Basic stats
        report.append("\n📊 BASIC STATISTICS")
        report.append("-"*40)
        stats_df = pd.DataFrame(self.results.get('basic_stats', {}))
        if not stats_df.empty:
            report.append(stats_df.round(4).to_string())
        
        # Variance
        report.append("\n📈 VARIANCE")
        report.append("-"*40)
        var_df = pd.DataFrame(self.results.get('variance', {}))
        if not var_df.empty:
            report.append(var_df.round(4).to_string())
        
        # Correlations
        if 'strong_correlations' in self.results:
            report.append("\n📉 STRONG CORRELATIONS (>0.5)")
            report.append("-"*40)
            for col1, col2, val in self.results['strong_correlations'][:5]:
                direction = "positive" if val > 0 else "negative"
                report.append(f"  {col1} ↔ {col2}: {val:.4f} ({direction})")
        
        # Outliers
        report.append("\n🎯 OUTLIERS")
        report.append("-"*40)
        for col, count in self.results.get('outliers', {}).items():
            report.append(f"  {col}: {count} outliers")
        
        # Normality
        report.append("\n📊 NORMALITY")
        report.append("-"*40)
        for col, data in self.results.get('normality', {}).items():
            status = "✅ Normal" if data['is_normal'] else "❌ Not normal"
            report.append(f"  {col}: {status} (skew={data['skewness']:.4f})")
        
        # PCA
        if 'pca' in self.results:
            report.append("\n📉 PCA")
            report.append("-"*40)
            explained = self.results['pca']['explained_variance']
            cum_var = np.cumsum(explained)
            report.append(f"Total explained variance: {explained.sum()*100:.1f}%")
            for i, (ev, cum) in enumerate(zip(explained[:5], cum_var[:5])):
                report.append(f"  PC{i+1}: {ev*100:.1f}% (cumulative: {cum*100:.1f}%)")
            report.append("\nLoadings (PC1):")
            loadings = self.results['pca']['loadings']['PC1'].abs().sort_values(ascending=False)
            for var, loading in list(loadings.items())[:5]:
                report.append(f"  {var}: {loading:.4f}")
        
        report.append("\n"+"="*80)
        report.append(f"📌 Report generated by Elemental Society Analysis v2.0")
        report.append("="*80)
        
        return "\n".join(report)

# ============================================================
# PART 4: VISUALIZATION FUNCTIONS
# ============================================================

def plot_correlation(df, title="Correlation Matrix", figsize=(10, 8)):
    """Plot correlation matrix heatmap"""
    numeric_df = df.select_dtypes(include=[np.number])
    if len(numeric_df.columns) < 2:
        print("⚠️ Need at least 2 numeric columns")
        return
    
    plt.figure(figsize=figsize)
    corr = numeric_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', center=0,
                fmt='.2f', square=True, linewidths=0.5)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_pca_scores(pca_results, figsize=(10, 6)):
    """Plot PCA scores"""
    if pca_results is None:
        print("⚠️ No PCA results available")
        return
    
    scores = pca_results['scores']
    explained = pca_results['explained_variance']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # PCA scores
    scatter = ax1.scatter(scores[:, 0], scores[:, 1],
                         c=np.linalg.norm(scores[:, :2], axis=1),
                         cmap='viridis', alpha=0.6,
                         edgecolors='black', linewidth=0.5)
    ax1.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
    ax1.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
    ax1.set_title('PCA Scores', fontweight='bold')
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax1.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax1, label='Distance')
    
    # Explained variance
    n_components = min(5, len(explained))
    ax2.bar(range(1, n_components+1), explained[:n_components]*100,
            color='steelblue', edgecolor='black')
    ax2.set_xlabel('Principal Component')
    ax2.set_ylabel('Explained Variance (%)')
    ax2.set_title('Explained Variance by Component', fontweight='bold')
    ax2.set_xticks(range(1, n_components+1))
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_distributions(df, cols=None, figsize=(12, 8)):
    """Plot distributions for selected columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:4]
    
    cols = [col for col in cols if col in numeric_df.columns]
    if not cols:
        print("⚠️ No valid columns")
        return
    
    n_cols = min(2, len(cols))
    n_rows = (len(cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1:
        if n_cols == 1:
            axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(cols):
        if i >= len(axes):
            break
        data = numeric_df[col].dropna()
        if len(data) > 0:
            axes[i].hist(data, bins=20, density=True, alpha=0.6,
                        color='steelblue', edgecolor='black')
            axes[i].set_xlabel(col, fontsize=10)
            axes[i].set_ylabel('Density', fontsize=10)
            axes[i].set_title(f'{col} Distribution', fontweight='bold')
            axes[i].axvline(data.mean(), color='red', linestyle='--',
                          label=f'Mean: {data.mean():.2f}')
            axes[i].axvline(data.median(), color='green', linestyle='--',
                          label=f'Median: {data.median():.2f}')
            axes[i].legend(fontsize=8)
            axes[i].grid(True, alpha=0.3)
    
    for i in range(len(cols), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

# ============================================================
# PART 5: COMPLETE WORKFLOW
# ============================================================

def analyze_geochemical_data(file_path, **kwargs):
    """
    Complete workflow: Load → Clean → Analyze → Report
    
    Parameters:
    -----------
    file_path : str
        Path to the geochemical data file
    **kwargs : additional arguments for cleaning
    
    Returns:
    --------
    tuple : (cleaned_df, eda_results)
    """
    print("\n" + "="*80)
    print("🚀 ELEMENTAL SOCIETY ANALYSIS - COMPLETE WORKFLOW")
    print("="*80)
    
    # Step 1: Load data
    df = load_file(file_path)
    if df is None:
        print("❌ Failed to load data")
        return None, None
    
    # Step 2: Clean data
    cleaner = DataCleaner(df)
    cleaned_df = cleaner.clean_all(**kwargs)
    
    # Show cleaning log
    print("\n📝 Cleaning Log:")
    print(cleaner.get_log())
    
    # Step 3: Analyze
    eda = ProfessionalEDA(cleaned_df)
    eda.run_all()
    
    # Step 4: Report
    print("\n" + eda.get_report())
    
    # Step 5: Visualizations
    print("\n📊 GENERATING VISUALIZATIONS...")
    
    # Correlation matrix
    plot_correlation(cleaned_df, title="Correlation Matrix")
    
    # PCA scores
    if 'pca' in eda.results:
        plot_pca_scores(eda.results['pca'])
    
    # Distributions
    plot_distributions(cleaned_df)
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    
    return cleaned_df, eda

# ============================================================
# PART 6: QUICK ANALYSIS FUNCTIONS
# ============================================================

def quick_analysis(df):
    """
    Quick analysis on existing DataFrame
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data to analyze
    
    Returns:
    --------
    tuple : (cleaned_df, eda_results)
    """
    print("\n" + "="*80)
    print("🚀 QUICK ANALYSIS")
    print("="*80)
    
    # Clean
    cleaner = DataCleaner(df)
    cleaned_df = cleaner.clean_all()
    
    # Analyze
    eda = ProfessionalEDA(cleaned_df)
    eda.run_all()
    
    # Report
    print("\n" + eda.get_report())
    
    return cleaned_df, eda

# ============================================================
# PART 7: DEMO WITH SAMPLE DATA (FULLY FIXED)
# ============================================================

def create_sample_data(n_samples=50):
    """Create sample geochemical data for testing"""
    np.random.seed(42)
    
    # Base patterns
    base = np.random.randn(n_samples) * 2
    
    # Create data as lists first (to handle mixed types)
    sample_ids = [f'Sample_{i+1:03d}' for i in range(n_samples)]
    au_data = []
    cu_data = []
    zn_data = []
    pb_data = []
    as_data = []
    lithologies = []
    
    for i in range(n_samples):
        # Au with some < values
        if i in np.random.choice(n_samples, size=5, replace=False):
            au_data.append(f"<{np.random.uniform(0.005, 0.02):.6f}")
        else:
            au_data.append(1.5 + 0.8 * base[i] + 0.2 * np.random.randn())
        
        # Cu with some outliers
        if i in np.random.choice(n_samples, size=3, replace=False):
            cu_data.append(np.random.uniform(200, 500))
        else:
            cu_data.append(30 + 15 * base[i] + 5 * np.random.randn())
        
        # Zn with some missing values
        if i in np.random.choice(n_samples, size=3, replace=False):
            zn_data.append(np.nan)
        else:
            zn_data.append(45 + 5 * base[i] + 10 * np.random.randn())
        
        # Pb
        pb_data.append(15 + 3 * np.random.randn())
        
        # As
        as_data.append(2 + 0.5 * base[i] + 0.3 * np.random.randn())
        
        # Lithology
        lithologies.append(np.random.choice(['Granite', 'Gabbro', 'Sandstone', 'Rhyolite']))
    
    # Create DataFrame
    df = pd.DataFrame({
        'Sample_ID': sample_ids,
        'Au_ppm': au_data,
        'Cu_ppm': cu_data,
        'Zn_ppm': zn_data,
        'Pb_ppm': pb_data,
        'As_ppm': as_data,
        'Lithology': lithologies
    })
    
    return df

def demo_analysis():
    """Run demo analysis on sample data"""
    print("\n" + "="*80)
    print("📊 DEMO ANALYSIS ON SAMPLE DATA")
    print("="*80)
    
    # Create sample data
    df = create_sample_data()
    print(f"\nSample data shape: {df.shape}")
    print("\nFirst 5 rows:")
    print(df.head())
    print("\nData types:")
    print(df.dtypes)
    
    # Run analysis
    cleaned_df, eda = quick_analysis(df)
    
    # Show some results
    print("\n📊 KEY INSIGHTS:")
    print("-" * 40)
    
    if 'strong_correlations' in eda.results:
        print("\nStrong Correlations:")
        for col1, col2, val in eda.results['strong_correlations'][:3]:
            print(f"  {col1} ↔ {col2}: {val:.4f}")
    
    if 'outliers' in eda.results:
        print("\nOutliers per column:")
        for col, count in eda.results['outliers'].items():
            if count > 0:
                print(f"  {col}: {count}")
    
    if 'normality' in eda.results:
        print("\nNormality:")
        for col, data in eda.results['normality'].items():
            status = "Normal" if data['is_normal'] else "Not normal"
            print(f"  {col}: {status}")
    
    return cleaned_df, eda

# ============================================================
# MAIN EXECUTION
# ============================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🧪 ELEMENTAL SOCIETY ANALYSIS - JUPYTER SUITE v2.0")
    print("="*80)
    print("""
📌 AVAILABLE FUNCTIONS:

    1. LOAD DATA:
       df = load_file('your_file.csv')
    
    2. CLEAN DATA:
       cleaner = DataCleaner(df)
       cleaned_df = cleaner.clean_all()
    
    3. ANALYZE DATA:
       eda = ProfessionalEDA(cleaned_df)
       eda.run_all()
    
    4. COMPLETE WORKFLOW:
       cleaned_df, eda = analyze_geochemical_data('your_file.csv')
    
    5. QUICK ANALYSIS:
       cleaned_df, eda = quick_analysis(df)
    
    6. DEMO:
       cleaned_df, eda = demo_analysis()
    
    7. VISUALIZATIONS:
       plot_correlation(df)
       plot_pca_scores(eda.results['pca'])
       plot_distributions(df)
    
    8. GET REPORT:
       report = eda.get_report()
       print(report)
    
    9. ACCESS RESULTS:
       eda.results['basic_stats']    # Statistics
       eda.results['correlation']    # Correlation matrix
       eda.results['pca']            # PCA results
       eda.results['outliers']       # Outliers
       eda.results['normality']      # Normality tests
    """)
    
    # Run demo
    print("\n" + "="*80)
    print("📊 RUNNING DEMO ANALYSIS")
    print("="*80)
    cleaned_df, eda = demo_analysis()

🔬 ELEMENTAL SOCIETY ANALYSIS - JUPYTER EDA SUITE
📌 Project Code: 1010 (Data Cleaning)
👨‍🔬 Developed by: Mohamed Almoufty Abaker

🧪 ELEMENTAL SOCIETY ANALYSIS - JUPYTER SUITE v2.0

📌 AVAILABLE FUNCTIONS:

    1. LOAD DATA:
       df = load_file('your_file.csv')
    
    2. CLEAN DATA:
       cleaner = DataCleaner(df)
       cleaned_df = cleaner.clean_all()
    
    3. ANALYZE DATA:
       eda = ProfessionalEDA(cleaned_df)
       eda.run_all()
    
    4. COMPLETE WORKFLOW:
       cleaned_df, eda = analyze_geochemical_data('your_file.csv')
    
    5. QUICK ANALYSIS:
       cleaned_df, eda = quick_analysis(df)
    
    6. DEMO:
       cleaned_df, eda = demo_analysis()
    
    7. VISUALIZATIONS:
       plot_correlation(df)
       plot_pca_scores(eda.results['pca'])
       plot_distributions(df)
    
    8. GET REPORT:
       report = eda.get_report()
       print(report)
    
    9. ACCESS RESULTS:
       eda.results['basic_stats']    # Statistics
       eda.results['correlation']    # C

In [11]:
df = clean_any_data('Li pb la mg ce.csv')
print(df)
analyzer = StatisticalAnalyzer(df)
plot_correlation_matrix(df)
plot_pca_scores(analyzer)

NameError: name 'clean_any_data' is not defined

In [25]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - JUPYTER COMPLETE VERSION (FULLY FIXED)
# Project Code: 1010 (Data Cleaning)
# Developed by: Mohamed Almoufty Abaker - Geologist & Writer
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("🔬 ELEMENTAL SOCIETY ANALYSIS - JUPYTER EDA SUITE")
print("="*80)
print("📌 Project Code: 1010 (Data Cleaning)")
print("👨‍🔬 Developed by: Mohamed Almoufty Abaker")
print("="*80)

# ============================================================
# PART 1: FILE LOADING ENGINE
# ============================================================

def load_file(file_path, **kwargs):
    """
    Universal file loader for geochemical data
    
    Parameters:
    -----------
    file_path : str
        Path to the file
    **kwargs : additional arguments for pd.read_csv or pd.read_excel
    
    Returns:
    --------
    pandas.DataFrame : Loaded data
    """
    print(f"\n📁 LOADING FILE: {file_path}")
    print("-" * 50)
    
    # Check file extension
    file_ext = file_path.split('.')[-1].lower()
    
    # Try different delimiters and encodings for CSV/text files
    if file_ext in ['csv', 'txt', 'dat', 'tsv']:
        delimiters = [';', ',', '\t', '|', ' ']
        encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1']
        
        df = None
        for encoding in encodings:
            for delim in delimiters:
                try:
                    df = pd.read_csv(file_path, sep=delim, encoding=encoding, 
                                   engine='python', on_bad_lines='skip', **kwargs)
                    if df is not None and not df.empty and df.shape[1] > 1:
                        print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                        print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                        return df
                except:
                    continue
        
        # If all fail, try manual parsing
        if df is None or df.empty:
            print("⚠️ Auto-detection failed, trying manual parsing...")
            df = manual_parse_file(file_path)
            if df is not None:
                print(f"✅ Manual parsing successful! Shape: {df.shape}")
                return df
    
    # Excel files
    elif file_ext in ['xlsx', 'xls', 'xlsm']:
        try:
            df = pd.read_excel(file_path, engine='openpyxl', **kwargs)
            print(f"✅ Read as Excel file (openpyxl)")
            print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
            return df
        except:
            try:
                df = pd.read_excel(file_path, engine='xlrd', **kwargs)
                print(f"✅ Read as Excel file (xlrd)")
                print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                return df
            except Exception as e:
                print(f"❌ Excel read failed: {e}")
                return None
    
    else:
        print(f"⚠️ Unknown file type: {file_ext}")
        df = manual_parse_file(file_path)
        if df is not None:
            print(f"✅ Manual parsing successful! Shape: {df.shape}")
            return df
    
    print("❌ Could not read file")
    return None

def manual_parse_file(file_path):
    """Manually parse text files when auto-detection fails"""
    try:
        with open(file_path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    # Detect delimiter
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        
        df = pd.DataFrame(data)
        
        # Check if first row is header
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    
    return None

# ============================================================
# PART 2: DATA CLEANING ENGINE
# ============================================================

class DataCleaner:
    """Professional Data Cleaning Engine for Geochemical Data"""
    
    def __init__(self, df):
        self.df = df.copy()
        self.cleaned_df = None
        self.cleaning_log = []
        self.results =}
        print(f"\n🧹 DATA CLEANER INITIALIZED")
        print(f"Original shape: {self.df.shape}")
        print(f"Columns: {len(self.df.columns)}")
        print(f"Rows: {len(self.df)}")
    
    def clean_all(self, handle_missing=True, handle_outliers=True, 
                  handle_detection_limits=True, remove_duplicates=True):
        """
        Run complete cleaning pipeline
        """
        self.cleaning_log = []
        print("\n🧹 RUNNING CLEANING PIPELINE")
        print("-" * 50)
        
        # 1. Clean column names
        self._clean_column_names()
        
        # 2. Handle detection limits BEFORE converting to numeric
        if handle_detection_limits:
            self._handle_detection_limits()
        
        # 3. Convert to numeric
        self._convert_to_numeric()
        
        # 4. Remove duplicates
        if remove_duplicates:
            self._remove_duplicates()
        
        # 5. Handle missing values
        if handle_missing:
            self._handle_missing_values()
        
        # 6. Handle outliers
        if handle_outliers:
            self._handle_outliers()
        print(f'{df.head(5)}, {df.info()}')
        
        self.cleaned_df = self.df.copy()
        
        print(f"\n✅ Cleaning complete!")
        print(f"Final shape: {self.cleaned_df.shape}")
        print(f"Rows: {len(self.cleaned_df)}")
        print(f"Columns: {len(self.cleaned_df.columns)}")
        
        return self.cleaned_df
    
    def _clean_column_names(self):
        """Clean column names"""
        self.df.columns = self.df.columns.str.strip()
        self.df.columns = self.df.columns.str.replace(' ', '_')
        self.df.columns = self.df.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
        self.df.columns = self.df.columns.str.replace('__', '_', regex=False)
        
        # Make unique
        seen = {}
        new_cols = []
        for col in self.df.columns:
            if col in seen:
                seen[col] += 1
                new_cols.append(f"{col}_{seen[col]}")
            else:
                seen[col] = 0
                new_cols.append(col)
        self.df.columns = new_cols
        
        self.cleaning_log.append("✅ Column names cleaned")
    
    def _handle_detection_limits(self):
        """Handle detection limits (< values) - BEFORE converting to numeric"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            
            # Check if column contains < values
            mask = self.df[col].astype(str).str.startswith('<', na=False)
            if mask.sum() > 0:
                for idx in self.df[mask].index:
                    val = str(self.df.loc[idx, col])
                    try:
                        # Extract number after <
                        limit = float(val.replace('<', '').strip())
                        # Replace with half detection limit
                        self.df.loc[idx, col] = limit / 2
                    except:
                        self.df.loc[idx, col] = np.nan
                self.cleaning_log.append(f"✅ Handled {mask.sum()} detection limits in {col}")
    
    def _convert_to_numeric(self):
        """Convert columns to numeric"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
        self.cleaning_log.append("✅ Converted columns to numeric")
    
    def _remove_duplicates(self):
        """Remove duplicate rows"""
        initial_len = len(self.df)
        self.df = self.df.drop_duplicates()
        removed = initial_len - len(self.df)
        if removed > 0:
            self.cleaning_log.append(f"✅ Removed {removed} duplicate rows")
    
    def _handle_missing_values(self):
        """Handle missing values"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            missing = self.df[col].isnull().sum()
            if missing > 0:
                if self.df[col].dtype in ['float64', 'int64']:
                    self.df[col] = self.df[col].fillna(self.df[col].median())
                    self.cleaning_log.append(f"✅ Filled {missing} missing values in {col}")
    
    def _handle_outliers(self, method='iqr'):
        """Handle outliers using IQR method"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            
            if self.df[col].dtype not in ['float64', 'int64']:
                continue
            
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            
            outliers = ((self.df[col] < lower) | (self.df[col] > upper)).sum()
            if outliers > 0:
                self.df[col] = self.df[col].clip(lower=lower, upper=upper)
                self.cleaning_log.append(f"✅ Capped {outliers} outliers in {col}")
    
    def get_log(self):
        """Get cleaning log"""
        return "\n".join(self.cleaning_log)

# ============================================================
# PART 3: PROFESSIONAL EDA ENGINE
# ============================================================

class ProfessionalEDA:
    """Comprehensive Exploratory Data Analysis Engine"""
    
    def __init__(self, df):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        print(f"\n📊 EDA ENGINE INITIALIZED")
        print(f"Numeric columns: {len(self.numeric_df.columns)}")
        print(f"Samples: {len(self.df)}")
    
    def run_all(self):
        """Run all analyses"""
        print("\n📊 RUNNING COMPREHENSIVE EDA")
        print("-" * 50)
        
        self.basic_statistics()
        self.variance_analysis()
        self.correlation_analysis()
        self.outlier_detection()
        self.normality_analysis()
        self.perform_pca()
        
        print("\n✅ EDA complete!")
        return self.results
    
    def basic_statistics(self):
        """Calculate basic statistics"""
        self.results['basic_stats'] = {
            'mean': self.numeric_df.mean(),
            'median': self.numeric_df.median(),
            'std': self.numeric_df.std(),
            'min': self.numeric_df.min(),
            'max': self.numeric_df.max(),
            'range': self.numeric_df.max() - self.numeric_df.min(),
            'q25': self.numeric_df.quantile(0.25),
            'q75': self.numeric_df.quantile(0.75),
            'iqr': self.numeric_df.quantile(0.75) - self.numeric_df.quantile(0.25),
            'skew': self.numeric_df.skew(),
            'kurtosis': self.numeric_df.kurtosis(),
            'count': self.numeric_df.count()
        }
        print("  ✅ Basic statistics")
        return self.results['basic_stats']
    
    def variance_analysis(self):
        """Calculate variance metrics"""
        self.results['variance'] = {
            'variance': self.numeric_df.var(),
            'std_dev': self.numeric_df.std(),
            'cv': (self.numeric_df.std() / self.numeric_df.mean()) * 100
        }
        print("  ✅ Variance analysis")
        return self.results['variance']
    
    def correlation_analysis(self):
        """Calculate correlation matrix"""
        if len(self.numeric_df.columns) > 1:
            self.results['correlation'] = self.numeric_df.corr()
            
            # Find strong correlations
            strong = []
            corr = self.results['correlation']
            for i, col1 in enumerate(corr.columns):
                for col2 in corr.columns[i+1:]:
                    val = corr.loc[col1, col2]
                    if abs(val) > 0.5:
                        strong.append((col1, col2, val))
            self.results['strong_correlations'] = sorted(strong, key=lambda x: abs(x[2]), reverse=True)
            print(f"  ✅ Correlation matrix ({len(strong)} strong correlations)")
        return self.results.get('correlation')
    
    def outlier_detection(self):
        """Detect outliers using IQR method"""
        self.results['outliers'] = {}
        for col in self.numeric_df.columns:
            Q1 = self.numeric_df[col].quantile(0.25)
            Q3 = self.numeric_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            self.results['outliers'][col] = ((self.numeric_df[col] < lower) | 
                                            (self.numeric_df[col] > upper)).sum()
        print("  ✅ Outlier detection")
        return self.results['outliers']
    
    def normality_analysis(self):
        """Analyze normality using skewness and kurtosis"""
        self.results['normality'] = {}
        for col in self.numeric_df.columns:
            skew_val = self.numeric_df[col].skew()
            kurt_val = self.numeric_df[col].kurtosis()
            is_normal = (-1 < skew_val < 1) and (-2 < kurt_val < 2)
            self.results['normality'][col] = {
                'skewness': skew_val,
                'kurtosis': kurt_val,
                'is_normal': is_normal
            }
        print("  ✅ Normality analysis")
        return self.results['normality']
    
    def perform_pca(self):
        """Perform Principal Component Analysis"""
        if len(self.numeric_df.columns) >= 2:
            clean_data = self.numeric_df.dropna()
            if len(clean_data) > 1:
                scaler = StandardScaler()
                scaled = scaler.fit_transform(clean_data)
                pca = SklearnPCA()
                scores = pca.fit_transform(scaled)
                self.results['pca'] = {
                    'explained_variance': pca.explained_variance_ratio_,
                    'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
                    'scores': scores,
                    'loadings': pd.DataFrame(
                        pca.components_.T,
                        columns=[f'PC{i+1}' for i in range(pca.n_components_)],
                        index=clean_data.columns
                    ),
                    'n_components': pca.n_components_,
                    'n_samples': len(clean_data)
                }
                print(f"  ✅ PCA ({pca.n_components_} components, "
                      f"{pca.explained_variance_ratio_.sum()*100:.1f}% variance)")
        return self.results.get('pca')
    
    def get_summary(self):
        """Get summary statistics"""
        return {
            'shape': self.df.shape,
            'numeric_cols': len(self.numeric_df.columns),
            'total_cols': len(self.df.columns),
            'basic_stats': self.results.get('basic_stats', {}),
            'variance': self.results.get('variance', {}),
            'strong_correlations': self.results.get('strong_correlations', []),
            'outliers': self.results.get('outliers', {}),
            'normality': self.results.get('normality', {})
        }
    
    def get_report(self):
        """Generate comprehensive report"""
        report = []
        report.append("="*80)
        report.append("📊 ELEMENTAL SOCIETY ANALYSIS - EDA REPORT")
        report.append("="*80)
        
        # Dataset info
        report.append(f"\n📋 Dataset: {len(self.df)} rows × {len(self.df.columns)} columns")
        report.append(f"📊 Numeric columns: {len(self.numeric_df.columns)}")
        
        # Basic stats
        report.append("\n📊 BASIC STATISTICS")
        report.append("-"*40)
        stats_df = pd.DataFrame(self.results.get('basic_stats', {}))
        if not stats_df.empty:
            report.append(stats_df.round(4).to_string())
        
        # Variance
        report.append("\n📈 VARIANCE")
        report.append("-"*40)
        var_df = pd.DataFrame(self.results.get('variance', {}))
        if not var_df.empty:
            report.append(var_df.round(4).to_string())
        
        # Correlations
        if 'strong_correlations' in self.results:
            report.append("\n📉 STRONG CORRELATIONS (>0.5)")
            report.append("-"*40)
            for col1, col2, val in self.results['strong_correlations'][:5]:
                direction = "positive" if val > 0 else "negative"
                report.append(f"  {col1} ↔ {col2}: {val:.4f} ({direction})")
        
        # Outliers
        report.append("\n🎯 OUTLIERS")
        report.append("-"*40)
        for col, count in self.results.get('outliers', {}).items():
            report.append(f"  {col}: {count} outliers")
        
        # Normality
        report.append("\n📊 NORMALITY")
        report.append("-"*40)
        for col, data in self.results.get('normality', {}).items():
            status = "✅ Normal" if data['is_normal'] else "❌ Not normal"
            report.append(f"  {col}: {status} (skew={data['skewness']:.4f})")
        
        # PCA
        if 'pca' in self.results:
            report.append("\n📉 PCA")
            report.append("-"*40)
            explained = self.results['pca']['explained_variance']
            cum_var = np.cumsum(explained)
            report.append(f"Total explained variance: {explained.sum()*100:.1f}%")
            for i, (ev, cum) in enumerate(zip(explained[:5], cum_var[:5])):
                report.append(f"  PC{i+1}: {ev*100:.1f}% (cumulative: {cum*100:.1f}%)")
            report.append("\nLoadings (PC1):")
            loadings = self.results['pca']['loadings']['PC1'].abs().sort_values(ascending=False)
            for var, loading in list(loadings.items())[:5]:
                report.append(f"  {var}: {loading:.4f}")
        
        report.append("\n"+"="*80)
        report.append(f"📌 Report generated by Elemental Society Analysis v2.0")
        report.append("="*80)
        
        return "\n".join(report)

# ============================================================
# PART 4: VISUALIZATION FUNCTIONS
# ============================================================

def plot_correlation(df, title="Correlation Matrix", figsize=(10, 8)):
    """Plot correlation matrix heatmap"""
    numeric_df = df.select_dtypes(include=[np.number])
    if len(numeric_df.columns) < 2:
        print("⚠️ Need at least 2 numeric columns")
        return
    
    plt.figure(figsize=figsize)
    corr = numeric_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', center=0,
                fmt='.2f', square=True, linewidths=0.5)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_pca_scores(pca_results, figsize=(10, 6)):
    """Plot PCA scores"""
    if pca_results is None:
        print("⚠️ No PCA results available")
        return
    
    scores = pca_results['scores']
    explained = pca_results['explained_variance']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # PCA scores
    scatter = ax1.scatter(scores[:, 0], scores[:, 1],
                         c=np.linalg.norm(scores[:, :2], axis=1),
                         cmap='viridis', alpha=0.6,
                         edgecolors='black', linewidth=0.5)
    ax1.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
    ax1.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
    ax1.set_title('PCA Scores', fontweight='bold')
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax1.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax1, label='Distance')
    
    # Explained variance
    n_components = min(5, len(explained))
    ax2.bar(range(1, n_components+1), explained[:n_components]*100,
            color='steelblue', edgecolor='black')
    ax2.set_xlabel('Principal Component')
    ax2.set_ylabel('Explained Variance (%)')
    ax2.set_title('Explained Variance by Component', fontweight='bold')
    ax2.set_xticks(range(1, n_components+1))
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_distributions(df, cols=None, figsize=(12, 8)):
    """Plot distributions for selected columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:4]
    
    cols = [col for col in cols if col in numeric_df.columns]
    if not cols:
        print("⚠️ No valid columns")
        return
    
    n_cols = min(2, len(cols))
    n_rows = (len(cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1:
        if n_cols == 1:
            axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(cols):
        if i >= len(axes):
            break
        data = numeric_df[col].dropna()
        if len(data) > 0:
            axes[i].hist(data, bins=20, density=True, alpha=0.6,
                        color='steelblue', edgecolor='black')
            axes[i].set_xlabel(col, fontsize=10)
            axes[i].set_ylabel('Density', fontsize=10)
            axes[i].set_title(f'{col} Distribution', fontweight='bold')
            axes[i].axvline(data.mean(), color='red', linestyle='--',
                          label=f'Mean: {data.mean():.2f}')
            axes[i].axvline(data.median(), color='green', linestyle='--',
                          label=f'Median: {data.median():.2f}')
            axes[i].legend(fontsize=8)
            axes[i].grid(True, alpha=0.3)
    
    for i in range(len(cols), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

# ============================================================
# PART 5: COMPLETE WORKFLOW
# ============================================================

def analyze_geochemical_data(file_path, **kwargs):
    """
    Complete workflow: Load → Clean → Analyze → Report
    
    Parameters:
    -----------
    file_path : str
        Path to the geochemical data file
    **kwargs : additional arguments for cleaning
    
    Returns:
    --------
    tuple : (cleaned_df, eda_results)
    """
    print("\n" + "="*80)
    print("🚀 ELEMENTAL SOCIETY ANALYSIS - COMPLETE WORKFLOW")
    print("="*80)
    
    # Step 1: Load data
    df = load_file(file_path)
    if df is None:
        print("❌ Failed to load data")
        return None, None
    
    # Step 2: Clean data
    cleaner = DataCleaner(df)
    cleaned_df = cleaner.clean_all(**kwargs)
    
    # Show cleaning log
    print("\n📝 Cleaning Log:")
    print(cleaner.get_log())
    
    # Step 3: Analyze
    eda = ProfessionalEDA(cleaned_df)
    eda.run_all()
    
    # Step 4: Report
    print("\n" + eda.get_report())
    
    # Step 5: Visualizations
    print("\n📊 GENERATING VISUALIZATIONS...")
    
    # Correlation matrix
    plot_correlation(cleaned_df, title="Correlation Matrix")
    
    # PCA scores
    if 'pca' in eda.results:
        plot_pca_scores(eda.results['pca'])
    
    # Distributions
    plot_distributions(cleaned_df)
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    
    return cleaned_df, eda

# ============================================================
# PART 6: QUICK ANALYSIS FUNCTIONS
# ============================================================

def quick_analysis(df):
    """
    Quick analysis on existing DataFrame
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data to analyze
    
    Returns:
    --------
    tuple : (cleaned_df, eda_results)
    """
    print("\n" + "="*80)
    print("🚀 QUICK ANALYSIS")
    print("="*80)
    
    # Clean
    cleaner = DataCleaner(df)
    cleaned_df = cleaner.clean_all()
    
    # Analyze
    eda = ProfessionalEDA(cleaned_df)
    eda.run_all()
    
    # Report
    print("\n" + eda.get_report())
    
    return cleaned_df, eda

# ============================================================
# PART 7: DEMO WITH SAMPLE DATA (FULLY FIXED)
# ============================================================

def create_sample_data(n_samples=50):
    """Create sample geochemical data for testing"""
    np.random.seed(42)
    
    # Base patterns
    base = np.random.randn(n_samples) * 2
    
    # Create data as lists first (to handle mixed types)
    sample_ids = [f'Sample_{i+1:03d}' for i in range(n_samples)]
    au_data = []
    cu_data = []
    zn_data = []
    pb_data = []
    as_data = []
    lithologies = []
    
    for i in range(n_samples):
        # Au with some < values
        if i in np.random.choice(n_samples, size=5, replace=False):
            au_data.append(f"<{np.random.uniform(0.005, 0.02):.6f}")
        else:
            au_data.append(1.5 + 0.8 * base[i] + 0.2 * np.random.randn())
        
        # Cu with some outliers
        if i in np.random.choice(n_samples, size=3, replace=False):
            cu_data.append(np.random.uniform(200, 500))
        else:
            cu_data.append(30 + 15 * base[i] + 5 * np.random.randn())
        
        # Zn with some missing values
        if i in np.random.choice(n_samples, size=3, replace=False):
            zn_data.append(np.nan)
        else:
            zn_data.append(45 + 5 * base[i] + 10 * np.random.randn())
        
        # Pb
        pb_data.append(15 + 3 * np.random.randn())
        
        # As
        as_data.append(2 + 0.5 * base[i] + 0.3 * np.random.randn())
        
        # Lithology
        lithologies.append(np.random.choice(['Granite', 'Gabbro', 'Sandstone', 'Rhyolite']))
    
    # Create DataFrame
    df = pd.DataFrame({
        'Sample_ID': sample_ids,
        'Au_ppm': au_data,
        'Cu_ppm': cu_data,
        'Zn_ppm': zn_data,
        'Pb_ppm': pb_data,
        'As_ppm': as_data,
        'Lithology': lithologies
    })
    
    return df

def demo_analysis():
    """Run demo analysis on sample data"""
    print("\n" + "="*80)
    print("📊 DEMO ANALYSIS ON SAMPLE DATA")
    print("="*80)
    
    # Create sample data
    df = create_sample_data()
    print(f"\nSample data shape: {df.shape}")
    print("\nFirst 5 rows:")
    print(df.head())
    print("\nData types:")
    print(df.dtypes)
    
    # Run analysis
    cleaned_df, eda = quick_analysis(df)
    
    # Show some results
    print("\n📊 KEY INSIGHTS:")
    print("-" * 40)
    
    if 'strong_correlations' in eda.results:
        print("\nStrong Correlations:")
        for col1, col2, val in eda.results['strong_correlations'][:3]:
            print(f"  {col1} ↔ {col2}: {val:.4f}")
    
    if 'outliers' in eda.results:
        print("\nOutliers per column:")
        for col, count in eda.results['outliers'].items():
            if count > 0:
                print(f"  {col}: {count}")
    
    if 'normality' in eda.results:
        print("\nNormality:")
        for col, data in eda.results['normality'].items():
            status = "Normal" if data['is_normal'] else "Not normal"
            print(f"  {col}: {status}")
    
    return cleaned_df, eda

# ============================================================
# MAIN EXECUTION
# ============================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🧪 ELEMENTAL SOCIETY ANALYSIS - JUPYTER SUITE v2.0")
    print("="*80)
    print("""
📌 AVAILABLE FUNCTIONS:

    1. LOAD DATA:
       df = load_file('your_file.csv')
    
    2. CLEAN DATA:
       cleaner = DataCleaner(df)
       cleaned_df = cleaner.clean_all()
    
    3. ANALYZE DATA:
       eda = ProfessionalEDA(cleaned_df)
       eda.run_all()
    
    4. COMPLETE WORKFLOW:
       cleaned_df, eda = analyze_geochemical_data('your_file.csv')
    
    5. QUICK ANALYSIS:
       cleaned_df, eda = quick_analysis(df)
    
    6. DEMO:
       cleaned_df, eda = demo_analysis()
    
    7. VISUALIZATIONS:
       plot_correlation(df)
       plot_pca_scores(eda.results['pca'])
       plot_distributions(df)
    
    8. GET REPORT:
       report = eda.get_report()
       print(report)
    
    9. ACCESS RESULTS:
       eda.results['basic_stats']    # Statistics
       eda.results['correlation']    # Correlation matrix
       eda.results['pca']            # PCA results
       eda.results['outliers']       # Outliers
       eda.results['normality']      # Normality tests
    """)
    
    # Run demo
    print("\n" + "="*80)
    print("📊 RUNNING DEMO ANALYSIS")
    print("="*80)
    cleaned_df, eda = demo_analysis()

SyntaxError: unmatched '}' (1911187301.py, line 163)

In [29]:
# ============================================================
# ELEMENTAL SOCIETY ANALYSIS - JUPYTER COMPLETE VERSION (FULLY FIXED)
# Project Code: 1010 (Data Cleaning)
# Developed by: Mohamed Almoufty Abaker - Geologist & Writer
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA as SklearnPCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("="*80)
print("🔬 ELEMENTAL SOCIETY ANALYSIS - JUPYTER EDA SUITE")
print("="*80)
print("📌 Project Code: 1010 (Data Cleaning)")
print("👨‍🔬 Developed by: Mohamed Almoufty Abaker")
print("="*80)

# ============================================================
# PART 1: FILE LOADING ENGINE
# ============================================================

def load_file(file_path, **kwargs):
    """
    Universal file loader for geochemical data
    
    Parameters:
    -----------
    file_path : str
        Path to the file
    **kwargs : additional arguments for pd.read_csv or pd.read_excel
    
    Returns:
    --------
    pandas.DataFrame : Loaded data
    """
    print(f"\n📁 LOADING FILE: {file_path}")
    print("-" * 50)
    
    # Check file extension
    file_ext = file_path.split('.')[-1].lower()
    
    # Try different delimiters and encodings for CSV/text files
    if file_ext in ['csv', 'txt', 'dat', 'tsv']:
        delimiters = [';', ',', '\t', '|', ' ']
        encodings = ['utf-8', 'cp1252', 'latin-1', 'iso-8859-1']
        
        df = None
        for encoding in encodings:
            for delim in delimiters:
                try:
                    df = pd.read_csv(file_path, sep=delim, encoding=encoding, 
                                   engine='python', on_bad_lines='skip', **kwargs)
                    if df is not None and not df.empty and df.shape[1] > 1:
                        print(f"✅ Read with delimiter: '{delim}' (encoding: {encoding})")
                        print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                        return df
                except:
                    continue
        
        # If all fail, try manual parsing
        if df is None or df.empty:
            print("⚠️ Auto-detection failed, trying manual parsing...")
            df = manual_parse_file(file_path)
            if df is not None:
                print(f"✅ Manual parsing successful! Shape: {df.shape}")
                return df
    
    # Excel files
    elif file_ext in ['xlsx', 'xls', 'xlsm']:
        try:
            df = pd.read_excel(file_path, engine='openpyxl', **kwargs)
            print(f"✅ Read as Excel file (openpyxl)")
            print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
            return df
        except:
            try:
                df = pd.read_excel(file_path, engine='xlrd', **kwargs)
                print(f"✅ Read as Excel file (xlrd)")
                print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                return df
            except Exception as e:
                print(f"❌ Excel read failed: {e}")
                return None
    
    else:
        print(f"⚠️ Unknown file type: {file_ext}")
        df = manual_parse_file(file_path)
        if df is not None:
            print(f"✅ Manual parsing successful! Shape: {df.shape}")
            return df
    
    print("❌ Could not read file")
    return None

def manual_parse_file(file_path):
    """Manually parse text files when auto-detection fails"""
    try:
        with open(file_path, 'r', encoding='cp1252') as f:
            lines = f.readlines()
    except:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                lines = f.readlines()
        except:
            return None
    
    lines = [line.strip() for line in lines if line.strip()]
    if not lines:
        return None
    
    # Detect delimiter
    first_line = lines[0]
    delimiters = [';', ',', '\t', '|']
    best_delim = None
    max_count = 0
    
    for delim in delimiters:
        count = first_line.count(delim)
        if count > max_count:
            max_count = count
            best_delim = delim
    
    if best_delim:
        data = [line.split(best_delim) for line in lines]
        max_cols = max(len(row) for row in data)
        for row in data:
            if len(row) < max_cols:
                row.extend([''] * (max_cols - len(row)))
        
        df = pd.DataFrame(data)
        
        # Check if first row is header
        try:
            pd.to_numeric(df.iloc[0, 0])
            return df
        except:
            df.columns = df.iloc[0].tolist()
            df = df.iloc[1:].reset_index(drop=True)
            return df
    
    return None

# ============================================================
# PART 2: DATA CLEANING ENGINE
# ============================================================

class DataCleaner:
    """Professional Data Cleaning Engine for Geochemical Data"""
    
    def __init__(self, df):
        self.df = df.copy()
        self.cleaned_df = None
        self.cleaning_log = []
        self.dc_result = {}
        print(f"\n🧹 DATA CLEANER INITIALIZED")
        print(f"Original shape: {self.df.shape}")
        print(f"Columns: {len(self.df.columns)}")
        print(f"Rows: {len(self.df)}")
    
    def clean_all(self, handle_missing=True, handle_outliers=True, 
                  handle_detection_limits=True, remove_duplicates=True):
        """
        Run complete cleaning pipeline
        """
        self.cleaning_log = []
        print("\n🧹 RUNNING CLEANING PIPELINE")
        print("-" * 50)
        
        # 1. Clean column names
        self._clean_column_names()
        
        # 2. Handle detection limits BEFORE converting to numeric
        if handle_detection_limits:
            self._handle_detection_limits()
        
        # 3. Convert to numeric
        self._convert_to_numeric()
        print(f'{self.df}')
        
        # 4. Remove duplicates
        if remove_duplicates:
            self._remove_duplicates()
            print(f'{self.df}')
        
        # 5. Handle missing values
        if handle_missing:
            self._handle_missing_values()
        
        # 6. Handle outliers
        if handle_outliers:
            self._handle_outliers()
        
        self.cleaned_df = self.df.copy()
        
        print(f"\n✅ Cleaning complete!")
        print(f"Final shape: {self.cleaned_df.shape}")
        print(f"Rows: {len(self.cleaned_df)}")
        print(f"Columns: {len(self.cleaned_df.columns)}")
        
        return self.cleaned_df
    
    def _clean_column_names(self):
        """Clean column names"""
        self.df.columns = self.df.columns.str.strip()
        self.df.columns = self.df.columns.str.replace(' ', '_')
        self.df.columns = self.df.columns.str.replace(r'[^a-zA-Z0-9_]', '_', regex=True)
        self.df.columns = self.df.columns.str.replace('__', '_', regex=False)
        
        # Make unique
        seen = {}
        new_cols = []
        for col in self.df.columns:
            if col in seen:
                seen[col] += 1
                new_cols.append(f"{col}_{seen[col]}")
            else:
                seen[col] = 0
                new_cols.append(col)
        self.df.columns = new_cols
        
        self.cleaning_log.append("✅ Column names cleaned")
    
    def _handle_detection_limits(self):
        """Handle detection limits (< values) - BEFORE converting to numeric"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            
            # Check if column contains < values
            mask = self.df[col].astype(str).str.startswith('<', na=False)
            if mask.sum() > 0:
                for idx in self.df[mask].index:
                    val = str(self.df.loc[idx, col])
                    try:
                        # Extract number after <
                        limit = float(val.replace('<', '').strip())
                        # Replace with half detection limit
                        self.df.loc[idx, col] = limit / 2
                    except:
                        self.df.loc[idx, col] = np.nan
                self.cleaning_log.append(f"✅ Handled {mask.sum()} detection limits in {col}")
    
    def _convert_to_numeric(self):
        """Convert columns to numeric"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            self.df[col] = pd.to_numeric(self.df[col], errors='coerce')
        self.cleaning_log.append("✅ Converted columns to numeric")
    
    def _remove_duplicates(self):
        """Remove duplicate rows"""
        initial_len = len(self.df)
        self.df = self.df.drop_duplicates()
        removed = initial_len - len(self.df)
        if removed > 0:
            self.cleaning_log.append(f"✅ Removed {removed} duplicate rows")
    
    def _handle_missing_values(self):
        """Handle missing values"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            missing = self.df[col].isnull().sum()
            if missing > 0:
                if self.df[col].dtype in ['float64', 'int64']:
                    self.df[col] = self.df[col].fillna(self.df[col].median())
                    self.cleaning_log.append(f"✅ Filled {missing} missing values in {col}")
    
    def _handle_outliers(self, method='iqr'):
        """Handle outliers using IQR method"""
        for col in self.df.columns:
            if col in ['Sample_ID', 'Lithology', 'Comments']:
                continue
            
            if self.df[col].dtype not in ['float64', 'int64']:
                continue
            
            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            
            outliers = ((self.df[col] < lower) | (self.df[col] > upper)).sum()
            if outliers > 0:
                self.df[col] = self.df[col].clip(lower=lower, upper=upper)
                self.cleaning_log.append(f"✅ Capped {outliers} outliers in {col}")
    
    def get_log(self):
        """Get cleaning log"""
        return "\n".join(self.cleaning_log)

# ============================================================
# PART 3: PROFESSIONAL EDA ENGINE
# ============================================================

class ProfessionalEDA:
    """Comprehensive Exploratory Data Analysis Engine"""
    
    def __init__(self, df):
        self.df = df
        self.numeric_df = df.select_dtypes(include=[np.number])
        self.results = {}
        print(f"\n📊 EDA ENGINE INITIALIZED")
        print(f"Numeric columns: {len(self.numeric_df.columns)}")
        print(f"Samples: {len(self.df)}")
    
    def run_all(self):
        """Run all analyses"""
        print("\n📊 RUNNING COMPREHENSIVE EDA")
        print("-" * 50)
        
        self.basic_statistics()
        self.variance_analysis()
        self.correlation_analysis()
        self.outlier_detection()
        self.normality_analysis()
        self.perform_pca()
        
        print("\n✅ EDA complete!")
        return self.results
    
    def basic_statistics(self):
        """Calculate basic statistics"""
        self.results['basic_stats'] = {
            'mean': self.numeric_df.mean(),
            'median': self.numeric_df.median(),
            'std': self.numeric_df.std(),
            'min': self.numeric_df.min(),
            'max': self.numeric_df.max(),
            'range': self.numeric_df.max() - self.numeric_df.min(),
            'q25': self.numeric_df.quantile(0.25),
            'q75': self.numeric_df.quantile(0.75),
            'iqr': self.numeric_df.quantile(0.75) - self.numeric_df.quantile(0.25),
            'skew': self.numeric_df.skew(),
            'kurtosis': self.numeric_df.kurtosis(),
            'count': self.numeric_df.count()
        }
        print("  ✅ Basic statistics")
        return self.results['basic_stats']
    
    def variance_analysis(self):
        """Calculate variance metrics"""
        self.results['variance'] = {
            'variance': self.numeric_df.var(),
            'std_dev': self.numeric_df.std(),
            'cv': (self.numeric_df.std() / self.numeric_df.mean()) * 100
        }
        print("  ✅ Variance analysis")
        return self.results['variance']
    
    def correlation_analysis(self):
        """Calculate correlation matrix"""
        if len(self.numeric_df.columns) > 1:
            self.results['correlation'] = self.numeric_df.corr()
            
            # Find strong correlations
            strong = []
            corr = self.results['correlation']
            for i, col1 in enumerate(corr.columns):
                for col2 in corr.columns[i+1:]:
                    val = corr.loc[col1, col2]
                    if abs(val) > 0.5:
                        strong.append((col1, col2, val))
            self.results['strong_correlations'] = sorted(strong, key=lambda x: abs(x[2]), reverse=True)
            print(f"  ✅ Correlation matrix ({len(strong)} strong correlations)")
        return self.results.get('correlation')
    
    def outlier_detection(self):
        """Detect outliers using IQR method"""
        self.results['outliers'] = {}
        for col in self.numeric_df.columns:
            Q1 = self.numeric_df[col].quantile(0.25)
            Q3 = self.numeric_df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            self.results['outliers'][col] = ((self.numeric_df[col] < lower) | 
                                            (self.numeric_df[col] > upper)).sum()
        print("  ✅ Outlier detection")
        return self.results['outliers']
    
    def normality_analysis(self):
        """Analyze normality using skewness and kurtosis"""
        self.results['normality'] = {}
        for col in self.numeric_df.columns:
            skew_val = self.numeric_df[col].skew()
            kurt_val = self.numeric_df[col].kurtosis()
            is_normal = (-1 < skew_val < 1) and (-2 < kurt_val < 2)
            self.results['normality'][col] = {
                'skewness': skew_val,
                'kurtosis': kurt_val,
                'is_normal': is_normal
            }
        print("  ✅ Normality analysis")
        return self.results['normality']
    
    def perform_pca(self):
        """Perform Principal Component Analysis"""
        if len(self.numeric_df.columns) >= 2:
            clean_data = self.numeric_df.dropna()
            if len(clean_data) > 1:
                scaler = StandardScaler()
                scaled = scaler.fit_transform(clean_data)
                pca = SklearnPCA()
                scores = pca.fit_transform(scaled)
                self.results['pca'] = {
                    'explained_variance': pca.explained_variance_ratio_,
                    'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
                    'scores': scores,
                    'loadings': pd.DataFrame(
                        pca.components_.T,
                        columns=[f'PC{i+1}' for i in range(pca.n_components_)],
                        index=clean_data.columns
                    ),
                    'n_components': pca.n_components_,
                    'n_samples': len(clean_data)
                }
                print(f"  ✅ PCA ({pca.n_components_} components, "
                      f"{pca.explained_variance_ratio_.sum()*100:.1f}% variance)")
        return self.results.get('pca')
    
    def get_summary(self):
        """Get summary statistics"""
        return {
            'shape': self.df.shape,
            'numeric_cols': len(self.numeric_df.columns),
            'total_cols': len(self.df.columns),
            'basic_stats': self.results.get('basic_stats', {}),
            'variance': self.results.get('variance', {}),
            'strong_correlations': self.results.get('strong_correlations', []),
            'outliers': self.results.get('outliers', {}),
            'normality': self.results.get('normality', {})
        }
    
    def get_report(self):
        """Generate comprehensive report"""
        report = []
        report.append("="*80)
        report.append("📊 ELEMENTAL SOCIETY ANALYSIS - EDA REPORT")
        report.append("="*80)
        
        # Dataset info
        report.append(f"\n📋 Dataset: {len(self.df)} rows × {len(self.df.columns)} columns")
        report.append(f"📊 Numeric columns: {len(self.numeric_df.columns)}")
        
        # Basic stats
        report.append("\n📊 BASIC STATISTICS")
        report.append("-"*40)
        stats_df = pd.DataFrame(self.results.get('basic_stats', {}))
        if not stats_df.empty:
            report.append(stats_df.round(4).to_string())
        
        # Variance
        report.append("\n📈 VARIANCE")
        report.append("-"*40)
        var_df = pd.DataFrame(self.results.get('variance', {}))
        if not var_df.empty:
            report.append(var_df.round(4).to_string())
        
        # Correlations
        if 'strong_correlations' in self.results:
            report.append("\n📉 STRONG CORRELATIONS (>0.5)")
            report.append("-"*40)
            for col1, col2, val in self.results['strong_correlations'][:5]:
                direction = "positive" if val > 0 else "negative"
                report.append(f"  {col1} ↔ {col2}: {val:.4f} ({direction})")
        
        # Outliers
        report.append("\n🎯 OUTLIERS")
        report.append("-"*40)
        for col, count in self.results.get('outliers', {}).items():
            report.append(f"  {col}: {count} outliers")
        
        # Normality
        report.append("\n📊 NORMALITY")
        report.append("-"*40)
        for col, data in self.results.get('normality', {}).items():
            status = "✅ Normal" if data['is_normal'] else "❌ Not normal"
            report.append(f"  {col}: {status} (skew={data['skewness']:.4f})")
        
        # PCA
        if 'pca' in self.results:
            report.append("\n📉 PCA")
            report.append("-"*40)
            explained = self.results['pca']['explained_variance']
            cum_var = np.cumsum(explained)
            report.append(f"Total explained variance: {explained.sum()*100:.1f}%")
            for i, (ev, cum) in enumerate(zip(explained[:5], cum_var[:5])):
                report.append(f"  PC{i+1}: {ev*100:.1f}% (cumulative: {cum*100:.1f}%)")
            report.append("\nLoadings (PC1):")
            loadings = self.results['pca']['loadings']['PC1'].abs().sort_values(ascending=False)
            for var, loading in list(loadings.items())[:5]:
                report.append(f"  {var}: {loading:.4f}")
        
        report.append("\n"+"="*80)
        report.append(f"📌 Report generated by Elemental Society Analysis v2.0")
        report.append("="*80)
        
        return "\n".join(report)

# ============================================================
# PART 4: VISUALIZATION FUNCTIONS
# ============================================================

def plot_correlation(df, title="Correlation Matrix", figsize=(10, 8)):
    """Plot correlation matrix heatmap"""
    numeric_df = df.select_dtypes(include=[np.number])
    if len(numeric_df.columns) < 2:
        print("⚠️ Need at least 2 numeric columns")
        return
    
    plt.figure(figsize=figsize)
    corr = numeric_df.corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', center=0,
                fmt='.2f', square=True, linewidths=0.5)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_pca_scores(pca_results, figsize=(10, 6)):
    """Plot PCA scores"""
    if pca_results is None:
        print("⚠️ No PCA results available")
        return
    
    scores = pca_results['scores']
    explained = pca_results['explained_variance']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize)
    
    # PCA scores
    scatter = ax1.scatter(scores[:, 0], scores[:, 1],
                         c=np.linalg.norm(scores[:, :2], axis=1),
                         cmap='viridis', alpha=0.6,
                         edgecolors='black', linewidth=0.5)
    ax1.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
    ax1.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
    ax1.set_title('PCA Scores', fontweight='bold')
    ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax1.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax1.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax1, label='Distance')
    
    # Explained variance
    n_components = min(5, len(explained))
    ax2.bar(range(1, n_components+1), explained[:n_components]*100,
            color='steelblue', edgecolor='black')
    ax2.set_xlabel('Principal Component')
    ax2.set_ylabel('Explained Variance (%)')
    ax2.set_title('Explained Variance by Component', fontweight='bold')
    ax2.set_xticks(range(1, n_components+1))
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_distributions(df, cols=None, figsize=(12, 8)):
    """Plot distributions for selected columns"""
    numeric_df = df.select_dtypes(include=[np.number])
    if cols is None:
        cols = numeric_df.columns[:4]
    
    cols = [col for col in cols if col in numeric_df.columns]
    if not cols:
        print("⚠️ No valid columns")
        return
    
    n_cols = min(2, len(cols))
    n_rows = (len(cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1:
        if n_cols == 1:
            axes = [axes]
    else:
        axes = axes.flatten()
    
    for i, col in enumerate(cols):
        if i >= len(axes):
            break
        data = numeric_df[col].dropna()
        if len(data) > 0:
            axes[i].hist(data, bins=20, density=True, alpha=0.6,
                        color='steelblue', edgecolor='black')
            axes[i].set_xlabel(col, fontsize=10)
            axes[i].set_ylabel('Density', fontsize=10)
            axes[i].set_title(f'{col} Distribution', fontweight='bold')
            axes[i].axvline(data.mean(), color='red', linestyle='--',
                          label=f'Mean: {data.mean():.2f}')
            axes[i].axvline(data.median(), color='green', linestyle='--',
                          label=f'Median: {data.median():.2f}')
            axes[i].legend(fontsize=8)
            axes[i].grid(True, alpha=0.3)
    
    for i in range(len(cols), len(axes)):
        fig.delaxes(axes[i])
    
    plt.tight_layout()
    plt.show()

# ============================================================
# PART 5: COMPLETE WORKFLOW
# ============================================================

def analyze_geochemical_data(file_path, **kwargs):
    """
    Complete workflow: Load → Clean → Analyze → Report
    
    Parameters:
    -----------
    file_path : str
        Path to the geochemical data file
    **kwargs : additional arguments for cleaning
    
    Returns:
    --------
    tuple : (cleaned_df, eda_results)
    """
    print("\n" + "="*80)
    print("🚀 ELEMENTAL SOCIETY ANALYSIS - COMPLETE WORKFLOW")
    print("="*80)
    
    # Step 1: Load data
    df = load_file(file_path)
    if df is None:
        print("❌ Failed to load data")
        return None, None
    
    # Step 2: Clean data
    cleaner = DataCleaner(df)
    cleaned_df = cleaner.clean_all(**kwargs)
    
    # Show cleaning log
    print("\n📝 Cleaning Log:")
    print(cleaner.get_log())
    
    # Step 3: Analyze
    eda = ProfessionalEDA(cleaned_df)
    eda.run_all()
    
    # Step 4: Report
    print("\n" + eda.get_report())
    
    # Step 5: Visualizations
    print("\n📊 GENERATING VISUALIZATIONS...")
    
    # Correlation matrix
    plot_correlation(cleaned_df, title="Correlation Matrix")
    
    # PCA scores
    if 'pca' in eda.results:
        plot_pca_scores(eda.results['pca'])
    
    # Distributions
    plot_distributions(cleaned_df)
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80)
    
    return cleaned_df, eda

# ============================================================
# PART 6: QUICK ANALYSIS FUNCTIONS
# ============================================================

def quick_analysis(df):
    """
    Quick analysis on existing DataFrame
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data to analyze
    
    Returns:
    --------
    tuple : (cleaned_df, eda_results)
    """
    print("\n" + "="*80)
    print("🚀 QUICK ANALYSIS")
    print("="*80)
    
    # Clean
    cleaner = DataCleaner(df)
    cleaned_df = cleaner.clean_all()
    
    # Analyze
    eda = ProfessionalEDA(cleaned_df)
    eda.run_all()
    
    # Report
    print("\n" + eda.get_report())
    
    return cleaned_df, eda

# ============================================================
# PART 7: DEMO WITH SAMPLE DATA (FULLY FIXED)
# ============================================================

def create_sample_data(n_samples=50):
    """Create sample geochemical data for testing"""
    np.random.seed(42)
    
    # Base patterns
    base = np.random.randn(n_samples) * 2
    
    # Create data as lists first (to handle mixed types)
    sample_ids = [f'Sample_{i+1:03d}' for i in range(n_samples)]
    au_data = []
    cu_data = []
    zn_data = []
    pb_data = []
    as_data = []
    lithologies = []
    
    for i in range(n_samples):
        # Au with some < values
        if i in np.random.choice(n_samples, size=5, replace=False):
            au_data.append(f"<{np.random.uniform(0.005, 0.02):.6f}")
        else:
            au_data.append(1.5 + 0.8 * base[i] + 0.2 * np.random.randn())
        
        # Cu with some outliers
        if i in np.random.choice(n_samples, size=3, replace=False):
            cu_data.append(np.random.uniform(200, 500))
        else:
            cu_data.append(30 + 15 * base[i] + 5 * np.random.randn())
        
        # Zn with some missing values
        if i in np.random.choice(n_samples, size=3, replace=False):
            zn_data.append(np.nan)
        else:
            zn_data.append(45 + 5 * base[i] + 10 * np.random.randn())
        
        # Pb
        pb_data.append(15 + 3 * np.random.randn())
        
        # As
        as_data.append(2 + 0.5 * base[i] + 0.3 * np.random.randn())
        
        # Lithology
        lithologies.append(np.random.choice(['Granite', 'Gabbro', 'Sandstone', 'Rhyolite']))
    
    # Create DataFrame
    df = pd.DataFrame({
        'Sample_ID': sample_ids,
        'Au_ppm': au_data,
        'Cu_ppm': cu_data,
        'Zn_ppm': zn_data,
        'Pb_ppm': pb_data,
        'As_ppm': as_data,
        'Lithology': lithologies
    })
    
    return df

def demo_analysis():
    """Run demo analysis on sample data"""
    print("\n" + "="*80)
    print("📊 DEMO ANALYSIS ON SAMPLE DATA")
    print("="*80)
    
    # Create sample data
    df = create_sample_data()
    print(f"\nSample data shape: {df.shape}")
    print("\nFirst 5 rows:")
    print(df.head())
    print("\nData types:")
    print(df.dtypes)
    
    # Run analysis
    cleaned_df, eda = quick_analysis(df)
    
    # Show some results
    print("\n📊 KEY INSIGHTS:")
    print("-" * 40)
    
    if 'strong_correlations' in eda.results:
        print("\nStrong Correlations:")
        for col1, col2, val in eda.results['strong_correlations'][:3]:
            print(f"  {col1} ↔ {col2}: {val:.4f}")
    
    if 'outliers' in eda.results:
        print("\nOutliers per column:")
        for col, count in eda.results['outliers'].items():
            if count > 0:
                print(f"  {col}: {count}")
    
    if 'normality' in eda.results:
        print("\nNormality:")
        for col, data in eda.results['normality'].items():
            status = "Normal" if data['is_normal'] else "Not normal"
            print(f"  {col}: {status}")
    
    return cleaned_df, eda

# ============================================================
# MAIN EXECUTION
# ============================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🧪 ELEMENTAL SOCIETY ANALYSIS - JUPYTER SUITE v2.0")
    print("="*80)
    print("""
📌 AVAILABLE FUNCTIONS:

    1. LOAD DATA:
       df = load_file('your_file.csv')
    
    2. CLEAN DATA:
       cleaner = DataCleaner(df)
       cleaned_df = cleaner.clean_all()
    
    3. ANALYZE DATA:
       eda = ProfessionalEDA(cleaned_df)
       eda.run_all()
    
    4. COMPLETE WORKFLOW:
       cleaned_df, eda = analyze_geochemical_data('your_file.csv')
    
    5. QUICK ANALYSIS:
       cleaned_df, eda = quick_analysis(df)
    
    6. DEMO:
       cleaned_df, eda = demo_analysis()
    
    7. VISUALIZATIONS:
       plot_correlation(df)
       plot_pca_scores(eda.results['pca'])
       plot_distributions(df)
    
    8. GET REPORT:
       report = eda.get_report()
       print(report)
    
    9. ACCESS RESULTS:
       eda.results['basic_stats']    # Statistics
       eda.results['correlation']    # Correlation matrix
       eda.results['pca']            # PCA results
       eda.results['outliers']       # Outliers
       eda.results['normality']      # Normality tests
    """)
    
    # Run demo
    print("\n" + "="*80)
    print("📊 RUNNING DEMO ANALYSIS")
    print("="*80)
    cleaned_df, eda = demo_analysis()

🔬 ELEMENTAL SOCIETY ANALYSIS - JUPYTER EDA SUITE
📌 Project Code: 1010 (Data Cleaning)
👨‍🔬 Developed by: Mohamed Almoufty Abaker

🧪 ELEMENTAL SOCIETY ANALYSIS - JUPYTER SUITE v2.0

📌 AVAILABLE FUNCTIONS:

    1. LOAD DATA:
       df = load_file('your_file.csv')
    
    2. CLEAN DATA:
       cleaner = DataCleaner(df)
       cleaned_df = cleaner.clean_all()
    
    3. ANALYZE DATA:
       eda = ProfessionalEDA(cleaned_df)
       eda.run_all()
    
    4. COMPLETE WORKFLOW:
       cleaned_df, eda = analyze_geochemical_data('your_file.csv')
    
    5. QUICK ANALYSIS:
       cleaned_df, eda = quick_analysis(df)
    
    6. DEMO:
       cleaned_df, eda = demo_analysis()
    
    7. VISUALIZATIONS:
       plot_correlation(df)
       plot_pca_scores(eda.results['pca'])
       plot_distributions(df)
    
    8. GET REPORT:
       report = eda.get_report()
       print(report)
    
    9. ACCESS RESULTS:
       eda.results['basic_stats']    # Statistics
       eda.results['correlation']    # C

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ============================================
# FUNCTION 1: Load Data from File Path
# ============================================
def load_data(file_path, file_type=None, **kwargs):
    """
    Load data from various file formats based on file path
    
    Parameters:
    -----------
    file_path : str or Path
        Path to the data file
    file_type : str, optional
        Type of file (csv, excel, json, parquet, etc.)
        If None, it will be inferred from file extension
    **kwargs : additional arguments passed to the read function
    
    Returns:
    --------
    pandas.DataFrame : Loaded data
    """
    file_path = Path(file_path)
    
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    
    # Infer file type from extension if not specified
    if file_type is None:
        file_type = file_path.suffix.lower().replace('.', '')
    
    print(f"📂 Loading file: {file_path.name}")
    print(f"📁 File type: {file_type}")
    
    try:
        # Load based on file type
        if file_type in ['csv', 'tsv', 'txt']:
            # Handle different delimiters
            if file_type == 'csv':
                df = pd.read_csv(file_path, **kwargs)
            elif file_type == 'tsv':
                df = pd.read_csv(file_path, sep='\t', **kwargs)
            else:  # txt - try to auto-detect delimiter
                df = pd.read_csv(file_path, sep=None, engine='python', **kwargs)
                
        elif file_type in ['xlsx', 'xls']:
            df = pd.read_excel(file_path, **kwargs)
            
        elif file_type == 'json':
            df = pd.read_json(file_path, **kwargs)
            
        elif file_type == 'parquet':
            df = pd.read_parquet(file_path, **kwargs)
            
        elif file_type == 'feather':
            df = pd.read_feather(file_path, **kwargs)
            
        elif file_type == 'pickle' or file_type == 'pkl':
            df = pd.read_pickle(file_path, **kwargs)
            
        else:
            raise ValueError(f"Unsupported file type: {file_type}")
        
        print(f"✅ Successfully loaded {len(df)} rows and {len(df.columns)} columns")
        print(f"📋 Columns: {df.columns.tolist()}")
        return df
        
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        raise

# ============================================
# FUNCTION 2: Visualize Detection Limits
# ============================================
def visualize_detection_limits(df, lod_value=0.5, column=None):
    """
    Automatically detect and visualize detection limits in numeric columns
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataset
    lod_value : float
        Limit of Detection value
    column : str, optional
        Specific column to visualize. If None, uses first numeric column
    """
    # Find all numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if not numeric_cols:
        print("⚠️ No numeric columns found in the dataset!")
        return
    
    # Use specified column or first numeric column
    if column is not None and column in numeric_cols:
        col_to_plot = column
    else:
        col_to_plot = numeric_cols[0]
        if column is not None:
            print(f"⚠️ Column '{column}' not found or not numeric. Using '{col_to_plot}' instead")
    
    print(f"\n🔍 Visualizing detection limits for column: '{col_to_plot}'")
    print(f"📊 Available numeric columns: {numeric_cols}")
    
    # Before handling - show original distribution
    plt.figure(figsize=(14, 6))
    
    plt.subplot(1, 2, 1)
    # Clean the data: remove NaN and infinite values
    clean_data = df[col_to_plot].replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(clean_data) > 0:
        sns.histplot(clean_data, bins=30, kde=True)
        plt.title(f'Before Detection Limit Handling\nColumn: {col_to_plot}')
        plt.axvline(x=lod_value, color='red', linestyle='--', linewidth=2, label=f'LOD = {lod_value}')
        plt.xlabel(col_to_plot)
        plt.ylabel('Frequency')
        plt.legend()
        
        # Add statistical info
        stats_text = f'Mean: {clean_data.mean():.2f}\nMedian: {clean_data.median():.2f}\nStd: {clean_data.std():.2f}\nN: {len(clean_data)}'
        plt.text(0.98, 0.95, stats_text, transform=plt.gca().transAxes, 
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    else:
        plt.text(0.5, 0.5, 'No valid data to plot', ha='center', va='center')
        plt.title(f'Column: {col_to_plot} - No Data')
    
    # After handling - simulate handling detection limits
    plt.subplot(1, 2, 2)
    
    # Apply LOD/2 handling to values below LOD
    data_after = clean_data.copy()
    # Replace values below LOD with LOD/2
    mask_below_lod = data_after < lod_value
    data_after[mask_below_lod] = lod_value / 2
    
    if len(data_after) > 0:
        sns.histplot(data_after, bins=30, kde=True, color='green')
        plt.title(f'After Detection Limit Handling\n(LOD replaced with LOD/2)')
        plt.axvline(x=lod_value, color='red', linestyle='--', linewidth=2, label=f'LOD = {lod_value}')
        plt.axvline(x=lod_value/2, color='orange', linestyle='--', linewidth=2, label=f'LOD/2 = {lod_value/2}')
        plt.xlabel(col_to_plot)
        plt.ylabel('Frequency')
        plt.legend()
        
        # Add statistical info after handling
        stats_text_after = f'Mean: {data_after.mean():.2f}\nMedian: {data_after.median():.2f}\nStd: {data_after.std():.2f}\nN: {len(data_after)}'
        plt.text(0.98, 0.95, stats_text_after, transform=plt.gca().transAxes, 
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
        
        # Show how many values were affected
        affected_count = mask_below_lod.sum()
        total_count = len(data_after)
        plt.text(0.02, 0.95, f'Values below LOD: {affected_count}/{total_count} ({affected_count/total_count*100:.1f}%)',
                transform=plt.gca().transAxes, verticalalignment='top', 
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    else:
        plt.text(0.5, 0.5, 'No valid data after processing', ha='center', va='center')
        plt.title(f'Column: {col_to_plot} - No Data')
    
    plt.tight_layout()
    plt.show()
    
    return numeric_cols

# ============================================
# FUNCTION 3: Complete EDA Visualization
# ============================================
def complete_eda_visualization(df, output_dir=None):
    """
    Complete EDA visualization showing all important aspects
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataset
    output_dir : str, optional
        Directory to save figures. If None, figures are displayed but not saved
    """
    if output_dir:
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
    
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Data Overview
    ax1 = plt.subplot(3, 3, 1)
    ax1.text(0.5, 0.8, f"Dataset Shape: {df.shape}", ha='center', fontsize=12)
    ax1.text(0.5, 0.6, f"Missing Values: {df.isnull().sum().sum()}", ha='center', fontsize=12)
    ax1.text(0.5, 0.4, f"Duplicate Rows: {df.duplicated().sum()}", ha='center', fontsize=12)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    ax1.text(0.5, 0.2, f"Numeric Columns: {len(numeric_cols)}", ha='center', fontsize=12)
    ax1.set_title('Data Overview')
    ax1.axis('off')
    
    # 2. Missing Values Heatmap
    ax2 = plt.subplot(3, 3, 2)
    if df.isnull().sum().sum() > 0:
        sns.heatmap(df.isnull(), cbar=False, cmap='viridis', ax=ax2)
        ax2.set_title('Missing Value Pattern')
    else:
        ax2.text(0.5, 0.5, 'No Missing Values', ha='center', va='center')
        ax2.set_title('Missing Value Pattern')
        ax2.axis('off')
    
    # 3. Numeric Columns Distribution
    ax3 = plt.subplot(3, 3, 3)
    if numeric_cols:
        # Show distribution of first numeric column
        first_col = numeric_cols[0]
        clean_data = df[first_col].replace([np.inf, -np.inf], np.nan).dropna()
        if len(clean_data) > 0:
            sns.histplot(clean_data, kde=True, ax=ax3)
            ax3.set_title(f'Distribution of {first_col}')
            ax3.set_xlabel(first_col)
    
    # 4. Correlation Matrix
    ax4 = plt.subplot(3, 3, 4)
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr()
        sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', 
                   square=True, ax=ax4, cbar_kws={'shrink': 0.8})
        ax4.set_title('Correlation Matrix')
    
    # 5. Box Plots for Numeric Columns
    ax5 = plt.subplot(3, 3, 5)
    if numeric_cols:
        # Show boxplots for first few numeric columns
        cols_to_plot = numeric_cols[:min(5, len(numeric_cols))]
        data_for_box = df[cols_to_plot].replace([np.inf, -np.inf], np.nan)
        if len(data_for_box.columns) > 0:
            data_for_box.boxplot(ax=ax5, rot=45)
            ax5.set_title('Boxplots of Numeric Columns')
    
    # 6. Value Counts for Categorical Columns
    ax6 = plt.subplot(3, 3, 6)
    cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if cat_cols:
        # Show first categorical column
        first_cat = cat_cols[0]
        top_categories = df[first_cat].value_counts().head(10)
        if len(top_categories) > 0:
            top_categories.plot(kind='barh', ax=ax6)
            ax6.set_title(f'Top Values in {first_cat}')
            ax6.set_xlabel('Count')
    
    # 7. Scatter Plot (if at least 2 numeric columns)
    ax7 = plt.subplot(3, 3, 7)
    if len(numeric_cols) >= 2:
        col1, col2 = numeric_cols[0], numeric_cols[1] if len(numeric_cols) > 1 else numeric_cols[0]
        if col1 != col2:
            # Clean data for scatter
            scatter_data = df[[col1, col2]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(scatter_data) > 0:
                sns.scatterplot(data=scatter_data, x=col1, y=col2, ax=ax7, alpha=0.5)
                ax7.set_title(f'{col1} vs {col2}')
    
    # 8. Data Types Distribution
    ax8 = plt.subplot(3, 3, 8)
    dtypes = df.dtypes.value_counts()
    dtypes.plot(kind='pie', ax=ax8, autopct='%1.1f%%')
    ax8.set_title('Data Types Distribution')
    ax8.set_ylabel('')
    
    # 9. Memory Usage
    ax9 = plt.subplot(3, 3, 9)
    memory_usage = df.memory_usage(deep=True)
    # Show top memory consumers
    top_memory = memory_usage.sort_values(ascending=False).head(10)
    if len(top_memory) > 1:
        top_memory[1:].plot(kind='barh', ax=ax9)
        ax9.set_title('Memory Usage by Column')
        ax9.set_xlabel('Memory (bytes)')
    
    plt.tight_layout()
    
    # Save figure if output directory is specified
    if output_dir:
        fig_path = output_dir / 'eda_visualization.png'
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f"💾 Figure saved to: {fig_path}")
    
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*60)
    print("📊 DATASET SUMMARY")
    print("="*60)
    print(f"Total Rows: {len(df)}")
    print(f"Total Columns: {len(df.columns)}")
    print(f"Missing Values: {df.isnull().sum().sum()}")
    print(f"Duplicate Rows: {df.duplicated().sum()}")
    print(f"Numeric Columns: {len(numeric_cols)}")
    print(f"Categorical Columns: {len(cat_cols)}")
    
    if numeric_cols:
        print("\n📈 Numeric Column Statistics:")
        print(df[numeric_cols].describe().to_string())
    
    # Save summary to file if output directory specified
    if output_dir:
        summary_path = output_dir / 'data_summary.txt'
        with open(summary_path, 'w') as f:
            f.write("DATA SUMMARY\n")
            f.write("="*60 + "\n")
            f.write(f"Total Rows: {len(df)}\n")
            f.write(f"Total Columns: {len(df.columns)}\n")
            f.write(f"Missing Values: {df.isnull().sum().sum()}\n")
            f.write(f"Duplicate Rows: {df.duplicated().sum()}\n")
            f.write(f"Numeric Columns: {len(numeric_cols)}\n")
            f.write(f"Categorical Columns: {len(cat_cols)}\n\n")
            if numeric_cols:
                f.write("Numeric Column Statistics:\n")
                f.write(df[numeric_cols].describe().to_string())
        print(f"💾 Summary saved to: {summary_path}")

# ============================================
# FUNCTION 4: Interactive Detection Limits (Jupyter only)
# ============================================
def interactive_detection_limits(df):
    """
    Interactive visualization with column selection (Jupyter only)
    """
    try:
        from ipywidgets import interact, widgets
    except ImportError:
        print("⚠️ ipywidgets not installed. Run: pip install ipywidgets")
        return
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    if not numeric_cols:
        print("No numeric columns found!")
        return
    
    def plot_column(col_name, lod_value=0.5, handling_method='LOD/2'):
        clean_data = df[col_name].replace([np.inf, -np.inf], np.nan).dropna()
        
        if len(clean_data) == 0:
            print(f"No valid data in column: {col_name}")
            return
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Before
        axes[0].hist(clean_data, bins=30, alpha=0.7, edgecolor='black')
        axes[0].axvline(x=lod_value, color='red', linestyle='--', linewidth=2, label=f'LOD = {lod_value}')
        axes[0].set_title(f'Before Handling\n{col_name}')
        axes[0].set_xlabel(col_name)
        axes[0].set_ylabel('Frequency')
        axes[0].legend()
        
        # After
        data_after = clean_data.copy()
        if handling_method == 'LOD/2':
            data_after[data_after < lod_value] = lod_value / 2
            label = f'LOD/2 = {lod_value/2}'
        elif handling_method == 'LOD/√2':
            data_after[data_after < lod_value] = lod_value / np.sqrt(2)
            label = f'LOD/√2 = {lod_value/np.sqrt(2):.3f}'
        else:  # Remove below LOD
            data_after = data_after[data_after >= lod_value]
            label = f'Remove < LOD'
        
        axes[1].hist(data_after, bins=30, alpha=0.7, color='green', edgecolor='black')
        axes[1].axvline(x=lod_value, color='red', linestyle='--', linewidth=2, label=f'LOD = {lod_value}')
        if handling_method in ['LOD/2', 'LOD/√2']:
            axes[1].axvline(x=lod_value/2, color='orange', linestyle='--', 
                          linewidth=2, label=label)
        axes[1].set_title(f'After Handling\n{col_name}')
        axes[1].set_xlabel(col_name)
        axes[1].set_ylabel('Frequency')
        axes[1].legend()
        
        plt.tight_layout()
        plt.show()
        
        # Print statistics
        print(f"\n📊 Statistics for '{col_name}':")
        print(f"Before: Mean={clean_data.mean():.3f}, Median={clean_data.median():.3f}, "
              f"Std={clean_data.std():.3f}")
        print(f"After:  Mean={data_after.mean():.3f}, Median={data_after.median():.3f}, "
              f"Std={data_after.std():.3f}")
        if isinstance(data_after, pd.Series):
            print(f"Values changed: {len(clean_data) - len(data_after)} ({100*(1-len(data_after)/len(clean_data)):.1f}%)")
    
    # Create interactive widgets
    interact(plot_column,
             col_name=widgets.Dropdown(options=numeric_cols, description='Column:'),
             lod_value=widgets.FloatSlider(min=0.1, max=10, step=0.1, value=0.5, description='LOD Value:'),
             handling_method=widgets.Dropdown(options=['LOD/2', 'LOD/√2', 'Remove'], 
                                             description='Method:'))

# ============================================
# MAIN EXECUTION - PASTE YOUR FILE PATH HERE
# ============================================

# ==========================================
# STEP 1: SET YOUR FILE PATH
# ==========================================

# PASTE YOUR FILE PATH HERE (Choose one):

# For Windows:
# FILE_PATH = r"C:\Users\YourName\Documents\your_data.csv"
# FILE_PATH = r"D:\Data\project\dataset.xlsx"

# For Mac/Linux:
# FILE_PATH = "/Users/username/data/your_file.csv"
# FILE_PATH = "/home/username/projects/data.json"

# For relative path (file in same folder):
# FILE_PATH = "your_data.csv"

# ==========================================
# STEP 2: CONFIGURE OPTIONS
# ==========================================

# LOD Value (Limit of Detection)
LOD_VALUE = 0.5

# Column to visualize for detection limits (None = auto-detect)
COLUMN = None  # Example: "concentration" or "measurement"

# Output directory for saving results (None = don't save)
OUTPUT_DIR = "analysis_results"

# File type (None = auto-detect from extension)
FILE_TYPE = None  # Example: "csv", "excel", "json", "parquet"

# Additional options for reading files
READ_KWARGS = {}  # Example: {'sheet_name': 'Sheet1', 'encoding': 'utf-8'}

# ==========================================
# STEP 3: RUN THE ANALYSIS
# ==========================================

# Uncomment ONE of the following sections based on what you want to do:

# OPTION A: Full Pipeline (Load data + All visualizations)
# -------------------------------------------------------------------
# df = load_data(FILE_PATH, FILE_TYPE, **READ_KWARGS)
# visualize_detection_limits(df, LOD_VALUE, COLUMN)
# complete_eda_visualization(df, OUTPUT_DIR)


# OPTION B: Just load data and see basic info
# -------------------------------------------------------------------
# df = load_data(FILE_PATH, FILE_TYPE, **READ_KWARGS)
# print("\n📊 Data Info:")
# print(df.info())
# print("\n📈 Statistical Summary:")
# print(df.describe())


# OPTION C: Just visualize detection limits (after loading data)
# -------------------------------------------------------------------
# df = load_data(FILE_PATH, FILE_TYPE, **READ_KWARGS)
# visualize_detection_limits(df, LOD_VALUE, COLUMN)


# OPTION D: Just complete EDA (after loading data)
# -------------------------------------------------------------------
# df = load_data(FILE_PATH, FILE_TYPE, **READ_KWARGS)
# complete_eda_visualization(df, OUTPUT_DIR)


# OPTION E: Interactive visualization (Jupyter Notebook only)
# -------------------------------------------------------------------
# df = load_data(FILE_PATH, FILE_TYPE, **READ_KWARGS)
# interactive_detection_limits(df)


# ==========================================
# STEP 4: MANUAL ANALYSIS (Alternative)
# ==========================================

# If you want to manually explore your data:
# df = load_data(FILE_PATH, FILE_TYPE, **READ_KWARGS)
# 
# # Check basic info
# print(df.head())
# print(df.info())
# print(df.describe())
# 
# # Check column names
# print("Columns:", df.columns.tolist())
# 
# # Check for missing values
# print("\nMissing Values:")
# print(df.isnull().sum())
# 
# # Visualize specific column
# plt.figure(figsize=(10, 6))
# df['your_column_name'].hist(bins=30)
# plt.title('Distribution of your_column_name')
# plt.show()